# 01 — ETL y preprocesamiento de las fuentes de datos

## Preparación, depuración y transformación de las fuentes para el análisis multifuente de la calidad del aire en Barcelona

Este notebook recoge el proceso de **extracción, transformación y preparación (ETL)** de las diferentes fuentes de datos utilizadas en el estudio de la calidad del aire en Barcelona.

El objetivo es transformar conjuntos de datos procedentes de distintas fuentes, con estructuras, resoluciones temporales y niveles de granularidad diferentes, en datasets **limpios, validados y reproducibles**, preparados para su posterior preintegración e incorporación al dataset maestro del proyecto.

El periodo general de estudio comprende los años **2018–2024**, permitiendo analizar tanto la evolución temporal de la contaminación atmosférica como los cambios experimentados en diferentes variables urbanas y de movilidad. Dentro de este intervalo se presta especial atención a tres etapas:

- **Pre-COVID:** 2018–2019.
- **COVID:** 2020–2021.
- **Post-COVID:** 2022–2024.

La disponibilidad temporal no es homogénea para todas las fuentes. Por este motivo, en cada bloque se documenta explícitamente su cobertura real y se evita generar información artificial para los periodos sin observaciones.

---

## Fuentes de información

El proyecto integra seis bloques principales de datos:

1. **Contaminación atmosférica**  
   Registros de las estaciones de vigilancia de la calidad del aire, con especial atención a los contaminantes relevantes para el estudio y a la construcción posterior de la variable objetivo.

2. **Meteorología**  
   Variables meteorológicas potencialmente relacionadas con los procesos de dispersión, acumulación y transformación de contaminantes.

3. **Tráfico viario**  
   Información procedente de los aforos de tráfico de Barcelona, transformada para obtener indicadores compatibles con la escala temporal del análisis.

4. **Contaminación acústica**  
   Datos de ruido ambiental utilizados como variable complementaria de presión y actividad urbana.

5. **Tráfico aéreo**  
   Información diaria de vuelos, compañías y zonas geográficas de operación, disponible para el periodo correspondiente de la fuente.

6. **Tráfico marítimo**  
   Registros de llegadas y salidas de buques del Port de Barcelona, incluyendo información sobre tipología y características físicas de las embarcaciones.

---

## Estrategia de procesamiento

Aunque cada fuente requiere un tratamiento específico, se mantiene una metodología común para garantizar la trazabilidad y reproducibilidad del proceso:

**Datos originales → inspección → control de calidad → limpieza → transformación → agregación → validación → exportación**

Para cada bloque se realizan, cuando resultan aplicables, las siguientes operaciones:

- identificación y carga de los archivos originales;
- análisis de estructura, tipos de datos y variables disponibles;
- comprobación de cobertura temporal;
- detección de registros duplicados y valores ausentes;
- normalización de fechas, identificadores y variables;
- comprobación de coherencia y rangos plausibles;
- transformación de la granularidad temporal cuando es necesario;
- generación de variables descriptivas útiles para el análisis posterior;
- clasificación de los periodos pre-COVID, COVID y post-COVID;
- preparación de estructuras específicas cuando el dashboard requiere mayor nivel de detalle;
- control final de calidad y exportación de los datasets procesados.

Los archivos definitivos de cada fuente se almacenan en su correspondiente carpeta **`DATOS LIMPIOS`**, utilizando una nomenclatura homogénea terminada en **`_limpio.csv`**.

---

## Criterio metodológico

En esta fase se prioriza la **calidad, consistencia y trazabilidad de las fuentes**, evitando introducir transformaciones dependientes del modelo antes de construir el dataset integrado.

Por este motivo, operaciones como la generación definitiva de retardos temporales (*lags*), medias móviles, escalado, selección de variables, PCA, balanceo de clases o transformaciones específicas de Machine Learning se reservan para etapas posteriores.

Esta separación permite reducir el riesgo de **fuga de información (*data leakage*)** y mantener claramente diferenciadas tres fases del pipeline:

**ETL y preprocesamiento → integración multifuente → análisis y modelado**

---

## Estructura del notebook

El notebook se organiza en los siguientes bloques:

### 1. Contaminación atmosférica
Preparación de las series procedentes de las estaciones de calidad del aire.

### 2. Meteorología
Tratamiento y transformación de las variables meteorológicas.

### 3. Tráfico viario
Preparación de los datos de aforos y construcción de indicadores de movilidad.

### 4. Contaminación acústica
Tratamiento de los registros de ruido ambiental.

### 5. Tráfico aéreo
Consolidación multianual y generación de las estructuras diaria y de dashboard.

### 6. Tráfico marítimo
Reconstrucción de llegadas y salidas, clasificación de embarcaciones y agregación diaria de la actividad portuaria.

---

> **Resultado esperado:** obtener un conjunto de datasets independientes, limpios y validados que conserven la máxima información útil de cada fuente y proporcionen una base consistente para la posterior construcción del dataset maestro espacio-temporal.

# 0 · CONFIGURACIÓN DEL ENTORNO DE TRABAJO

In [ ]:
from google.colab import drive
import os

# 1. Ejecuta este comando para montar la unidad
drive.mount('/content/drive')

# 2. (Opcional) Verifica que puedes ver tu carpeta de TFM
# Esto te confirmará si la ruta es correcta
base_path = '/content/drive/MyDrive/TFM'

if os.path.exists(base_path):
    print(f"¡Éxito! Ruta encontrada. Contenido de la carpeta TFM:")
    print(os.listdir(base_path))
else:
    print(f"No se encuentra la carpeta en: {base_path}. Verifica que 'TFM' sea el nombre exacto.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
¡Éxito! Ruta encontrada. Contenido de la carpeta TFM:
['01_Modelo_Digital_del_Terreno', '02_Trafico_Rodado_y_Aforos', '03_Transporte_Maritimo', '04_Otros_Transportes', '05_Transporte_Aereo', '06_Contaminacion_Atmosferica', '07_Contaminacion_Acustica', '08_Meteorologia', '09_Indicadores_Socioeconomicos', 'MODELO BASE LINE', '06_Dataset_Maestro_V4', 'NOTEBOOKS_GEO', 'CAPAS_GEO_LIMPIAS', 'RESULTADOS_AFOROS', '06_Analisis_Geoespacial', '10_Machine_Learning_Geoespacial', '12_Dashboard', '11_Machine_Learning_Temporal']


# 1 · PREPARACIÓN DE LOS DATOS DE CONTAMINACIÓN ATMOSFÉRICA

Los datos de contaminación atmosférica constituyen la **variable ambiental principal del estudio** y proporcionan la base sobre la que posteriormente se integrarán las variables meteorológicas, de movilidad y de actividad urbana.

El periodo analizado comprende desde **junio de 2018 hasta diciembre de 2024**, lo que permite estudiar la evolución de la calidad del aire antes, durante y después de las restricciones de movilidad asociadas a la COVID-19.

## Fuente y estructura de los datos

Los datos de contaminación atmosférica utilizados en este estudio proceden del **Portal Open Data del Ayuntamiento de Barcelona**, concretamente del conjunto de datos de detalle de calidad del aire de la ciudad.

**Fuente:** Ajuntament de Barcelona – Open Data BCN. *Qualitat de l'aire de Barcelona*.  
**Conjunto de datos:** Datos de detalle de calidad del aire de Barcelona.  
**Enlace:** https://opendata-ajuntament.barcelona.cat/data/es/dataset/qualitat-aire-detall-bcn

Los datos originales contienen las mediciones registradas por las estaciones de vigilancia de la calidad del aire de Barcelona.

Durante la inspección inicial de los archivos se identificó un **cambio en su estructura de publicación** a lo largo del periodo analizado:

- **Junio de 2018 – marzo de 2019:** formato histórico de 18 columnas, con los diferentes contaminantes almacenados en columnas independientes.
- **Desde abril de 2019:** formato de 57 columnas, con identificación explícita del contaminante y hasta 24 observaciones horarias diarias (`H01–H24`), acompañadas de sus correspondientes indicadores de validez.

Esta heterogeneidad hizo necesario desarrollar procedimientos diferenciados de lectura y transformación antes de obtener una estructura común para todo el periodo de estudio.

## Homogeneización y control de calidad

Los dos formatos históricos se transformaron a una estructura homogénea definida por **fecha, estación, contaminante y concentración**.

En el formato histórico se eliminaron las unidades y caracteres no numéricos incorporados en los valores de concentración y se realizó su conversión a formato numérico. En el formato moderno, las 24 observaciones horarias se transformaron a formato largo y se utilizaron las correspondientes **banderas de validez** para excluir las mediciones consideradas no válidas en los archivos originales.

Asimismo, se realizaron controles de duplicidad a escala horaria y se descartaron concentraciones negativas. La auditoría del bloque moderno confirmó la existencia de un máximo de **un único registro por combinación de fecha, estación, contaminante y hora**, descartando problemas de duplicación de las observaciones horarias.

A partir de las observaciones consideradas válidas se calculó la **concentración media diaria para cada estación y contaminante**, conservando adicionalmente el número de horas válidas utilizado para obtener cada promedio diario.

## Armonización de las estaciones de medida

La inspección de los datos mostró la coexistencia de diferentes sistemas de codificación de las estaciones a lo largo del periodo analizado. Por este motivo, los códigos históricos y modernos se asociaron con su correspondiente estación física, manteniendo la trazabilidad de los identificadores originales.

Se prestó especial atención a la estación de **Sants**, para la que aparecen distintos códigos en diferentes periodos de la serie. Estos identificadores se mantuvieron diferenciados durante las etapas de control de calidad con el objetivo de evitar fusiones prematuras y posibles duplicidades.

Adicionalmente, se construyó una **tabla maestra de estaciones de calidad del aire** que incorpora su denominación, tipología y coordenadas geográficas. Las coordenadas originales en **WGS84 (EPSG:4326)** se reproyectaron a **ETRS89 / UTM zona 31N (EPSG:25831)**, proporcionando coordenadas métricas compatibles con los posteriores análisis geoespaciales.

## Criterio de cobertura temporal

Con el objetivo de garantizar una representatividad suficiente de las concentraciones medias diarias, se analizó el número de observaciones horarias válidas disponible para cada combinación de fecha, estación y contaminante.

Tras eliminar los registros diarios sin ninguna observación válida, se obtuvieron **45.253 registros con información efectiva**. Sobre este conjunto se evaluaron diferentes umbrales de cobertura:

- ≥ 12 horas válidas: **98,76 %** de los registros.
- ≥ 18 horas válidas: **96,44 %** de los registros.
- ≥ 20 horas válidas: **94,33 %** de los registros.
- 24 horas válidas: **66,66 %** de los registros.

Finalmente, se adoptó un mínimo de **18 horas válidas por día** como criterio de inclusión en el dataset analítico. Este umbral permite conservar una elevada proporción de las observaciones disponibles al tiempo que garantiza una cobertura horaria suficiente para la estimación de las concentraciones medias diarias.

## Análisis de valores extremos

Como control adicional de calidad se realizó una detección exploratoria de valores extremos mediante el **criterio del rango intercuartílico (IQR)**.

Los valores identificados como extremos no se eliminaron automáticamente, ya que concentraciones elevadas pueden corresponder a **episodios reales de contaminación atmosférica** y constituyen información potencialmente relevante para los objetivos del estudio.

La proporción de observaciones identificadas mediante este criterio fue:

- **NO₂:** 1,40 %.
- **O₃:** 0,57 %.
- **PM10:** 2,95 %.
- **PM2.5:** 5,42 %.

Por tanto, el criterio IQR se utilizó como herramienta de diagnóstico y no como procedimiento automático de eliminación de observaciones.

## Dataset resultante

Tras las operaciones de limpieza, homogeneización, control de calidad y aplicación del umbral mínimo de 18 horas válidas, se obtuvo un dataset definitivo de **43.642 observaciones diarias**, correspondiente al periodo comprendido entre junio de 2018 y diciembre de 2024.

El dataset final no presenta **valores nulos ni duplicados** y contiene información de cuatro contaminantes:

- **NO₂:** 17.188 registros.
- **O₃:** 12.766 registros.
- **PM10:** 11.954 registros.
- **PM2.5:** 1.734 registros.

## Estrategia de análisis por contaminante

La disponibilidad temporal de los contaminantes no es homogénea durante todo el periodo de estudio.

El **NO₂ se adopta como contaminante principal del análisis temporal** debido a su mayor continuidad y cobertura histórica y a su especial interés para el estudio de la contaminación atmosférica asociada a la movilidad y a la actividad urbana.

**O₃ y PM10** se mantienen como contaminantes complementarios, permitiendo ampliar la caracterización de la calidad del aire y contrastar comportamientos diferenciados entre contaminantes.

**PM2.5** se conserva como variable específica, pero su interpretación se realizará teniendo en cuenta su menor disponibilidad histórica dentro del periodo analizado.

## Salidas del bloque

Como resultado del procesamiento se generan los datasets limpios y auditados de contaminación atmosférica, junto con la tabla maestra de estaciones y sus coordenadas.

Estos archivos constituyen las **salidas definitivas del bloque de contaminación atmosférica**. Su integración con los datos meteorológicos, de movilidad y con el resto de fuentes temporales se realizará posteriormente en un notebook independiente dedicado a la **construcción del dataset maestro temporal**.

## 1.1 Inspección de la estructura de los archivos de contaminación por año

In [ ]:
from pathlib import Path
import pandas as pd

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")

# Diccionario para almacenar las columnas por año
columnas_por_anio = {}

# Recorremos las carpetas de los años (ej. 2018, 2019, etc.)
for carpeta_anio in sorted(ruta_base.glob("*")):
    if carpeta_anio.is_dir():
        anio = carpeta_anio.name
        csvs = list(carpeta_anio.glob("*.csv"))

        if csvs:
            # Leemos el primer CSV de ese año como muestra de estructura
            try:
                df_muestra = pd.read_csv(csvs[0], sep=None, engine="python", nrows=5)
                columnas_por_anio[anio] = list(df_muestra.columns)
            except Exception as e:
                columnas_por_anio[anio] = [f"Error al leer: {e}"]

# Mostramos el resultado por pantalla
for anio, cols in columnas_por_anio.items():
    print(f"=== AÑO {anio} (Total columnas: {len(cols)}) ===")
    print(cols[:10], "...\n")  # Mostramos las primeras 10 columnas para no saturar

=== AÑO 2018 (Total columnas: 18) ===
['nom_cabina', 'qualitat_aire', 'codi_dtes', 'zqa', 'codi_eoi', 'longitud', 'latitud', 'hora_o3', 'qualitat_o3', 'valor_o3'] ...

=== AÑO 2019 (Total columnas: 18) ===
['nom_cabina', 'qualitat_aire', 'codi_dtes', 'zqa', 'codi_eoi', 'longitud', 'latitud', 'hora_o3', 'qualitat_o3', 'valor_o3'] ...

=== AÑO 2020 (Total columnas: 57) ===
['CODI_PROVINCIA', 'PROVINCIA', 'CODI_MUNICIPI', 'MUNICIPI', 'ESTACIO', 'CODI_CONTAMINANT', 'ANY', 'MES', 'DIA', 'H01'] ...

=== AÑO 2021 (Total columnas: 57) ===
['CODI_PROVINCIA', 'PROVINCIA', 'CODI_MUNICIPI', 'MUNICIPI', 'ESTACIO', 'CODI_CONTAMINANT', 'ANY', 'MES', 'DIA', 'H01'] ...

=== AÑO 2022 (Total columnas: 57) ===
['CODI_PROVINCIA', 'PROVINCIA', 'CODI_MUNICIPI', 'MUNICIPI', 'ESTACIO', 'CODI_CONTAMINANT', 'ANY', 'MES', 'DIA', 'H01'] ...

=== AÑO 2023 (Total columnas: 57) ===
['CODI_PROVINCIA', 'PROVINCIA', 'CODI_MUNICIPI', 'MUNICIPI', 'ESTACIO', 'CODI_CONTAMINANT', 'ANY', 'MES', 'DIA', 'H01'] ...

=== AÑO 2024

### Resultado de la inspección

Se identifica un cambio en la estructura de los datos de contaminación atmosférica a lo largo del periodo de estudio:

- **2018–2019:** archivos con 18 columnas y una estructura específica.
- **2020–2024:** archivos con 57 columnas y una estructura homogénea entre años.
- Los datos ya procesados presentan una estructura normalizada de 8 variables.

Esta heterogeneidad hace necesario aplicar procedimientos de lectura y transformación diferenciados antes de unificar las series temporales.

## 1.2 Identificación del cambio de estructura de los archivos históricos

In [ ]:
from pathlib import Path
import pandas as pd

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")

for anio in ["2018", "2019"]:
    print(f"\n--- ARCHIVOS DEL AÑO {anio} ---")
    carpeta_anio = ruta_base / anio
    for f in sorted(carpeta_anio.glob("*.csv")):
        try:
            # Leemos solo las cabeceras y 1 fila para ver sus columnas
            df_temp = pd.read_csv(f, sep=None, engine="python", nrows=1)
            print(f"  > {f.name} -> {len(df_temp.columns)} columnas")
        except Exception as e:
            print(f"  > {f.name} -> Error: {e}")


--- ARCHIVOS DEL AÑO 2018 ---
  > 2018_06_Juny_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_07_Juliol_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_08_Agost_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_09_Setembre_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_10_Octubre_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_11_Novembre_qualitat_aire_BCN.csv -> 18 columnas
  > 2018_12_Desembre_qualitat_aire_BCN.csv -> 18 columnas

--- ARCHIVOS DEL AÑO 2019 ---
  > 2019_01_Gener_qualitat_aire_BCN.csv -> 18 columnas
  > 2019_02_Febrer_qualitat_aire_BCN.csv -> 18 columnas
  > 2019_03_Marc_qualitat_aire_BCN.csv -> 18 columnas
  > 2019_04_Abril_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_05_Maig_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_06_Juny_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_07_Juliol_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_08_Agost_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_09_Setembre_qualitat_aire_BCN.csv -> 57 columnas
  > 2019_11_Novembre_qualitat_aire_BCN.csv ->

### Resultado de la inspección

La revisión individual de los archivos muestra un cambio en el esquema de publicación de los datos a partir de **abril de 2019**. Los archivos comprendidos entre junio de 2018 y marzo de 2019 presentan una estructura de 18 columnas, mientras que desde abril de 2019 se utiliza una estructura de 57 columnas, que se mantiene durante el resto del periodo analizado.

Por tanto, la preparación de los datos requiere dos procedimientos de transformación diferenciados antes de realizar su homogeneización y concatenación.

## 1.3 Inspección detallada del formato histórico de 18 columnas

In [ ]:
import pandas as pd
from pathlib import Path

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")
archivo_muestra = ruta_base / "2018" / "2018_06_Juny_qualitat_aire_BCN.csv"

df_18 = pd.read_csv(archivo_muestra, sep=None, engine="python")
print("Columnas del formato de 18 columnas:")
print(df_18.columns.tolist())
print("\nMuestra de datos:")
display(df_18.head(2))

Columnas del formato de 18 columnas:
['nom_cabina', 'qualitat_aire', 'codi_dtes', 'zqa', 'codi_eoi', 'longitud', 'latitud', 'hora_o3', 'qualitat_o3', 'valor_o3', 'hora_no2', 'qualitat_no2', 'valor_no2', 'hora_pm10', 'qualitat_pm10', 'valor_pm10', 'generat', 'dateTime']

Muestra de datos:


,nom_cabina,qualitat_aire,codi_dtes,zqa,codi_eoi,longitud,latitud,hora_o3,qualitat_o3,valor_o3,hora_no2,qualitat_no2,valor_no2,hora_pm10,qualitat_pm10,valor_pm10,generat,dateTime
0,Barcelona - Sants,--,ID,1,8019042,2.1331,41.3788,NaN,NaN,NaN,7h,--,--,NaN,NaN,NaN,11/06/2018 9:00,1528700703
1,Barcelona - Eixample,Bona,IH,1,8019043,2.1538,41.3853,8h,Bona,37 µg/m³,8h,Bona,62 µg/m³,9h,Bona,26 µg/m³,11/06/2018 9:00,1528700703


### Resultado de la inspección del formato histórico

El formato utilizado entre junio de 2018 y marzo de 2019 presenta una estructura **ancha**, en la que cada registro contiene simultáneamente información de varios contaminantes.

Las variables principales identificadas son:

- `nom_cabina`: nombre de la estación.
- `codi_eoi`: identificador de la estación.
- `longitud`, `latitud`: coordenadas geográficas.
- `valor_o3`, `valor_no2`, `valor_pm10`: concentraciones de contaminantes.
- `qualitat_o3`, `qualitat_no2`, `qualitat_pm10`: clasificación cualitativa asociada.
- `hora_o3`, `hora_no2`, `hora_pm10`: hora de referencia de cada medida.
- `generat` y `dateTime`: información temporal del registro.

Los valores de concentración se almacenan como texto e incluyen la unidad (`µg/m³`), por lo que requieren limpieza y conversión a formato numérico antes de su utilización.

Asimismo, la organización en formato ancho deberá transformarse posteriormente a una estructura homogénea compatible con los archivos publicados desde abril de 2019.

## 1.4 Carga de la tabla de referencia para la identificación de contaminantes

In [ ]:
from pathlib import Path
import pandas as pd

r = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/TIPOS CONTAMIANATES_DESIGNACION/qualitat_aire_contaminants.csv")

try:
    df = pd.read_csv(r, encoding="utf-8", sep=",")
except Exception:
    df = pd.read_csv(r, encoding="latin1", sep=";")

print("Tabla de referencia de contaminantes")
print("=" * 60)
print(f"Dimensiones: {df.shape}")
print(f"\nColumnas:\n{df.columns.tolist()}")

display(df.head(10))

Tabla de referencia de contaminantes
Dimensiones: (21, 3)

Columnas:
['Codi_Contaminant', 'Desc_Contaminant', 'Unitats']


,Codi_Contaminant,Desc_Contaminant,Unitats
0,1,SO2,µg/m³
1,6,CO,mg/m³
2,7,NO,µg/m³
3,8,NO2,µg/m³
4,9,PM2.5,µg/m³
5,10,PM10,µg/m³
6,12,NOx,µg/m³
7,14,O3,µg/m³
8,22,Black Carbon,µg/m³
9,101,SO2*,µg/m³


## 1.5 Homogeneización de los formatos históricos y construcción del dataset unificado de contaminación

In [ ]:
import pandas as pd
from pathlib import Path

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")

# 1. Diccionarios de mapeo oficiales para el bloque moderno (57 columnas)
dic_contaminantes = {
    1: 'SO2', 6: 'CO', 7: 'NO', 8: 'NO2', 9: 'PM2.5', 10: 'PM10',
    12: 'NOx', 14: 'O3', 22: 'Black Carbon', 101: 'SO2*', 106: 'CO*',
    107: 'NO*', 108: 'NO2*', 109: 'PM2.5*', 110: 'PM10*', 112: 'Nox*', 114: 'O3*'
}

dic_estaciones = {
    4: 'Barcelona - Sants', 42: 'Barcelona - Sants', 43: 'Barcelona - Eixample',
    44: 'Barcelona - Gràcia', 50: 'Barcelona - Ciutadella', 54: 'Barcelona - Vall Hebron',
    57: 'Barcelona - Palau Reial', 58: 'Barcelona - Poblenou', 60: 'Barcelona - Observ Fabra'
}

dfs_total = []

# Recorremos todos los CSVs de todas las subcarpetas
for csv_file in sorted(ruta_base.glob("**/*.csv")):
    try:
        df = pd.read_csv(csv_file, sep=None, engine="python")
        num_cols = len(df.columns)

        # --- CASO A: FORMATO 18 COLUMNAS (Jun 2018 - Mar 2019) ---
        if num_cols == 18:
            # Parseamos la fecha desde dateTime
            df['fecha'] = pd.to_datetime(df['dateTime'], unit='s', errors='coerce').dt.date
            # Si dateTime viene como string o formato fecha normal:
            if df['fecha'].isnull().all():
                df['fecha'] = pd.to_datetime(df['dateTime'], errors='coerce').dt.date

            # Pasamos las columnas de contaminantes a formato largo (melt)
            # Los contaminantes principales son valor_o3, valor_no2, valor_pm10
            cols_valor = [c for c in df.columns if c.startswith('valor_')]
            if cols_valor:
                df_melt = df.melt(
                    id_vars=['nom_cabina', 'fecha'],
                    value_vars=cols_valor,
                    var_name='contaminant',
                    value_name='valor'
                )
                df_melt['contaminant'] = df_melt['contaminant'].str.replace('valor_', '').str.upper()
                dfs_total.append(df_melt[['fecha', 'nom_cabina', 'contaminant', 'valor']])

        # --- CASO B: FORMATO 57 COLUMNAS (Abr 2019 - Dic 2024) ---
        elif num_cols == 57 or 'H01' in df.columns:
            df['nom_cabina'] = df['ESTACIO'].map(dic_estaciones)
            df['contaminant'] = df['CODI_CONTAMINANT'].map(dic_contaminantes)

            # Construimos la fecha
            df['fecha'] = pd.to_datetime(
                df['ANY'].astype(str).str.replace('.0', '', regex=False) + '-' +
                df['MES'].astype(str).str.zfill(2).str.replace('.0', '', regex=False) + '-' +
                df['DIA'].astype(str).str.zfill(2).str.replace('.0', '', regex=False),
                errors='coerce'
            ).dt.date

            # Pasamos las 24 horas (H01 a H24) a formato largo
            cols_horas = [f'H{str(i).zfill(2)}' for i in range(1, 25)]
            df_melt = df.melt(
                id_vars=['fecha', 'nom_cabina', 'contaminant'],
                value_vars=cols_horas,
                var_name='hora',
                value_name='valor'
            )
            dfs_total.append(df_melt[['fecha', 'nom_cabina', 'contaminant', 'valor']])

    except Exception as e:
        print(f"Error procesando {csv_file.name}: {e}")

# Unimos todo en un gran DataFrame maestro
if dfs_total:
    df_maestro = pd.concat(dfs_total, ignore_index=True)
    # Limpiamos nulos en fecha o estación
    df_maestro = df_maestro.dropna(subset=['fecha', 'nom_cabina', 'contaminant'])

    print(f"¡Dataset maestro generado con éxito!")
    print(f"Dimensiones totales: {df_maestro.shape}")
    display(df_maestro.head(5))

    # Opcional: Guardarlo directamente como tu CSV unificado final
    # ruta_salida = ruta_base / "contaminacion_barcelona_2018_2024_unificado.csv"
    # df_maestro.to_csv(ruta_salida, index=False)
    # print(f"Guardado en: {ruta_salida}")

¡Dataset maestro generado con éxito!
Dimensiones totales: (2594229, 4)


,fecha,nom_cabina,contaminant,valor
0,2018-06-11,Barcelona - Sants,O3,NaN
1,2018-06-11,Barcelona - Eixample,O3,37 µg/m³
2,2018-06-11,Barcelona - Gràcia,O3,51 µg/m³
3,2018-06-11,Barcelona - Ciutadella,O3,27 µg/m³
4,2018-06-11,Barcelona - Vall Hebron,O3,22 µg/m³


### Resultado de la homogeneización

La integración de los diferentes formatos históricos genera un dataset unificado de 2.594.229 registros y cuatro variables comunes: fecha, estación de medida, contaminante y valor observado.

En esta etapa se conserva el valor en su formato original. Por tanto, antes de realizar la agregación temporal será necesario normalizar las concentraciones, convertirlas a formato numérico y analizar la presencia de valores ausentes o no válidos.

## 1.6 Limpieza de las concentraciones y agregación diaria por estación y contaminante

En esta etapa se completa la normalización de los dos formatos históricos. Para los archivos antiguos se eliminan las unidades y caracteres no numéricos de las concentraciones, mientras que en el formato moderno se convierten las 24 observaciones horarias a tipo numérico. Posteriormente, todas las observaciones se integran en una estructura común y se calcula la concentración media diaria para cada combinación de fecha, estación y contaminante.

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")

# Diccionarios oficiales
dic_contaminantes = {
    1: 'SO2', 6: 'CO', 7: 'NO', 8: 'NO2', 9: 'PM2.5', 10: 'PM10',
    12: 'NOx', 14: 'O3', 22: 'Black Carbon', 101: 'SO2*', 106: 'CO*',
    107: 'NO*', 108: 'NO2*', 109: 'PM2.5*', 110: 'PM10*', 112: 'Nox*', 114: 'O3*'
}

dic_estaciones = {
    4: 'Barcelona - Sants', 42: 'Barcelona - Sants', 43: 'Barcelona - Eixample',
    44: 'Barcelona - Gràcia', 50: 'Barcelona - Ciutadella', 54: 'Barcelona - Vall Hebron',
    57: 'Barcelona - Palau Reial', 58: 'Barcelona - Poblenou', 60: 'Barcelona - Observ Fabra'
}

dfs_total = []

for csv_file in sorted(ruta_base.glob("**/*.csv")):
    try:
        df = pd.read_csv(csv_file, sep=None, engine="python")
        num_cols = len(df.columns)

        # --- CASO A: FORMATO 18 COLUMNAS ---
        if num_cols == 18:
            df['fecha'] = pd.to_datetime(df['dateTime'], unit='s', errors='coerce').dt.date
            if df['fecha'].isnull().all():
                df['fecha'] = pd.to_datetime(df['dateTime'], errors='coerce').dt.date

            cols_valor = [c for c in df.columns if c.startswith('valor_')]
            if cols_valor:
                # Limpiamos las columnas de valor individualmente antes del melt
                for col in cols_valor:
                    df[col] = (
                        df[col].astype(str)
                        .str.replace(r'[^0-9.,]', '', regex=True)
                        .str.replace(',', '.')
                    )
                    df[col] = pd.to_numeric(df[col], errors='coerce')

                df_melt = df.melt(
                    id_vars=['nom_cabina', 'fecha'],
                    value_vars=cols_valor,
                    var_name='contaminant',
                    value_name='valor'
                )
                df_melt['contaminant'] = df_melt['contaminant'].str.replace('valor_', '').str.upper()
                dfs_total.append(df_melt[['fecha', 'nom_cabina', 'contaminant', 'valor']])

        # --- CASO B: FORMATO 57 COLUMNAS ---
        elif num_cols == 57 or 'H01' in df.columns:
            df['nom_cabina'] = df['ESTACIO'].map(dic_estaciones)
            df['contaminant'] = df['CODI_CONTAMINANT'].map(dic_contaminantes)

            df['fecha'] = pd.to_datetime(
                df['ANY'].astype(str).str.replace('.0', '', regex=False) + '-' +
                df['MES'].astype(str).str.zfill(2).str.replace('.0', '', regex=False) + '-' +
                df['DIA'].astype(str).str.zfill(2).str.replace('.0', '', regex=False),
                errors='coerce'
            ).dt.date

            cols_horas = [f'H{str(i).zfill(2)}' for i in range(1, 25)]
            for col in cols_horas:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            df_melt = df.melt(
                id_vars=['fecha', 'nom_cabina', 'contaminant'],
                value_vars=cols_horas,
                var_name='hora',
                value_name='valor'
            )
            dfs_total.append(df_melt[['fecha', 'nom_cabina', 'contaminant', 'valor']])

    except Exception as e:
        print(f"Error procesando {csv_file.name}: {e}")

# Unimos todo y agrupamos a nivel diario de forma limpia
if dfs_total:
    df_maestro = pd.concat(dfs_total, ignore_index=True)
    df_maestro = df_maestro.dropna(subset=['fecha', 'nom_cabina', 'contaminant'])

    # Agrupación diaria final
    df_maestro_diario = (
        df_maestro
        .groupby(['fecha', 'nom_cabina', 'contaminant'], as_index=False)['valor']
        .mean()
    )

    ruta_salida_diaria = ruta_base / "contaminacion_barcelona_2018_2024_diario.csv"
    df_maestro_diario.to_csv(ruta_salida_diaria, index=False)
    print(f"¡Proceso completado con éxito! Archivo guardado en:\n{ruta_salida_diaria}")
    print(f"Dimensiones finales: {df_maestro_diario.shape}")

¡Proceso completado con éxito! Archivo guardado en:
/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS/contaminacion_barcelona_2018_2024_diario.csv
Dimensiones finales: (102050, 4)


### Resultado de la agregación diaria

Tras la limpieza y normalización de las concentraciones, las observaciones se agregan mediante la concentración media diaria para cada combinación de fecha, estación y contaminante.

El dataset resultante contiene **102.050 registros** y constituye la base diaria de contaminación atmosférica que se utilizará en las siguientes etapas del análisis temporal.

## 1.7 Depuración avanzada, control de calidad y construcción del dataset diario auditado

Esta etapa refina el procesamiento previo mediante la deduplicación de registros, la aplicación de las banderas de validez horaria, la exclusión de valores físicamente imposibles, el filtrado de los contaminantes de interés y el cómputo del número de horas válidas utilizadas en cada media diaria. El resultado constituye la versión auditada del dataset diario de contaminación atmosférica.

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")

# Diccionarios oficiales
dic_contaminantes = {
    7: 'NO', 8: 'NO2', 9: 'PM2.5', 10: 'PM10', 12: 'NOx', 14: 'O3', 1: 'SO2', 6: 'CO'
}
dic_estaciones = {
    4: 'Barcelona - Sants', 42: 'Barcelona - Sants', 43: 'Barcelona - Eixample',
    44: 'Barcelona - Gràcia', 50: 'Barcelona - Ciutadella', 54: 'Barcelona - Vall Hebron',
    57: 'Barcelona - Palau Reial', 58: 'Barcelona - Poblenou', 60: 'Barcelona - Observ Fabra'
}

contaminantes_interes = ['NO2', 'PM10', 'PM2.5', 'O3']

dfs_modernos = []
dfs_antiguos = []

print("Iniciando procesamiento refinado con control de duplicados y formato antiguo...")

# Excluimos expresamente cualquier CSV generado previamente en la ruta
archivos_csv = [f for f in ruta_base.glob("**/*.csv") if "unificado" not in f.name and "auditado" not in f.name and "diario" not in f.name]

for csv_file in sorted(archivos_csv):
    try:
        df = pd.read_csv(csv_file, sep=None, engine="python")
        num_cols = len(df.columns)

        # --- BLOQUE ANTIGUO (Jun 2018 - Mar 2019): Formato 18 columnas ---
        if num_cols == 18:
            df['fecha'] = pd.to_datetime(df['dateTime'], unit='s', errors='coerce').dt.date
            if df['fecha'].isnull().all():
                df['fecha'] = pd.to_datetime(df['dateTime'], errors='coerce').dt.date

            # Conservamos el código de estación original (codi_dtes o codi_eoi)
            estacion_col = 'codi_dtes' if 'codi_dtes' in df.columns else 'nom_cabina'

            # Mapeo de contaminantes presentes en formato antiguo
            mapeo_antiguo = {'o3': 'O3', 'no2': 'NO2', 'pm10': 'PM10'}

            filas_antiguas = []
            for prefix, cont_name in mapeo_antiguo.items():
                col_val = f'valor_{prefix}'
                col_hora = f'hora_{prefix}'

                if col_val in df.columns and col_hora in df.columns:
                    sub_df = df[['fecha', estacion_col, 'nom_cabina', col_hora, col_val]].copy()
                    sub_df.columns = ['fecha', 'codi_estacio', 'nom_cabina', 'hora', 'valor']
                    sub_df['contaminant'] = cont_name
                    filas_antiguas.append(sub_df)

            if filas_antiguas:
                df_antiguo_long = pd.concat(filas_antiguas, ignore_index=True)

                # Limpieza de valores
                df_antiguo_long['valor'] = (
                    df_antiguo_long['valor'].astype(str)
                    .str.replace(r'[^0-9.,]', '', regex=True)
                    .str.replace(',', '.')
                )
                df_antiguo_long['valor'] = pd.to_numeric(df_antiguo_long['valor'], errors='coerce')

                # Deduplicación por fecha + estación + contaminante + hora real
                df_antiguo_long = df_antiguo_long.drop_duplicates(
                    subset=['fecha', 'codi_estacio', 'contaminant', 'hora'],
                    keep='last'
                )

                dfs_antiguos.append(df_antiguo_long[['fecha', 'codi_estacio', 'nom_cabina', 'contaminant', 'hora', 'valor']])

        # --- BLOQUE MODERNO (Abr 2019 - 2024): Formato 57 columnas ---
        elif num_cols == 57 or 'H01' in df.columns:
            # Conservamos el código de estación original (ESTACIO) y mapeamos nombre
            df['codi_estacio'] = df['ESTACIO']
            df['nom_cabina'] = df['ESTACIO'].map(dic_estaciones)
            df['contaminant'] = df['CODI_CONTAMINANT'].map(dic_contaminantes)

            df['fecha'] = pd.to_datetime(
                df['ANY'].astype(str).str.replace('.0', '', regex=False) + '-' +
                df['MES'].astype(str).str.zfill(2).str.replace('.0', '', regex=False) + '-' +
                df['DIA'].astype(str).str.zfill(2).str.replace('.0', '', regex=False),
                errors='coerce'
            ).dt.date

            # Aplicar banderas V01...V24 sobre H01...H24
            for i in range(1, 25):
                h_col = f'H{str(i).zfill(2)}'
                v_col = f'V{str(i).zfill(2)}'

                if h_col in df.columns and v_col in df.columns:
                    df[h_col] = pd.to_numeric(df[h_col], errors='coerce')
                    df.loc[df[v_col].astype(str).str.strip().str.upper() != 'V', h_col] = np.nan
                elif h_col in df.columns:
                    df[h_col] = pd.to_numeric(df[h_col], errors='coerce')

            cols_horas = [f'H{str(i).zfill(2)}' for i in range(1, 25) if f'H{str(i).zfill(2)}' in df.columns]

            df_melt = df.melt(
                id_vars=['fecha', 'codi_estacio', 'nom_cabina', 'contaminant'],
                value_vars=cols_horas,
                var_name='hora',
                value_name='valor'
            )
            dfs_modernos.append(df_melt[['fecha', 'codi_estacio', 'nom_cabina', 'contaminant', 'hora', 'valor']])

    except Exception as e:
        print(f"Aviso procesando {csv_file.name}: {e}")

# Auditoría previa de duplicados en el bloque moderno
if dfs_modernos:
    df_mod_concat = pd.concat(dfs_modernos, ignore_index=True)
    duplicados_modernos = (
        df_mod_concat
        .groupby(["fecha", "codi_estacio", "contaminant", "hora"])
        .size()
        .sort_values(ascending=False)
    )
    print("\n--- AUDITORÍA PREVIA: Máximo número de registros por hora/estación/fecha (Moderno) ---")
    print(duplicados_modernos.head(10))

# Consolidamos bloques
lista_unir = []
if dfs_modernos:
    lista_unir.append(pd.concat(dfs_modernos, ignore_index=True))
if dfs_antiguos:
    lista_unir.append(pd.concat(dfs_antiguos, ignore_index=True))

df_total = pd.concat(lista_unir, ignore_index=True) if lista_unir else pd.DataFrame()

# Limpieza general de nulos y filtro de contaminantes de interés
df_total = df_total.dropna(subset=['fecha', 'codi_estacio', 'contaminant'])
df_total = df_total[df_total['contaminant'].isin(contaminantes_interes)]

# Limpieza de valores físicamente imposibles (solo negativos)
df_total.loc[df_total['valor'] < 0, 'valor'] = np.nan

# Agregación diaria robusta (calculando la media real y contando horas/registros válidos únicos)
df_diario = (
    df_total
    .groupby(['fecha', 'codi_estacio', 'nom_cabina', 'contaminant'], as_index=False)
    .agg(
        valor=('valor', 'mean'),
        horas_validas=('valor', lambda x: x.notnull().sum())
    )
)

# Guardar resultado definitivo
ruta_salida_auditada = ruta_base / "contaminacion_barcelona_2018_2024_auditado_v2.csv"
df_diario.to_csv(ruta_salida_auditada, index=False)

# Auditoría Final
print("\n" + "="*50)
print("       AUDITORÍA FINAL REVISADA")
print("="*50)
print(f"** Ruta de guardado:** {ruta_salida_auditada}")
print(f"** Dimensiones finales:** {df_diario.shape}")
print(f"** Rango temporal:** Desde {df_diario['fecha'].min()} hasta {df_diario['fecha'].max()}")
print(f"** Número de códigos de estación únicos:** {df_diario['codi_estacio'].nunique()}")
print(f"   -> Estaciones: {df_diario[['codi_estacio', 'nom_cabina']].drop_duplicates().values.tolist()}")
print(f"** Contaminantes incluidos:** {list(df_diario['contaminant'].unique())}")
print("\n** Distribución de registros por contaminante:**")
print(df_diario['contaminant'].value_counts())
print("\n** Resumen estricto de horas válidas por día (Máximo esperado <= 24):**")
print(df_diario['horas_validas'].describe())

Iniciando procesamiento refinado con control de duplicados y formato antiguo...

--- AUDITORÍA PREVIA: Máximo número de registros por hora/estación/fecha (Moderno) ---
fecha       codi_estacio  contaminant  hora
2024-12-31  58            PM10         H24     1
2019-04-02  4             NO           H01     1
                                       H02     1
                                       H03     1
                                       H04     1
                                       H05     1
                                       H06     1
                                       H07     1
2024-12-31  58            PM10         H08     1
                                       H07     1
dtype: int64

       AUDITORÍA FINAL REVISADA
** Ruta de guardado:** /content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS/contaminacion_barcelona_2018_2024_auditado_v2.csv
** Dimensiones finales:** (49014, 6)
** Rango temporal:** Desde 2018-06-11 hasta 2024-12-31
** Númer

### Resultado de la auditoría

La auditoría confirma la ausencia de duplicidades horarias en el bloque moderno, con un máximo de un registro por combinación de fecha, estación, contaminante y hora.

El dataset auditado contiene 49.014 registros diarios correspondientes al periodo comprendido entre junio de 2018 y diciembre de 2024. Se conservan cuatro contaminantes de interés: NO₂, O₃, PM10 y PM2.5.

La disponibilidad horaria es elevada, con una mediana de 24 observaciones válidas por día y un primer cuartil de 23. No obstante, se identifican registros diarios con cobertura insuficiente o nula, que deberán ser considerados durante el control de calidad posterior.

La presencia de 16 códigos de estación no corresponde a 16 ubicaciones físicas diferentes, sino al cambio del sistema de codificación entre los formatos históricos y modernos. Estos identificadores deberán homogeneizarse antes de la integración definitiva.

## 1.8 Armonización de estaciones físicas y análisis de cobertura temporal

En esta etapa se revisa la coexistencia de distintos códigos históricos para una misma estación de medida, se armonizan dichos identificadores mediante una variable común de estación física y se analiza la cobertura temporal de los registros diarios. Asimismo, se evalúa el número de horas válidas disponibles por día y la cobertura específica de cada contaminante y estación antes de la integración definitiva en el dataset temporal.

In [ ]:
import pandas as pd
from pathlib import Path

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")
ruta_auditado = ruta_base / "contaminacion_barcelona_2018_2024_auditado_v2.csv"

# Cargamos el archivo auditado v2
df_diario = pd.read_csv(ruta_auditado)

# Aseguramos formato fecha
df_diario['fecha'] = pd.to_datetime(df_diario['fecha']).dt.date

print("="*60)
print("1. AUDITORÍA DE CÓDIGOS DE ESTACIÓN Y CRUCE SÁNTS (4 y 42)")
print("="*60)

auditoria_codigos = (
    df_diario
    .groupby(["codi_estacio", "nom_cabina"])["fecha"]
    .agg(fecha_inicio="min", fecha_fin="max", registros="count")
    .reset_index()
    .sort_values(["nom_cabina", "fecha_inicio"])
)
display(auditoria_codigos)

print("\n--- Comprobación específica códigos Sants (4 y 42) ---")
display(
    df_diario[df_diario["codi_estacio"].astype(str).isin(["4", "42"])]
    .groupby("codi_estacio")["fecha"]
    .agg(["min", "max", "count"])
)

print("\n" + "="*60)
print("2. ARMONIZACIÓN A ESTACIÓN FÍSICA ÚNICA")
print("="*60)

df_diario["codi_estacio_str"] = df_diario["codi_estacio"].astype(str)
mapeo_estacion_fisica = {
    "I2": "Poblenou",
    "58": "Poblenou",
    "ID": "Sants",
    "4": "Sants",
    "42": "Sants",
    "IH": "Eixample",
    "43": "Eixample",
    "IJ": "Gràcia",
    "44": "Gràcia",
    "IL": "Ciutadella",
    "50": "Ciutadella",
    "IN": "Vall Hebron",
    "54": "Vall Hebron",
    "IZ": "Palau Reial",
    "57": "Palau Reial",
    "OF": "Observ Fabra",
    "60": "Observ Fabra"
}

df_diario["estacion_fisica"] = df_diario["codi_estacio_str"].map(mapeo_estacion_fisica)
print(f"Estaciones físicas mapeadas con éxito. Total únicas: {df_diario['estacion_fisica'].nunique()}")
print(df_diario['estacion_fisica'].value_counts())

print("\n" + "="*60)
print("3. ANÁLISIS DE COBERTURA Y HORAS VÁLIDAS")
print("="*60)

print("Días por nivel de cobertura (horas_validas):")
print(df_diario["horas_validas"].value_counts().sort_index().head(25))

print("\nPorcentajes según umbral de horas válidas:")
for umbral in [1, 12, 18, 20, 24]:
    n = (df_diario["horas_validas"] >= umbral).sum()
    pct = n / len(df_diario) * 100
    print(f"≥ {umbral:2d} horas: {n:,} registros ({pct:.2f}%)")

print("\n" + "="*60)
print("4. COBERTURA DETALLADA POR CONTAMINANTE Y ESTACIÓN FÍSICA")
print("="*60)

cobertura_contaminante = (
    df_diario
    .groupby(["contaminant", "estacion_fisica"])
    .agg(
        fecha_inicio=("fecha", "min"),
        fecha_fin=("fecha", "max"),
        dias=("fecha", "count"),
        dias_con_dato=("valor", "count"),
        media_horas=("horas_validas", "mean")
    )
    .reset_index()
    .sort_values(["contaminant", "estacion_fisica"])
)
display(cobertura_contaminante)

# Guardamos la versión con la estación física unificada para el siguiente paso
ruta_salida_fisica = ruta_base / "contaminacion_barcelona_2018_2024_estacion_fisica.csv"
df_diario.to_csv(ruta_salida_fisica, index=False)
print(f"\nDataset actualizado con 'estacion_fisica' guardado en:\n{ruta_salida_fisica}")

1. AUDITORÍA DE CÓDIGOS DE ESTACIÓN Y CRUCE SÁNTS (4 y 42)


,codi_estacio,nom_cabina,fecha_inicio,fecha_fin,registros
12,IL,Barcelona - Ciutadella,2018-06-11,2019-03-31,855
4,50,Barcelona - Ciutadella,2019-04-02,2024-12-31,4100
10,IH,Barcelona - Eixample,2018-06-11,2019-03-31,855
2,43,Barcelona - Eixample,2019-04-02,2024-12-31,6805
11,IJ,Barcelona - Gràcia,2018-06-11,2019-03-31,855
3,44,Barcelona - Gràcia,2019-04-02,2024-12-31,6150
15,OF,Barcelona - Observ Fabra,2018-07-09,2019-03-31,771
14,IZ,Barcelona - Palau Reial,2018-06-11,2019-03-31,855
6,57,Barcelona - Palau Reial,2019-04-02,2024-12-31,7298
8,I2,Barcelona - Poblenou,2018-06-11,2019-03-31,855



--- Comprobación específica códigos Sants (4 y 42) ---


,min,max,count
codi_estacio,,,
4,2019-04-02,2024-12-31,4094
42,2019-04-02,2024-12-31,2050



2. ARMONIZACIÓN A ESTACIÓN FÍSICA ÚNICA
Estaciones físicas mapeadas con éxito. Total únicas: 8
estacion_fisica
Palau Reial     8153
Eixample        7660
Vall Hebron     7051
Gràcia          7005
Sants           6999
Poblenou        6420
Ciutadella      4955
Observ Fabra     771
Name: count, dtype: int64

3. ANÁLISIS DE COBERTURA Y HORAS VÁLIDAS
Días por nivel de cobertura (horas_validas):
horas_validas
0      3761
1        92
2        47
3        11
4         8
5        10
6        20
7        27
8        72
9        55
10      101
11      118
12      116
13      190
14      183
15      187
16      199
17      175
18      320
19      637
20     1020
21     1225
22     1874
23     8399
24    30167
Name: count, dtype: int64

Porcentajes según umbral de horas válidas:
≥  1 horas: 45,253 registros (92.33%)
≥ 12 horas: 44,692 registros (91.18%)
≥ 18 horas: 43,642 registros (89.04%)
≥ 20 horas: 42,685 registros (87.09%)
≥ 24 horas: 30,167 registros (61.55%)

4. COBERTURA DETALLADA POR CONTA

,contaminant,estacion_fisica,fecha_inicio,fecha_fin,dias,dias_con_dato,media_horas
0,NO2,Ciutadella,2018-06-11,2024-12-31,2335,2272,22.565310
1,NO2,Eixample,2018-06-11,2024-12-31,2335,2262,22.068951
2,NO2,Gràcia,2018-06-11,2024-12-31,2335,2249,22.112206
3,NO2,Observ Fabra,2018-07-09,2019-03-31,257,194,15.712062
4,NO2,Palau Reial,2018-06-11,2024-12-31,2335,2228,21.862099
5,NO2,Poblenou,2018-06-11,2024-12-31,2140,2056,22.226636
6,NO2,Sants,2018-06-11,2024-12-31,4385,4299,22.793387
7,NO2,Vall Hebron,2018-06-11,2024-12-31,2335,2239,22.268522
8,O3,Ciutadella,2018-06-11,2024-12-31,2335,2264,22.470236
9,O3,Eixample,2018-06-11,2024-12-31,2335,2252,21.969593



Dataset actualizado con 'estacion_fisica' guardado en:
/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS/contaminacion_barcelona_2018_2024_estacion_fisica.csv


### Resultado de la armonización y análisis de cobertura

La armonización de los diferentes sistemas de codificación permite identificar **8 estaciones físicas** a lo largo del periodo de estudio, evitando considerar como estaciones independientes los cambios históricos en sus códigos de identificación.

El análisis de cobertura muestra una elevada disponibilidad de información horaria. El **92,33 %** de los registros diarios dispone de al menos una observación válida, el **91,18 %** alcanza al menos 12 horas y el **89,04 %** dispone de 18 o más horas válidas. Un **61,55 %** de los registros presenta las 24 observaciones horarias completas.

La disponibilidad no es homogénea entre contaminantes. NO₂ presenta una cobertura temporal amplia en las principales estaciones durante prácticamente todo el periodo 2018–2024, mientras que PM2.5 presenta una disponibilidad mucho más limitada y concentrada en los últimos años.

Asimismo, se identifica la coexistencia de los códigos 4 y 42 para la estación de Sants durante el periodo moderno, circunstancia que deberá controlarse para evitar duplicidades al trabajar a nivel de estación física.

## 1.9 Selección del dataset analítico y definición de la estrategia por contaminante

En esta etapa se depura el dataset auditado eliminando registros sin observaciones horarias válidas y se evalúan distintos umbrales de cobertura diaria. Se adopta un criterio mínimo de 18 horas válidas para definir el dataset analítico principal.

A partir de esta selección se establece la estrategia de modelado del TFM: el NO₂ se utiliza como contaminante principal debido a su mayor continuidad temporal, mientras que PM10 y O₃ se conservan como variables complementarias y PM2.5 se trata de forma específica debido a su menor disponibilidad histórica.

In [ ]:
import pandas as pd
from pathlib import Path

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")
ruta_auditado = ruta_base / "contaminacion_barcelona_2018_2024_auditado_v2.csv"

# 1. Cargamos el archivo base
df_diario = pd.read_csv(ruta_auditado)
df_diario['fecha'] = pd.to_datetime(df_diario['fecha']).dt.date
df_diario["codi_estacio_str"] = df_diario["codi_estacio"].astype(str)

# 2. Mapeo refinado: Sants separado (ID -> Sants_hist, 4 -> Sants_4, 42 -> Sants_42) y resto unificado
mapeo_estacion_fisica_refinado = {
    "I2": "Poblenou",
    "58": "Poblenou",
    "ID": "Sants_hist",
    "4": "Sants_4",
    "42": "Sants_42",
    "IH": "Eixample",
    "43": "Eixample",
    "IJ": "Gràcia",
    "44": "Gràcia",
    "IL": "Ciutadella",
    "50": "Ciutadella",
    "IN": "Vall Hebron",
    "54": "Vall Hebron",
    "IZ": "Palau Reial",
    "57": "Palau Reial",
    "OF": "Observ Fabra"
}

df_diario["estacion_fisica"] = df_diario["codi_estacio_str"].map(mapeo_estacion_fisica_refinado)

# 3. Limpieza de combinaciones sin datos reales (horas_validas = 0 o NaN en valor)
df_valido = df_diario[
    (df_diario["horas_validas"] > 0) &
    (df_diario["valor"].notna())
].copy()

print(f"Registros originales: {len(df_diario):,}")
print(f"Registros tras eliminar ceros/nulos reales: {len(df_valido):,}")

# 4. Evaluación de umbrales sobre el dataset válido
print("\n--- Evaluación de cobertura (horas válidas) sobre dataset válido ---")
for umbral in [1, 12, 18, 20, 24]:
    n = (df_valido["horas_validas"] >= umbral).sum()
    pct = n / len(df_valido) * 100
    print(f"≥ {umbral:2d} horas: {n:,} registros ({pct:.2f}%)")

# 5. Dataset analítico principal con umbral de 18 horas
df_modelo_18h = df_valido[df_valido["horas_validas"] >= 18].copy()

# 6. Separación por estrategias de contaminantes para el TFM
df_no2 = df_modelo_18h[df_modelo_18h["contaminant"] == "NO2"].copy()
df_complementarios = df_modelo_18h[df_modelo_18h["contaminant"].isin(["PM10", "O3"])].copy()
df_pm25 = df_modelo_18h[df_modelo_18h["contaminant"] == "PM2.5"].copy()

# Guardamos los datasets limpios y estructurados según el plan
ruta_no2 = ruta_base / "contaminacion_barcelona_NO2_principal.csv"
ruta_comp = ruta_base / "contaminacion_barcelona_PM10_O3_complementario.csv"
ruta_pm25 = ruta_base / "contaminacion_barcelona_PM25_especifico.csv"

df_no2.to_csv(ruta_no2, index=False)
df_complementarios.to_csv(ruta_comp, index=False)
df_pm25.to_csv(ruta_pm25, index=False)

print("\n" + "="*50)
print("       ESTRATEGIA TFM APLICADA CON ÉXITO")
print("="*50)
print(f"-> Dataset Principal (NO₂ - ≥18h): {len(df_no2):,} registros (Guardado en {ruta_no2.name})")
print(f"-> Dataset Complementario (PM₁₀ & O₃ - ≥18h): {len(df_complementarios):,} registros (Guardado en {ruta_comp.name})")
print(f"-> Dataset Específico (PM₂.₅): {len(df_pm25):,} registros (Guardado en {ruta_pm25.name})")

Registros originales: 49,014
Registros tras eliminar ceros/nulos reales: 45,253

--- Evaluación de cobertura (horas válidas) sobre dataset válido ---
≥  1 horas: 45,253 registros (100.00%)
≥ 12 horas: 44,692 registros (98.76%)
≥ 18 horas: 43,642 registros (96.44%)
≥ 20 horas: 42,685 registros (94.33%)
≥ 24 horas: 30,167 registros (66.66%)

       ESTRATEGIA TFM APLICADA CON ÉXITO
-> Dataset Principal (NO₂ - ≥18h): 17,188 registros (Guardado en contaminacion_barcelona_NO2_principal.csv)
-> Dataset Complementario (PM₁₀ & O₃ - ≥18h): 24,720 registros (Guardado en contaminacion_barcelona_PM10_O3_complementario.csv)
-> Dataset Específico (PM₂.₅): 1,734 registros (Guardado en contaminacion_barcelona_PM25_especifico.csv)


### Resultado de la selección del dataset analítico

Tras eliminar los registros diarios sin observaciones válidas, el dataset se reduce de **49.014 a 45.253 registros**. Dentro de este conjunto, el **96,44 %** de los registros dispone de al menos **18 horas válidas**, por lo que se adopta este umbral como criterio de calidad para la construcción del dataset analítico.

La aplicación de este criterio conserva **43.642 registros diarios** y permite establecer tres conjuntos de análisis:

- **NO₂:** 17.188 registros, utilizado como contaminante principal.
- **PM10 y O₃:** 24.720 registros, empleados como contaminantes complementarios.
- **PM2.5:** 1.734 registros, tratado de forma específica debido a su menor cobertura temporal.

El umbral seleccionado permite mantener una elevada cobertura de la muestra al tiempo que garantiza una representación horaria suficiente para el cálculo de las concentraciones medias diarias.

## 1.10 Validación final y cierre del dataset diario de contaminación

En esta etapa se realiza la validación final del dataset analítico tras aplicar el umbral mínimo de 18 horas válidas. Se revisan las estadísticas descriptivas, la presencia de valores nulos, posibles duplicados a nivel de fecha–estación–contaminante y la existencia de valores extremos mediante el criterio del rango intercuartílico (IQR).

El objetivo es comprobar la consistencia del conjunto de datos antes de considerarlo cerrado y utilizarlo como entrada en la construcción del dataset maestro temporal.

### Resultado de la validación final

El dataset definitivo de contaminación atmosférica contiene **43.642 registros diarios** correspondientes al periodo comprendido entre junio de 2018 y diciembre de 2024.

Tras aplicar un criterio mínimo de 18 horas válidas por día, el conjunto final no presenta valores nulos ni duplicados. Se conservan cuatro contaminantes: NO₂, O₃, PM10 y PM2.5.

El análisis mediante el criterio IQR identifica una proporción reducida de valores extremos. Estos registros no se eliminan de forma automática, dado que pueden corresponder a episodios reales de elevada contaminación y constituyen información relevante para el análisis de eventos adversos.

El archivo resultante se considera la versión cerrada del bloque de contaminación atmosférica y se utilizará como entrada para la construcción del dataset maestro temporal.

In [ ]:
import pandas as pd
from pathlib import Path

ruta_base = Path("/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS")
ruta_auditado = ruta_base / "contaminacion_barcelona_2018_2024_auditado_v2.csv"

# 1. Cargamos el archivo base

df_diario = pd.read_csv(ruta_auditado)
df_diario['fecha'] = pd.to_datetime(df_diario['fecha']).dt.date
df_diario["codi_estacio_str"] = df_diario["codi_estacio"].astype(str)

# 2. Mapeo refinado (Sants separado y resto unificado)

mapeo_estacion_fisica_refinado = {
    "I2": "Poblenou",
    "58": "Poblenou",
    "ID": "Sants_hist",
    "4": "Sants_4",
    "42": "Sants_42",
    "IH": "Eixample",
    "43": "Eixample",
    "IJ": "Gràcia",
    "44": "Gràcia",
    "IL": "Ciutadella",
    "50": "Ciutadella",
    "IN": "Vall Hebron",
    "54": "Vall Hebron",
    "IZ": "Palau Reial",
    "57": "Palau Reial",
    "OF": "Observ Fabra"
}

df_diario["estacion_fisica"] = df_diario["codi_estacio_str"].map(
    mapeo_estacion_fisica_refinado
)

# 3. Filtrado estricto
# (eliminación de ceros y nulos reales, y aplicación del umbral de ≥18 horas válidas)

df_valido = df_diario[
    (df_diario["horas_validas"] > 0) &
    (df_diario["valor"].notna())
].copy()

df_modelo_18h = df_valido[
    df_valido["horas_validas"] >= 18
].copy()

# ==========================================
# LAS 5 COMPROBACIONES FINALES DE CIERRE
# ==========================================

print("\n--- 1. ESTADÍSTICAS DESCRIPTIVAS ---")
display(
    df_modelo_18h
    .groupby("contaminant")["valor"]
    .describe()
)

print("\n--- 2. VALORES NULOS ---")
print(df_modelo_18h.isna().sum())

print("\n--- 3. DUPLICADOS ---")

duplicados = df_modelo_18h.duplicated(
    subset=["fecha", "estacion_fisica", "contaminant"]
).sum()

print(f"Duplicados encontrados: {duplicados}")

print("\n--- 4. DETECCIÓN DE OUTLIERS (IQR) ---")

for cont in df_modelo_18h["contaminant"].unique():

    datos = df_modelo_18h[
        df_modelo_18h["contaminant"] == cont
    ]["valor"]

    Q1 = datos.quantile(0.25)
    Q3 = datos.quantile(0.75)
    IQR = Q3 - Q1

    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR

    n_out = (
        (datos < lim_inf) |
        (datos > lim_sup)
    ).sum()

    print(
        f"{cont}: {n_out} outliers "
        f"({100 * n_out / len(datos):.2f}%)"
    )

print("\n--- 5. RESUMEN FINAL ---")

print(
    f"Periodo: "
    f"{df_modelo_18h['fecha'].min()} - "
    f"{df_modelo_18h['fecha'].max()}"
)

print(
    f"Estaciones: "
    f"{df_modelo_18h['estacion_fisica'].nunique()}"
)

print(
    f"Contaminantes: "
    f"{df_modelo_18h['contaminant'].unique()}"
)

print(
    f"Registros finales: "
    f"{len(df_modelo_18h):,}"
)

print("\nRegistros por contaminante:")

print(
    df_modelo_18h["contaminant"].value_counts()
)

# ==========================================
# GUARDADO DEFINITIVO EN DATOS LIMPIOS
# ==========================================

ruta_datos_limpios = ruta_base / "DATOS LIMPIOS"
ruta_datos_limpios.mkdir(parents=True, exist_ok=True)

ruta_definitiva = (
    ruta_datos_limpios /
    "df_contaminacion_atm_2018_2024_Limpio.csv"
)

df_modelo_18h.to_csv(
    ruta_definitiva,
    index=False
)

print(
    f"\nDataset definitivo guardado en: "
    f"{ruta_definitiva}"
)


--- 1. ESTADÍSTICAS DESCRIPTIVAS ---


,count,mean,std,min,25%,50%,75%,max
contaminant,,,,,,,,
NO2,17188.0,25.139627,14.358170,1.0,14.000000,22.708333,33.666667,113.400000
O3,12766.0,54.672389,21.544355,1.0,40.336957,55.250000,68.739130,233.583333
PM10,11954.0,21.527592,10.983317,1.0,14.291667,19.526316,26.583333,123.695652
PM2.5,1734.0,9.916487,5.602957,1.0,6.250000,8.645833,12.156250,46.166667



--- 2. VALORES NULOS ---
fecha               0
codi_estacio        0
nom_cabina          0
contaminant         0
valor               0
horas_validas       0
codi_estacio_str    0
estacion_fisica     0
dtype: int64

--- 3. DUPLICADOS ---
Duplicados encontrados: 0

--- 4. DETECCIÓN DE OUTLIERS (IQR) ---
NO2: 241 outliers (1.40%)
PM10: 353 outliers (2.95%)
O3: 73 outliers (0.57%)
PM2.5: 94 outliers (5.42%)

--- 5. RESUMEN FINAL ---
Periodo: 2018-06-12 - 2024-12-31
Estaciones: 10
Contaminantes: ['NO2' 'PM10' 'O3' 'PM2.5']
Registros finales: 43,642

Registros por contaminante:
contaminant
NO2      17188
O3       12766
PM10     11954
PM2.5     1734
Name: count, dtype: int64

Dataset definitivo guardado en: /content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/DATOS_DIARIOS_HORARIOS/DATOS LIMPIOS/df_contaminacion_atm_2018_2024_Limpio.csv


In [ ]:
# ============================================================
# PREINTEGRACIÓN — CONTAMINACIÓN ATMOSFÉRICA
# CARGA DEL DATASET DE 11 + GEOLOCALIZACIÓN DE ESTACIONES
# ============================================================

import pandas as pd
import geopandas as gpd
from pathlib import Path


# ============================================================
# 1. DEFINIR RUTA DE TRABAJO
# ============================================================
# IMPORTANTE:
# Se trabaja EXCLUSIVAMENTE con la copia situada dentro de:
#
# 11_Machine_Learning_Temporal/
# └── DATOS LIMPIOS/
#     └── 06_Contaminacion_Atmosferica/
# ============================================================

RUTA_CSV_CONTAMINACION = Path(
    "/content/drive/MyDrive/TFM/06_Contaminacion_Atmosferica/"
    "DATOS_DIARIOS_HORARIOS/DATOS LIMPIOS/"
    "df_contaminacion_atm_2018_2024_Limpio.csv"
)

RUTA_CSV_CONTAMINACION = (
    RUTA_CONTAMINACION /
    "df_contaminacion_atm_2018_2024_Limpio.csv"
)

RUTA_CSV_ESTACIONES = (
    RUTA_CONTAMINACION /
    "estaciones_calidad_aire.csv"
)


# ============================================================
# 2. COMPROBAR Y CARGAR EL CSV DE CONTAMINACIÓN DE 11
# ============================================================

if not RUTA_CSV_CONTAMINACION.exists():
    raise FileNotFoundError(
        "No se encuentra el dataset limpio de contaminación "
        "dentro de 11_Machine_Learning_Temporal:\n\n"
        f"{RUTA_CSV_CONTAMINACION}"
    )

df_contaminacion = pd.read_csv(
    RUTA_CSV_CONTAMINACION
)

print("=" * 80)
print("DATASET DE CONTAMINACIÓN CARGADO DESDE 11")
print("=" * 80)

print("\nRuta utilizada:")
print(RUTA_CSV_CONTAMINACION)

print(
    f"\nDimensiones: "
    f"{df_contaminacion.shape}"
)

print(
    f"Registros: "
    f"{len(df_contaminacion):,}"
)


# ============================================================
# 3. CREAR TABLA DE ESTACIONES DE CALIDAD DEL AIRE
# ============================================================

datos_estaciones = [
    {
        "estacion_fisica": "Eixample",
        "lat": 41.3853,
        "lon": 2.1538,
        "tipo": "Tráfico"
    },
    {
        "estacion_fisica": "Gràcia",
        "lat": 41.3987,
        "lon": 2.1534,
        "tipo": "Tráfico"
    },
    {
        "estacion_fisica": "Ciutadella",
        "lat": 41.3864,
        "lon": 2.1874,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Palau Reial",
        "lat": 41.3875,
        "lon": 2.1153,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Vall d'Hebron",
        "lat": 41.4261,
        "lon": 2.1478,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Poblenou",
        "lat": 41.4039,
        "lon": 2.2045,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Sants",
        "lat": 41.3791,
        "lon": 2.1328,
        "tipo": "Fondo"
    },
    {
        "estacion_fisica": "Observatori Fabra",
        "lat": 41.4184,
        "lon": 2.1239,
        "tipo": "Fondo regional"
    }
]

df_estaciones = pd.DataFrame(
    datos_estaciones
)


# ============================================================
# 4. GEOLOCALIZAR ESTACIONES
# ============================================================
# Coordenadas originales:
# EPSG:4326 — WGS84
#
# Sistema proyectado de trabajo:
# EPSG:25831 — ETRS89 / UTM zona 31N
# ============================================================

gdf_estaciones = gpd.GeoDataFrame(
    df_estaciones,
    geometry=gpd.points_from_xy(
        df_estaciones["lon"],
        df_estaciones["lat"]
    ),
    crs="EPSG:4326"
)

gdf_estaciones = gdf_estaciones.to_crs(
    "EPSG:25831"
)


# ============================================================
# 5. EXTRAER COORDENADAS PROYECTADAS
# ============================================================

gdf_estaciones["X_ETRS89"] = (
    gdf_estaciones.geometry.x
)

gdf_estaciones["Y_ETRS89"] = (
    gdf_estaciones.geometry.y
)


# ============================================================
# 6. CREAR TABLA FINAL DE ESTACIONES
# ============================================================

df_estaciones_final = (
    gdf_estaciones
    .drop(columns=["geometry"])
    .copy()
)


# ============================================================
# 7. GUARDAR ESTACIONES JUNTO AL CSV DE CONTAMINACIÓN DE 11
# ============================================================

df_estaciones_final.to_csv(
    RUTA_CSV_ESTACIONES,
    index=False,
    encoding="utf-8"
)


# ============================================================
# 8. CONTROL FINAL
# ============================================================

print("\n" + "=" * 80)
print("CONTROL FINAL — BLOQUE CONTAMINACIÓN")
print("=" * 80)

print("\nDataset utilizado para la preintegración:")
print(RUTA_CSV_CONTAMINACION)

print(
    "\n✓ Existe:",
    RUTA_CSV_CONTAMINACION.exists()
)

print("\nTabla de estaciones:")
print(RUTA_CSV_ESTACIONES)

print(
    "\n✓ Existe:",
    RUTA_CSV_ESTACIONES.exists()
)


# ============================================================
# 9. CONTENIDO DE LA CARPETA DE TRABAJO
# ============================================================

print("\n" + "=" * 80)
print("ARCHIVOS DISPONIBLES EN EL BLOQUE")
print("=" * 80)

for archivo in sorted(RUTA_CONTAMINACION.iterdir()):
    print("✓", archivo.name)


# ============================================================
# 10. VISUALIZACIÓN DE LAS ESTACIONES
# ============================================================

print("\nTabla de estaciones geolocalizadas:")

display(df_estaciones_final)


# ============================================================
# 11. CIERRE
# ============================================================

print("\n" + "=" * 80)
print("BLOQUE PREPARADO PARA PREINTEGRACIÓN")
print("=" * 80)

print(
    f"\n✓ Contaminación cargada: "
    f"{len(df_contaminacion):,} registros."
)

print(
    f"✓ Estaciones geolocalizadas: "
    f"{len(df_estaciones_final)}."
)

print(
    "✓ Fuente de trabajo: "
    "11_Machine_Learning_Temporal."
)

print(
    "✓ CRS espacial: EPSG:25831."
)

FileNotFoundError: No se encuentra el dataset limpio de contaminación dentro de 11_Machine_Learning_Temporal:

/content/drive/MyDrive/TFM/11_Machine_Learning_Temporal/DATOS LIMPIOS/06_Contaminacion_Atmosferica/df_contaminacion_atm_2018_2024_Limpio.csv

## 1.12 Cierre del bloque de contaminación atmosférica

### Decisiones metodológicas adoptadas

- **Periodo disponible:** 12/06/2018–31/12/2024.
- Se identificaron **dos estructuras diferentes en los archivos originales**:
  - junio de 2018–marzo de 2019: formato histórico de 18 columnas;
  - desde abril de 2019: formato de 57 columnas.
- Ambos formatos se transformaron a una estructura homogénea a escala diaria.
- En el formato moderno se utilizaron las **banderas de validez horaria** para descartar observaciones no válidas.
- Se eliminaron concentraciones negativas y registros sin observaciones válidas.
- Se comprobó la ausencia de duplicidades horarias antes de realizar la agregación diaria.
- La concentración diaria se obtuvo como la **media de las observaciones horarias válidas**.
- Se adoptó un umbral mínimo de **18 horas válidas por día**. Este criterio conserva el 96,44 % de los registros con datos válidos y proporciona un equilibrio entre cobertura temporal y calidad de la media diaria.
- No se eliminaron automáticamente los valores extremos identificados mediante IQR, ya que pueden representar episodios reales de contaminación.
- Los distintos códigos históricos de las estaciones fueron identificados y armonizados, manteniendo temporalmente separados los códigos asociados a Sants para preservar la trazabilidad.
- Se generó una tabla independiente con las coordenadas y la tipología de las estaciones en WGS84 y ETRS89 / UTM 31N.

### Dataset resultante

El dataset definitivo contiene **43.642 observaciones diarias**, sin valores nulos ni duplicados, correspondientes a cuatro contaminantes:

- NO₂: 17.188 registros.
- O₃: 12.766 registros.
- PM10: 11.954 registros.
- PM2.5: 1.734 registros.

### Estrategia para el análisis posterior

El **NO₂ se considera el contaminante principal** del análisis debido a su mayor continuidad y cobertura temporal. O₃ y PM10 se mantienen como contaminantes complementarios, mientras que PM2.5 se analizará teniendo en cuenta su menor disponibilidad histórica.

La integración de este bloque con meteorología, movilidad y el resto de fuentes temporales se realizará posteriormente en el notebook específico de construcción del dataset maestro temporal.

### Aspectos pendientes para la integración

1. Homogeneizar definitivamente la nomenclatura de estaciones antes de realizar los cruces entre fuentes (`Vall Hebron` / `Vall d'Hebron`, `Observ Fabra` / `Observatori Fabra`).

2. Resolver de forma explícita la correspondencia temporal de los códigos asociados a Sants (`ID`, `4` y `42`) antes de construir una única serie por estación física.

# 2 · PREPARACIÓN DE LOS DATOS METEOROLÓGICOS

Los datos meteorológicos utilizados en este estudio proceden del **Portal Open Data del Ayuntamiento de Barcelona**, concretamente del conjunto de datos de medidas registradas por las estaciones meteorológicas de la ciudad:

**Fuente:** Ajuntament de Barcelona – Open Data BCN, *Mesures de les estacions meteorològiques*.  
https://opendata-ajuntament.barcelona.cat/data/es/dataset/mesures-estacions-meteorologiques

**Conjunto de datos:** Medidas de las estaciones meteorológicas de Barcelona.

Los datos contienen registros meteorológicos diarios procedentes de cuatro estaciones (`D5`, `X2`, `X4` y `X8`) y proporcionan información sobre temperatura, humedad relativa, presión atmosférica, precipitación, radiación solar y características del viento.

El periodo utilizado en este trabajo comprende desde **enero de 2018 hasta diciembre de 2024**.

Durante la inspección inicial de los archivos se identificaron diferencias en la estructura y disponibilidad de las variables entre los distintos periodos:

- **2018–2019:** los datos se encuentran distribuidos en archivos independientes por estación y presentan una disponibilidad de variables inferior a la existente en los años posteriores.
- **2020–2024:** los registros presentan inicialmente una estructura en formato largo, con observaciones identificadas mediante fecha, estación, acrónimo de la variable meteorológica y valor registrado.

Esta heterogeneidad hizo necesario desarrollar procedimientos diferenciados de lectura y transformación antes de obtener una estructura común para todo el periodo de estudio..

## 2.1 Carga e inspección inicial de los datos meteorológicos

Se realiza una primera inspección de los datos meteorológicos disponibles, utilizando como muestra el archivo correspondiente a 2024. El objetivo es identificar la estructura del dataset, las variables disponibles y el formato de los registros antes de abordar su selección, limpieza y homogeneización temporal.

In [ ]:
import pandas as pd

# Ruta al archivo seleccionado dentro de la carpeta 08_Meteorologia
ruta_meteo = '/content/drive/My Drive/TFM/08_Meteorologia/2024_MeteoCat_Detall_Estacions.csv'

# 1. Leer el archivo CSV (probando separador estándar o punto y coma)
try:
    df_meteo = pd.read_csv(ruta_meteo, encoding='utf-8')
except Exception:
    try:
        df_meteo = pd.read_csv(ruta_meteo, encoding='latin1', sep=';')
    except Exception:
        df_meteo = pd.read_csv(ruta_meteo, encoding='utf-8', sep=';')

# 2. Mostrar las variables (columnas) disponibles
print("--- VARIABLES / COLUMNAS (METEOROLOGÍA) ---")
print(df_meteo.columns.tolist())
print("\n" + "="*50 + "\n")

# 3. Mostrar la información general del DataFrame (tipos de datos y nulos)
print("--- INFORMACIÓN GENERAL (INFO) ---")
df_meteo.info()
print("\n" + "="*50 + "\n")

# 4. Mostrar el resumen estadístico
print("--- RESUMEN ESTADÍSTICO (DESCRIBE) ---")
print(df_meteo.describe(include='all'))
print("\n" + "="*50 + "\n")

# 5. Mostrar las 10 primeras filas
print("--- 10 PRIMERAS FILAS ---")
print(df_meteo.head(10))

--- VARIABLES / COLUMNAS (METEOROLOGÍA) ---
['DATA_LECTURA', 'DATA_EXTREM', 'CODI_ESTACIO', 'ACRÒNIM', 'VALOR']


--- INFORMACIÓN GENERAL (INFO) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18094 entries, 0 to 18093
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   DATA_LECTURA  18094 non-null  object 
 1   DATA_EXTREM   9870 non-null   object 
 2   CODI_ESTACIO  18094 non-null  object 
 3   ACRÒNIM       18094 non-null  object 
 4   VALOR         18094 non-null  float64
dtypes: float64(1), object(4)
memory usage: 706.9+ KB


--- RESUMEN ESTADÍSTICO (DESCRIBE) ---
       DATA_LECTURA DATA_EXTREM CODI_ESTACIO ACRÒNIM         VALOR
count         18094        9870        18094   18094  18094.000000
unique          366        1346            4      15           NaN
top      2024-09-30    00:00:00           X4      TM           NaN
freq             51         268         5490    1372           NaN
mean        

### Resultado de la inspección inicial

El archivo meteorológico correspondiente a 2024 contiene **18.094 registros y 5 variables**, organizados en formato largo. Cada observación queda identificada mediante la fecha de lectura (`DATA_LECTURA`), el código de estación (`CODI_ESTACIO`), el acrónimo de la variable meteorológica (`ACRÒNIM`) y su valor (`VALOR`).

Se identifican **4 estaciones meteorológicas**, **15 variables distintas** y una cobertura de **366 fechas**, correspondiente al año bisiesto 2024.

La variable `DATA_EXTREM`, que registra la hora asociada a determinados valores extremos diarios, presenta información únicamente en 9.870 registros. Esta ausencia no se interpreta inicialmente como dato perdido, ya que su disponibilidad depende del tipo de variable meteorológica considerada.

La estructura observada confirma que los datos se encuentran en formato largo y deberán transformarse posteriormente para obtener una tabla diaria con las variables meteorológicas organizadas en columnas.

## 2.2 Unificación temporal y transformación del bloque meteorológico

In [ ]:
import pandas as pd
import glob
import os

# 1. Definir la ruta base de la carpeta de meteorología en tu Google Drive
ruta_base = '/content/drive/My Drive/TFM/08_Meteorologia/'

# Lista de años que queremos unificar
anos = [2020, 2021, 2022, 2023, 2024]
dfs = []

# 2. Leer y concatenar los CSVs de 2020 a 2024
for ano in anos:
    # Construir el nombre del archivo según el patrón que se ve en tu imagen
    nombre_archivo = f"{ano}_MeteoCat_Detall_Estacions.csv"
    ruta_completa = os.path.join(ruta_base, nombre_archivo)

    if os.path.exists(ruta_completa):
        print(f"Leyendo archivo: {nombre_archivo}...")
        # Probamos lectura estándar o con separador punto y coma por seguridad
        try:
            df_temp = pd.read_csv(ruta_completa, encoding='utf-8')
            if 'DATA_LECTURA' not in df_temp.columns:
                df_temp = pd.read_csv(ruta_completa, encoding='latin1', sep=';')
        except Exception:
            df_temp = pd.read_csv(ruta_completa, encoding='latin1', sep=';')

        dfs.append(df_temp)
    else:
        print(f"No se encontró el archivo para el año {ano} en la ruta: {ruta_completa}")

# Unir todos los años en un único DataFrame
if dfs:
    df_meteo_total = pd.concat(dfs, ignore_index=True)
    print("\n¡Archivos de 2020 a 2024 unificados con éxito!")

    # 3. Estandarizar formatos y eliminar DATA_EXTREM
    if 'DATA_EXTREM' in df_meteo_total.columns:
        df_meteo_total = df_meteo_total.drop(columns=['DATA_EXTREM'])

    df_meteo_total['DATA_LECTURA'] = pd.to_datetime(df_meteo_total['DATA_LECTURA'])
    df_meteo_total['CODI_ESTACIO'] = df_meteo_total['CODI_ESTACIO'].astype('category')
    df_meteo_total['VALOR'] = pd.to_numeric(df_meteo_total['VALOR'], errors='coerce')

    # 4. Pivotar la tabla (pasarla de formato largo a ancho)
    # Usamos pivot_table para manejar posibles duplicados de una misma variable en un mismo día/estación
    df_meteo_pivot = df_meteo_total.pivot_table(
        index=['DATA_LECTURA', 'CODI_ESTACIO'],
        columns='ACRÒNIM',
        values='VALOR',
        aggfunc='mean'
    ).reset_index()

    # Renombrar columnas clave para mayor comodidad
    df_meteo_pivot.rename(columns={'DATA_LECTURA': 'Fecha', 'CODI_ESTACIO': 'Estacion'}, inplace=True)

    print("\n--- VISTA PREVIA DEL DATASET METEOROLÓGICO PIVOTADO ---")
    display(df_meteo_pivot.head(10))

    # Opcional: Guardar el resultado procesado en tu Drive para no tener que recalcularlo
    # df_meteo_pivot.to_csv(os.path.join(ruta_base, 'Meteo_2020_2024_Pivotado.csv'), index=False)
else:
    print("No se pudo cargar ningún archivo. Revisa las rutas.")

Leyendo archivo: 2020_MeteoCat_Detall_Estacions.csv...
Leyendo archivo: 2021_MeteoCat_Detall_Estacions.csv...
Leyendo archivo: 2022_MeteoCat_Detall_Estacions.csv...
Leyendo archivo: 2023_MeteoCat_Detall_Estacions.csv...
Leyendo archivo: 2024_MeteoCat_Detall_Estacions.csv...

¡Archivos de 2020 a 2024 unificados con éxito!

--- VISTA PREVIA DEL DATASET METEOROLÓGICO PIVOTADO ---


/tmp/ipykernel_1272/2092426163.py:47: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df_meteo_pivot = df_meteo_total.pivot_table(


ACRÒNIM,Fecha,Estacion,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
0,2020-01-01,D5,63.0,321.0,81.0,64.0,91.0,981.8,980.8,0.0,983.2,8.8,7.0,4.6,11.5,2.6,11.2
1,2020-01-01,X2,NaN,NaN,81.0,62.0,95.0,NaN,NaN,NaN,NaN,NaN,8.3,4.5,12.0,NaN,NaN
2,2020-01-01,X4,221.0,288.0,68.0,55.0,77.0,1027.4,1026.5,0.0,1028.9,8.8,10.0,6.7,13.3,1.0,4.2
3,2020-01-01,X8,275.0,293.0,79.0,57.0,94.0,1021.8,1020.9,0.0,1023.3,8.6,8.2,3.8,12.9,1.1,4.9
4,2020-01-02,D5,258.0,267.0,82.0,65.0,89.0,981.5,980.6,0.0,983.0,7.8,7.7,5.5,12.3,5.2,10.3
5,2020-01-02,X2,NaN,NaN,81.0,60.0,96.0,NaN,NaN,NaN,NaN,NaN,8.9,4.1,14.0,NaN,NaN
6,2020-01-02,X4,281.0,285.0,69.0,55.0,79.0,1027.1,1025.7,0.0,1028.8,7.7,10.6,7.1,14.4,1.6,7.8
7,2020-01-02,X8,283.0,230.0,81.0,58.0,96.0,1021.5,1020.3,0.0,1023.1,7.3,8.9,4.6,14.4,1.7,6.4
8,2020-01-03,D5,293.0,267.0,73.0,46.0,92.0,980.7,978.8,0.0,982.2,6.9,8.1,4.2,10.8,5.4,12.8
9,2020-01-03,X2,NaN,NaN,77.0,51.0,97.0,NaN,NaN,NaN,NaN,NaN,9.1,6.1,13.7,NaN,NaN


En esta etapa se integran los archivos meteorológicos correspondientes al periodo **2020–2024** en un único conjunto de datos. Tras la concatenación anual, se normalizan los tipos de las variables principales y se elimina `DATA_EXTREM`, al no resultar necesaria para la construcción del dataset diario.

Posteriormente, los registros se transforman de formato largo a formato ancho mediante una tabla dinámica, de forma que cada combinación de **fecha y estación meteorológica** quede representada por una única fila y cada variable meteorológica pase a ocupar una columna independiente.

Este formato facilita la posterior selección de variables, el análisis de cobertura y la integración con el resto de fuentes temporales.

## 2.3 Auditoría de calidad y cobertura del dataset meteorológico

Una vez unificados los archivos anuales y transformados los datos meteorológicos a formato ancho, se realiza una auditoría del dataset resultante antes de continuar con su procesamiento.

Se examinan los tipos de datos y dimensiones del conjunto, la distribución estadística de las variables meteorológicas, la presencia y proporción de valores nulos y la disponibilidad de registros por estación.

Este análisis permite evaluar la continuidad y calidad de cada variable meteorológica, detectar diferencias de cobertura entre estaciones y determinar qué variables pueden incorporarse de forma robusta al posterior dataset maestro temporal.

In [ ]:
# Usamos el nombre exacto de tu DataFrame pivotado
df = df_meteo_pivot

print("="*60)
print("1. INFORMACIÓN GENERAL (TIPOS Y MEMORIA)")
print("="*60)
print(df.info())

print("\n" + "="*60)
print("2. RESUMEN ESTADÍSTICO")
print("="*60)
display(df.describe())

print("\n" + "="*60)
print("3. CONTEO DE VALORES NULOS POR VARIABLE")
print("="*60)
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().mean() * 100).round(2)
df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos})
print(df_nulos)

print("\n" + "="*60)
print("4. CONTEO DE REGISTROS POR ESTACIÓN")
print("="*60)
print(df.groupby("Estacion").count())

1. INFORMACIÓN GENERAL (TIPOS Y MEMORIA)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7216 entries, 0 to 7215
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Fecha     7216 non-null   datetime64[ns]
 1   Estacion  7216 non-null   category      
 2   DVM10     5461 non-null   float64       
 3   DVVX10    5470 non-null   float64       
 4   HRM       7215 non-null   float64       
 5   HRN       7214 non-null   float64       
 6   HRX       7214 non-null   float64       
 7   PM        5481 non-null   float64       
 8   PN        5481 non-null   float64       
 9   PPT       5481 non-null   float64       
 10  PX        5481 non-null   float64       
 11  RS24h     5474 non-null   float64       
 12  TM        7215 non-null   float64       
 13  TN        7216 non-null   float64       
 14  TX        7216 non-null   float64       
 15  VVM10     5473 non-null   float64       
 16  VVX10     5474 non-

ACRÒNIM,Fecha,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
count,7216,5461.000000,5470.000000,7215.000000,7214.000000,7214.000000,5481.000000,5481.000000,5481.000000,5481.000000,5474.000000,7215.000000,7216.000000,7216.000000,5473.000000,5474.00000
mean,2022-06-20 22:33:23.547672064,201.101080,205.963254,68.208455,47.586498,87.379817,996.746816,994.569513,1.314851,999.074621,15.836609,17.658933,14.281846,21.990382,2.687338,9.21867
min,2020-01-01 00:00:00,0.000000,0.000000,24.000000,7.000000,36.000000,942.500000,936.600000,0.000000,946.800000,0.200000,1.700000,-1.400000,4.100000,0.500000,3.00000
25%,2021-03-26 18:00:00,118.000000,125.000000,60.000000,38.000000,80.000000,972.900000,971.000000,0.000000,975.000000,9.000000,12.800000,9.500000,16.800000,1.700000,7.00000
50%,2022-06-20 12:00:00,230.000000,213.000000,69.000000,47.000000,90.000000,1006.100000,1003.900000,0.000000,1008.300000,14.900000,17.000000,13.700000,21.500000,2.300000,8.70000
75%,2023-09-14 06:00:00,270.000000,298.000000,77.000000,57.000000,97.000000,1012.500000,1010.600000,0.000000,1014.700000,22.900000,22.850000,19.400000,27.400000,3.400000,10.90000
max,2024-12-31 00:00:00,359.000000,359.000000,100.000000,100.000000,100.000000,1033.700000,1032.800000,93.200000,1035.700000,32.000000,33.400000,29.500000,39.500000,15.300000,30.30000
std,NaN,89.529605,94.368556,12.296193,13.275606,11.508267,20.587990,20.712482,5.660429,20.537422,8.041284,6.000567,5.979697,6.239265,1.400116,3.13627



3. CONTEO DE VALORES NULOS POR VARIABLE
          Nulos  Porcentaje (%)
ACRÒNIM                        
Fecha         0            0.00
Estacion      0            0.00
DVM10      1755           24.32
DVVX10     1746           24.20
HRM           1            0.01
HRN           2            0.03
HRX           2            0.03
PM         1735           24.04
PN         1735           24.04
PPT        1735           24.04
PX         1735           24.04
RS24h      1742           24.14
TM            1            0.01
TN            0            0.00
TX            0            0.00
VVM10      1743           24.15
VVX10      1742           24.14

4. CONTEO DE REGISTROS POR ESTACIÓN
ACRÒNIM   Fecha  DVM10  DVVX10   HRM   HRN   HRX    PM    PN   PPT    PX  \
Estacion                                                                   
D5         1827   1811    1818  1826  1825  1825  1827  1827  1827  1827   
X2         1735      0       0  1735  1735  1735     0     0     0     0   
X4        

/tmp/ipykernel_1272/561168229.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("Estacion").count())


### Resultado de la auditoría de calidad y cobertura

El dataset meteorológico pivotado contiene **7.216 observaciones diarias por estación** y **15 variables meteorológicas**, además de la fecha y el identificador de estación. El periodo disponible comprende desde el **1 de enero de 2020 hasta el 31 de diciembre de 2024**.

La cobertura de las variables de **temperatura y humedad relativa** es prácticamente completa. Las variables `TM`, `TN`, `TX`, `HRM`, `HRN` y `HRX` presentan porcentajes de valores ausentes próximos al 0 %.

Por el contrario, las variables asociadas a **presión atmosférica, precipitación, radiación solar y viento** presentan aproximadamente un **24 % de valores ausentes** cuando se analiza el dataset de forma global. La inspección por estación muestra que esta ausencia se concentra fundamentalmente en la estación `X2`, que dispone de registros de temperatura y humedad, pero no proporciona las restantes variables consideradas.

Las estaciones `D5`, `X4` y `X8` presentan, en cambio, una cobertura prácticamente completa de las variables meteorológicas durante el periodo disponible.

Por tanto, los valores ausentes observados no deben interpretarse inicialmente como fallos aleatorios de medición, sino principalmente como consecuencia de la **diferente disponibilidad de variables entre estaciones meteorológicas**. Esta característica deberá tenerse en cuenta en la posterior selección y tratamiento de las variables antes de su integración en el dataset maestro temporal.

## 2.4 Incorporación y homogeneización de los datos meteorológicos de 2018–2019

El periodo 2018–2019 se procesa de forma independiente debido a que sus archivos presentan una estructura distinta a la utilizada entre 2020 y 2024.

En esta etapa se cargan y concatenan los archivos disponibles para ambos años, se normaliza el formato de fecha, se filtran exclusivamente los registros correspondientes a 2018 y 2019 y se armonizan los nombres de las variables para hacerlos compatibles con el dataset meteorológico ya preparado para 2020–2024.

Asimismo, se estandarizan los tipos de datos y, cuando es posible, se reordenan las columnas siguiendo exactamente la estructura del dataset meteorológico moderno. De este modo, ambos periodos quedan preparados para su posterior concatenación en una única serie meteorológica homogénea.

In [ ]:
import pandas as pd
import glob
import os

# 1. Ruta a la subcarpeta 2018-2019
ruta_carpeta = '/content/drive/My Drive/TFM/08_Meteorologia/2018-2019/'
patron = os.path.join(ruta_carpeta, "*_*.csv")
archivos = glob.glob(patron)

dfs = []
for archivo in archivos:
    nombre_archivo = os.path.basename(archivo)
    print(f"Leyendo archivo: {nombre_archivo}...")
    try:
        df_temp = pd.read_csv(archivo, encoding='utf-8')
        if len(df_temp.columns) <= 1:
            df_temp = pd.read_csv(archivo, encoding='latin1', sep=';')
    except Exception:
        df_temp = pd.read_csv(archivo, encoding='latin1', sep=';')
    dfs.append(df_temp)

if dfs:
    df_18_19_total = pd.concat(dfs, ignore_index=True)

    # 2. Estandarizar fecha y filtrar estrictamente 2018 y 2019
    df_18_19_total['DATA_LECTURA'] = pd.to_datetime(df_18_19_total['DATA_LECTURA'], format='%d/%m/%Y', errors='coerce')
    df_18_19_total = df_18_19_total[df_18_19_total['DATA_LECTURA'].dt.year.isin([2018, 2019])]

    # 3. Renombrar columnas principales para que coincidan con el otro dataset ('Fecha', 'Estacion')
    df_18_19_total.rename(columns={'DATA_LECTURA': 'Fecha', 'CODI_ESTACIO': 'Estacion'}, inplace=True)

    # 4. Ajustar nombres de variables si difieren (ej: PPT24H -> PPT) para idéntica estructura
    if 'PPT24H' in df_18_19_total.columns and 'PPT' not in df_18_19_total.columns:
        df_18_19_total.rename(columns={'PPT24H': 'PPT'}, inplace=True)

    # 5. Estandarizar tipos de datos numéricos y de categoría
    df_18_19_total['Estacion'] = df_18_19_total['Estacion'].astype('category')
    cols_numericas = [col for col in df_18_19_total.columns if col not in ['Fecha', 'Estacion']]
    for col in cols_numericas:
        df_18_19_total[col] = pd.to_numeric(df_18_19_total[col], errors='coerce')

    # Opcional: Reordenar columnas alfabéticamente o igualar al DataFrame pivotado anterior si lo tienes en memoria (`df_meteo_pivot`)
    if 'df_meteo_pivot' in locals():
        df_18_19_total = df_18_19_total.reindex(columns=df_meteo_pivot.columns)

    print("\n¡Dataset de 2018-2019 ajustado al formato exacto!")
    print(f"Rango de fechas: {df_18_19_total['Fecha'].min().date()} a {df_18_19_total['Fecha'].max().date()}")

    display(df_18_19_total.head(10))
else:
    print("No se encontraron archivos en la ruta especificada.")

Leyendo archivo: 2019_x8_barcelona_zona_universitaria.csv...
Leyendo archivo: 2019_x4_barcelona_el_raval.csv...
Leyendo archivo: 2019_x2_barcelona_zoo.csv...
Leyendo archivo: 2019_d5_observatori_fabra.csv...

¡Dataset de 2018-2019 ajustado al formato exacto!
Rango de fechas: 2018-01-01 a 2019-12-31


ACRÒNIM,Fecha,Estacion,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
3532,2018-01-01,X8,299.0,NaN,46.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,12.4,9.2,15.8,5.3,17.3
3533,2018-01-02,X8,297.0,NaN,58.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,13.9,10.3,18.1,3.5,14.2
3534,2018-01-03,X8,305.0,NaN,58.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,17.5,14.0,21.5,6.4,17.7
3535,2018-01-04,X8,302.0,NaN,53.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,18.3,14.9,22.0,4.8,16.6
3536,2018-01-05,X8,283.0,NaN,50.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,15.0,10.5,19.2,3.0,11.9
3537,2018-01-06,X8,75.0,NaN,78.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,12.9,8.7,16.5,3.1,15.1
3538,2018-01-07,X8,159.0,NaN,81.0,NaN,NaN,NaN,NaN,5.5,NaN,NaN,10.8,6.6,14.5,2.1,9.6
3539,2018-01-08,X8,306.0,NaN,71.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,9.0,5.1,15.4,2.4,9.3
3540,2018-01-09,X8,288.0,NaN,55.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,9.1,5.2,15.0,3.1,10.2
3541,2018-01-10,X8,290.0,NaN,53.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,10.5,7.3,14.7,2.8,8.9


### Resultado de la homogeneización del periodo 2018–2019

Los archivos históricos de 2018–2019 se han adaptado correctamente a la misma estructura utilizada para el periodo 2020–2024, permitiendo trabajar con una serie meteorológica homogénea a escala diaria.

El periodo disponible comprende desde el **1 de enero de 2018 hasta el 31 de diciembre de 2019**.

La inspección de los registros muestra, sin embargo, que la cobertura histórica no es uniforme entre variables. En particular, algunas magnitudes como la presión atmosférica (`PM`, `PN`, `PX`), la radiación solar (`RS24h`) y determinados indicadores extremos de humedad o viento presentan valores ausentes en parte del periodo 2018–2019, mientras que variables como temperatura, humedad media, precipitación y algunas medidas de viento muestran una mayor continuidad.

Por tanto, aunque ambos periodos han quedado estructuralmente homogeneizados, será necesario evaluar posteriormente la cobertura temporal de cada variable antes de construir el dataset meteorológico definitivo.

## 2.5 Auditoría de calidad y cobertura del periodo 2018–2019

Una vez homogeneizada la estructura de los datos meteorológicos correspondientes a 2018–2019, se realiza una auditoría específica de este periodo antes de integrarlo con la serie 2020–2024.

Se examinan las dimensiones y tipos de datos, los principales estadísticos descriptivos, la cantidad y proporción de valores ausentes de cada variable y la cobertura disponible por estación meteorológica.

Este análisis permite identificar posibles diferencias en la disponibilidad histórica de las variables y determinar cuáles presentan una continuidad temporal suficiente para su utilización en el posterior análisis temporal.

In [ ]:
# Usamos el nombre exacto de tu DataFrame de 2018-2019
df = df_18_19_total

print("="*60)
print("1. INFORMACIÓN GENERAL (TIPOS Y MEMORIA) - 2018-2019")
print("="*60)
print(df.info())

print("\n" + "="*60)
print("2. RESUMEN ESTADÍSTICO - 2018-2019")
print("="*60)
display(df.describe())

print("\n" + "="*60)
print("3. CONTEO DE VALORES NULOS POR VARIABLE - 2018-2019")
print("="*60)
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().mean() * 100).round(2)
df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje_nulos})
print(df_nulos)

print("\n" + "="*60)
print("4. CONTEO DE REGISTROS POR ESTACIÓN - 2018-2019")
print("="*60)
print(df.groupby("Estacion").count())

1. INFORMACIÓN GENERAL (TIPOS Y MEMORIA) - 2018-2019
<class 'pandas.core.frame.DataFrame'>
Index: 2920 entries, 3532 to 22668
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Fecha     2920 non-null   datetime64[ns]
 1   Estacion  2920 non-null   category      
 2   DVM10     2190 non-null   float64       
 3   DVVX10    0 non-null      float64       
 4   HRM       2920 non-null   float64       
 5   HRN       0 non-null      float64       
 6   HRX       0 non-null      float64       
 7   PM        0 non-null      float64       
 8   PN        0 non-null      float64       
 9   PPT       2190 non-null   float64       
 10  PX        0 non-null      float64       
 11  RS24h     0 non-null      float64       
 12  TM        2920 non-null   float64       
 13  TN        2920 non-null   float64       
 14  TX        2920 non-null   float64       
 15  VVM10     2190 non-null   float64       
 16  VVX10   

ACRÒNIM,Fecha,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
count,2920,2190.000000,0.0,2920.000000,0.0,0.0,0.0,0.0,2190.000000,0.0,0.0,2920.000000,2920.000000,2920.000000,2190.000000,2190.000000
mean,2018-12-31 12:00:00,208.374429,NaN,67.938014,NaN,NaN,NaN,NaN,2.133744,NaN,NaN,17.088836,13.730651,21.277774,2.808402,9.577169
min,2018-01-01 00:00:00,0.000000,NaN,22.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,-0.700000,-2.400000,1.200000,0.700000,3.700000
25%,2018-07-02 00:00:00,131.000000,NaN,60.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,12.200000,8.700000,16.200000,1.800000,7.200000
50%,2018-12-31 12:00:00,230.000000,NaN,69.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,16.350000,13.100000,20.600000,2.300000,9.000000
75%,2019-07-02 00:00:00,287.000000,NaN,76.000000,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,22.400000,19.000000,26.525000,3.500000,11.300000
max,2019-12-31 00:00:00,358.000000,NaN,98.000000,NaN,NaN,NaN,NaN,139.900000,NaN,NaN,32.200000,29.400000,39.800000,11.100000,28.000000
std,NaN,90.884690,NaN,12.005163,NaN,NaN,NaN,NaN,8.657333,NaN,NaN,6.091057,6.119140,6.238893,1.456440,3.282247



3. CONTEO DE VALORES NULOS POR VARIABLE - 2018-2019
          Nulos  Porcentaje (%)
ACRÒNIM                        
Fecha         0             0.0
Estacion      0             0.0
DVM10       730            25.0
DVVX10     2920           100.0
HRM           0             0.0
HRN        2920           100.0
HRX        2920           100.0
PM         2920           100.0
PN         2920           100.0
PPT         730            25.0
PX         2920           100.0
RS24h      2920           100.0
TM            0             0.0
TN            0             0.0
TX            0             0.0
VVM10       730            25.0
VVX10       730            25.0

4. CONTEO DE REGISTROS POR ESTACIÓN - 2018-2019
ACRÒNIM   Fecha  DVM10  DVVX10  HRM  HRN  HRX  PM  PN  PPT  PX  RS24h   TM  \
Estacion                                                                     
D5          730    730       0  730    0    0   0   0  730   0      0  730   
X2          730      0       0  730    0    0   0   0   

/tmp/ipykernel_1272/1841973606.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("Estacion").count())


### Resultado de la auditoría del periodo 2018–2019

El bloque meteorológico correspondiente a 2018–2019 contiene **2.920 registros diarios**, distribuidos de forma uniforme entre las cuatro estaciones disponibles (`D5`, `X2`, `X4` y `X8`), con **730 observaciones por estación**. La cobertura temporal comprende íntegramente ambos años.

La disponibilidad de las variables meteorológicas presenta diferencias importantes respecto al periodo 2020–2024:

- Las variables de **temperatura** (`TM`, `TN`, `TX`) y la **humedad relativa media** (`HRM`) presentan cobertura completa en las cuatro estaciones.
- La **dirección media del viento** (`DVM10`), la **velocidad media del viento** (`VVM10`), la **velocidad máxima del viento** (`VVX10`) y la **precipitación** (`PPT`) presentan una cobertura del **75 %**, debido a que no se encuentran disponibles en la estación `X2`.
- Las variables `DVVX10`, `HRN`, `HRX`, `PM`, `PN`, `PX` y `RS24h` no presentan observaciones disponibles durante 2018–2019.

Por tanto, la estructura de los datos puede homogeneizarse con la correspondiente al periodo 2020–2024, pero la **cobertura histórica de las variables no es equivalente**. Esta diferencia deberá mantenerse explícita durante la construcción del dataset meteorológico definitivo, evitando interpretar los valores ausentes como fallos aleatorios de medición.

Las variables de temperatura y humedad media son las que presentan una mayor continuidad para todo el periodo 2018–2024, mientras que otras magnitudes meteorológicas únicamente disponen de una cobertura completa a partir de 2020.

## 2.6 Unificación final de la serie meteorológica 2018–2024

Una vez homogeneizados por separado los periodos 2018–2019 y 2020–2024, ambos bloques se concatenan en una única serie meteorológica diaria.

La tabla resultante se ordena cronológicamente por fecha y estación, manteniendo una estructura común de variables para todo el periodo de estudio. Esta etapa permite disponer de una base meteorológica única y coherente que posteriormente será sometida a controles finales de cobertura, consistencia y calidad antes de su utilización en la construcción del dataset maestro temporal.

In [ ]:
import pandas as pd

# 1. Asegurarnos de que ambos DataFrames tienen exactamente las mismas columnas y tipos
# (El script de 2018-2019 ya lo alineamos antes con reindex o nombres idénticos)

# Concatenar ambos periodos (2018-2019 y 2020-2024)
df_meteo_2018_2024 = pd.concat([df_18_19_total, df_meteo_pivot], ignore_index=True)

# 2. Ordenar cronológicamente por Fecha y por Estación
df_meteo_2018_2024 = df_meteo_2018_2024.sort_values(by=['Fecha', 'Estacion']).reset_index(drop=True)

# 3. Comprobar el resultado final
print("="*60)
print("¡TABLA MAESTRA METEOROLÓGICA UNIFICADA (2018-2024)!")
print("="*60)
print(f"Dimensiones de la tabla: {df_meteo_2018_2024.shape}")
print(f"Rango temporal: desde {df_meteo_2018_2024['Fecha'].min().date()} hasta {df_meteo_2018_2024['Fecha'].max().date()}")
print(f"Estaciones presentes: {df_meteo_2018_2024['Estacion'].unique().tolist()}")

print("\n--- VISTA PREVIA DE LAS 10 PRIMERAS FILAS ---")
display(df_meteo_2018_2024.head(10))

# Opcional: Guardar la tabla maestra unificada en tu Drive para tenerla lista para los siguientes bloques
# ruta_base = '/content/drive/My Drive/TFM/08_Meteorologia/'
# df_meteo_2018_2024.to_csv(ruta_base + 'Meteo_Master_2018_2024.csv', index=False)

¡TABLA MAESTRA METEOROLÓGICA UNIFICADA (2018-2024)!
Dimensiones de la tabla: (10136, 17)
Rango temporal: desde 2018-01-01 hasta 2024-12-31
Estaciones presentes: ['D5', 'X2', 'X4', 'X8']

--- VISTA PREVIA DE LAS 10 PRIMERAS FILAS ---


ACRÒNIM,Fecha,Estacion,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
0,2018-01-01,D5,273.0,NaN,56.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,9.1,6.0,13.0,5.0,18.2
1,2018-01-01,X2,NaN,NaN,44.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,9.8,15.5,NaN,NaN
2,2018-01-01,X4,304.0,NaN,44.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,13.1,10.2,15.8,5.1,17.2
3,2018-01-01,X8,299.0,NaN,46.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,12.4,9.2,15.8,5.3,17.3
4,2018-01-02,D5,303.0,NaN,68.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,10.9,7.8,15.8,4.1,17.2
5,2018-01-02,X2,NaN,NaN,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.9,8.7,17.6,NaN,NaN
6,2018-01-02,X4,296.0,NaN,55.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,14.6,11.5,18.2,2.6,11.6
7,2018-01-02,X8,297.0,NaN,58.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,13.9,10.3,18.1,3.5,14.2
8,2018-01-03,D5,305.0,NaN,68.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,14.4,11.2,18.7,6.3,17.9
9,2018-01-03,X2,NaN,NaN,57.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.6,11.7,21.8,NaN,NaN


### Resultado de la unificación temporal

La concatenación de los bloques 2018–2019 y 2020–2024 genera una serie meteorológica unificada con **10.136 registros diarios por estación** y **17 variables**, correspondiente al periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

Se mantienen las cuatro estaciones meteorológicas disponibles (`D5`, `X2`, `X4` y `X8`) y una estructura común para todo el periodo de estudio.

La inspección de las primeras observaciones confirma que las variables con menor cobertura histórica permanecen como valores ausentes en los años en los que no estaban disponibles, evitando introducir imputaciones prematuras durante la fase de preparación de la fuente.

De este modo, la serie 2018–2024 queda estructuralmente homogeneizada y preparada para la posterior auditoría final de calidad, cobertura temporal y consistencia física antes de su integración en el dataset maestro temporal.

## 2.7 Auditoría final del dataset meteorológico unificado

Una vez construida la serie meteorológica unificada para el periodo **2018–2024**, se realiza una auditoría global del dataset antes de definir su versión definitiva.

Se revisan las dimensiones y tipos de datos, los principales estadísticos descriptivos, el número de valores únicos de cada variable y la cantidad y proporción de valores ausentes.

Este análisis permite cuantificar la cobertura final de cada variable meteorológica en el conjunto completo, comprobar la correcta conservación de la estructura temporal y detectar aquellas variables cuya disponibilidad histórica es parcial antes de adoptar decisiones de selección o tratamiento para el posterior modelado.

In [ ]:
# Usamos tu tabla maestra unificada
df = df_meteo_2018_2024

print("="*60)
print("1. INFORMACIÓN GENERAL (INFO)")
print("="*60)
print(df.info())

print("\n" + "="*60)
print("2. RESUMEN ESTADÍSTICO (DESCRIBE)")
print("="*60)
display(df.describe(include='all'))

print("\n" + "="*60)
print("3. VALORES ÚNICOS POR COLUMNA")
print("="*60)
for col in df.columns:
    print(f"Columna '{col}': {df[col].nunique()} valores únicos")

print("\n" + "="*60)
print("4. CONTEO DE NULOS Y PORCENTAJE")
print("="*60)
nulos = df.isnull().sum()
porcentaje = (df.isnull().mean() * 100).round(2)
df_nulos_master = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': porcentaje})
print(df_nulos_master)

1. INFORMACIÓN GENERAL (INFO)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10136 entries, 0 to 10135
Data columns (total 17 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Fecha     10136 non-null  datetime64[ns]
 1   Estacion  10136 non-null  category      
 2   DVM10     7651 non-null   float64       
 3   DVVX10    5470 non-null   float64       
 4   HRM       10135 non-null  float64       
 5   HRN       7214 non-null   float64       
 6   HRX       7214 non-null   float64       
 7   PM        5481 non-null   float64       
 8   PN        5481 non-null   float64       
 9   PPT       7671 non-null   float64       
 10  PX        5481 non-null   float64       
 11  RS24h     5474 non-null   float64       
 12  TM        10135 non-null  float64       
 13  TN        10136 non-null  float64       
 14  TX        10136 non-null  float64       
 15  VVM10     7663 non-null   float64       
 16  VVX10     7664 non-null   fl

ACRÒNIM,Fecha,Estacion,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
count,10136,10136,7651.000000,5470.000000,10135.000000,7214.000000,7214.000000,5481.000000,5481.000000,7671.000000,5481.000000,5474.000000,10135.000000,10136.000000,10136.000000,7663.000000,7664.000000
unique,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,D5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,2557,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2021-06-20 19:30:55.406471936,NaN,203.182983,205.963254,68.130538,47.586498,87.379817,996.746816,994.569513,1.548638,999.074621,15.836609,17.494682,14.123056,21.785093,2.721937,9.321112
min,2018-01-01 00:00:00,NaN,0.000000,0.000000,22.000000,7.000000,36.000000,942.500000,936.600000,0.000000,946.800000,0.200000,-0.700000,-2.400000,1.200000,0.500000,3.000000
25%,2019-09-26 00:00:00,NaN,122.000000,125.000000,60.000000,38.000000,80.000000,972.900000,971.000000,0.000000,975.000000,9.000000,12.600000,9.300000,16.600000,1.700000,7.000000
50%,2021-06-20 12:00:00,NaN,230.000000,213.000000,69.000000,47.000000,90.000000,1006.100000,1003.900000,0.000000,1008.300000,14.900000,16.900000,13.600000,21.200000,2.300000,8.800000
75%,2023-03-16 00:00:00,NaN,274.000000,298.000000,77.000000,57.000000,97.000000,1012.500000,1010.600000,0.000000,1014.700000,22.900000,22.700000,19.300000,27.100000,3.400000,11.000000
max,2024-12-31 00:00:00,NaN,359.000000,359.000000,100.000000,100.000000,100.000000,1033.700000,1032.800000,139.900000,1035.700000,32.000000,33.400000,29.500000,39.800000,15.300000,30.300000



3. VALORES ÚNICOS POR COLUMNA
Columna 'Fecha': 2557 valores únicos
Columna 'Estacion': 4 valores únicos
Columna 'DVM10': 359 valores únicos
Columna 'DVVX10': 359 valores únicos
Columna 'HRM': 78 valores únicos
Columna 'HRN': 92 valores únicos
Columna 'HRX': 62 valores únicos
Columna 'PM': 707 valores únicos
Columna 'PN': 750 valores únicos
Columna 'PPT': 311 valores únicos
Columna 'PX': 683 valores únicos
Columna 'RS24h': 310 valores únicos
Columna 'TM': 298 valores únicos
Columna 'TN': 295 valores únicos
Columna 'TX': 319 valores únicos
Columna 'VVM10': 106 valores únicos
Columna 'VVX10': 198 valores únicos

4. CONTEO DE NULOS Y PORCENTAJE
          Nulos  Porcentaje (%)
ACRÒNIM                        
Fecha         0            0.00
Estacion      0            0.00
DVM10      2485           24.52
DVVX10     4666           46.03
HRM           1            0.01
HRN        2922           28.83
HRX        2922           28.83
PM         4655           45.93
PN         4655           45.9

### Resultado de la auditoría global

El dataset meteorológico unificado contiene **10.136 registros**, correspondientes a **2.557 fechas únicas** y cuatro estaciones meteorológicas (`D5`, `X2`, `X4` y `X8`) durante el periodo 2018–2024.

Las variables de **temperatura** presentan una cobertura prácticamente completa: `TN` y `TX` no contienen valores ausentes y `TM` presenta únicamente una observación sin dato. La **humedad relativa media (`HRM`)** muestra igualmente una cobertura prácticamente total, con un único valor ausente.

La disponibilidad del resto de variables es heterogénea como consecuencia de las diferencias existentes entre estaciones y periodos históricos. Las variables `DVM10`, `PPT`, `VVM10` y `VVX10` presentan aproximadamente un **24 % de valores ausentes**, mientras que `HRN` y `HRX` alcanzan aproximadamente el **29 %**.

Las mayores discontinuidades corresponden a `DVVX10`, las variables de presión atmosférica (`PM`, `PN`, `PX`) y la radiación solar (`RS24h`), con porcentajes de ausencia próximos al **46 %**. La auditoría realizada previamente muestra que esta falta de información responde principalmente a la diferente disponibilidad de variables entre estaciones y a cambios en la cobertura histórica de la fuente, especialmente durante 2018–2019.

Los rangos observados en la inspección estadística son compatibles, a priori, con las magnitudes meteorológicas consideradas. No obstante, antes de cerrar el bloque se realizarán controles adicionales de duplicidad, continuidad temporal y coherencia interna entre variables relacionadas.

La ausencia de determinadas variables se mantiene explícitamente en esta fase, evitando realizar imputaciones antes de definir la estrategia de integración y modelado.

## 2.8 Imputación condicionada de valores ausentes y preservación de nulos estructurales

La imputación meteorológica se realiza mediante KNN utilizando información temporal y de estación como contexto para identificar observaciones comparables.

Se incorporan como variables auxiliares el **año, mes, día, día del año** y la identificación de la **estación meteorológica** mediante codificación *One-Hot*. De este modo, la imputación tiene en cuenta tanto la estacionalidad como la pertenencia a una determinada estación.

No obstante, se distingue entre valores ausentes puntuales y ausencias estructurales. Estas últimas, asociadas a variables no registradas por determinadas estaciones o no disponibles en determinados periodos históricos, se preservan expresamente para evitar generar observaciones artificiales.

In [ ]:
# ==============================================================================
# 2.8 IMPUTACIÓN CONDICIONADA DE VALORES AUSENTES
#     Y PRESERVACIÓN DE NULOS ESTRUCTURALES
# ==============================================================================

import pandas as pd
import numpy as np

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------------------
# 1. PARTIR DEL DATASET ANTERIOR A CUALQUIER IMPUTACIÓN
# ------------------------------------------------------------------------------

df = df_meteo_2018_2024.copy()

df["Fecha"] = pd.to_datetime(df["Fecha"])

# ------------------------------------------------------------------------------
# 2. VARIABLES TEMPORALES AUXILIARES
# ------------------------------------------------------------------------------

df["Anio"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.month
df["Dia"] = df["Fecha"].dt.day
df["DiaDelAno"] = df["Fecha"].dt.dayofyear

# ------------------------------------------------------------------------------
# 3. DEFINICIÓN DE AUSENCIAS ESTRUCTURALES
# ------------------------------------------------------------------------------

# Variables que la estación X2 no registra
variables_no_medidas_x2 = [
    "PM",
    "PN",
    "PX",
    "PPT",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10",
    "RS24h"
]

# Variables no disponibles históricamente en 2018-2019
variables_no_disponibles_1819 = [
    "DVVX10",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "RS24h"
]

# Matriz booleana que marcará las posiciones que NO deben imputarse
mascara_estructural = pd.DataFrame(
    False,
    index=df.index,
    columns=df.columns
)

# ------------------------------------------------------------------------------
# 3.1 Ausencias estructurales de X2
# ------------------------------------------------------------------------------

mask_x2 = df["Estacion"].astype(str).eq("X2")

for variable in variables_no_medidas_x2:

    if variable in df.columns:

        mascara_estructural.loc[
            mask_x2,
            variable
        ] = True

# ------------------------------------------------------------------------------
# 3.2 Ausencias estructurales históricas 2018-2019
# ------------------------------------------------------------------------------

mask_1819 = df["Anio"].isin([2018, 2019])

for variable in variables_no_disponibles_1819:

    if variable in df.columns:

        mascara_estructural.loc[
            mask_1819,
            variable
        ] = True

# IMPORTANTE:
#
# DVM10, VVM10, VVX10 y PPT sí existen en 2018-2019
# para D5, X4 y X8.
#
# Por tanto, NO se consideran ausencias estructurales históricas
# generales para ese periodo.

# ------------------------------------------------------------------------------
# 4. CODIFICACIÓN ONE-HOT DE LA ESTACIÓN
# ------------------------------------------------------------------------------

df_encoded = pd.get_dummies(
    df,
    columns=["Estacion"],
    drop_first=False,
    dtype=float
)

cols_estacion = [
    col
    for col in df_encoded.columns
    if col.startswith("Estacion_")
]

# ------------------------------------------------------------------------------
# 5. VARIABLES METEOROLÓGICAS
# ------------------------------------------------------------------------------

variables_meteo = [
    "DVM10",
    "DVVX10",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PPT",
    "PX",
    "RS24h",
    "TM",
    "TN",
    "TX",
    "VVM10",
    "VVX10"
]

variables_meteo = [
    variable
    for variable in variables_meteo
    if variable in df_encoded.columns
]

# Contexto utilizado por el KNN:
# proximidad temporal + identificación de estación
variables_contexto = [
    "Anio",
    "Mes",
    "Dia",
    "DiaDelAno"
] + cols_estacion

# ------------------------------------------------------------------------------
# 6. IMPUTACIÓN VARIABLE A VARIABLE
# ------------------------------------------------------------------------------

df_final_imputed = df_encoded.copy()

resumen_imputacion = []

for variable in variables_meteo:

    # Filas donde esa variable es estructuralmente inexistente
    mask_struct = mascara_estructural[variable]

    # Filas donde sí está permitido imputar
    mask_permitida = ~mask_struct

    # Variable a imputar + contexto temporal y de estación
    features = [
        variable
    ] + variables_contexto

    subset = df_encoded.loc[
        mask_permitida,
        features
    ].copy()

    # Número de huecos reales susceptibles de imputación
    n_nulos_imputables = subset[variable].isna().sum()

    if n_nulos_imputables > 0:

        # ----------------------------------------------------------
        # Escalado
        # ----------------------------------------------------------

        scaler = StandardScaler()

        datos_escalados = scaler.fit_transform(
            subset
        )

        # ----------------------------------------------------------
        # KNN
        # ----------------------------------------------------------

        imputer = KNNImputer(
            n_neighbors=5,
            weights="distance"
        )

        datos_imputados_escalados = imputer.fit_transform(
            datos_escalados
        )

        # ----------------------------------------------------------
        # Volver a escala original
        # ----------------------------------------------------------

        datos_imputados = pd.DataFrame(
            scaler.inverse_transform(
                datos_imputados_escalados
            ),
            columns=features,
            index=subset.index
        )

        # Solo sustituimos la variable meteorológica
        df_final_imputed.loc[
            mask_permitida,
            variable
        ] = datos_imputados[variable]

    # --------------------------------------------------------------
    # Guardar resumen de la imputación
    # --------------------------------------------------------------

    resumen_imputacion.append({
        "Variable": variable,
        "Nulos_imputables": int(n_nulos_imputables),
        "Nulos_estructurales": int(mask_struct.sum())
    })

# ------------------------------------------------------------------------------
# 7. RESTAURAR EXPLÍCITAMENTE LOS NULOS ESTRUCTURALES
# ------------------------------------------------------------------------------

for variable in variables_meteo:

    mask_struct = mascara_estructural[variable]

    if mask_struct.any():

        df_final_imputed.loc[
            mask_struct,
            variable
        ] = np.nan

# ------------------------------------------------------------------------------
# 8. RECUPERAR FECHA Y ESTACIÓN ORIGINALES
# ------------------------------------------------------------------------------

df_final_imputed["Fecha"] = df["Fecha"]
df_final_imputed["Estacion"] = df["Estacion"]

# ------------------------------------------------------------------------------
# 9. RESUMEN FINAL DE IMPUTACIÓN
# ------------------------------------------------------------------------------

df_resumen_imputacion = pd.DataFrame(
    resumen_imputacion
)

print("=" * 90)
print("IMPUTACIÓN METEOROLÓGICA CONDICIONADA COMPLETADA")
print("=" * 90)

print(
    f"\nDimensiones: "
    f"{df_final_imputed.shape}"
)

print(
    f"Periodo: "
    f"{df_final_imputed['Fecha'].min().date()} "
    f"- "
    f"{df_final_imputed['Fecha'].max().date()}"
)

print("\nResumen de la imputación:")

display(
    df_resumen_imputacion
)

print(
    "\nSe han preservado los nulos estructurales "
    "de X2 y del periodo 2018-2019."
)

IMPUTACIÓN METEOROLÓGICA CONDICIONADA COMPLETADA

Dimensiones: (10136, 25)
Periodo: 2018-01-01 - 2024-12-31

Resumen de la imputación:


,Variable,Nulos_imputables,Nulos_estructurales
0,DVM10,20,2465
1,DVVX10,11,4655
2,HRM,1,0
3,HRN,2,2920
4,HRX,2,2920
5,PM,0,4655
6,PN,0,4655
7,PPT,0,2465
8,PX,0,4655
9,RS24h,7,4655



Se han preservado los nulos estructurales de X2 y del periodo 2018-2019.


### Resultado de la imputación condicionada

La imputación meteorológica se realizó distinguiendo entre **ausencias puntuales potencialmente recuperables** y **ausencias estructurales**, asociadas a variables no registradas por determinadas estaciones o no disponibles durante ciertos periodos históricos.

La estrategia permitió imputar únicamente un número reducido de observaciones puntuales en variables con cobertura efectiva, por ejemplo:

- `DVM10`: 20 valores imputados.
- `DVVX10`: 11 valores imputados.
- `HRM`: 1 valor imputado.
- `HRN`: 2 valores imputados.
- `HRX`: 2 valores imputados.
- `RS24h`: 7 valores imputados.
- `TM`: 1 valor imputado.
- `VVM10`: 8 valores imputados.
- `VVX10`: 7 valores imputados.

Por el contrario, se preservaron de forma explícita las ausencias estructurales. Entre ellas destacan las correspondientes a la estación `X2` y las variables no disponibles históricamente durante 2018–2019.

En particular, las variables de presión (`PM`, `PN`, `PX`) mantienen **4.655 ausencias estructurales**, mientras que `PPT`, `DVM10`, `VVM10` y `VVX10` conservan **2.465 ausencias estructurales** asociadas principalmente a la estación `X2`.

De esta forma se evita reconstruir artificialmente variables que no fueron realmente observadas, manteniendo la trazabilidad de la disponibilidad original de la fuente meteorológica.

## 2.9 Auditoría y validación de la imputación

Tras aplicar la imputación condicionada se verifica que los valores ausentes puntuales hayan sido completados correctamente y que las ausencias estructurales permanezcan preservadas.

La auditoría analiza la cobertura final por variable y estación, comprueba la ausencia de duplicados y verifica específicamente los patrones estructurales identificados para la estación `X2` y para el periodo histórico 2018–2019.

In [ ]:
# ==============================================================================
# 2.9 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN
# ==============================================================================

import pandas as pd
import numpy as np

df = df_final_imputed.copy()

print("=" * 90)
print("2.9 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN METEOROLÓGICA")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. DIMENSIONES Y COBERTURA TEMPORAL
# ------------------------------------------------------------------------------

print("\n1. DIMENSIONES Y COBERTURA TEMPORAL")
print("-" * 90)

print(f"Dimensiones        : {df.shape}")
print(f"Fecha inicial      : {df['Fecha'].min().date()}")
print(f"Fecha final        : {df['Fecha'].max().date()}")
print(f"Fechas únicas      : {df['Fecha'].nunique()}")
print(f"Estaciones         : {df['Estacion'].nunique()}")
print(
    f"Listado estaciones : "
    f"{df['Estacion'].astype(str).unique().tolist()}"
)

# ------------------------------------------------------------------------------
# 2. DUPLICADOS FECHA + ESTACIÓN
# ------------------------------------------------------------------------------

print("\n2. DUPLICADOS FECHA + ESTACIÓN")
print("-" * 90)

duplicados = df.duplicated(
    subset=["Fecha", "Estacion"]
).sum()

print(f"Duplicados encontrados: {duplicados}")

# ------------------------------------------------------------------------------
# 3. NULOS FINALES POR VARIABLE
# ------------------------------------------------------------------------------

print("\n3. NULOS FINALES POR VARIABLE")
print("-" * 90)

variables_meteo = [
    "DVM10",
    "DVVX10",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PPT",
    "PX",
    "RS24h",
    "TM",
    "TN",
    "TX",
    "VVM10",
    "VVX10"
]

tabla_nulos = pd.DataFrame({
    "Nulos": df[variables_meteo].isna().sum(),
    "Porcentaje (%)": (
        df[variables_meteo]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(tabla_nulos)

# ------------------------------------------------------------------------------
# 4. NULOS POR ESTACIÓN
# ------------------------------------------------------------------------------

print("\n4. NULOS POR ESTACIÓN")
print("-" * 90)

tabla_nulos_estacion = (
    df.groupby(
        "Estacion",
        observed=True
    )[variables_meteo]
    .apply(
        lambda x: x.isna().sum()
    )
)

display(tabla_nulos_estacion)

# ------------------------------------------------------------------------------
# 5. VERIFICACIÓN ESPECÍFICA DE X2
# ------------------------------------------------------------------------------

print("\n5. VERIFICACIÓN DE NULOS ESTRUCTURALES DE X2")
print("-" * 90)

variables_x2 = [
    "PM",
    "PN",
    "PX",
    "PPT",
    "DVM10",
    "DVVX10",
    "VVM10",
    "VVX10",
    "RS24h"
]

df_x2 = df[
    df["Estacion"].astype(str).eq("X2")
].copy()

tabla_x2 = pd.DataFrame({
    "Nulos_X2": df_x2[variables_x2].isna().sum(),
    "Registros_X2": len(df_x2)
})

tabla_x2["Porcentaje_nulo_X2"] = (
    tabla_x2["Nulos_X2"]
    / tabla_x2["Registros_X2"]
    * 100
).round(2)

display(tabla_x2)

# ------------------------------------------------------------------------------
# 6. VERIFICACIÓN DE AUSENCIAS ESTRUCTURALES 2018-2019
# ------------------------------------------------------------------------------

print("\n6. VERIFICACIÓN DE AUSENCIAS ESTRUCTURALES 2018-2019")
print("-" * 90)

variables_1819 = [
    "DVVX10",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PX",
    "RS24h"
]

df_1819 = df[
    df["Fecha"].dt.year.isin(
        [2018, 2019]
    )
].copy()

tabla_1819 = pd.DataFrame({
    "Nulos_2018_2019": (
        df_1819[variables_1819]
        .isna()
        .sum()
    ),
    "Registros_2018_2019": len(df_1819)
})

tabla_1819["Porcentaje_nulo_2018_2019"] = (
    tabla_1819["Nulos_2018_2019"]
    / tabla_1819["Registros_2018_2019"]
    * 100
).round(2)

display(tabla_1819)

# ------------------------------------------------------------------------------
# 7. COBERTURA POR AÑO Y ESTACIÓN
# ------------------------------------------------------------------------------

print("\n7. COBERTURA POR AÑO Y ESTACIÓN")
print("-" * 90)

cobertura_anual = (
    df.groupby(
        ["Anio", "Estacion"],
        observed=True
    )
    .agg(
        registros=("Fecha", "count"),
        fecha_inicio=("Fecha", "min"),
        fecha_fin=("Fecha", "max")
    )
    .reset_index()
)

display(cobertura_anual)

# ------------------------------------------------------------------------------
# 8. RESUMEN FINAL
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN DE LA AUDITORÍA")
print("=" * 90)

print(f"Duplicados Fecha-Estación : {duplicados}")
print(
    f"Nulos totales meteorología: "
    f"{int(df[variables_meteo].isna().sum().sum()):,}"
)

print(
    "\nLos valores ausentes restantes deben corresponder "
    "exclusivamente a ausencias estructurales previamente identificadas."
)

print(
    "\nAuditoría de la imputación finalizada correctamente."
)

2.9 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN METEOROLÓGICA

1. DIMENSIONES Y COBERTURA TEMPORAL
------------------------------------------------------------------------------------------
Dimensiones        : (10136, 25)
Fecha inicial      : 2018-01-01
Fecha final        : 2024-12-31
Fechas únicas      : 2557
Estaciones         : 4
Listado estaciones : ['D5', 'X2', 'X4', 'X8']

2. DUPLICADOS FECHA + ESTACIÓN
------------------------------------------------------------------------------------------
Duplicados encontrados: 0

3. NULOS FINALES POR VARIABLE
------------------------------------------------------------------------------------------


,Nulos,Porcentaje (%)
DVM10,2465,24.32
DVVX10,4655,45.93
HRM,0,0.00
HRN,2920,28.81
HRX,2920,28.81
PM,4655,45.93
PN,4655,45.93
PPT,2465,24.32
PX,4655,45.93
RS24h,4655,45.93



4. NULOS POR ESTACIÓN
------------------------------------------------------------------------------------------


,DVM10,DVVX10,HRM,HRN,HRX,PM,PN,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10
Estacion,,,,,,,,,,,,,,,
D5,0,730,0,730,730,730,730,0,730,730,0,0,0,0,0
X2,2465,2465,0,730,730,2465,2465,2465,2465,2465,0,0,0,2465,2465
X4,0,730,0,730,730,730,730,0,730,730,0,0,0,0,0
X8,0,730,0,730,730,730,730,0,730,730,0,0,0,0,0



5. VERIFICACIÓN DE NULOS ESTRUCTURALES DE X2
------------------------------------------------------------------------------------------


...


6. VERIFICACIÓN DE AUSENCIAS ESTRUCTURALES 2018-2019
------------------------------------------------------------------------------------------


,Nulos_2018_2019,Registros_2018_2019,Porcentaje_nulo_2018_2019
DVVX10,2920,2920,100.0
HRN,2920,2920,100.0
HRX,2920,2920,100.0
PM,2920,2920,100.0
PN,2920,2920,100.0
PX,2920,2920,100.0
RS24h,2920,2920,100.0



7. COBERTURA POR AÑO Y ESTACIÓN
------------------------------------------------------------------------------------------


,Anio,Estacion,registros,fecha_inicio,fecha_fin
0,2018,D5,365,2018-01-01,2018-12-31
1,2018,X2,365,2018-01-01,2018-12-31
2,2018,X4,365,2018-01-01,2018-12-31
3,2018,X8,365,2018-01-01,2018-12-31
4,2019,D5,365,2019-01-01,2019-12-31
5,2019,X2,365,2019-01-01,2019-12-31
6,2019,X4,365,2019-01-01,2019-12-31
7,2019,X8,365,2019-01-01,2019-12-31
8,2020,D5,366,2020-01-01,2020-12-31
9,2020,X2,366,2020-01-01,2020-12-31



RESUMEN DE LA AUDITORÍA
Duplicados Fecha-Estación : 0
Nulos totales meteorología: 38,975

Los valores ausentes restantes deben corresponder exclusivamente a ausencias estructurales previamente identificadas.

Auditoría de la imputación finalizada correctamente.


### Resultado de la auditoría

La auditoría posterior a la imputación confirma la consistencia del dataset meteorológico procesado. La tabla resultante contiene 10.136 observaciones correspondientes a cuatro estaciones meteorológicas y cubre el periodo comprendido entre 2018 y 2024, sin duplicados para la combinación fecha–estación.

Las variables fundamentales de temperatura (`TM`, `TN`, `TX`) y humedad media (`HRM`) presentan cobertura completa tras la imputación de los escasos valores ausentes puntuales.

Los valores nulos restantes corresponden a patrones de ausencia estructural previamente identificados y se han conservado deliberadamente para evitar la generación artificial de observaciones en variables no registradas por determinadas estaciones o periodos.

En particular, se preservan las ausencias asociadas a la estación `X2` en determinadas variables meteorológicas, así como las correspondientes a variables no disponibles en los datos históricos de 2018–2019.

La cobertura temporal es prácticamente completa para las cuatro estaciones durante el periodo de estudio. Como particularidad, la estación `X2` dispone de registros hasta el 30 de septiembre de 2024.

Por tanto, el dataset se considera consistente y adecuado para continuar con la selección y preparación de las variables meteorológicas que se incorporarán posteriormente al dataset maestro temporal.

## 2.10 Control de coherencia física y detección de valores extremos

Una vez completada la imputación y verificada la conservación de las ausencias estructurales, se realiza un control de coherencia física de las variables meteorológicas.

El objetivo es detectar posibles valores incompatibles con la naturaleza de cada variable y analizar la presencia de observaciones extremas. Los valores extremos no se eliminan automáticamente, ya que pueden corresponder a episodios meteorológicos reales relevantes para explicar la variabilidad de la contaminación atmosférica.

In [ ]:
# ==============================================================================
# 2.10 CONTROL DE COHERENCIA FÍSICA Y DETECCIÓN DE VALORES EXTREMOS
# ==============================================================================

import pandas as pd
import numpy as np

df = df_final_imputed.copy()

print("=" * 90)
print("2.10 CONTROL DE COHERENCIA FÍSICA Y VALORES EXTREMOS")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. VARIABLES METEOROLÓGICAS
# ------------------------------------------------------------------------------

variables_meteo = [
    "DVM10",
    "DVVX10",
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PPT",
    "PX",
    "RS24h",
    "TM",
    "TN",
    "TX",
    "VVM10",
    "VVX10"
]

variables_meteo = [
    col for col in variables_meteo
    if col in df.columns
]

# ------------------------------------------------------------------------------
# 2. ESTADÍSTICAS DESCRIPTIVAS DESPUÉS DE LA IMPUTACIÓN
# ------------------------------------------------------------------------------

print("\n1. ESTADÍSTICAS DESCRIPTIVAS")
print("-" * 90)

display(
    df[variables_meteo]
    .describe()
    .T
    .round(3)
)

# ------------------------------------------------------------------------------
# 3. CONTROL DE RANGOS FÍSICOS BÁSICOS
# ------------------------------------------------------------------------------

print("\n2. CONTROL DE RANGOS FÍSICOS")
print("-" * 90)

# Estos límites se utilizan como controles de coherencia,
# no como criterio automático de eliminación.

rangos_control = {
    "DVM10":  (0, 359),       # dirección media del viento (grados)
    "DVVX10": (0, 359),       # dirección asociada al viento máximo (grados)

    "HRM":    (0, 100),       # humedad relativa (%)
    "HRN":    (0, 100),
    "HRX":    (0, 100),

    "PPT":    (0, None),      # precipitación: no puede ser negativa
    "RS24h":  (0, None),      # radiación: no puede ser negativa

    "VVM10":  (0, None),      # velocidad del viento
    "VVX10":  (0, None),

    # Rangos amplios de control para detectar errores evidentes.
    # NO se utilizan para eliminar extremos meteorológicos reales.
    "TM":     (-30, 50),
    "TN":     (-30, 50),
    "TX":     (-30, 50),

    "PM":     (850, 1100),
    "PN":     (850, 1100),
    "PX":     (850, 1100)
}

resultados_rangos = []

for variable, (lim_inf, lim_sup) in rangos_control.items():

    if variable not in df.columns:
        continue

    serie = df[variable].dropna()

    fuera_inf = (
        (serie < lim_inf).sum()
        if lim_inf is not None
        else 0
    )

    fuera_sup = (
        (serie > lim_sup).sum()
        if lim_sup is not None
        else 0
    )

    resultados_rangos.append({
        "Variable": variable,
        "Min_observado": serie.min(),
        "Max_observado": serie.max(),
        "Limite_inferior": lim_inf,
        "Limite_superior": lim_sup,
        "Fuera_por_abajo": int(fuera_inf),
        "Fuera_por_arriba": int(fuera_sup),
        "Total_fuera_rango": int(fuera_inf + fuera_sup)
    })

df_control_rangos = pd.DataFrame(resultados_rangos)

display(df_control_rangos)

# ------------------------------------------------------------------------------
# 4. COMPROBACIONES DE COHERENCIA ENTRE VARIABLES
# ------------------------------------------------------------------------------

print("\n3. COHERENCIA ENTRE VARIABLES METEOROLÓGICAS")
print("-" * 90)

# Temperaturas:
# TN <= TM <= TX

mask_temp = (
    df["TN"].notna() &
    df["TM"].notna() &
    df["TX"].notna()
)

incoherencia_temp = (
    mask_temp &
    (
        (df["TN"] > df["TM"]) |
        (df["TM"] > df["TX"])
    )
)

print(
    f"Incoherencias TN <= TM <= TX: "
    f"{incoherencia_temp.sum():,}"
)

# Humedad:
# HRN <= HRM <= HRX
# Solo donde las tres variables existen.

mask_hr = (
    df["HRN"].notna() &
    df["HRM"].notna() &
    df["HRX"].notna()
)

incoherencia_hr = (
    mask_hr &
    (
        (df["HRN"] > df["HRM"]) |
        (df["HRM"] > df["HRX"])
    )
)

print(
    f"Incoherencias HRN <= HRM <= HRX: "
    f"{incoherencia_hr.sum():,}"
)

# Presión:
# PN <= PM <= PX
# Solo donde existen las tres variables.

mask_presion = (
    df["PN"].notna() &
    df["PM"].notna() &
    df["PX"].notna()
)

incoherencia_presion = (
    mask_presion &
    (
        (df["PN"] > df["PM"]) |
        (df["PM"] > df["PX"])
    )
)

print(
    f"Incoherencias PN <= PM <= PX: "
    f"{incoherencia_presion.sum():,}"
)

# ------------------------------------------------------------------------------
# 5. DETECCIÓN EXPLORATORIA DE OUTLIERS MEDIANTE IQR
# ------------------------------------------------------------------------------

print("\n4. DETECCIÓN EXPLORATORIA DE OUTLIERS (IQR)")
print("-" * 90)

# Excluimos las direcciones del viento del IQR:
# son variables circulares (0° y 359° son prácticamente la misma dirección).

variables_iqr = [
    "HRM",
    "HRN",
    "HRX",
    "PM",
    "PN",
    "PPT",
    "PX",
    "RS24h",
    "TM",
    "TN",
    "TX",
    "VVM10",
    "VVX10"
]

resultados_iqr = []

for variable in variables_iqr:

    if variable not in df.columns:
        continue

    serie = df[variable].dropna()

    if len(serie) == 0:
        continue

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = (
        (serie < limite_inferior) |
        (serie > limite_superior)
    )

    n_outliers = outliers.sum()

    resultados_iqr.append({
        "Variable": variable,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Limite_IQR_inferior": limite_inferior,
        "Limite_IQR_superior": limite_superior,
        "Outliers": int(n_outliers),
        "Porcentaje_outliers": round(
            100 * n_outliers / len(serie), 2
        )
    })

df_outliers_iqr = pd.DataFrame(resultados_iqr)

display(df_outliers_iqr)

# ------------------------------------------------------------------------------
# 6. RESUMEN
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN DEL CONTROL DE COHERENCIA")
print("=" * 90)

print(
    f"Valores fuera de los rangos físicos de control: "
    f"{df_control_rangos['Total_fuera_rango'].sum():,}"
)

print(
    f"Incoherencias de temperatura: "
    f"{incoherencia_temp.sum():,}"
)

print(
    f"Incoherencias de humedad: "
    f"{incoherencia_hr.sum():,}"
)

print(
    f"Incoherencias de presión: "
    f"{incoherencia_presion.sum():,}"
)

print(
    "\nLos outliers detectados mediante IQR se consideran inicialmente "
    "observaciones potencialmente reales y NO se eliminan automáticamente."
)

2.10 CONTROL DE COHERENCIA FÍSICA Y VALORES EXTREMOS

1. ESTADÍSTICAS DESCRIPTIVAS
------------------------------------------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
DVM10,7671.0,203.062,89.939,-0.0,122.0,229.0,273.5,359.0
DVVX10,5481.0,205.955,94.318,-0.0,125.0,213.0,298.0,359.0
HRM,10136.0,68.131,12.213,22.0,60.0,69.0,77.0,100.0
HRN,7216.0,47.587,13.274,7.0,38.0,47.0,57.0,100.0
HRX,7216.0,87.380,11.507,36.0,80.0,90.0,97.0,100.0
PM,5481.0,996.747,20.588,942.5,972.9,1006.1,1012.5,1033.7
PN,5481.0,994.570,20.712,936.6,971.0,1003.9,1010.6,1032.8
PPT,7671.0,1.549,6.665,0.0,0.0,0.0,0.0,139.9
PX,5481.0,999.075,20.537,946.8,975.0,1008.3,1014.7,1035.7
RS24h,5481.0,15.836,8.036,0.2,9.0,14.9,22.8,32.0



2. CONTROL DE RANGOS FÍSICOS
------------------------------------------------------------------------------------------


,Variable,Min_observado,Max_observado,Limite_inferior,Limite_superior,Fuera_por_abajo,Fuera_por_arriba,Total_fuera_rango
0,DVM10,-2.842171e-14,359.0,0,359.0,6,0,6
1,DVVX10,-2.842171e-14,359.0,0,359.0,5,0,5
2,HRM,2.200000e+01,100.0,0,100.0,0,0,0
3,HRN,7.000000e+00,100.0,0,100.0,0,0,0
4,HRX,3.600000e+01,100.0,0,100.0,0,0,0
5,PPT,0.000000e+00,139.9,0,NaN,0,0,0
6,RS24h,2.000000e-01,32.0,0,NaN,0,0,0
7,VVM10,5.000000e-01,15.3,0,NaN,0,0,0
8,VVX10,3.000000e+00,30.3,0,NaN,0,0,0
9,TM,-7.000000e-01,33.4,-30,50.0,0,0,0



3. COHERENCIA ENTRE VARIABLES METEOROLÓGICAS
------------------------------------------------------------------------------------------
Incoherencias TN <= TM <= TX: 0
Incoherencias HRN <= HRM <= HRX: 0
Incoherencias PN <= PM <= PX: 0

4. DETECCIÓN EXPLORATORIA DE OUTLIERS (IQR)
------------------------------------------------------------------------------------------


,Variable,Q1,Q3,IQR,Limite_IQR_inferior,Limite_IQR_superior,Outliers,Porcentaje_outliers
0,HRM,60.0,77.0,17.0,34.50,102.50,65,0.64
1,HRN,38.0,57.0,19.0,9.50,85.50,48,0.67
2,HRX,80.0,97.0,17.0,54.50,122.50,87,1.21
3,PM,972.9,1012.5,39.6,913.50,1071.90,0,0.00
4,PN,971.0,1010.6,39.6,911.60,1070.00,0,0.00
5,PPT,0.0,0.0,0.0,0.00,0.00,1783,23.24
6,PX,975.0,1014.7,39.7,915.45,1074.25,0,0.00
7,RS24h,9.0,22.8,13.8,-11.70,43.50,0,0.00
8,TM,12.6,22.7,10.1,-2.55,37.85,0,0.00
9,TN,9.3,19.3,10.0,-5.70,34.30,0,0.00



RESUMEN DEL CONTROL DE COHERENCIA
Valores fuera de los rangos físicos de control: 11
Incoherencias de temperatura: 0
Incoherencias de humedad: 0
Incoherencias de presión: 0

Los outliers detectados mediante IQR se consideran inicialmente observaciones potencialmente reales y NO se eliminan automáticamente.


### Resultado del Control de coherencia física y valores extremos

Una vez realizada la imputación de los valores ausentes, se lleva a cabo una auditoría de coherencia física del dataset meteorológico con el objetivo de comprobar que el proceso de imputación no haya generado valores incompatibles con el comportamiento esperado de las variables.

El control incluye:

- análisis de las estadísticas descriptivas de las variables meteorológicas;
- comprobación de rangos físicos plausibles;
- verificación de las relaciones internas entre variables:
  - temperatura mínima ≤ temperatura media ≤ temperatura máxima (`TN ≤ TM ≤ TX`);
  - humedad mínima ≤ humedad media ≤ humedad máxima (`HRN ≤ HRM ≤ HRX`);
  - presión mínima ≤ presión media ≤ presión máxima (`PN ≤ PM ≤ PX`);
- detección exploratoria de valores extremos mediante el rango intercuartílico (IQR).

No se detectaron incoherencias en las relaciones entre temperatura, humedad o presión, lo que indica que la imputación mantiene la consistencia interna de las variables meteorológicas.

Los únicos valores situados ligeramente fuera de los rangos físicos definidos corresponden a las variables de dirección del viento (`DVM10` y `DVVX10`), con valores negativos del orden de \(10^{-14}\). Estas desviaciones son consecuencia de la precisión numérica asociada a las operaciones de escalado e inversión realizadas durante la imputación y equivalen, en la práctica, a 0°.

La detección mediante IQR identifica algunos valores extremos, principalmente en precipitación y velocidad del viento. Estos registros no se eliminan automáticamente, ya que pueden corresponder a episodios meteorológicos reales. En el caso de la precipitación, la elevada frecuencia de días sin lluvia provoca un rango intercuartílico igual a cero, por lo que el criterio IQR clasifica como extremos numerosos días con precipitación positiva.

Por tanto, los valores extremos meteorológicamente plausibles se conservan y únicamente se corregirán las pequeñas desviaciones derivadas de precisión numérica antes de generar el dataset meteorológico definitivo.

## 2.11 Corrección de precisión numérica y cierre del control de calidad

La auditoría de coherencia física permitió identificar pequeñas desviaciones numéricas en las variables de dirección del viento (`DVM10` y `DVVX10`), con valores negativos del orden de \(10^{-14}\). Estas desviaciones no representan observaciones meteorológicas anómalas, sino errores de precisión derivados de las operaciones de escalado e inversión realizadas durante el proceso de imputación.

Dado que las direcciones del viento deben encontrarse en el intervalo comprendido entre 0° y 359°, se corrigen exclusivamente estos residuos numéricos mediante acotación al rango físico correspondiente.

No se modifican el resto de variables ni se eliminan los valores extremos detectados mediante IQR, ya que estos pueden representar episodios meteorológicos reales relevantes para el análisis posterior de la contaminación atmosférica.

Tras esta corrección se realiza una comprobación final de los valores mínimos y máximos de las variables de dirección del viento.

In [ ]:
# ==============================================================================
# 2.11 CORRECCIÓN DE PRECISIÓN NUMÉRICA Y CIERRE DEL CONTROL DE CALIDAD
# ==============================================================================

# Partimos del dataset imputado
df_meteo_final = df_final_imputed.copy()

# --------------------------------------------------------------------------
# 1. Corrección de pequeños residuos de precisión numérica
# --------------------------------------------------------------------------

# Las direcciones del viento deben encontrarse entre 0 y 359 grados.
# Se corrigen únicamente los pequeños residuos numéricos detectados
# durante la auditoría, sin modificar el resto de variables.

for col in ['DVM10', 'DVVX10']:
    if col in df_meteo_final.columns:
        df_meteo_final[col] = df_meteo_final[col].clip(
            lower=0,
            upper=359
        )

# --------------------------------------------------------------------------
# 2. Comprobación final
# --------------------------------------------------------------------------

print("=" * 80)
print("2.11 CONTROL FINAL DE LAS DIRECCIONES DEL VIENTO")
print("=" * 80)

for col in ['DVM10', 'DVVX10']:
    print(
        f"{col}: "
        f"mín = {df_meteo_final[col].min():.6f} | "
        f"máx = {df_meteo_final[col].max():.6f}"
    )

# Comprobar que ya no existen valores fuera del intervalo físico

fuera_rango = {}

for col in ['DVM10', 'DVVX10']:
    n = (
        (df_meteo_final[col] < 0) |
        (df_meteo_final[col] > 359)
    ).sum()

    fuera_rango[col] = n

print("\nValores fuera del intervalo 0–359°:")
for col, n in fuera_rango.items():
    print(f"{col}: {n}")

print("\nCorrección de precisión numérica completada.")
print("No se han eliminado valores extremos meteorológicos.")

2.11 CONTROL FINAL DE LAS DIRECCIONES DEL VIENTO
DVM10: mín = 0.000000 | máx = 359.000000
DVVX10: mín = 0.000000 | máx = 359.000000

Valores fuera del intervalo 0–359°:
DVM10: 0
DVVX10: 0

Corrección de precisión numérica completada.
No se han eliminado valores extremos meteorológicos.


### Resultado Corrección de precisión numérica y cierre del control de calidad

Tras la corrección de los pequeños residuos asociados a la precisión numérica, las variables de dirección del viento (`DVM10` y `DVVX10`) presentan valores comprendidos íntegramente entre **0° y 359°**, sin observaciones fuera de su rango físico.

No se han eliminado valores extremos meteorológicos, dado que pueden representar episodios reales de interés para el análisis de la contaminación atmosférica.

Con esta comprobación se da por finalizado el control de coherencia física del bloque meteorológico.

## 2.12 Construcción y auditoría final del dataset meteorológico

Una vez completados los procesos de homogeneización, imputación y control de coherencia física, se construye el dataset meteorológico definitivo que será utilizado posteriormente en la integración con el resto de bloques del estudio.

Las variables auxiliares generadas exclusivamente para facilitar la imputación mediante KNN —componentes temporales y codificaciones *one-hot* de las estaciones— no forman parte de las variables meteorológicas originales y, por tanto, se eliminan del dataset final.

Se conservan como identificadores la fecha y la estación meteorológica, junto con las variables meteorológicas disponibles. Antes del almacenamiento definitivo se realiza una última auditoría destinada a comprobar:

- las dimensiones y el periodo temporal del dataset;
- la ausencia de registros duplicados por fecha y estación;
- la estructura y los tipos de las variables;
- la distribución de registros por estación;
- la presencia de valores ausentes;
- y la conservación exclusiva de las ausencias estructurales previamente identificadas.

Los valores ausentes estructurales se mantienen deliberadamente y no se imputan, ya que corresponden a variables no medidas por determinadas estaciones o no disponibles durante determinados periodos históricos.

In [ ]:
# ==============================================================================
# 2.12 CONSTRUCCIÓN Y AUDITORÍA FINAL DEL DATASET METEOROLÓGICO
# ==============================================================================

import pandas as pd
from pathlib import Path

# Partimos del dataset ya imputado y corregido en 2.11
df_meteo_definitivo = df_meteo_final.copy()

# ------------------------------------------------------------------------------
# 1. ELIMINACIÓN DE VARIABLES AUXILIARES UTILIZADAS EN LA IMPUTACIÓN
# ------------------------------------------------------------------------------

columnas_auxiliares = [
    'Anio',
    'Mes',
    'Dia',
    'DiaDelAno',
    'Estacion_D5',
    'Estacion_X2',
    'Estacion_X4',
    'Estacion_X8'
]

columnas_eliminar = [
    col for col in columnas_auxiliares
    if col in df_meteo_definitivo.columns
]

df_meteo_definitivo = df_meteo_definitivo.drop(
    columns=columnas_eliminar
)

# ------------------------------------------------------------------------------
# 2. ORDENAR COLUMNAS
# ------------------------------------------------------------------------------

variables_meteo = [
    'TM', 'TN', 'TX',
    'HRM', 'HRN', 'HRX',
    'PM', 'PN', 'PX',
    'PPT', 'RS24h',
    'DVM10', 'DVVX10',
    'VVM10', 'VVX10'
]

columnas_finales = (
    ['Fecha', 'Estacion'] +
    [col for col in variables_meteo
     if col in df_meteo_definitivo.columns]
)

df_meteo_definitivo = df_meteo_definitivo[columnas_finales]

# ------------------------------------------------------------------------------
# 3. ORDEN CRONOLÓGICO
# ------------------------------------------------------------------------------

df_meteo_definitivo = (
    df_meteo_definitivo
    .sort_values(['Fecha', 'Estacion'])
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# 4. AUDITORÍA GENERAL
# ------------------------------------------------------------------------------

print("=" * 90)
print("2.12 AUDITORÍA FINAL DEL DATASET METEOROLÓGICO")
print("=" * 90)

print(f"\nDimensiones finales: {df_meteo_definitivo.shape}")

print(
    f"Periodo: "
    f"{df_meteo_definitivo['Fecha'].min().date()} - "
    f"{df_meteo_definitivo['Fecha'].max().date()}"
)

print(
    f"Estaciones: "
    f"{df_meteo_definitivo['Estacion'].nunique()}"
)

print(
    f"Códigos de estación: "
    f"{df_meteo_definitivo['Estacion'].unique().tolist()}"
)

# ------------------------------------------------------------------------------
# 5. DUPLICADOS FECHA-ESTACIÓN
# ------------------------------------------------------------------------------

duplicados = df_meteo_definitivo.duplicated(
    subset=['Fecha', 'Estacion']
).sum()

print("\n" + "-" * 90)
print("DUPLICADOS FECHA-ESTACIÓN")
print("-" * 90)

print(f"Duplicados encontrados: {duplicados}")

# ------------------------------------------------------------------------------
# 6. REGISTROS POR ESTACIÓN
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("REGISTROS POR ESTACIÓN")
print("-" * 90)

print(
    df_meteo_definitivo
    .groupby('Estacion', observed=True)
    .size()
)

# ------------------------------------------------------------------------------
# 7. AUDITORÍA FINAL DE NULOS
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("VALORES NULOS DEL DATASET DEFINITIVO")
print("-" * 90)

nulos = df_meteo_definitivo.isna().sum()

porcentaje_nulos = (
    df_meteo_definitivo
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

df_auditoria_nulos = pd.DataFrame({
    'Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos
})

display(df_auditoria_nulos)

# ------------------------------------------------------------------------------
# 8. NULOS POR ESTACIÓN
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("NULOS POR ESTACIÓN")
print("-" * 90)

variables_con_nulos = [
    col for col in variables_meteo
    if (
        col in df_meteo_definitivo.columns
        and df_meteo_definitivo[col].isna().any()
    )
]

nulos_por_estacion = (
    df_meteo_definitivo
    .groupby('Estacion', observed=True)[variables_con_nulos]
    .apply(lambda x: x.isna().sum())
)

display(nulos_por_estacion)

# ------------------------------------------------------------------------------
# 9. COMPROBACIÓN DE TIPOS
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("TIPOS DE DATOS")
print("-" * 90)

print(df_meteo_definitivo.dtypes)

# ------------------------------------------------------------------------------
# 10. ESTADÍSTICAS DESCRIPTIVAS FINALES
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("ESTADÍSTICAS DESCRIPTIVAS FINALES")
print("-" * 90)

display(
    df_meteo_definitivo[variables_meteo]
    .describe()
    .T
    .round(3)
)

# ------------------------------------------------------------------------------
# 11. GUARDADO DEL DATASET DEFINITIVO
# ------------------------------------------------------------------------------

from pathlib import Path

# Archivo limpio definitivo del bloque de Meteorología
ruta_salida = Path(
    "/content/drive/MyDrive/TFM/08_Meteorologia/DATOS LIMPIOS/"
    "df_meteorologia_2018_2024_Limpio.csv"
)

# Guardado.
# Si el archivo ya existe, pandas lo sobrescribe automáticamente.
df_meteo_definitivo.to_csv(
    ruta_salida,
    index=False
)

print("\n" + "=" * 90)
print("DATASET METEOROLÓGICO DEFINITIVO")
print("=" * 90)

print(f"Registros finales: {len(df_meteo_definitivo):,}")
print(f"Columnas finales: {df_meteo_definitivo.shape[1]}")
print(f"Duplicados fecha-estación: {duplicados}")

print("\nArchivo definitivo guardado/sustituido en:")
print(ruta_salida)

2.12 AUDITORÍA FINAL DEL DATASET METEOROLÓGICO

Dimensiones finales: (10136, 17)
Periodo: 2018-01-01 - 2024-12-31
Estaciones: 4
Códigos de estación: ['D5', 'X2', 'X4', 'X8']

------------------------------------------------------------------------------------------
DUPLICADOS FECHA-ESTACIÓN
------------------------------------------------------------------------------------------
Duplicados encontrados: 0

------------------------------------------------------------------------------------------
REGISTROS POR ESTACIÓN
------------------------------------------------------------------------------------------
Estacion
D5    2557
X2    2465
X4    2557
X8    2557
dtype: int64

------------------------------------------------------------------------------------------
VALORES NULOS DEL DATASET DEFINITIVO
------------------------------------------------------------------------------------------


,Nulos,Porcentaje (%)
Fecha,0,0.00
Estacion,0,0.00
TM,0,0.00
TN,0,0.00
TX,0,0.00
HRM,0,0.00
HRN,2920,28.81
HRX,2920,28.81
PM,4655,45.93
PN,4655,45.93



------------------------------------------------------------------------------------------
NULOS POR ESTACIÓN
------------------------------------------------------------------------------------------


,HRN,HRX,PM,PN,PX,PPT,RS24h,DVM10,DVVX10,VVM10,VVX10
Estacion,,,,,,,,,,,
D5,730,730,730,730,730,0,730,0,730,0,0
X2,730,730,2465,2465,2465,2465,2465,2465,2465,2465,2465
X4,730,730,730,730,730,0,730,0,730,0,0
X8,730,730,730,730,730,0,730,0,730,0,0



------------------------------------------------------------------------------------------
TIPOS DE DATOS
------------------------------------------------------------------------------------------
Fecha       datetime64[ns]
Estacion          category
TM                 float64
TN                 float64
TX                 float64
HRM                float64
HRN                float64
HRX                float64
PM                 float64
PN                 float64
PX                 float64
PPT                float64
RS24h              float64
DVM10              float64
DVVX10             float64
VVM10              float64
VVX10              float64
dtype: object

------------------------------------------------------------------------------------------
ESTADÍSTICAS DESCRIPTIVAS FINALES
------------------------------------------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
TM,10136.0,17.495,6.032,-0.7,12.6,16.9,22.7,33.4
TN,10136.0,14.123,6.025,-2.4,9.3,13.6,19.3,29.5
TX,10136.0,21.785,6.247,1.2,16.6,21.2,27.1,39.8
HRM,10136.0,68.131,12.213,22.0,60.0,69.0,77.0,100.0
HRN,7216.0,47.587,13.274,7.0,38.0,47.0,57.0,100.0
HRX,7216.0,87.380,11.507,36.0,80.0,90.0,97.0,100.0
PM,5481.0,996.747,20.588,942.5,972.9,1006.1,1012.5,1033.7
PN,5481.0,994.570,20.712,936.6,971.0,1003.9,1010.6,1032.8
PX,5481.0,999.075,20.537,946.8,975.0,1008.3,1014.7,1035.7
PPT,7671.0,1.549,6.665,0.0,0.0,0.0,0.0,139.9



DATASET METEOROLÓGICO DEFINITIVO
Registros finales: 10,136
Columnas finales: 17
Duplicados fecha-estación: 0

Archivo definitivo guardado/sustituido en:
/content/drive/MyDrive/TFM/08_Meteorologia/DATOS LIMPIOS/df_meteorologia_2018_2024_Limpio.csv


### Resultado Construcción y auditoría final del dataset meteorológico

Tras completar los procesos de homogeneización, imputación y control de coherencia física, se obtiene el dataset meteorológico definitivo para el periodo **2018–2024**.

El conjunto final contiene **10.136 observaciones correspondientes a cuatro estaciones meteorológicas (D5, X2, X4 y X8)** y **17 variables**, incluyendo los identificadores temporal y espacial y las principales variables de temperatura, humedad, presión atmosférica, precipitación, radiación y viento.

La auditoría final confirma la **ausencia de registros duplicados por fecha y estación** y la correcta conservación de los valores ausentes de carácter estructural. Estas ausencias corresponden a variables no medidas por determinadas estaciones o no disponibles durante determinados periodos históricos, por lo que se mantienen deliberadamente sin imputación.

Las variables meteorológicas fundamentales para el análisis —especialmente temperatura media (`TM`) y humedad relativa media (`HRM`)— presentan cobertura prácticamente completa durante todo el periodo. Asimismo, los controles realizados previamente confirman la coherencia física de las variables y la conservación de los episodios meteorológicos extremos potencialmente relevantes.

El dataset resultante constituye la **entrada definitiva del bloque meteorológico para su posterior integración con los datos de contaminación atmosférica y el resto de variables temporales del estudio**.

## 2.12 Cierre del bloque de meteorología

Tras las etapas de carga, inspección, homogeneización, integración temporal, tratamiento de valores ausentes y control final de calidad, se obtiene un **dataset meteorológico consolidado para el periodo 2018–2024**.

La información procede de las estaciones meteorológicas **D5, X2, X4 y X8** y se encuentra estructurada con resolución diaria, incluyendo variables representativas de **temperatura, humedad relativa, precipitación, presión atmosférica, radiación solar y viento**.

Durante el preprocesado se identificaron diferencias en la disponibilidad de variables entre estaciones y periodos. En particular, se distinguieron los **valores ausentes puntuales**, susceptibles de imputación, de los **nulos estructurales** derivados de variables no registradas por determinadas estaciones o no disponibles en determinados periodos. Estos últimos se conservaron explícitamente para evitar la generación artificial de información inexistente.

Los valores ausentes considerados imputables fueron tratados mediante un procedimiento basado en **KNN**, incorporando información temporal y de estación para preservar, en la medida de lo posible, la estructura meteorológica de las observaciones. Posteriormente se eliminaron las variables auxiliares utilizadas exclusivamente durante el proceso de imputación.

Como control final, se verificó la coherencia de las variables meteorológicas y, específicamente, el rango de las direcciones del viento, limitado al intervalo físico **0–359°**. No se eliminaron valores extremos meteorológicos, al considerarse observaciones potencialmente reales y relevantes para el posterior análisis de su relación con la contaminación atmosférica.

El bloque queda así preparado para su integración con las restantes fuentes de información del estudio mediante la dimensión temporal y, cuando resulte necesario, mediante criterios de correspondencia espacial entre estaciones.

# 3 · PREPARACIÓN DE LOS DATOS DE TRÁFICO RODADO Y AFOROS

## 3.1 Fuente y descripción de los datos

Los datos de tráfico rodado utilizados en este estudio proceden del portal **Open Data BCN del Ajuntament de Barcelona**, concretamente del conjunto de datos de **aforamientos de tráfico: detalle de los valores de Intensidad Media Diaria (IMD)**.

**Fuente oficial:**  
https://opendata-ajuntament.barcelona.cat/data/es/dataset/aforaments-detall

La fuente proporciona información de tráfico asociada a los distintos puntos de aforo de la ciudad de Barcelona. Para cada punto se dispone de valores de **Intensidad Media Diaria (`Valor_IMD`)**, diferenciados por año, mes y tipología de día.

Las variables originales empleadas son:

- `Any`: año de referencia;
- `Id_aforament`: identificador del punto de aforo;
- `Mes`: mes de referencia;
- `Codi_tipus_dia`: código correspondiente al tipo de día;
- `Desc_tipus_dia`: descripción del tipo de día;
- `Valor_IMD`: Intensidad Media Diaria de tráfico.

Para el estudio se utilizan los archivos correspondientes al periodo **2018–2024**, permitiendo analizar la evolución del tráfico durante tres etapas de especial interés:

- **2018–2019:** periodo previo a la pandemia;
- **2020–2021:** periodo afectado por las restricciones y cambios de movilidad asociados a la COVID-19;
- **2022–2024:** periodo posterior y de recuperación de la movilidad.

Dado que la información original presenta una resolución mensual diferenciada por tipo de día, se desarrolla posteriormente un procedimiento de reconstrucción temporal que permite obtener una serie de resolución diaria compatible con los restantes bloques de información utilizados en el estudio.

El tratamiento incluye la unificación de los archivos anuales, normalización de variables, análisis de cobertura, reconstrucción del calendario diario, tratamiento controlado de valores ausentes y auditorías sucesivas destinadas a preservar la estructura temporal y la trazabilidad de los datos.

## 3.1 Carga e inspección inicial de los datos de tráfico rodado
Se realiza una primera inspección de los datos de tráfico rodado disponibles, utilizando como muestra el archivo correspondiente a **2024**. El objetivo es identificar la estructura del dataset, las variables disponibles, los tipos de datos y el formato de los registros antes de abordar su selección, limpieza y homogeneización temporal.

In [ ]:
# ==============================================================================
# 3.1 CARGA E INSPECCIÓN INICIAL DE LOS DATOS DE TRÁFICO RODADO
# ==============================================================================

import pandas as pd

# ------------------------------------------------------------------------------
# 1. Ruta al archivo de muestra correspondiente a 2024
# ------------------------------------------------------------------------------

ruta_aforos = (
    "/content/drive/My Drive/TFM/"
    "02_Trafico_Rodado_y_Aforos/"
    "2024_aforament_detall_valor.csv"
)

# ------------------------------------------------------------------------------
# 2. Lectura robusta del archivo CSV
# ------------------------------------------------------------------------------

try:
    df_aforos = pd.read_csv(
        ruta_aforos,
        encoding="utf-8"
    )

    # Comprobación básica por si el separador no se detecta correctamente
    if len(df_aforos.columns) <= 1:
        df_aforos = pd.read_csv(
            ruta_aforos,
            encoding="utf-8",
            sep=";"
        )

except Exception:

    try:
        df_aforos = pd.read_csv(
            ruta_aforos,
            encoding="latin1",
            sep=";"
        )

    except Exception:
        df_aforos = pd.read_csv(
            ruta_aforos,
            encoding="latin1"
        )

# ------------------------------------------------------------------------------
# 3. Variables disponibles
# ------------------------------------------------------------------------------

print("=" * 80)
print("1. VARIABLES / COLUMNAS")
print("=" * 80)

print(df_aforos.columns.tolist())

# ------------------------------------------------------------------------------
# 4. Información general del DataFrame
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. INFORMACIÓN GENERAL")
print("=" * 80)

df_aforos.info()

# ------------------------------------------------------------------------------
# 5. Valores nulos
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. VALORES NULOS")
print("=" * 80)

tabla_nulos = pd.DataFrame({
    "Nulos": df_aforos.isna().sum(),
    "Porcentaje (%)": (
        df_aforos
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(tabla_nulos)

# ------------------------------------------------------------------------------
# 6. Valores únicos por variable
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. VALORES ÚNICOS POR VARIABLE")
print("=" * 80)

for col in df_aforos.columns:
    print(
        f"{col}: "
        f"{df_aforos[col].nunique(dropna=False):,} valores únicos"
    )

# ------------------------------------------------------------------------------
# 7. Categorías de tipo de día
# ------------------------------------------------------------------------------

if "Desc_tipus_dia" in df_aforos.columns:

    print("\n" + "=" * 80)
    print("5. TIPOS DE DÍA DISPONIBLES")
    print("=" * 80)

    tipos_dia = (
        df_aforos[
            ["Codi_tipus_dia", "Desc_tipus_dia"]
        ]
        .drop_duplicates()
        .sort_values("Codi_tipus_dia")
        .reset_index(drop=True)
    )

    display(tipos_dia)

# ------------------------------------------------------------------------------
# 8. Resumen estadístico
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. RESUMEN ESTADÍSTICO")
print("=" * 80)

display(
    df_aforos.describe(include="all")
)

# ------------------------------------------------------------------------------
# 9. Primeras observaciones
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. PRIMERAS 10 FILAS")
print("=" * 80)

display(
    df_aforos.head(10)
)

# ------------------------------------------------------------------------------
# 10. Resumen básico de la estructura temporal y espacial
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. RESUMEN DE LA ESTRUCTURA DEL DATASET")
print("=" * 80)

if "Any" in df_aforos.columns:
    print(
        f"Años presentes: "
        f"{sorted(df_aforos['Any'].dropna().unique().tolist())}"
    )

if "Mes" in df_aforos.columns:
    print(
        f"Meses presentes: "
        f"{sorted(df_aforos['Mes'].dropna().unique().tolist())}"
    )

if "Id_aforament" in df_aforos.columns:
    print(
        f"Puntos de aforo únicos: "
        f"{df_aforos['Id_aforament'].nunique():,}"
    )

if "Desc_tipus_dia" in df_aforos.columns:
    print(
        f"Tipos de día: "
        f"{df_aforos['Desc_tipus_dia'].nunique():,}"
    )

1. VARIABLES / COLUMNAS
['Any', 'Id_aforament', 'Mes', 'Codi_tipus_dia', 'Desc_tipus_dia', 'Valor_IMD']

2. INFORMACIÓN GENERAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50280 entries, 0 to 50279
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Any             50280 non-null  int64 
 1   Id_aforament    50280 non-null  object
 2   Mes             50280 non-null  int64 
 3   Codi_tipus_dia  50280 non-null  int64 
 4   Desc_tipus_dia  50280 non-null  object
 5   Valor_IMD       50280 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.3+ MB

3. VALORES NULOS


,Nulos,Porcentaje (%)
Any,0,0.0
Id_aforament,0,0.0
Mes,0,0.0
Codi_tipus_dia,0,0.0
Desc_tipus_dia,0,0.0
Valor_IMD,0,0.0



4. VALORES ÚNICOS POR VARIABLE
Any: 1 valores únicos
Id_aforament: 838 valores únicos
Mes: 12 valores únicos
Codi_tipus_dia: 5 valores únicos
Desc_tipus_dia: 5 valores únicos
Valor_IMD: 18,846 valores únicos

5. TIPOS DE DÍA DISPONIBLES


,Codi_tipus_dia,Desc_tipus_dia
0,1,dilluns
1,2,laborables
2,3,divendres
3,4,dissabte
4,5,diumenge



6. RESUMEN ESTADÍSTICO


,Any,Id_aforament,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD
count,50280.0,50280,50280.000000,50280.000000,50280,50280
unique,NaN,838,NaN,NaN,5,18846
top,NaN,9010,NaN,NaN,dilluns,Mesura no disponible
freq,NaN,60,NaN,NaN,10056,5565
mean,2024.0,NaN,6.500000,3.000000,NaN,NaN
std,0.0,NaN,3.452087,1.414228,NaN,NaN
min,2024.0,NaN,1.000000,1.000000,NaN,NaN
25%,2024.0,NaN,3.750000,2.000000,NaN,NaN
50%,2024.0,NaN,6.500000,3.000000,NaN,NaN
75%,2024.0,NaN,9.250000,4.000000,NaN,NaN



7. PRIMERAS 10 FILAS


,Any,Id_aforament,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD
0,2024,10001,1,1,dilluns,8665
1,2024,10001,1,2,laborables,8867
2,2024,10001,1,3,divendres,9127
3,2024,10001,1,4,dissabte,8063
4,2024,10001,1,5,diumenge,6234
5,2024,10001,2,1,dilluns,9176
6,2024,10001,2,2,laborables,9454
7,2024,10001,2,3,divendres,10016
8,2024,10001,2,4,dissabte,8240
9,2024,10001,2,5,diumenge,6574



8. RESUMEN DE LA ESTRUCTURA DEL DATASET
Años presentes: [2024]
Meses presentes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Puntos de aforo únicos: 838
Tipos de día: 5


### Resultado de la inspección inicial

El archivo correspondiente a 2024 contiene **50.280 registros**, distribuidos entre **838 puntos de aforo**, los **12 meses del año** y **cinco categorías de tipo de día**: lunes, laborables, viernes, sábados y domingos.

La información se encuentra agregada por combinación de **año, punto de aforo, mes y tipo de día**, proporcionando como variable principal la **Intensidad Media Diaria (`Valor_IMD`)**.

Aunque la inspección mediante valores nulos no detecta registros `NaN`, se observa que `Valor_IMD` se encuentra almacenada como variable de tipo texto (`object`). El análisis de sus valores revela la presencia de **5.565 registros identificados mediante la cadena `"Mesura no disponible"`**, que representan valores ausentes codificados explícitamente en la fuente original.

Por tanto, antes de realizar cualquier análisis estadístico será necesario transformar estos registros a valores ausentes reales (`NaN`) y convertir `Valor_IMD` a formato numérico.

La estructura temporal del dataset no representa observaciones correspondientes a fechas individuales, sino valores de IMD característicos de cada combinación de mes y tipo de día. Esta particularidad deberá considerarse posteriormente en la construcción de la serie temporal diaria.

## 3.2 Carga y unificación de los archivos anuales de aforos (2018–2024)

In [ ]:
# ==============================================================================
# 3.2 CARGA Y UNIFICACIÓN DE LOS ARCHIVOS ANUALES DE AFOROS
# ==============================================================================

import pandas as pd
import glob
import os

# ------------------------------------------------------------------------------
# 1. Ruta de los datos
# ------------------------------------------------------------------------------

ruta_base = (
    "/content/drive/MyDrive/TFM/"
    "02_Trafico_Rodado_y_Aforos/"
)

patron_archivos = os.path.join(
    ruta_base,
    "*_aforament_detall_valor.csv"
)

archivos_csv = sorted(glob.glob(patron_archivos))

# ------------------------------------------------------------------------------
# 2. Comprobar los archivos encontrados
# ------------------------------------------------------------------------------

print("=" * 80)
print("ARCHIVOS ENCONTRADOS")
print("=" * 80)

print(f"Número de archivos: {len(archivos_csv)}\n")

for archivo in archivos_csv:
    print(os.path.basename(archivo))

# ------------------------------------------------------------------------------
# 3. Lectura individual y comprobación de estructura
# ------------------------------------------------------------------------------

lista_df = []
resumen_archivos = []

for archivo in archivos_csv:

    nombre_archivo = os.path.basename(archivo)

    try:
        df_temp = pd.read_csv(
            archivo,
            encoding="utf-8"
        )

        if len(df_temp.columns) <= 1:
            df_temp = pd.read_csv(
                archivo,
                encoding="utf-8",
                sep=";"
            )

    except Exception:

        df_temp = pd.read_csv(
            archivo,
            encoding="latin1",
            sep=";"
        )

    # Año indicado en el propio dataset
    anos_archivo = sorted(
        df_temp["Any"]
        .dropna()
        .unique()
        .tolist()
    )

    resumen_archivos.append({
        "Archivo": nombre_archivo,
        "Registros": len(df_temp),
        "Columnas": len(df_temp.columns),
        "Años_contenidos": anos_archivo
    })

    lista_df.append(df_temp)

# ------------------------------------------------------------------------------
# 4. Resumen de los archivos
# ------------------------------------------------------------------------------

df_resumen_archivos = pd.DataFrame(
    resumen_archivos
)

print("\n" + "=" * 80)
print("RESUMEN DE LOS ARCHIVOS")
print("=" * 80)

display(df_resumen_archivos)

# ------------------------------------------------------------------------------
# 5. Comprobar si todos tienen las mismas columnas
# ------------------------------------------------------------------------------

estructuras = {
    tuple(df.columns)
    for df in lista_df
}

print("\n" + "=" * 80)
print("COMPROBACIÓN DE ESTRUCTURA")
print("=" * 80)

print(
    f"Número de estructuras diferentes encontradas: "
    f"{len(estructuras)}"
)

if len(estructuras) == 1:
    print("Todos los archivos presentan la misma estructura.")
else:
    print(
        "ATENCIÓN: existen diferencias de estructura "
        "entre los archivos."
    )

# ------------------------------------------------------------------------------
# 6. Concatenación
# ------------------------------------------------------------------------------

df_aforos_total = pd.concat(
    lista_df,
    ignore_index=True
)

# ------------------------------------------------------------------------------
# 7. Comprobación del dataset unificado
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET DE AFOROS UNIFICADO")
print("=" * 80)

print(
    f"Dimensiones: "
    f"{df_aforos_total.shape}"
)

print(
    f"Años presentes: "
    f"{sorted(df_aforos_total['Any'].dropna().unique().tolist())}"
)

print(
    f"Puntos de aforo únicos: "
    f"{df_aforos_total['Id_aforament'].nunique():,}"
)

print(
    f"Tipos de día: "
    f"{df_aforos_total['Desc_tipus_dia'].nunique()}"
)

print("\nPrimeras observaciones:")

display(
    df_aforos_total.head(10)
)

ARCHIVOS ENCONTRADOS
Número de archivos: 7

2018_aforament_detall_valor.csv
2019_aforament_detall_valor.csv
2020_aforament_detall_valor.csv
2021_aforament_detall_valor.csv
2022_aforament_detall_valor.csv
2023_aforament_detall_valor.csv
2024_aforament_detall_valor.csv

RESUMEN DE LOS ARCHIVOS


,Archivo,Registros,Columnas,Años_contenidos
0,2018_aforament_detall_valor.csv,36780,6,[2018]
1,2019_aforament_detall_valor.csv,41100,6,[2019]
2,2020_aforament_detall_valor.csv,42840,6,[2020]
3,2021_aforament_detall_valor.csv,43440,6,[2021]
4,2022_aforament_detall_valor.csv,49435,6,[2022]
5,2023_aforament_detall_valor.csv,49920,6,[2023]
6,2024_aforament_detall_valor.csv,50280,6,[2024]



COMPROBACIÓN DE ESTRUCTURA
Número de estructuras diferentes encontradas: 1
Todos los archivos presentan la misma estructura.

DATASET DE AFOROS UNIFICADO
Dimensiones: (313795, 6)
Años presentes: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
Puntos de aforo únicos: 922
Tipos de día: 10

Primeras observaciones:


,Any,Id_aforament,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD
0,2018,10001,1,1,Dilluns,8625
1,2018,10001,1,2,Laborable,8776
2,2018,10001,1,3,Divendres,9480
3,2018,10001,1,4,Dissabte,6415
4,2018,10001,1,5,Diumenge,4633
5,2018,10001,2,1,Dilluns,8940
6,2018,10001,2,2,Laborable,9172
7,2018,10001,2,3,Divendres,10985
8,2018,10001,2,4,Dissabte,6238
9,2018,10001,2,5,Diumenge,4572


### Resultado de la unificación anual

Se localizaron y unificaron correctamente los **siete archivos anuales de aforos correspondientes al periodo 2018–2024**. Todos presentan una estructura homogénea de seis variables, por lo que su concatenación puede realizarse directamente sin necesidad de adaptar el esquema entre años.

El dataset conjunto contiene **313.795 registros** y **922 puntos de aforo únicos**.

La cobertura temporal incluye de forma continua los años **2018, 2019, 2020, 2021, 2022, 2023 y 2024**, permitiendo analizar tanto el periodo previo a la pandemia como los cambios registrados durante 2020 y la evolución posterior.

La inspección del campo `Desc_tipus_dia` identifica **10 denominaciones diferentes**, aunque conceptualmente los datos representan cinco categorías de tipo de día. Esta diferencia parece asociada a cambios de nomenclatura entre años —por ejemplo, variaciones de mayúsculas, minúsculas o singular/plural—, por lo que será necesario normalizar dichas etiquetas antes de construir la serie temporal.

La estructura de los archivos permanece estable durante todo el periodo de estudio. Las siguientes etapas se centrarán en la normalización de las categorías de día, la identificación y conversión de los valores de IMD no disponibles y el análisis de la cobertura temporal de los distintos puntos de aforo.

## 3.3 Auditoría inicial del dataset unificado de aforos

Una vez unificados los archivos correspondientes al periodo 2018–2024, se realiza una auditoría general del dataset resultante.

Se examinan las dimensiones y tipos de las variables, el número de valores únicos y la presencia de valores ausentes. Esta comprobación permite evaluar la consistencia de la tabla maestra e identificar posibles problemas de calidad antes de proceder a la normalización de categorías y al tratamiento de la variable de Intensidad Media Diaria (`Valor_IMD`).

In [ ]:
# ==============================================================================
# 3.3 AUDITORÍA INICIAL DEL DATASET UNIFICADO DE AFOROS
# ==============================================================================

df = df_aforos_total

# ------------------------------------------------------------------------------
# 1. Información general
# ------------------------------------------------------------------------------

print("=" * 80)
print("1. INFORMACIÓN GENERAL")
print("=" * 80)

df.info()

print(f"\nDimensiones del dataset: {df.shape}")

# ------------------------------------------------------------------------------
# 2. Primeras observaciones
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. PRIMERAS OBSERVACIONES")
print("=" * 80)

display(df.head(10))

# ------------------------------------------------------------------------------
# 3. Valores únicos por variable
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. VALORES ÚNICOS POR VARIABLE")
print("=" * 80)

for col in df.columns:
    print(
        f"{col}: "
        f"{df[col].nunique(dropna=False):,} valores únicos"
    )

# ------------------------------------------------------------------------------
# 4. Valores nulos
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. VALORES NULOS")
print("=" * 80)

df_nulos = pd.DataFrame({
    "Nulos": df.isna().sum(),
    "Porcentaje (%)": (
        df.isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(df_nulos)

# ------------------------------------------------------------------------------
# 5. Cobertura temporal
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. COBERTURA TEMPORAL")
print("=" * 80)

print(
    "Años disponibles:",
    sorted(df["Any"].dropna().unique().tolist())
)

print(
    "Meses disponibles:",
    sorted(df["Mes"].dropna().unique().tolist())
)

# ------------------------------------------------------------------------------
# 6. Número de puntos de aforo por año
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. PUNTOS DE AFORO POR AÑO")
print("=" * 80)

aforos_por_ano = (
    df.groupby("Any")["Id_aforament"]
    .nunique()
    .rename("Puntos_aforo")
    .reset_index()
)

display(aforos_por_ano)

1. INFORMACIÓN GENERAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313795 entries, 0 to 313794
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   Any             313795 non-null  int64 
 1   Id_aforament    313795 non-null  object
 2   Mes             313795 non-null  int64 
 3   Codi_tipus_dia  313795 non-null  int64 
 4   Desc_tipus_dia  313795 non-null  object
 5   Valor_IMD       313795 non-null  object
dtypes: int64(3), object(3)
memory usage: 14.4+ MB

Dimensiones del dataset: (313795, 6)

2. PRIMERAS OBSERVACIONES


,Any,Id_aforament,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD
0,2018,10001,1,1,Dilluns,8625
1,2018,10001,1,2,Laborable,8776
2,2018,10001,1,3,Divendres,9480
3,2018,10001,1,4,Dissabte,6415
4,2018,10001,1,5,Diumenge,4633
5,2018,10001,2,1,Dilluns,8940
6,2018,10001,2,2,Laborable,9172
7,2018,10001,2,3,Divendres,10985
8,2018,10001,2,4,Dissabte,6238
9,2018,10001,2,5,Diumenge,4572



3. VALORES ÚNICOS POR VARIABLE
Any: 7 valores únicos
Id_aforament: 922 valores únicos
Mes: 12 valores únicos
Codi_tipus_dia: 5 valores únicos
Desc_tipus_dia: 10 valores únicos
Valor_IMD: 71,896 valores únicos

4. VALORES NULOS


,Nulos,Porcentaje (%)
Any,0,0.0
Id_aforament,0,0.0
Mes,0,0.0
Codi_tipus_dia,0,0.0
Desc_tipus_dia,0,0.0
Valor_IMD,0,0.0



5. COBERTURA TEMPORAL
Años disponibles: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
Meses disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

6. PUNTOS DE AFORO POR AÑO


,Any,Puntos_aforo
0,2018,613
1,2019,685
2,2020,714
3,2021,724
4,2022,824
5,2023,832
6,2024,838


### Resultado de la auditoría inicial

El dataset unificado contiene **313.795 registros** correspondientes al periodo completo **2018–2024**, con información para los doce meses de cada año.

Se identifican **922 puntos de aforo distintos** en el conjunto del periodo, aunque su número varía temporalmente, aumentando desde **613 puntos en 2018 hasta 838 en 2024**. Esta evolución indica que la cobertura espacial de la red de aforos no permanece constante durante todo el periodo de estudio, aspecto que deberá considerarse en las comparaciones interanuales.

La variable `Codi_tipus_dia` presenta **cinco categorías**, mientras que `Desc_tipus_dia` contiene **diez denominaciones diferentes**. Esta discrepancia apunta a variaciones de nomenclatura entre los archivos anuales y requiere una normalización previa de las etiquetas.

No se detectan valores nulos (`NaN`) en las variables originales. Sin embargo, `Valor_IMD` permanece almacenada como variable de tipo texto (`object`), por lo que será necesario comprobar la existencia de valores no disponibles codificados mediante cadenas de texto antes de convertirla a formato numérico.

La estructura general del dataset se considera adecuada para continuar con las etapas de normalización y control de calidad.

## 3.4 Auditoría de `Valor_IMD` y de las categorías de tipo de día

Antes de proceder a la limpieza y transformación de los datos de tráfico, se analiza la variable de Intensidad Media Diaria (`Valor_IMD`) para identificar posibles registros no numéricos codificados como texto.

Paralelamente, se examina la correspondencia entre `Codi_tipus_dia` y `Desc_tipus_dia`, con el objetivo de determinar el origen de las distintas denominaciones detectadas durante la auditoría inicial y comprobar si representan diferencias reales de categoría o únicamente variaciones de nomenclatura entre años.

In [ ]:
# ==============================================================================
# 3.4 AUDITORÍA DE VALOR_IMD Y DE LAS CATEGORÍAS DE TIPO DE DÍA
# ==============================================================================

df = df_aforos_total

# ------------------------------------------------------------------------------
# 1. Identificación de valores no numéricos en Valor_IMD
# ------------------------------------------------------------------------------

valor_imd_num = pd.to_numeric(
    df["Valor_IMD"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

mask_no_numericos = valor_imd_num.isna()

print("=" * 80)
print("1. VALORES NO NUMÉRICOS EN Valor_IMD")
print("=" * 80)

print(
    f"Registros no numéricos: "
    f"{mask_no_numericos.sum():,}"
)

print(
    f"Porcentaje sobre el total: "
    f"{mask_no_numericos.mean() * 100:.2f}%"
)

# Mostrar los valores responsables
if mask_no_numericos.any():

    valores_problematicos = (
        df.loc[mask_no_numericos, "Valor_IMD"]
        .value_counts(dropna=False)
        .rename_axis("Valor_original")
        .reset_index(name="Registros")
    )

    print("\nValores no numéricos detectados:")
    display(valores_problematicos)

# ------------------------------------------------------------------------------
# 2. Distribución de valores no numéricos por año
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. VALORES NO NUMÉRICOS POR AÑO")
print("=" * 80)

no_numericos_ano = (
    df.loc[mask_no_numericos]
    .groupby("Any")
    .size()
    .rename("Registros_no_numericos")
    .reset_index()
)

display(no_numericos_ano)

# ------------------------------------------------------------------------------
# 3. Correspondencia entre código y descripción del tipo de día
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. CATEGORÍAS DE TIPO DE DÍA")
print("=" * 80)

tipos_dia = (
    df[
        ["Codi_tipus_dia", "Desc_tipus_dia"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Codi_tipus_dia", "Desc_tipus_dia"]
    )
    .reset_index(drop=True)
)

display(tipos_dia)

# ------------------------------------------------------------------------------
# 4. Categorías utilizadas en cada año
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. DENOMINACIONES DE TIPO DE DÍA POR AÑO")
print("=" * 80)

tipos_dia_ano = (
    df[
        ["Any", "Codi_tipus_dia", "Desc_tipus_dia"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Any", "Codi_tipus_dia"]
    )
    .reset_index(drop=True)
)

display(tipos_dia_ano)

1. VALORES NO NUMÉRICOS EN Valor_IMD
Registros no numéricos: 37,615
Porcentaje sobre el total: 11.99%

Valores no numéricos detectados:


,Valor_original,Registros
0,Mesura no disponible,37615



2. VALORES NO NUMÉRICOS POR AÑO


,Any,Registros_no_numericos
0,2018,7799
1,2019,6175
2,2020,3575
3,2021,3783
4,2022,6023
5,2023,4695
6,2024,5565



3. CATEGORÍAS DE TIPO DE DÍA


,Codi_tipus_dia,Desc_tipus_dia
0,1,Dilluns
1,1,dilluns
2,2,Laborable
3,2,laborables
4,3,Divendres
5,3,divendres
6,4,Dissabte
7,4,dissabte
8,5,Diumenge
9,5,diumenge



4. DENOMINACIONES DE TIPO DE DÍA POR AÑO


,Any,Codi_tipus_dia,Desc_tipus_dia
0,2018,1,Dilluns
1,2018,2,Laborable
2,2018,3,Divendres
3,2018,4,Dissabte
4,2018,5,Diumenge
5,2019,1,Dilluns
6,2019,2,Laborable
7,2019,3,Divendres
8,2019,4,Dissabte
9,2019,5,Diumenge


### Resultado de la auditoría

La auditoría de `Valor_IMD` identifica **37.615 registros no numéricos**, equivalentes al **11,99 %** del conjunto de datos. La totalidad de estos registros corresponde a la expresión `"Mesura no disponible"`, utilizada por la fuente original para representar observaciones sin medición disponible.

Los valores no disponibles aparecen en todos los años del periodo analizado, aunque con diferente frecuencia, por lo que deberán transformarse en valores ausentes reales (`NaN`) antes de convertir `Valor_IMD` a formato numérico.

Por otra parte, las diez denominaciones detectadas inicialmente en `Desc_tipus_dia` corresponden en realidad a **cinco categorías de día**, identificadas de forma consistente mediante `Codi_tipus_dia`:

- 1: lunes
- 2: laborables
- 3: viernes
- 4: sábado
- 5: domingo

Las diferencias observadas son exclusivamente de nomenclatura. Durante 2018–2019 se utilizan etiquetas con mayúscula inicial y la categoría `Laborable` en singular, mientras que desde 2020 se emplean minúsculas y `laborables` en plural.

Por tanto, se considera adecuado normalizar las denominaciones utilizando `Codi_tipus_dia` como referencia y convertir `"Mesura no disponible"` en `NaN`, preservando estos registros para evaluar posteriormente el procedimiento de tratamiento de valores ausentes.

## 3.5 Limpieza y normalización de las variables de aforo

Una vez identificados los valores no numéricos y las diferencias de nomenclatura entre años, se procede a normalizar las variables principales del bloque de tráfico rodado.

La variable `Valor_IMD` se transforma a formato numérico, convirtiendo automáticamente en valores ausentes (`NaN`) aquellos registros codificados en la fuente como `"Mesura no disponible"`. De este modo, las ausencias quedan representadas explícitamente y pueden ser analizadas posteriormente mediante procedimientos estadísticos.

Asimismo, la variable `Desc_tipus_dia` se normaliza mediante conversión a minúsculas y eliminación de espacios adicionales, con el objetivo de reducir las diferencias de nomenclatura existentes entre los distintos años.

La correspondencia entre el código numérico `Codi_tipus_dia` y la descripción del tipo de día se mantiene como referencia para garantizar la consistencia de las categorías durante las etapas posteriores de transformación temporal.

In [ ]:
import numpy as np
import pandas as pd

# ==============================================================================
# 3.5 LIMPIEZA Y NORMALIZACIÓN DE LAS VARIABLES DE AFORO
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Conversión de Valor_IMD a formato numérico
# ------------------------------------------------------------------------------

df_aforos_total["Valor_IMD"] = (
    df_aforos_total["Valor_IMD"]
    .astype(str)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

df_aforos_total["Valor_IMD"] = pd.to_numeric(
    df_aforos_total["Valor_IMD"],
    errors="coerce"
)

# ------------------------------------------------------------------------------
# 2. Normalización de las categorías de tipo de día
# ------------------------------------------------------------------------------

mapeo_dias = {
    1: "lunes",
    2: "laborables",
    3: "viernes",
    4: "sábado",
    5: "domingo"
}

df_aforos_total["Desc_tipus_dia"] = (
    df_aforos_total["Codi_tipus_dia"]
    .map(mapeo_dias)
)

# ------------------------------------------------------------------------------
# 3. Comprobaciones
# ------------------------------------------------------------------------------

print("=" * 80)
print("LIMPIEZA Y NORMALIZACIÓN COMPLETADA")
print("=" * 80)

print(
    f"\nValores ausentes en Valor_IMD: "
    f"{df_aforos_total['Valor_IMD'].isna().sum():,}"
)

print(
    f"Porcentaje de valores ausentes: "
    f"{df_aforos_total['Valor_IMD'].isna().mean() * 100:.2f}%"
)

print("\nTipos de día normalizados:")

display(
    df_aforos_total[
        ["Codi_tipus_dia", "Desc_tipus_dia"]
    ]
    .drop_duplicates()
    .sort_values("Codi_tipus_dia")
    .reset_index(drop=True)
)

print("\nTipo de dato final de Valor_IMD:")
print(df_aforos_total["Valor_IMD"].dtype)

LIMPIEZA Y NORMALIZACIÓN COMPLETADA

Valores ausentes en Valor_IMD: 37,615
Porcentaje de valores ausentes: 11.99%

Tipos de día normalizados:


,Codi_tipus_dia,Desc_tipus_dia
0,1,lunes
1,2,laborables
2,3,viernes
3,4,sábado
4,5,domingo



Tipo de dato final de Valor_IMD:
float64


### Resultado de la limpieza y normalización

La variable `Valor_IMD` se convirtió correctamente a formato numérico (`float64`). Los registros identificados originalmente mediante la expresión `"Mesura no disponible"` se transformaron en valores ausentes reales (`NaN`).

Tras esta conversión se contabilizan **37.615 valores ausentes**, equivalentes al **11,99 %** del dataset.

Asimismo, la variable `Desc_tipus_dia` se normalizó utilizando `Codi_tipus_dia` como referencia, obteniéndose cinco categorías homogéneas para todo el periodo 2018–2024:

- lunes
- laborables
- viernes
- sábado
- domingo

Con esta transformación se eliminan las diferencias de nomenclatura existentes entre los archivos anuales y se obtiene una codificación temporal consistente para las etapas posteriores del análisis.

## 3.6 Auditoría de la distribución y cobertura de los valores ausentes de `Valor_IMD`

Tras la limpieza y normalización de las variables, se analiza la distribución de los valores ausentes de `Valor_IMD` antes de aplicar cualquier procedimiento de tratamiento o imputación.

El objetivo es determinar si las ausencias presentan patrones temporales o asociados a determinados puntos de aforo. Para ello, se estudia su distribución por año, mes y tipo de día, así como la cobertura disponible para cada punto de medida.

Esta auditoría permite diferenciar entre ausencias puntuales potencialmente imputables y situaciones de cobertura insuficiente o ausencia sistemática de información, evitando generar artificialmente datos en puntos o periodos sin soporte observacional suficiente.

In [ ]:
# ==============================================================================
# 3.6 AUDITORÍA DE LA DISTRIBUCIÓN Y COBERTURA DE LOS VALORES AUSENTES DE VALOR_IMD
# ==============================================================================

df = df_aforos_total.copy()

# Crear indicador de ausencia
df["IMD_ausente"] = df["Valor_IMD"].isna()

# ------------------------------------------------------------------------------
# 1. Resumen general
# ------------------------------------------------------------------------------

print("=" * 80)
print("1. RESUMEN GENERAL DE VALORES AUSENTES")
print("=" * 80)

n_total = len(df)
n_ausentes = df["IMD_ausente"].sum()
pct_ausentes = df["IMD_ausente"].mean() * 100

print(f"Registros totales: {n_total:,}")
print(f"Valores ausentes en Valor_IMD: {n_ausentes:,}")
print(f"Porcentaje de ausencia: {pct_ausentes:.2f}%")

# ------------------------------------------------------------------------------
# 2. Distribución de ausentes por año
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. DISTRIBUCIÓN DE AUSENTES POR AÑO")
print("=" * 80)

ausentes_ano = (
    df.groupby("Any")["IMD_ausente"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

ausentes_ano.columns = [
    "Año",
    "Registros",
    "Ausentes",
    "Porcentaje_ausentes"
]

ausentes_ano["Porcentaje_ausentes"] = (
    ausentes_ano["Porcentaje_ausentes"] * 100
).round(2)

display(ausentes_ano)

# ------------------------------------------------------------------------------
# 3. Distribución de ausentes por mes
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. DISTRIBUCIÓN DE AUSENTES POR MES")
print("=" * 80)

ausentes_mes = (
    df.groupby("Mes")["IMD_ausente"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

ausentes_mes.columns = [
    "Mes",
    "Registros",
    "Ausentes",
    "Porcentaje_ausentes"
]

ausentes_mes["Porcentaje_ausentes"] = (
    ausentes_mes["Porcentaje_ausentes"] * 100
).round(2)

display(ausentes_mes)

# ------------------------------------------------------------------------------
# 4. Distribución de ausentes por tipo de día
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. DISTRIBUCIÓN DE AUSENTES POR TIPO DE DÍA")
print("=" * 80)

ausentes_tipo_dia = (
    df.groupby(
        ["Codi_tipus_dia", "Desc_tipus_dia"],
        observed=True
    )["IMD_ausente"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

ausentes_tipo_dia.columns = [
    "Codigo",
    "Tipo_dia",
    "Registros",
    "Ausentes",
    "Porcentaje_ausentes"
]

ausentes_tipo_dia["Porcentaje_ausentes"] = (
    ausentes_tipo_dia["Porcentaje_ausentes"] * 100
).round(2)

display(ausentes_tipo_dia)

# ------------------------------------------------------------------------------
# 5. Cobertura por punto de aforo
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. COBERTURA POR PUNTO DE AFORO")
print("=" * 80)

cobertura_aforos = (
    df.groupby("Id_aforament")["Valor_IMD"]
    .agg(
        Registros="size",
        Disponibles="count"
    )
    .reset_index()
)

cobertura_aforos["Ausentes"] = (
    cobertura_aforos["Registros"] -
    cobertura_aforos["Disponibles"]
)

cobertura_aforos["Cobertura_pct"] = (
    cobertura_aforos["Disponibles"] /
    cobertura_aforos["Registros"] * 100
).round(2)

cobertura_aforos = cobertura_aforos.sort_values(
    "Cobertura_pct"
).reset_index(drop=True)

print(f"Puntos de aforo analizados: {len(cobertura_aforos):,}")

print("\nPuntos con menor cobertura:")
display(cobertura_aforos.head(20))

# ------------------------------------------------------------------------------
# 6. Clasificación de los puntos según su cobertura
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. DISTRIBUCIÓN DE LOS PUNTOS SEGÚN SU COBERTURA")
print("=" * 80)

intervalos = [-0.01, 25, 50, 75, 90, 99.999, 100.01]

etiquetas = [
    "≤25%",
    "25–50%",
    "50–75%",
    "75–90%",
    "90–<100%",
    "100%"
]

cobertura_aforos["Grupo_cobertura"] = pd.cut(
    cobertura_aforos["Cobertura_pct"],
    bins=intervalos,
    labels=etiquetas,
    include_lowest=True
)

resumen_cobertura = (
    cobertura_aforos["Grupo_cobertura"]
    .value_counts(sort=False)
    .rename_axis("Cobertura")
    .reset_index(name="Puntos_aforo")
)

display(resumen_cobertura)

# ------------------------------------------------------------------------------
# 7. Comprobación final
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. COMPROBACIÓN FINAL")
print("=" * 80)

print(
    f"Total de ausentes contabilizados: "
    f"{int(ausentes_ano['Ausentes'].sum()):,}"
)

print(
    f"Coincide con los NaN originales: "
    f"{int(ausentes_ano['Ausentes'].sum()) == n_ausentes}"
)

1. RESUMEN GENERAL DE VALORES AUSENTES
Registros totales: 313,795
Valores ausentes en Valor_IMD: 37,615
Porcentaje de ausencia: 11.99%

2. DISTRIBUCIÓN DE AUSENTES POR AÑO


,Año,Registros,Ausentes,Porcentaje_ausentes
0,2018,36780,7799,21.20
1,2019,41100,6175,15.02
2,2020,42840,3575,8.35
3,2021,43440,3783,8.71
4,2022,49435,6023,12.18
5,2023,49920,4695,9.41
6,2024,50280,5565,11.07



3. DISTRIBUCIÓN DE AUSENTES POR MES


,Mes,Registros,Ausentes,Porcentaje_ausentes
0,1,26150,4596,17.58
1,2,26150,4286,16.39
2,3,26150,3623,13.85
3,4,26150,3551,13.58
4,5,26150,3002,11.48
5,6,26145,3231,12.36
6,7,26150,2672,10.22
7,8,26150,2451,9.37
8,9,26150,2581,9.87
9,10,26150,2475,9.46



4. DISTRIBUCIÓN DE AUSENTES POR TIPO DE DÍA


...


5. COBERTURA POR PUNTO DE AFORO
Puntos de aforo analizados: 922

Puntos con menor cobertura:


,Id_aforament,Registros,Disponibles,Ausentes,Cobertura_pct
0,12002,60,5,55,8.33
1,10007,180,25,155,13.89
2,20446,60,10,50,16.67
3,20445,60,10,50,16.67
4,20443,60,15,45,25.00
5,20444,60,15,45,25.00
6,7040,60,15,45,25.00
7,4104,180,50,130,27.78
8,20433,60,20,40,33.33
9,13005,60,20,40,33.33



6. DISTRIBUCIÓN DE LOS PUNTOS SEGÚN SU COBERTURA


,Cobertura,Puntos_aforo
0,≤25%,7
1,25–50%,34
2,50–75%,145
3,75–90%,296
4,90–<100%,370
5,100%,70



7. COMPROBACIÓN FINAL
Total de ausentes contabilizados: 37,615
Coincide con los NaN originales: True


### Resultado de la auditoría de cobertura

El análisis identifica **37.615 valores ausentes de `Valor_IMD`**, equivalentes al **11,99 %** de los 313.795 registros del dataset.

La distribución temporal de las ausencias no es homogénea. Los mayores porcentajes se concentran en **2018 (21,20 %) y 2019 (15,02 %)**, mientras que durante el periodo 2020–2024 los porcentajes se sitúan aproximadamente entre el 8 % y el 12 %. También se observa una mayor proporción de valores ausentes durante los primeros meses del año, especialmente enero y febrero.

Por el contrario, la ausencia de información es prácticamente idéntica entre las cinco categorías de tipo de día, con porcentajes próximos al **12 %** en todos los casos. Por tanto, no se identifica un sesgo relevante asociado a esta variable.

El análisis por punto de aforo muestra una mayor heterogeneidad. De los **922 puntos analizados**, únicamente 70 presentan una cobertura completa del 100 %, mientras que existen estaciones con una disponibilidad de datos considerablemente inferior.

La distribución de la cobertura es la siguiente:

- 7 puntos con cobertura ≤ 25 %.
- 34 puntos con cobertura entre 25–50 %.
- 145 puntos con cobertura entre 50–75 %.
- 296 puntos con cobertura entre 75–90 %.
- 370 puntos con cobertura entre 90–<100 %.
- 70 puntos con cobertura completa.

Estos resultados indican que los valores ausentes no deben imputarse de forma indiscriminada. Antes de aplicar un procedimiento de imputación será necesario establecer un criterio mínimo de cobertura por punto de aforo y distinguir entre ausencias puntuales potencialmente recuperables y series con información insuficiente.

## 3.7 Análisis de continuidad temporal y periodo de actividad de los puntos de aforo

Antes de reconstruir la serie diaria de tráfico, se analiza la continuidad temporal de cada punto de aforo con el objetivo de identificar el periodo en el que dispone realmente de información.

Este paso permite distinguir entre valores ausentes dentro de una serie activa y periodos anteriores o posteriores a la existencia del punto de medida. De este modo, se evita interpretar como datos faltantes aquellos años o meses en los que un determinado aforo todavía no formaba parte de la red disponible.

Para cada `Id_aforament` se calcula el primer y último año con registros, el número de años presentes, el número total de combinaciones año-mes disponibles y la cobertura de `Valor_IMD` dentro de su periodo observado.

In [ ]:
# ==============================================================================
# 3.7 ANÁLISIS DE CONTINUIDAD TEMPORAL Y PERIODO DE ACTIVIDAD
#     DE LOS PUNTOS DE AFORO
# ==============================================================================

import pandas as pd
import numpy as np

df = df_aforos_total.copy()

# ------------------------------------------------------------------------------
# 1. Resumen temporal por punto de aforo
# ------------------------------------------------------------------------------

resumen_actividad = (
    df.groupby("Id_aforament")
    .agg(
        Anio_inicio=("Any", "min"),
        Anio_fin=("Any", "max"),
        Anios_presentes=("Any", "nunique"),
        Meses_presentes=("Mes", "nunique"),
        Registros_totales=("Valor_IMD", "size"),
        Registros_disponibles=("Valor_IMD", "count")
    )
    .reset_index()
)

# Número de años teóricos entre inicio y fin
resumen_actividad["Anios_intervalo"] = (
    resumen_actividad["Anio_fin"]
    - resumen_actividad["Anio_inicio"]
    + 1
)

# Detectar posibles discontinuidades entre año inicial y final
resumen_actividad["Anios_faltantes_intervalo"] = (
    resumen_actividad["Anios_intervalo"]
    - resumen_actividad["Anios_presentes"]
)

# Valores ausentes dentro del periodo observado
resumen_actividad["Registros_ausentes"] = (
    resumen_actividad["Registros_totales"]
    - resumen_actividad["Registros_disponibles"]
)

# Cobertura de IMD
resumen_actividad["Cobertura_IMD_pct"] = (
    resumen_actividad["Registros_disponibles"]
    / resumen_actividad["Registros_totales"]
    * 100
).round(2)

# ------------------------------------------------------------------------------
# 2. Resumen general
# ------------------------------------------------------------------------------

print("=" * 90)
print("1. RESUMEN DEL PERIODO DE ACTIVIDAD DE LOS PUNTOS DE AFORO")
print("=" * 90)

print(
    f"Puntos de aforo analizados: "
    f"{len(resumen_actividad):,}"
)

print(
    f"Puntos presentes desde 2018: "
    f"{(resumen_actividad['Anio_inicio'] == 2018).sum():,}"
)

print(
    f"Puntos presentes hasta 2024: "
    f"{(resumen_actividad['Anio_fin'] == 2024).sum():,}"
)

print(
    f"Puntos presentes durante los 7 años: "
    f"{(resumen_actividad['Anios_presentes'] == 7).sum():,}"
)

# ------------------------------------------------------------------------------
# 3. Distribución según año de incorporación
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("2. AÑO DE INCORPORACIÓN DE LOS PUNTOS")
print("=" * 90)

inicio_por_ano = (
    resumen_actividad["Anio_inicio"]
    .value_counts()
    .sort_index()
    .rename_axis("Anio_inicio")
    .reset_index(name="Puntos_aforo")
)

display(inicio_por_ano)

# ------------------------------------------------------------------------------
# 4. Distribución según último año disponible
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. ÚLTIMO AÑO CON REGISTROS")
print("=" * 90)

fin_por_ano = (
    resumen_actividad["Anio_fin"]
    .value_counts()
    .sort_index()
    .rename_axis("Anio_fin")
    .reset_index(name="Puntos_aforo")
)

display(fin_por_ano)

# ------------------------------------------------------------------------------
# 5. Número de años presentes por punto
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("4. NÚMERO DE AÑOS PRESENTES POR PUNTO")
print("=" * 90)

distribucion_anios = (
    resumen_actividad["Anios_presentes"]
    .value_counts()
    .sort_index()
    .rename_axis("Anios_presentes")
    .reset_index(name="Puntos_aforo")
)

display(distribucion_anios)

# ------------------------------------------------------------------------------
# 6. Detección de discontinuidades internas
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("5. DISCONTINUIDADES TEMPORALES INTERNAS")
print("=" * 90)

puntos_discontinuos = resumen_actividad[
    resumen_actividad["Anios_faltantes_intervalo"] > 0
].copy()

print(
    f"Puntos con años intermedios ausentes: "
    f"{len(puntos_discontinuos):,}"
)

if len(puntos_discontinuos) > 0:

    display(
        puntos_discontinuos
        .sort_values(
            ["Anios_faltantes_intervalo", "Cobertura_IMD_pct"],
            ascending=[False, True]
        )
        .head(30)
    )

# ------------------------------------------------------------------------------
# 7. Cobertura detallada por punto
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("6. PUNTOS CON MENOR COBERTURA DENTRO DE SU PERIODO OBSERVADO")
print("=" * 90)

display(
    resumen_actividad
    .sort_values("Cobertura_IMD_pct")
    .head(30)
)

# ------------------------------------------------------------------------------
# 8. Matriz de presencia anual
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("7. MATRIZ DE PRESENCIA ANUAL")
print("=" * 90)

presencia_anual = (
    df.assign(Presente=1)
    .pivot_table(
        index="Id_aforament",
        columns="Any",
        values="Presente",
        aggfunc="max",
        fill_value=0
    )
)

display(presencia_anual.head(20))

# ------------------------------------------------------------------------------
# 9. Resumen final
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN FINAL DE CONTINUIDAD")
print("=" * 90)

print(
    f"Puntos totales: "
    f"{len(resumen_actividad):,}"
)

print(
    f"Puntos con los 7 años presentes: "
    f"{(resumen_actividad['Anios_presentes'] == 7).sum():,}"
)

print(
    f"Puntos con discontinuidades internas: "
    f"{len(puntos_discontinuos):,}"
)

print(
    "\nEsta auditoría permitirá construir posteriormente la serie diaria "
    "únicamente dentro del periodo real de actividad de cada punto."
)

1. RESUMEN DEL PERIODO DE ACTIVIDAD DE LOS PUNTOS DE AFORO
Puntos de aforo analizados: 922
Puntos presentes desde 2018: 613
Puntos presentes hasta 2024: 838
Puntos presentes durante los 7 años: 529

2. AÑO DE INCORPORACIÓN DE LOS PUNTOS


,Anio_inicio,Puntos_aforo
0,2018,613
1,2019,95
2,2020,19
3,2021,21
4,2022,96
5,2023,30
6,2024,48



3. ÚLTIMO AÑO CON REGISTROS


,Anio_fin,Puntos_aforo
0,2018,5
1,2019,1
2,2020,6
3,2021,4
4,2022,20
5,2023,48
6,2024,838



4. NÚMERO DE AÑOS PRESENTES POR PUNTO


...


6. PUNTOS CON MENOR COBERTURA DENTRO DE SU PERIODO OBSERVADO


,Id_aforament,Anio_inicio,Anio_fin,Anios_presentes,Meses_presentes,Registros_totales,Registros_disponibles,Anios_intervalo,Anios_faltantes_intervalo,Registros_ausentes,Cobertura_IMD_pct
83,12002,2024,2024,1,12,60,5,1,0,55,8.33
5,10007,2018,2024,3,12,180,25,7,4,155,13.89
566,20446,2024,2024,1,12,60,10,1,0,50,16.67
565,20445,2024,2024,1,12,60,10,1,0,50,16.67
563,20443,2024,2024,1,12,60,15,1,0,45,25.00
564,20444,2024,2024,1,12,60,15,1,0,45,25.00
858,7040,2024,2024,1,12,60,15,1,0,45,25.00
721,4104,2022,2024,3,12,180,50,3,0,130,27.78
553,20433,2024,2024,1,12,60,20,1,0,40,33.33
92,13005,2018,2018,1,12,60,20,1,0,40,33.33



7. MATRIZ DE PRESENCIA ANUAL


Any,2018,2019,2020,2021,2022,2023,2024
Id_aforament,,,,,,,
10-SMD-1,1,1,1,1,1,1,1
10-SMD-2,1,1,1,1,1,1,1
10001,1,1,1,1,1,1,1
10002,1,1,1,1,1,1,1
10005,1,1,1,1,1,1,1
10007,1,0,0,0,0,1,1
10008,1,1,1,1,1,1,1
10009,1,1,1,1,1,1,1
1001,1,1,1,1,1,1,1



RESUMEN FINAL DE CONTINUIDAD
Puntos totales: 922
Puntos con los 7 años presentes: 529
Puntos con discontinuidades internas: 36

Esta auditoría permitirá construir posteriormente la serie diaria únicamente dentro del periodo real de actividad de cada punto.


### Resultado del análisis de continuidad temporal

La red de aforos presenta una composición variable a lo largo del periodo 2018–2024. De los **922 puntos de aforo** identificados, **613 están presentes desde 2018**, mientras que el resto se incorpora progresivamente durante los años posteriores. En 2024 permanecen representados **838 puntos**, y únicamente **529 puntos disponen de registros en los siete años analizados**.

La auditoría identifica además **36 puntos con discontinuidades internas**, es decir, puntos presentes en años no consecutivos dentro del periodo de estudio. Por tanto, el intervalo comprendido entre el primer y el último año de aparición de un aforo no puede interpretarse automáticamente como un periodo continuo de funcionamiento.

Asimismo, existen diferencias importantes en la cobertura de `Valor_IMD` dentro de los periodos observados, incluyendo algunos puntos con una disponibilidad de información muy reducida.

En consecuencia, para la posterior reconstrucción de las series diarias se conservarán únicamente las combinaciones `Id_aforament`–año realmente presentes en los datos originales. Los años en los que un punto no aparece en la fuente no serán considerados valores ausentes ni serán objeto de imputación.

Este criterio permite diferenciar la evolución real de la red de medida de los valores faltantes existentes dentro de los periodos efectivamente observados y evita generar artificialmente información para puntos de aforo sin cobertura documental.

## 3.8 Reconstrucción de la serie diaria de tráfico a partir de la IMD mensual por tipo de día

Una vez analizada la continuidad temporal de los puntos de aforo, se reconstruye una serie diaria de tráfico a partir de los valores mensuales de Intensidad Media Diaria (`Valor_IMD`) disponibles para cada tipo de día.

La reconstrucción se realiza exclusivamente para las combinaciones `Id_aforament`–año realmente presentes en la fuente original, evitando generar observaciones para periodos en los que un determinado punto de aforo no formaba parte de la red disponible.

Para cada fecha se asigna el código de tipo de día correspondiente según el calendario real:

- lunes → código 1;
- martes, miércoles y jueves → código 2;
- viernes → código 3;
- sábado → código 4;
- domingo → código 5.

Posteriormente, cada fecha se cruza con el valor de `Valor_IMD` correspondiente a su año, mes, punto de aforo y tipo de día. De este modo se obtiene una serie diaria coherente con la estructura temporal de la fuente, sin introducir interpolaciones artificiales entre categorías de día.

In [ ]:
# ==============================================================================
# 3.8 RECONSTRUCCIÓN DE LA SERIE DIARIA DE TRÁFICO
# ==============================================================================

import pandas as pd
import numpy as np

df = df_aforos_total.copy()

# ------------------------------------------------------------------------------
# 1. Identificar combinaciones Id_aforament - año realmente presentes
# ------------------------------------------------------------------------------

pares_activos = (
    df[["Id_aforament", "Any"]]
    .drop_duplicates()
    .sort_values(["Id_aforament", "Any"])
    .reset_index(drop=True)
)

print("=" * 90)
print("1. COMBINACIONES PUNTO-AÑO ACTIVAS")
print("=" * 90)

print(
    f"Combinaciones Id_aforament-año presentes: "
    f"{len(pares_activos):,}"
)

# ------------------------------------------------------------------------------
# 2. Construir calendario diario únicamente para los años activos
# ------------------------------------------------------------------------------

bloques_diarios = []

for _, fila in pares_activos.iterrows():

    id_aforo = fila["Id_aforament"]
    anio = int(fila["Any"])

    fechas = pd.date_range(
        start=f"{anio}-01-01",
        end=f"{anio}-12-31",
        freq="D"
    )

    temp = pd.DataFrame({
        "Fecha": fechas,
        "Id_aforament": id_aforo,
        "Any": anio
    })

    bloques_diarios.append(temp)

df_diario_base = pd.concat(
    bloques_diarios,
    ignore_index=True
)

# ------------------------------------------------------------------------------
# 3. Variables temporales
# ------------------------------------------------------------------------------

df_diario_base["Mes"] = (
    df_diario_base["Fecha"].dt.month
)

df_diario_base["Dia_semana"] = (
    df_diario_base["Fecha"].dt.dayofweek
)

# ------------------------------------------------------------------------------
# 4. Asignar código de tipo de día
# ------------------------------------------------------------------------------

# Python:
# 0 = lunes
# 1 = martes
# 2 = miércoles
# 3 = jueves
# 4 = viernes
# 5 = sábado
# 6 = domingo

def asignar_codigo_tipo_dia(dia_semana):

    if dia_semana == 0:
        return 1

    elif dia_semana in [1, 2, 3]:
        return 2

    elif dia_semana == 4:
        return 3

    elif dia_semana == 5:
        return 4

    elif dia_semana == 6:
        return 5


df_diario_base["Codi_tipus_dia"] = (
    df_diario_base["Dia_semana"]
    .apply(asignar_codigo_tipo_dia)
)

# ------------------------------------------------------------------------------
# 5. Descripción normalizada
# ------------------------------------------------------------------------------

mapeo_dias = {
    1: "lunes",
    2: "laborable",
    3: "viernes",
    4: "sábado",
    5: "domingo"
}

df_diario_base["Desc_tipus_dia"] = (
    df_diario_base["Codi_tipus_dia"]
    .map(mapeo_dias)
)

# ------------------------------------------------------------------------------
# 6. Preparar tabla mensual original para el cruce
# ------------------------------------------------------------------------------

df_imd_lookup = (
    df[
        [
            "Any",
            "Mes",
            "Id_aforament",
            "Codi_tipus_dia",
            "Valor_IMD"
        ]
    ]
    .copy()
)

# Comprobar previamente si existen duplicados
duplicados_lookup = (
    df_imd_lookup
    .duplicated(
        subset=[
            "Any",
            "Mes",
            "Id_aforament",
            "Codi_tipus_dia"
        ]
    )
    .sum()
)

print("\n" + "=" * 90)
print("2. CONTROL PREVIO AL MERGE")
print("=" * 90)

print(
    f"Duplicados en la clave "
    f"Año-Mes-Aforo-TipoDia: {duplicados_lookup:,}"
)

# ------------------------------------------------------------------------------
# 7. Cruce con Valor_IMD
# ------------------------------------------------------------------------------

df_aforos_diario = pd.merge(
    df_diario_base,
    df_imd_lookup,
    on=[
        "Any",
        "Mes",
        "Id_aforament",
        "Codi_tipus_dia"
    ],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------------------
# 8. Eliminar variable auxiliar
# ------------------------------------------------------------------------------

df_aforos_diario = (
    df_aforos_diario
    .drop(columns=["Dia_semana"])
    .sort_values(
        ["Fecha", "Id_aforament"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# 9. Auditoría inicial del resultado
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("3. RESULTADO DE LA RECONSTRUCCIÓN DIARIA")
print("=" * 90)

print(
    f"Dimensiones del dataset diario: "
    f"{df_aforos_diario.shape}"
)

print(
    f"Periodo: "
    f"{df_aforos_diario['Fecha'].min().date()} - "
    f"{df_aforos_diario['Fecha'].max().date()}"
)

print(
    f"Puntos de aforo: "
    f"{df_aforos_diario['Id_aforament'].nunique():,}"
)

print(
    f"Valores IMD ausentes: "
    f"{df_aforos_diario['Valor_IMD'].isna().sum():,}"
)

print(
    f"Porcentaje de IMD ausente: "
    f"{df_aforos_diario['Valor_IMD'].isna().mean() * 100:.2f}%"
)

print("\nPrimeras observaciones:")

display(
    df_aforos_diario.head(10)
)

1. COMBINACIONES PUNTO-AÑO ACTIVAS
Combinaciones Id_aforament-año presentes: 5,230

2. CONTROL PREVIO AL MERGE
Duplicados en la clave Año-Mes-Aforo-TipoDia: 0

3. RESULTADO DE LA RECONSTRUCCIÓN DIARIA
Dimensiones del dataset diario: (1910502, 7)
Periodo: 2018-01-01 - 2024-12-31
Puntos de aforo: 922
Valores IMD ausentes: 228,436
Porcentaje de IMD ausente: 11.96%

Primeras observaciones:


,Fecha,Id_aforament,Any,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD
0,2018-01-01,10-SMD-1,2018,1,1,lunes,76505.0
1,2018-01-01,10-SMD-2,2018,1,1,lunes,69354.0
2,2018-01-01,10001,2018,1,1,lunes,8625.0
3,2018-01-01,10002,2018,1,1,lunes,13777.0
4,2018-01-01,10005,2018,1,1,lunes,14675.0
5,2018-01-01,10007,2018,1,1,lunes,2116.0
6,2018-01-01,10008,2018,1,1,lunes,3479.0
7,2018-01-01,10009,2018,1,1,lunes,7310.0
8,2018-01-01,1001,2018,1,1,lunes,34153.0
9,2018-01-01,10010,2018,1,1,lunes,10718.0


In [ ]:
# 1. Auditoría de Valores Nulos
total_filas = len(df_aforos_diario)
nulos_total = df_aforos_diario['Valor_IMD'].isnull().sum()
pct_nulos = (nulos_total / total_filas) * 100

print(f"--- AUDITORÍA DE NULOS ---")
print(f"Total de registros diarios: {total_filas:,}")
print(f"Registros nulos en Valor_IMD: {nulos_total:,} ({pct_nulos:.2f}%)\n")

# 2. Estadísticos Descriptivos
print(f"--- ESTADÍSTICOS DESCRIPTIVOS (Valor_IMD) ---")
print(df_aforos_diario['Valor_IMD'].describe())

# 3. Detección de Outliers usando el rango intercuartílico (IQR)
Q1 = df_aforos_diario['Valor_IMD'].quantile(0.25)
Q3 = df_aforos_diario['Valor_IMD'].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_aforos_diario[(df_aforos_diario['Valor_IMD'] < limite_inferior) | (df_aforos_diario['Valor_IMD'] > limite_superior)]

print(f"\n--- DETECCIÓN DE OUTLIERS (Método IQR) ---")
print(f"Límite inferior teórico: {limite_inferior:.2f}")
print(f"Límite superior teórico: {limite_superior:.2f}")
print(f"Número de outliers detectados: {len(outliers):,} ({(len(outliers)/total_filas)*100:.2f}%)")

# Opcional: Mostrar los valores más altos extremos por si son autopistas muy transitadas
print("\nTop 5 valores más altos de IMD registrados:")
display(df_aforos_diario.nlargest(5, 'Valor_IMD')[['Fecha', 'Id_aforament', 'Desc_tipus_dia', 'Valor_IMD']])

### Resultado de la reconstrucción diaria

La reconstrucción temporal se realizó sobre **5.230 combinaciones `Id_aforament`–año realmente presentes en la fuente original**, evitando generar observaciones para periodos en los que un determinado punto de aforo no disponía de registros.

No se detectaron duplicados en la clave formada por año, mes, punto de aforo y tipo de día, lo que garantiza una correspondencia unívoca entre cada fecha reconstruida y su valor de Intensidad Media Diaria.

El proceso genera un dataset diario de **1.910.502 observaciones**, correspondiente al periodo **2018–2024** y a **922 puntos de aforo**.

La proporción de valores ausentes de `Valor_IMD` en el dataset diario es del **11,96 %**, prácticamente idéntica a la observada en los datos originales agregados. Esto indica que la desagregación temporal no introduce nuevas ausencias de forma relevante, sino que conserva y propaga los patrones de disponibilidad existentes en la fuente original.

Cada fecha queda asociada al valor mensual de IMD correspondiente a su punto de aforo y a la categoría de día definida por el calendario real, permitiendo disponer de una serie diaria compatible con la escala temporal utilizada en el resto del estudio.

## 3.9 Auditoría del dataset diario reconstruido

Antes de aplicar cualquier procedimiento de imputación sobre la serie diaria de tráfico, se realiza una auditoría específica del dataset reconstruido.

El objetivo es comprobar que la desagregación temporal conserva la estructura original de la fuente, que no se han generado duplicados por fecha y punto de aforo, que la asignación del tipo de día es coherente con el calendario y que los valores ausentes mantienen patrones compatibles con los observados en los datos agregados originales.

Asimismo, se analiza la cobertura diaria por año y por punto de aforo para distinguir entre huecos puntuales potencialmente imputables y series con disponibilidad insuficiente.

In [ ]:
# ==============================================================================
# 3.9 AUDITORÍA DEL DATASET DIARIO RECONSTRUIDO
# ==============================================================================

import pandas as pd
import numpy as np

df = df_aforos_diario.copy()

print("=" * 90)
print("3.9 AUDITORÍA DEL DATASET DIARIO RECONSTRUIDO")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Dimensiones y cobertura temporal
# ------------------------------------------------------------------------------

print("\n1. DIMENSIONES Y COBERTURA TEMPORAL")
print("-" * 90)

print(f"Dimensiones: {df.shape}")
print(f"Fecha inicial: {df['Fecha'].min().date()}")
print(f"Fecha final: {df['Fecha'].max().date()}")
print(f"Fechas únicas: {df['Fecha'].nunique():,}")
print(f"Puntos de aforo: {df['Id_aforament'].nunique():,}")

# ------------------------------------------------------------------------------
# 2. Duplicados Fecha + Id_aforament
# ------------------------------------------------------------------------------

print("\n2. DUPLICADOS FECHA + PUNTO DE AFORO")
print("-" * 90)

duplicados = df.duplicated(
    subset=["Fecha", "Id_aforament"]
).sum()

print(f"Duplicados encontrados: {duplicados:,}")

# ------------------------------------------------------------------------------
# 3. Comprobación del tipo de día
# ------------------------------------------------------------------------------

print("\n3. COMPROBACIÓN DEL TIPO DE DÍA")
print("-" * 90)

def codigo_esperado(fecha):
    dia = fecha.dayofweek

    if dia == 0:
        return 1
    elif dia in [1, 2, 3]:
        return 2
    elif dia == 4:
        return 3
    elif dia == 5:
        return 4
    else:
        return 5

codigo_calendario = df["Fecha"].apply(codigo_esperado)

errores_tipo_dia = (
    codigo_calendario != df["Codi_tipus_dia"]
).sum()

print(
    f"Incoherencias entre calendario y Codi_tipus_dia: "
    f"{errores_tipo_dia:,}"
)

# ------------------------------------------------------------------------------
# 4. Valores ausentes generales
# ------------------------------------------------------------------------------

print("\n4. VALORES AUSENTES DE Valor_IMD")
print("-" * 90)

n_ausentes = df["Valor_IMD"].isna().sum()
pct_ausentes = df["Valor_IMD"].isna().mean() * 100

print(f"Valores ausentes: {n_ausentes:,}")
print(f"Porcentaje de ausencia: {pct_ausentes:.2f}%")

# ------------------------------------------------------------------------------
# 5. Ausentes por año
# ------------------------------------------------------------------------------

print("\n5. AUSENTES POR AÑO")
print("-" * 90)

ausentes_ano = (
    df.assign(IMD_ausente=df["Valor_IMD"].isna())
    .groupby("Any")["IMD_ausente"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

ausentes_ano.columns = [
    "Año",
    "Registros",
    "Ausentes",
    "Porcentaje_ausentes"
]

ausentes_ano["Porcentaje_ausentes"] = (
    ausentes_ano["Porcentaje_ausentes"] * 100
).round(2)

display(ausentes_ano)

# ------------------------------------------------------------------------------
# 6. Ausentes por tipo de día
# ------------------------------------------------------------------------------

print("\n6. AUSENTES POR TIPO DE DÍA")
print("-" * 90)

ausentes_tipo = (
    df.assign(IMD_ausente=df["Valor_IMD"].isna())
    .groupby(
        ["Codi_tipus_dia", "Desc_tipus_dia"],
        observed=True
    )["IMD_ausente"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

ausentes_tipo.columns = [
    "Codigo",
    "Tipo_dia",
    "Registros",
    "Ausentes",
    "Porcentaje_ausentes"
]

ausentes_tipo["Porcentaje_ausentes"] = (
    ausentes_tipo["Porcentaje_ausentes"] * 100
).round(2)

display(ausentes_tipo)

# ------------------------------------------------------------------------------
# 7. Cobertura diaria por punto
# ------------------------------------------------------------------------------

print("\n7. COBERTURA DIARIA POR PUNTO DE AFORO")
print("-" * 90)

cobertura_diaria = (
    df.groupby("Id_aforament")["Valor_IMD"]
    .agg(
        Registros="size",
        Disponibles="count"
    )
    .reset_index()
)

cobertura_diaria["Ausentes"] = (
    cobertura_diaria["Registros"]
    - cobertura_diaria["Disponibles"]
)

cobertura_diaria["Cobertura_pct"] = (
    cobertura_diaria["Disponibles"]
    / cobertura_diaria["Registros"]
    * 100
).round(2)

display(
    cobertura_diaria
    .sort_values("Cobertura_pct")
    .head(30)
)

# ------------------------------------------------------------------------------
# 8. Cobertura por punto y año
# ------------------------------------------------------------------------------

print("\n8. COBERTURA POR PUNTO Y AÑO")
print("-" * 90)

cobertura_punto_ano = (
    df.groupby(
        ["Id_aforament", "Any"]
    )["Valor_IMD"]
    .agg(
        Registros="size",
        Disponibles="count"
    )
    .reset_index()
)

cobertura_punto_ano["Ausentes"] = (
    cobertura_punto_ano["Registros"]
    - cobertura_punto_ano["Disponibles"]
)

cobertura_punto_ano["Cobertura_pct"] = (
    cobertura_punto_ano["Disponibles"]
    / cobertura_punto_ano["Registros"]
    * 100
).round(2)

print(
    "Combinaciones punto-año con cobertura inferior al 50 %:"
)

display(
    cobertura_punto_ano[
        cobertura_punto_ano["Cobertura_pct"] < 50
    ]
    .sort_values("Cobertura_pct")
    .head(50)
)

# ------------------------------------------------------------------------------
# 9. Distribución de cobertura punto-año
# ------------------------------------------------------------------------------

print("\n9. DISTRIBUCIÓN DE COBERTURA PUNTO-AÑO")
print("-" * 90)

bins = [-0.01, 25, 50, 75, 90, 99.999, 100.01]

labels = [
    "≤25%",
    "25–50%",
    "50–75%",
    "75–90%",
    "90–<100%",
    "100%"
]

cobertura_punto_ano["Grupo_cobertura"] = pd.cut(
    cobertura_punto_ano["Cobertura_pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

resumen_cobertura = (
    cobertura_punto_ano["Grupo_cobertura"]
    .value_counts(sort=False)
    .rename_axis("Cobertura")
    .reset_index(name="Combinaciones_punto_año")
)

display(resumen_cobertura)

# ------------------------------------------------------------------------------
# 10. Resumen final
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN FINAL DE LA AUDITORÍA")
print("=" * 90)

print(f"Duplicados Fecha-Aforo: {duplicados:,}")
print(f"Incoherencias de tipo de día: {errores_tipo_dia:,}")
print(f"Valores IMD ausentes: {n_ausentes:,}")
print(f"Porcentaje de ausencia: {pct_ausentes:.2f}%")

3.9 AUDITORÍA DEL DATASET DIARIO RECONSTRUIDO

1. DIMENSIONES Y COBERTURA TEMPORAL
------------------------------------------------------------------------------------------
Dimensiones: (1910502, 7)
Fecha inicial: 2018-01-01
Fecha final: 2024-12-31
Fechas únicas: 2,557
Puntos de aforo: 922

2. DUPLICADOS FECHA + PUNTO DE AFORO
------------------------------------------------------------------------------------------
Duplicados encontrados: 0

3. COMPROBACIÓN DEL TIPO DE DÍA
------------------------------------------------------------------------------------------
Incoherencias entre calendario y Codi_tipus_dia: 0

4. VALORES AUSENTES DE Valor_IMD
------------------------------------------------------------------------------------------
Valores ausentes: 228,436
Porcentaje de ausencia: 11.96%

5. AUSENTES POR AÑO
------------------------------------------------------------------------------------------


,Año,Registros,Ausentes,Porcentaje_ausentes
0,2018,223745,47312,21.15
1,2019,250025,37398,14.96
2,2020,261324,21752,8.32
3,2021,264260,22955,8.69
4,2022,300760,36528,12.15
5,2023,303680,28573,9.41
6,2024,306708,33918,11.06



6. AUSENTES POR TIPO DE DÍA
------------------------------------------------------------------------------------------


,Codigo,Tipo_dia,Registros,Ausentes,Porcentaje_ausentes
0,1,lunes,273411,32663,11.95
1,2,laborable,818831,97982,11.97
2,3,viernes,272684,32556,11.94
3,4,sábado,272784,32670,11.98
4,5,domingo,272792,32565,11.94



7. COBERTURA DIARIA POR PUNTO DE AFORO
------------------------------------------------------------------------------------------


...


8. COBERTURA POR PUNTO Y AÑO
------------------------------------------------------------------------------------------
Combinaciones punto-año con cobertura inferior al 50 %:


,Id_aforament,Any,Registros,Disponibles,Ausentes,Cobertura_pct
4500,6026,2018,365,0,365,0.00
1936,20162,2018,365,27,338,7.40
3346,4004,2024,366,29,337,7.92
3597,4042,2024,366,30,336,8.20
163,1008,2020,366,30,336,8.20
3162,3021,2019,365,30,335,8.22
1485,2010,2021,365,30,335,8.22
468,11024,2024,366,31,335,8.47
1268,20067,2024,366,31,335,8.47
3769,4069,2024,366,31,335,8.47



9. DISTRIBUCIÓN DE COBERTURA PUNTO-AÑO
------------------------------------------------------------------------------------------


,Cobertura,Combinaciones_punto_año
0,≤25%,151
1,25–50%,247
2,50–75%,524
3,75–90%,520
4,90–<100%,571
5,100%,3217



RESUMEN FINAL DE LA AUDITORÍA
Duplicados Fecha-Aforo: 0
Incoherencias de tipo de día: 0
Valores IMD ausentes: 228,436
Porcentaje de ausencia: 11.96%


### Resultado de la auditoría del dataset diario

La auditoría confirma la consistencia estructural del dataset diario reconstruido, compuesto por **1.910.502 observaciones**, **922 puntos de aforo** y **2.557 fechas** correspondientes al periodo 2018–2024.

No se detectaron duplicados para la combinación `Fecha`–`Id_aforament` ni incoherencias entre el calendario real y el código de tipo de día asignado durante la reconstrucción.

El dataset presenta **228.436 valores ausentes de `Valor_IMD` (11,96 %)**. La proporción de ausencia es prácticamente idéntica entre las diferentes categorías de día, por lo que no se observa un patrón sistemático de datos faltantes asociado al día de la semana.

La disponibilidad presenta, sin embargo, diferencias relevantes entre años y entre puntos de aforo. De las **5.230 combinaciones punto-año** analizadas, **3.217 presentan cobertura completa**, mientras que existen combinaciones con información parcial o muy reducida.

En particular, 151 combinaciones presentan una cobertura igual o inferior al 25 % y 247 se sitúan entre el 25 % y el 50 %. Estos casos no deben tratarse mediante una imputación indiscriminada, ya que podría generarse una proporción excesiva de información sintética.

Por este motivo, la posterior imputación se realizará de forma controlada, preservando la estructura temporal de las series y evitando utilizar indiscriminadamente información procedente de otros años. Este criterio resulta especialmente relevante para conservar las alteraciones de movilidad asociadas al periodo COVID-19.

## 3.10 Imputación temporal controlada de `Valor_IMD`

La imputación de los valores ausentes se realiza sobre la resolución temporal original de la fuente, definida por la combinación de **punto de aforo, año, mes y tipo de día**, antes de trasladar los resultados al dataset diario reconstruido.

Con el objetivo de preservar las variaciones reales del tráfico —especialmente las asociadas al periodo COVID-19— no se utilizan observaciones de otros años para completar valores ausentes.

La imputación se restringe a series correspondientes al mismo:

- punto de aforo;
- año;
- tipo de día.

Dentro de cada serie se utilizan exclusivamente los meses temporalmente próximos. Además, únicamente se permite la imputación cuando existe una cobertura mínima del **75 %** tanto para la combinación punto-año como para la serie específica punto-año-tipo de día.

Para evitar reconstruir periodos prolongados sin información, la interpolación se limita a un máximo de **dos meses consecutivos ausentes**. Las series con cobertura insuficiente o huecos de mayor longitud conservan sus valores ausentes.

Se mantiene adicionalmente el valor original de `Valor_IMD` y se incorpora una variable indicadora que permite identificar de forma explícita las observaciones imputadas.

In [ ]:
# ==============================================================================
# 3.10 IMPUTACIÓN TEMPORAL CONTROLADA DE Valor_IMD
# ==============================================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------------------------
# 1. Partimos de los datos originales mensuales ya limpios
# ------------------------------------------------------------------------------

df_mensual = df_aforos_total.copy()

# Conservamos explícitamente el valor original
df_mensual["Valor_IMD_original"] = df_mensual["Valor_IMD"]

# ------------------------------------------------------------------------------
# 2. Cobertura original por punto-año
# ------------------------------------------------------------------------------

cobertura_punto_ano = (
    df_mensual
    .groupby(["Id_aforament", "Any"])["Valor_IMD"]
    .agg(
        Registros="size",
        Disponibles="count"
    )
    .reset_index()
)

cobertura_punto_ano["Cobertura_punto_ano_pct"] = (
    cobertura_punto_ano["Disponibles"]
    / cobertura_punto_ano["Registros"]
    * 100
)

# Añadimos esta información al dataset
df_mensual = df_mensual.merge(
    cobertura_punto_ano[
        [
            "Id_aforament",
            "Any",
            "Cobertura_punto_ano_pct"
        ]
    ],
    on=["Id_aforament", "Any"],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------------------
# 3. Cobertura por punto-año-tipo de día
# ------------------------------------------------------------------------------

cobertura_tipo = (
    df_mensual
    .groupby(
        [
            "Id_aforament",
            "Any",
            "Codi_tipus_dia"
        ]
    )["Valor_IMD"]
    .agg(
        Registros_tipo="size",
        Disponibles_tipo="count"
    )
    .reset_index()
)

cobertura_tipo["Cobertura_tipo_pct"] = (
    cobertura_tipo["Disponibles_tipo"]
    / cobertura_tipo["Registros_tipo"]
    * 100
)

df_mensual = df_mensual.merge(
    cobertura_tipo[
        [
            "Id_aforament",
            "Any",
            "Codi_tipus_dia",
            "Cobertura_tipo_pct"
        ]
    ],
    on=[
        "Id_aforament",
        "Any",
        "Codi_tipus_dia"
    ],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------------------
# 4. Definir qué registros pertenecen a series imputables
# ------------------------------------------------------------------------------

UMBRAL_COBERTURA = 75

df_mensual["Serie_imputable"] = (
    (df_mensual["Cobertura_punto_ano_pct"] >= UMBRAL_COBERTURA)
    &
    (df_mensual["Cobertura_tipo_pct"] >= UMBRAL_COBERTURA)
)

# ------------------------------------------------------------------------------
# 5. Interpolación temporal dentro del mismo punto-año-tipo de día
# ------------------------------------------------------------------------------

# Inicializamos con los valores originales
df_mensual["Valor_IMD_imputado"] = df_mensual["Valor_IMD"]

# Trabajamos grupo a grupo para NO mezclar años ni tipos de día
grupos = [
    "Id_aforament",
    "Any",
    "Codi_tipus_dia"
]

for _, indices in df_mensual.groupby(grupos).groups.items():

    idx = list(indices)

    sub = (
        df_mensual.loc[idx]
        .sort_values("Mes")
        .copy()
    )

    # Solo se imputa si la serie supera los criterios de cobertura
    if not sub["Serie_imputable"].all():
        continue

    serie = (
        sub
        .set_index("Mes")["Valor_IMD"]
        .reindex(range(1, 13))
    )

    # Interpolación lineal exclusivamente entre meses próximos.
    # limit=2 evita rellenar huecos largos.
    serie_interp = serie.interpolate(
        method="linear",
        limit=2,
        limit_direction="both"
    )

    # Recuperamos únicamente los meses realmente presentes
    valores_nuevos = (
        sub["Mes"]
        .map(serie_interp)
        .to_numpy()
    )

    df_mensual.loc[
        sub.index,
        "Valor_IMD_imputado"
    ] = valores_nuevos

# ------------------------------------------------------------------------------
# 6. Identificar exactamente qué observaciones fueron imputadas
# ------------------------------------------------------------------------------

df_mensual["IMD_imputado"] = (
    df_mensual["Valor_IMD_original"].isna()
    &
    df_mensual["Valor_IMD_imputado"].notna()
).astype(int)

# Sustituimos Valor_IMD por la versión controlada
df_mensual["Valor_IMD"] = df_mensual["Valor_IMD_imputado"]

# ------------------------------------------------------------------------------
# 7. Resumen de la imputación mensual
# ------------------------------------------------------------------------------

nulos_antes = (
    df_mensual["Valor_IMD_original"]
    .isna()
    .sum()
)

n_imputados = (
    df_mensual["IMD_imputado"]
    .sum()
)

nulos_despues = (
    df_mensual["Valor_IMD"]
    .isna()
    .sum()
)

print("=" * 90)
print("3.10 IMPUTACIÓN TEMPORAL CONTROLADA")
print("=" * 90)

print(f"\nNulos originales: {nulos_antes:,}")
print(f"Valores imputados: {n_imputados:,}")
print(f"Nulos conservados: {nulos_despues:,}")

print(
    f"Porcentaje de ausentes recuperados: "
    f"{100 * n_imputados / nulos_antes:.2f}%"
)

print(
    f"Series con umbral de cobertura >= {UMBRAL_COBERTURA}%: "
    f"{df_mensual['Serie_imputable'].mean() * 100:.2f}% "
    f"de los registros"
)

# ------------------------------------------------------------------------------
# 8. Preparar lookup mensual imputado para trasladarlo al dataset diario
# ------------------------------------------------------------------------------

lookup_imputado = df_mensual[
    [
        "Any",
        "Mes",
        "Id_aforament",
        "Codi_tipus_dia",
        "Valor_IMD_original",
        "Valor_IMD",
        "IMD_imputado",
        "Serie_imputable"
    ]
].copy()

# ------------------------------------------------------------------------------
# 9. Trasladar la imputación al dataset diario reconstruido
# ------------------------------------------------------------------------------

# Quitamos Valor_IMD para sustituirlo por la versión auditada/imputada
df_diario_base_imputacion = (
    df_aforos_diario
    .drop(columns=["Valor_IMD"])
    .copy()
)

df_aforos_diario_imputado = pd.merge(
    df_diario_base_imputacion,
    lookup_imputado,
    on=[
        "Any",
        "Mes",
        "Id_aforament",
        "Codi_tipus_dia"
    ],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------------------------
# 10. Auditoría rápida del resultado diario
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESULTADO TRASLADADO AL DATASET DIARIO")
print("=" * 90)

print(
    f"Dimensiones: "
    f"{df_aforos_diario_imputado.shape}"
)

print(
    f"Valores diarios con IMD disponible: "
    f"{df_aforos_diario_imputado['Valor_IMD'].notna().sum():,}"
)

print(
    f"Valores diarios todavía ausentes: "
    f"{df_aforos_diario_imputado['Valor_IMD'].isna().sum():,}"
)

print(
    f"Porcentaje diario todavía ausente: "
    f"{df_aforos_diario_imputado['Valor_IMD'].isna().mean() * 100:.2f}%"
)

print("\nMuestra del resultado:")

display(
    df_aforos_diario_imputado.head(10)
)

3.10 IMPUTACIÓN TEMPORAL CONTROLADA

Nulos originales: 37,615
Valores imputados: 9,514
Nulos conservados: 28,101
Porcentaje de ausentes recuperados: 25.29%
Series con umbral de cobertura >= 75%: 84.00% de los registros

RESULTADO TRASLADADO AL DATASET DIARIO
Dimensiones: (1910502, 10)
Valores diarios con IMD disponible: 1,739,692
Valores diarios todavía ausentes: 170,810
Porcentaje diario todavía ausente: 8.94%

Muestra del resultado:


,Fecha,Id_aforament,Any,Mes,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD_original,Valor_IMD,IMD_imputado,Serie_imputable
0,2018-01-01,10-SMD-1,2018,1,1,lunes,76505.0,76505.0,0.0,True
1,2018-01-01,10-SMD-2,2018,1,1,lunes,69354.0,69354.0,0.0,True
2,2018-01-01,10001,2018,1,1,lunes,8625.0,8625.0,0.0,True
3,2018-01-01,10002,2018,1,1,lunes,13777.0,13777.0,0.0,True
4,2018-01-01,10005,2018,1,1,lunes,14675.0,14675.0,0.0,False
5,2018-01-01,10007,2018,1,1,lunes,2116.0,2116.0,0.0,False
6,2018-01-01,10008,2018,1,1,lunes,3479.0,3479.0,0.0,False
7,2018-01-01,10009,2018,1,1,lunes,7310.0,7310.0,0.0,True
8,2018-01-01,1001,2018,1,1,lunes,34153.0,34153.0,0.0,True
9,2018-01-01,10010,2018,1,1,lunes,10718.0,10718.0,0.0,True


### Resultado de la imputación temporal controlada

La estrategia de imputación permitió recuperar **9.514 de los 37.615 valores ausentes** existentes en la resolución original de los datos, lo que representa el **25,29 % de las ausencias inicialmente identificadas**.

La imputación se restringió a series con una cobertura mínima del **75 %**, manteniendo separadas las observaciones correspondientes a cada punto de aforo, año y tipo de día. De este modo, no se emplea información de otros años para reconstruir valores ausentes, evitando suavizar artificialmente las variaciones temporales asociadas, entre otros factores, al periodo COVID-19.

Tras el proceso permanecen **28.101 valores ausentes** en la tabla de resolución original. Estos registros se conservan deliberadamente al corresponder a series con cobertura insuficiente o a huecos que no cumplen los criterios establecidos para una imputación suficientemente fiable.

Una vez trasladados los resultados al calendario diario, el dataset contiene **1.739.692 observaciones con IMD disponible** y **170.810 observaciones aún ausentes**, equivalentes al **8,94 %** de la tabla diaria.

Se conservan adicionalmente `Valor_IMD_original`, `Valor_IMD` e `IMD_imputado`, permitiendo distinguir en todo momento los valores procedentes directamente de la fuente de aquellos obtenidos mediante imputación.

## 3.11 Auditoría y validación de la imputación

Tras aplicar la imputación temporal controlada se evalúa su impacto sobre la distribución de `Valor_IMD`.

La validación compara los valores originales y los valores resultantes después de la imputación, analiza el número de observaciones estimadas por año y comprueba que el procedimiento no genere valores físicamente imposibles ni altere de forma significativa los patrones temporales del tráfico.

Se presta especial atención al periodo 2020–2021 para comprobar que la imputación no suaviza artificialmente las variaciones de movilidad asociadas a la pandemia.

In [ ]:
# ==============================================================================
# 3.11 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN
# ==============================================================================

import pandas as pd
import numpy as np

df = df_mensual.copy()

print("=" * 90)
print("3.11 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Resumen general
# ------------------------------------------------------------------------------

print("\n1. RESUMEN GENERAL")
print("-" * 90)

n_originales = df["Valor_IMD_original"].notna().sum()
n_imputados = df["IMD_imputado"].sum()
n_finales = df["Valor_IMD"].notna().sum()
n_nulos = df["Valor_IMD"].isna().sum()

print(f"Valores originales disponibles: {n_originales:,}")
print(f"Valores imputados: {n_imputados:,}")
print(f"Valores finales disponibles: {n_finales:,}")
print(f"Valores ausentes conservados: {n_nulos:,}")

# ------------------------------------------------------------------------------
# 2. Comprobar que ningún valor original ha sido modificado
# ------------------------------------------------------------------------------

print("\n2. CONSERVACIÓN DE LOS VALORES ORIGINALES")
print("-" * 90)

mask_original = df["Valor_IMD_original"].notna()

cambios_originales = (
    ~np.isclose(
        df.loc[mask_original, "Valor_IMD_original"],
        df.loc[mask_original, "Valor_IMD"],
        equal_nan=True
    )
).sum()

print(
    f"Valores originales modificados por la imputación: "
    f"{cambios_originales:,}"
)

# ------------------------------------------------------------------------------
# 3. Valores imputados por año
# ------------------------------------------------------------------------------

print("\n3. VALORES IMPUTADOS POR AÑO")
print("-" * 90)

imputacion_ano = (
    df.groupby("Any")
    .agg(
        Registros=("Valor_IMD", "size"),
        Originales=("Valor_IMD_original", "count"),
        Imputados=("IMD_imputado", "sum"),
        Finales=("Valor_IMD", "count")
    )
    .reset_index()
)

imputacion_ano["Pct_imputado_sobre_total"] = (
    imputacion_ano["Imputados"]
    / imputacion_ano["Registros"]
    * 100
).round(2)

display(imputacion_ano)

# ------------------------------------------------------------------------------
# 4. Comparación estadística original vs final
# ------------------------------------------------------------------------------

print("\n4. ESTADÍSTICOS ORIGINALES VS RESULTADO FINAL")
print("-" * 90)

comparacion = pd.DataFrame({
    "Original": df["Valor_IMD_original"].describe(),
    "Final": df["Valor_IMD"].describe()
})

comparacion["Diferencia"] = (
    comparacion["Final"] -
    comparacion["Original"]
)

display(comparacion.round(2))

# ------------------------------------------------------------------------------
# 5. Media anual antes y después de la imputación
# ------------------------------------------------------------------------------

print("\n5. MEDIA ANUAL DE IMD: ORIGINAL VS FINAL")
print("-" * 90)

media_anual = (
    df.groupby("Any")
    .agg(
        IMD_original=("Valor_IMD_original", "mean"),
        IMD_final=("Valor_IMD", "mean")
    )
    .reset_index()
)

media_anual["Diferencia"] = (
    media_anual["IMD_final"] -
    media_anual["IMD_original"]
)

media_anual["Diferencia_pct"] = (
    media_anual["Diferencia"]
    / media_anual["IMD_original"]
    * 100
).round(3)

display(media_anual.round(2))

# ------------------------------------------------------------------------------
# 6. Comprobación específica COVID
# ------------------------------------------------------------------------------

print("\n6. CONTROL ESPECÍFICO DEL PERIODO COVID")
print("-" * 90)

covid_control = media_anual[
    media_anual["Any"].isin([2019, 2020, 2021, 2022])
].copy()

display(covid_control.round(2))

# ------------------------------------------------------------------------------
# 7. Rangos y posibles valores imposibles
# ------------------------------------------------------------------------------

print("\n7. CONTROL DE RANGOS")
print("-" * 90)

print(
    f"Valor mínimo final: "
    f"{df['Valor_IMD'].min():,.2f}"
)

print(
    f"Valor máximo final: "
    f"{df['Valor_IMD'].max():,.2f}"
)

valores_negativos = (
    df["Valor_IMD"] < 0
).sum()

print(
    f"Valores negativos: "
    f"{valores_negativos:,}"
)

# ------------------------------------------------------------------------------
# 8. Estadísticos exclusivos de los valores imputados
# ------------------------------------------------------------------------------

print("\n8. DISTRIBUCIÓN DE LOS VALORES IMPUTADOS")
print("-" * 90)

solo_imputados = df.loc[
    df["IMD_imputado"] == 1,
    "Valor_IMD"
]

display(
    solo_imputados
    .describe()
    .to_frame("Valores_imputados")
    .round(2)
)

# ------------------------------------------------------------------------------
# 9. Nulos conservados por año
# ------------------------------------------------------------------------------

print("\n9. AUSENCIAS CONSERVADAS POR AÑO")
print("-" * 90)

nulos_ano = (
    df.assign(
        IMD_ausente_final=df["Valor_IMD"].isna()
    )
    .groupby("Any")["IMD_ausente_final"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

nulos_ano.columns = [
    "Año",
    "Registros",
    "Ausentes_finales",
    "Porcentaje_ausentes"
]

nulos_ano["Porcentaje_ausentes"] = (
    nulos_ano["Porcentaje_ausentes"] * 100
).round(2)

display(nulos_ano)

# ------------------------------------------------------------------------------
# 10. Resumen final
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN DE VALIDACIÓN")
print("=" * 90)

print(f"Valores originales modificados: {cambios_originales:,}")
print(f"Valores imputados: {n_imputados:,}")
print(f"Valores ausentes conservados: {n_nulos:,}")
print(f"Valores negativos generados: {valores_negativos:,}")

3.11 AUDITORÍA Y VALIDACIÓN DE LA IMPUTACIÓN

1. RESUMEN GENERAL
------------------------------------------------------------------------------------------
Valores originales disponibles: 276,180
Valores imputados: 9,514
Valores finales disponibles: 285,694
Valores ausentes conservados: 28,101

2. CONSERVACIÓN DE LOS VALORES ORIGINALES
------------------------------------------------------------------------------------------
Valores originales modificados por la imputación: 0

3. VALORES IMPUTADOS POR AÑO
------------------------------------------------------------------------------------------


,Any,Registros,Originales,Imputados,Finales,Pct_imputado_sobre_total
0,2018,36780,28981,1288,30269,3.50
1,2019,41100,34925,1450,36375,3.53
2,2020,42840,39265,1525,40790,3.56
3,2021,43440,39657,1238,40895,2.85
4,2022,49435,43412,1103,44515,2.23
5,2023,49920,45225,1420,46645,2.84
6,2024,50280,44715,1490,46205,2.96



4. ESTADÍSTICOS ORIGINALES VS RESULTADO FINAL
------------------------------------------------------------------------------------------


,Original,Final,Diferencia
count,276180.00,285694.00,9514.00
mean,11420.33,11421.62,1.29
std,16493.81,16443.81,-50.00
min,0.00,0.00,0.00
25%,926.00,937.00,11.00
50%,4832.00,4878.00,46.00
75%,14210.00,14250.00,40.00
max,503838.00,503838.00,0.00



5. MEDIA ANUAL DE IMD: ORIGINAL VS FINAL
------------------------------------------------------------------------------------------


...


7. CONTROL DE RANGOS
------------------------------------------------------------------------------------------
Valor mínimo final: 0.00
Valor máximo final: 503,838.00
Valores negativos: 0

8. DISTRIBUCIÓN DE LOS VALORES IMPUTADOS
------------------------------------------------------------------------------------------


,Valores_imputados
count,9514.00
mean,11458.99
std,14920.16
min,7.00
25%,1256.50
50%,6185.12
75%,15209.75
max,96346.00



9. AUSENCIAS CONSERVADAS POR AÑO
------------------------------------------------------------------------------------------


,Año,Registros,Ausentes_finales,Porcentaje_ausentes
0,2018,36780,6511,17.70
1,2019,41100,4725,11.50
2,2020,42840,2050,4.79
3,2021,43440,2545,5.86
4,2022,49435,4920,9.95
5,2023,49920,3275,6.56
6,2024,50280,4075,8.10



RESUMEN DE VALIDACIÓN
Valores originales modificados: 0
Valores imputados: 9,514
Valores ausentes conservados: 28,101
Valores negativos generados: 0


### Resultado de la validación de la imputación

La auditoría confirma que el procedimiento de imputación presenta un impacto reducido sobre la estructura estadística original de los datos.

Se imputaron **9.514 observaciones**, manteniéndose **28.101 valores ausentes** al no cumplir los criterios establecidos de cobertura y continuidad temporal. Ninguno de los **276.180 valores originalmente disponibles** fue modificado durante el proceso.

La distribución global de `Valor_IMD` permanece prácticamente inalterada: la media pasa de **11.420,33 a 11.421,62 vehículos/día**, mientras que los valores mínimo y máximo se mantienen en **0 y 503.838 vehículos/día**, respectivamente. No se generan valores negativos.

El análisis por año confirma igualmente un impacto limitado de la imputación. Las variaciones de la IMD media anual se mantienen entre aproximadamente **−1,03 % y +0,61 %** respecto a los datos originales.

Especialmente relevante resulta la conservación del patrón asociado al periodo COVID-19. La IMD media desciende desde aproximadamente **12.526 vehículos/día en 2019 hasta 9.520 en 2020**, seguida de una recuperación progresiva en 2021 y 2022. La imputación modifica la media de 2020 únicamente en torno al **0,61 %**, por lo que no altera de forma sustancial la señal temporal que constituye uno de los elementos de interés del estudio.

En consecuencia, se considera validada la estrategia de imputación temporal controlada y se mantiene el criterio de conservar como ausentes aquellos registros cuya reconstrucción no dispone de soporte observacional suficiente.

## 3.12 Preparación y auditoría del dataset diario definitivo

Una vez validado el procedimiento de imputación, se prepara el dataset diario definitivo de tráfico rodado.

Se conservan las variables necesarias para el posterior análisis temporal y la integración con el resto de bloques del estudio. Asimismo, se mantienen `Valor_IMD_original` e `IMD_imputado` como variables de trazabilidad, permitiendo distinguir entre las observaciones procedentes directamente de la fuente y aquellas reconstruidas mediante el procedimiento de imputación temporal controlada.

Las variables auxiliares utilizadas exclusivamente durante el proceso de imputación se eliminan del dataset final.

Finalmente, se realiza una auditoría estructural para comprobar el orden de las variables, los tipos de datos, la existencia de duplicados, los valores ausentes y la coherencia temporal antes de proceder al guardado definitivo.

In [ ]:
# ==============================================================================
# 3.12 PREPARACIÓN Y AUDITORÍA DEL DATASET DIARIO DEFINITIVO
# ==============================================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------------------------
# 1. Crear copia definitiva
# ------------------------------------------------------------------------------

df_aforos_final = df_aforos_diario_imputado.copy()

# ------------------------------------------------------------------------------
# 2. Eliminar variables auxiliares del proceso de imputación
# ------------------------------------------------------------------------------

columnas_auxiliares = [
    "Serie_imputable"
]

df_aforos_final = df_aforos_final.drop(
    columns=[
        col for col in columnas_auxiliares
        if col in df_aforos_final.columns
    ]
)

# ------------------------------------------------------------------------------
# 3. Ajustar tipos de datos
# ------------------------------------------------------------------------------

df_aforos_final["Fecha"] = pd.to_datetime(
    df_aforos_final["Fecha"]
)

df_aforos_final["Any"] = (
    df_aforos_final["Any"]
    .astype("int16")
)

df_aforos_final["Mes"] = (
    df_aforos_final["Mes"]
    .astype("int8")
)

df_aforos_final["Codi_tipus_dia"] = (
    df_aforos_final["Codi_tipus_dia"]
    .astype("int8")
)

# IMD_imputado debe ser 0/1
df_aforos_final["IMD_imputado"] = (
    df_aforos_final["IMD_imputado"]
    .fillna(0)
    .astype("int8")
)

# ------------------------------------------------------------------------------
# 4. Orden definitivo de las columnas
# ------------------------------------------------------------------------------

columnas_finales = [
    "Fecha",
    "Any",
    "Mes",
    "Id_aforament",
    "Codi_tipus_dia",
    "Desc_tipus_dia",
    "Valor_IMD_original",
    "Valor_IMD",
    "IMD_imputado"
]

df_aforos_final = df_aforos_final[
    columnas_finales
]

# ------------------------------------------------------------------------------
# 5. Ordenar cronológicamente
# ------------------------------------------------------------------------------

df_aforos_final = (
    df_aforos_final
    .sort_values(
        ["Fecha", "Id_aforament"]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------------------------
# 6. Auditoría estructural final
# ------------------------------------------------------------------------------

print("=" * 90)
print("3.12 AUDITORÍA DEL DATASET DIARIO DEFINITIVO")
print("=" * 90)

print(f"\nDimensiones: {df_aforos_final.shape}")

print(
    f"Periodo: "
    f"{df_aforos_final['Fecha'].min().date()} - "
    f"{df_aforos_final['Fecha'].max().date()}"
)

print(
    f"Puntos de aforo únicos: "
    f"{df_aforos_final['Id_aforament'].nunique():,}"
)

# ------------------------------------------------------------------------------
# 7. Duplicados
# ------------------------------------------------------------------------------

duplicados = df_aforos_final.duplicated(
    subset=["Fecha", "Id_aforament"]
).sum()

print(
    f"Duplicados Fecha-Id_aforament: "
    f"{duplicados:,}"
)

# ------------------------------------------------------------------------------
# 8. Valores ausentes
# ------------------------------------------------------------------------------

print("\nValores ausentes por variable:")

nulos_finales = (
    df_aforos_final
    .isna()
    .sum()
    .to_frame("Nulos")
)

nulos_finales["Porcentaje"] = (
    nulos_finales["Nulos"]
    / len(df_aforos_final)
    * 100
).round(2)

display(nulos_finales)

# ------------------------------------------------------------------------------
# 9. Control de la trazabilidad
# ------------------------------------------------------------------------------

n_imputados_diarios = (
    df_aforos_final["IMD_imputado"] == 1
).sum()

errores_trazabilidad = (
    (df_aforos_final["IMD_imputado"] == 1)
    &
    (
        df_aforos_final["Valor_IMD_original"].notna()
        |
        df_aforos_final["Valor_IMD"].isna()
    )
).sum()

print("\nControl de trazabilidad:")
print(
    f"Observaciones diarias marcadas como imputadas: "
    f"{n_imputados_diarios:,}"
)

print(
    f"Incoherencias en la bandera de imputación: "
    f"{errores_trazabilidad:,}"
)

# ------------------------------------------------------------------------------
# 10. Control de rangos
# ------------------------------------------------------------------------------

valores_negativos = (
    df_aforos_final["Valor_IMD"] < 0
).sum()

print("\nControl de Valor_IMD:")

print(
    f"Mínimo: "
    f"{df_aforos_final['Valor_IMD'].min():,.2f}"
)

print(
    f"Máximo: "
    f"{df_aforos_final['Valor_IMD'].max():,.2f}"
)

print(
    f"Valores negativos: "
    f"{valores_negativos:,}"
)

# ------------------------------------------------------------------------------
# 11. Tipos de datos
# ------------------------------------------------------------------------------

print("\nTipos de datos finales:")
print(df_aforos_final.dtypes)

# ------------------------------------------------------------------------------
# 12. Muestra del dataset definitivo
# ------------------------------------------------------------------------------

print("\nPrimeras observaciones:")

display(
    df_aforos_final.head(10)
)

# ------------------------------------------------------------------------------
# 13. Resumen
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESULTADO FINAL")
print("=" * 90)

print(
    f"Registros finales: "
    f"{len(df_aforos_final):,}"
)

print(
    f"Valores IMD disponibles: "
    f"{df_aforos_final['Valor_IMD'].notna().sum():,}"
)

print(
    f"Valores IMD ausentes conservados: "
    f"{df_aforos_final['Valor_IMD'].isna().sum():,}"
)

print(
    f"Porcentaje de IMD disponible: "
    f"{df_aforos_final['Valor_IMD'].notna().mean() * 100:.2f}%"
)

print(
    f"Duplicados Fecha-Aforo: "
    f"{duplicados:,}"
)

print(
    f"Incoherencias de trazabilidad: "
    f"{errores_trazabilidad:,}"
)

print(
    f"Valores negativos: "
    f"{valores_negativos:,}"
)

3.12 AUDITORÍA DEL DATASET DIARIO DEFINITIVO

Dimensiones: (1910502, 9)
Periodo: 2018-01-01 - 2024-12-31
Puntos de aforo únicos: 922
Duplicados Fecha-Id_aforament: 0

Valores ausentes por variable:


,Nulos,Porcentaje
Fecha,0,0.00
Any,0,0.00
Mes,0,0.00
Id_aforament,0,0.00
Codi_tipus_dia,0,0.00
Desc_tipus_dia,0,0.00
Valor_IMD_original,228436,11.96
Valor_IMD,170810,8.94
IMD_imputado,0,0.00



Control de trazabilidad:
Observaciones diarias marcadas como imputadas: 57,626
Incoherencias en la bandera de imputación: 0

Control de Valor_IMD:
Mínimo: 0.00
Máximo: 503,838.00
Valores negativos: 0

Tipos de datos finales:
Fecha                 datetime64[ns]
Any                            int16
Mes                             int8
Id_aforament                  object
Codi_tipus_dia                  int8
Desc_tipus_dia                object
Valor_IMD_original           float64
Valor_IMD                    float64
IMD_imputado                    int8
dtype: object

Primeras observaciones:


,Fecha,Any,Mes,Id_aforament,Codi_tipus_dia,Desc_tipus_dia,Valor_IMD_original,Valor_IMD,IMD_imputado
0,2018-01-01,2018,1,10-SMD-1,1,lunes,76505.0,76505.0,0
1,2018-01-01,2018,1,10-SMD-2,1,lunes,69354.0,69354.0,0
2,2018-01-01,2018,1,10001,1,lunes,8625.0,8625.0,0
3,2018-01-01,2018,1,10002,1,lunes,13777.0,13777.0,0
4,2018-01-01,2018,1,10005,1,lunes,14675.0,14675.0,0
5,2018-01-01,2018,1,10007,1,lunes,2116.0,2116.0,0
6,2018-01-01,2018,1,10008,1,lunes,3479.0,3479.0,0
7,2018-01-01,2018,1,10009,1,lunes,7310.0,7310.0,0
8,2018-01-01,2018,1,1001,1,lunes,34153.0,34153.0,0
9,2018-01-01,2018,1,10010,1,lunes,10718.0,10718.0,0



RESULTADO FINAL
Registros finales: 1,910,502
Valores IMD disponibles: 1,739,692
Valores IMD ausentes conservados: 170,810
Porcentaje de IMD disponible: 91.06%
Duplicados Fecha-Aforo: 0
Incoherencias de trazabilidad: 0
Valores negativos: 0


### Resultado de la auditoría final

El dataset diario definitivo de tráfico rodado queda compuesto por **1.910.502 observaciones**, correspondientes a **922 puntos de aforo** y al periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

La auditoría estructural no detecta duplicados para la combinación `Fecha`–`Id_aforament`, confirmando la unicidad de cada observación diaria por punto de aforo.

Tras el procedimiento de imputación temporal controlada, `Valor_IMD` presenta información disponible en **1.739.692 observaciones (91,06 %)**, mientras que **170.810 registros (8,94 %)** permanecen como valores ausentes. Estas ausencias se conservan deliberadamente al no cumplir los criterios establecidos para una reconstrucción suficientemente fiable.

La variable `Valor_IMD_original` mantiene **228.436 valores ausentes (11,96 %)**, correspondientes a la disponibilidad original de la fuente antes de la imputación. La diferencia entre ambas variables permite conservar la trazabilidad completa del tratamiento realizado.

En el dataset diario, **57.626 observaciones están identificadas mediante `IMD_imputado = 1`** como procedentes de valores mensuales reconstruidos. La comprobación de consistencia de esta variable no detecta ninguna incoherencia entre la información original, el valor final y la bandera de imputación.

El control de rangos muestra valores de `Valor_IMD` comprendidos entre **0 y 503.838 vehículos/día**, sin presencia de valores negativos. Los valores extremos originales se mantienen tras el tratamiento.

Por tanto, el dataset supera satisfactoriamente los controles finales de estructura, integridad, trazabilidad y coherencia de los valores, quedando preparado para su almacenamiento definitivo y posterior integración con los restantes bloques de información del estudio.

## 3.13 Guardado del dataset diario definitivo

Una vez completadas las etapas de limpieza, reconstrucción temporal, imputación controlada y auditoría final, se almacena el dataset definitivo de tráfico rodado.

El archivo conserva tanto el valor original de Intensidad Media Diaria como el valor final utilizado en el análisis y la variable de trazabilidad de la imputación.

El dataset se guarda en formato CSV con codificación `UTF-8-SIG`, facilitando su posterior lectura e integración con los restantes bloques de información del estudio.

In [ ]:
# ==============================================================================
# 3.13 GUARDADO DEL DATASET DIARIO DEFINITIVO
# ==============================================================================

import os
import pandas as pd

# ------------------------------------------------------------------------------
# 1. Ruta y nombre definitivo
# ------------------------------------------------------------------------------

ruta_base = "/content/drive/MyDrive/TFM/02_Trafico_Rodado_y_Aforos/DATOS LIMPIOS"

nombre_archivo = "df_aforos_2018_2024_Diario_Limpio.csv"

ruta_salida = os.path.join(
    ruta_base,
    nombre_archivo
)

# ------------------------------------------------------------------------------
# 2. Guardar el dataset definitivo
# ------------------------------------------------------------------------------

df_aforos_final.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------------------------
# 3. Verificar que el archivo se ha creado correctamente
# ------------------------------------------------------------------------------

if os.path.exists(ruta_salida):

    tamano_mb = os.path.getsize(ruta_salida) / (1024 ** 2)

    print("=" * 90)
    print("3.13 GUARDADO DEL DATASET DEFINITIVO")
    print("=" * 90)

    print("\nArchivo guardado correctamente.")
    print(f"Nombre: {nombre_archivo}")
    print(f"Ruta: {ruta_salida}")
    print(f"Tamaño: {tamano_mb:.2f} MB")

else:
    raise FileNotFoundError(
        "El archivo no se ha generado correctamente."
    )

# ------------------------------------------------------------------------------
# 4. Comprobación mediante lectura del archivo guardado
# ------------------------------------------------------------------------------

df_comprobacion = pd.read_csv(
    ruta_salida,
    encoding="utf-8-sig",
    parse_dates=["Fecha"]
)

print("\n" + "-" * 90)
print("COMPROBACIÓN DEL ARCHIVO GUARDADO")
print("-" * 90)

print(
    f"Dimensiones guardadas: "
    f"{df_comprobacion.shape}"
)

print(
    f"Periodo: "
    f"{df_comprobacion['Fecha'].min().date()} - "
    f"{df_comprobacion['Fecha'].max().date()}"
)

print(
    f"Puntos de aforo: "
    f"{df_comprobacion['Id_aforament'].nunique():,}"
)

print(
    f"Valores IMD disponibles: "
    f"{df_comprobacion['Valor_IMD'].notna().sum():,}"
)

print(
    f"Valores IMD ausentes: "
    f"{df_comprobacion['Valor_IMD'].isna().sum():,}"
)

print(
    f"Duplicados Fecha-Aforo: "
    f"{df_comprobacion.duplicated(['Fecha', 'Id_aforament']).sum():,}"
)

# ------------------------------------------------------------------------------
# 5. Verificación exacta de dimensiones
# ------------------------------------------------------------------------------

if df_comprobacion.shape == df_aforos_final.shape:
    print("\n✓ Las dimensiones coinciden con el dataset original en memoria.")
else:
    print("\n⚠ Las dimensiones no coinciden. Revisar el archivo.")

print("\nGuardado y verificación completados.")

3.13 GUARDADO DEL DATASET DEFINITIVO

Archivo guardado correctamente.
Nombre: df_aforos_2018_2024_Diario_Limpio.csv
Ruta: /content/drive/MyDrive/TFM/02_Trafico_Rodado_y_Aforos/DATOS LIMPIOS/df_aforos_2018_2024_Diario_Limpio.csv
Tamaño: 90.29 MB

------------------------------------------------------------------------------------------
COMPROBACIÓN DEL ARCHIVO GUARDADO
------------------------------------------------------------------------------------------
Dimensiones guardadas: (1910502, 9)
Periodo: 2018-01-01 - 2024-12-31
Puntos de aforo: 922
Valores IMD disponibles: 1,739,692
Valores IMD ausentes: 170,810
Duplicados Fecha-Aforo: 0

✓ Las dimensiones coinciden con el dataset original en memoria.

Guardado y verificación completados.


## 3.14 Cierre del bloque de tráfico rodado y aforos

El procesamiento del bloque de tráfico rodado permite obtener una serie temporal diaria homogénea para el periodo **2018–2024**, construida a partir de los datos de Intensidad Media Diaria (IMD) publicados por Open Data BCN.

Los siete archivos anuales presentan una estructura homogénea y contienen información correspondiente a **922 puntos de aforo**. Tras su integración se analizaron la estructura de los datos, los tipos de día, la continuidad temporal de los puntos y los patrones de disponibilidad de `Valor_IMD`.

Debido a que la fuente original proporciona valores mensuales diferenciados por tipo de día, se reconstruyó un calendario diario asignando a cada fecha el valor de IMD correspondiente a su punto de aforo, año, mes y categoría de día. La reconstrucción se realizó exclusivamente para las combinaciones punto–año realmente presentes en la fuente, evitando generar artificialmente periodos de actividad inexistentes.

El análisis de cobertura permitió identificar diferencias importantes en la disponibilidad temporal de los distintos puntos. Por este motivo, los valores ausentes no se imputaron de forma indiscriminada.

Se aplicó una **imputación temporal controlada** sobre la resolución original de los datos, restringida al mismo punto de aforo, año y tipo de día. Únicamente se consideraron series con una cobertura mínima del **75 %**, limitando además la reconstrucción a huecos temporales cortos. Este procedimiento evita utilizar información procedente de otros años y permite preservar las alteraciones reales de movilidad asociadas al periodo COVID-19.

De los **37.615 valores ausentes** existentes en la tabla original, se reconstruyeron **9.514 (25,29 %)**, mientras que los restantes se conservaron como ausentes al no disponer de soporte observacional suficiente.

Las auditorías posteriores confirmaron que la imputación no modifica ningún valor originalmente observado, no genera valores negativos y produce alteraciones mínimas en los estadísticos globales y anuales. Asimismo, se mantiene claramente la reducción de la movilidad observada durante 2020 y su posterior recuperación.

El dataset diario definitivo contiene:

- **1.910.502 observaciones**;
- **922 puntos de aforo**;
- cobertura temporal entre **2018 y 2024**;
- **1.739.692 observaciones con `Valor_IMD` disponible (91,06 %)**;
- **170.810 observaciones ausentes conservadas (8,94 %)**;
- **0 duplicados** para la combinación `Fecha`–`Id_aforament`;
- **0 valores negativos**;
- **0 incoherencias en la trazabilidad de la imputación**.

Se conservan `Valor_IMD_original`, `Valor_IMD` e `IMD_imputado`, permitiendo distinguir los datos procedentes directamente de la fuente de aquellos reconstruidos mediante imputación.

El bloque queda así preparado para su posterior integración temporal y espacial con los datos de contaminación atmosférica, meteorología y el resto de variables explicativas consideradas en el estudio.

# 4 · CONTAMINACIÓN ACÚSTICA — RED DE MONITORIZACIÓN DE RUIDO DE BARCELONA

## 4.0 Fuente, alcance y estrategia de procesamiento

Este bloque analiza los registros de **contaminación acústica procedentes de la red de equipos de monitorización de ruido de Barcelona**, publicados por el **Ajuntament de Barcelona a través del portal Open Data BCN**.

**Fuente oficial:**  
Open Data BCN — *Xarxa de Soroll. Equips de monitoratge. Dades*  
https://opendata-ajuntament.barcelona.cat/data/es/dataset/xarxasoroll-equipsmonitor-dades

El periodo de estudio comprende los años **2018–2024**. La información presenta un cambio de resolución temporal en la fuente original:

- **2018–2023:** registros acústicos con resolución **horaria** (`Nivell_LAeq_1h`).
- **2024:** registros acústicos con resolución **minutal** (`Nivell_LAeq_1min`).

Esta heterogeneidad temporal requiere un tratamiento específico antes de integrar el ruido con el resto de variables del proyecto. Por este motivo, ambos periodos se procesan inicialmente de forma independiente y posteriormente se homogeneizan a una escala temporal común.

### Objetivo

El objetivo del procesamiento es construir un **dataset acústico diario homogéneo para 2018–2024**, manteniendo la máxima trazabilidad posible respecto a los registros originales.

La variable acústica utilizada es el **nivel sonoro continuo equivalente ponderado A (LAeq)**. Dado que los niveles sonoros se expresan en decibelios y corresponden a una escala logarítmica, la obtención del indicador diario se realiza mediante **agregación energética**, evitando el uso de una media aritmética directa de los niveles horarios o minutales.

De forma general, el nivel equivalente diario se obtiene mediante:

\[
LAeq_{día} =
10 \log_{10}
\left(
\frac{1}{N}
\sum_{i=1}^{N}
10^{LAeq_i/10}
\right)
\]

donde \(N\) representa el número de observaciones acústicas válidas disponibles para cada combinación de fecha e instalación.

### Control de calidad temporal

Además del `LAeq_dia`, se calcula para cada combinación `Fecha–Id_Instal` la cobertura temporal disponible respecto al número teórico de observaciones del día.

El cálculo considera las particularidades derivadas de los **cambios oficiales de horario**, de manera que la duración teórica de determinados días puede diferir de las 24 horas o 1.440 minutos habituales.

Se establece como criterio de calidad una **cobertura temporal mínima del 75 %**. Las observaciones que cumplen este requisito se identifican mediante la variable `Dia_valido`.

Las observaciones con cobertura inferior al umbral no se eliminan del dataset maestro, sino que se conservan identificadas para mantener la trazabilidad y permitir su exclusión únicamente cuando el análisis posterior requiera observaciones temporalmente representativas.

### Criterios metodológicos

Durante el procesamiento se siguen los siguientes principios:

1. conservación de los registros originales siempre que sea posible;
2. agregación energética del LAeq a escala diaria;
3. control explícito de la cobertura temporal;
4. consideración de los cambios oficiales de horario;
5. ausencia de imputación de niveles acústicos;
6. ausencia de eliminación automática de valores extremos únicamente por criterios estadísticos;
7. conservación de la resolución temporal de origen como variable de trazabilidad;
8. comprobación sistemática de nulos, duplicados, rangos y coherencia temporal.

El resultado esperado es un dataset acústico diario preparado para su posterior **preintegración temporal y espacial con contaminación atmosférica, meteorología, tráfico rodado y el resto de variables explicativas del proyecto**.

## 4.1 Inspección inicial de los datos

Se inicia el tratamiento del bloque de **contaminación acústica** mediante la inspección de un archivo representativo de la fuente original.

El objetivo de esta primera etapa es identificar la estructura de los datos, las variables disponibles, los tipos de datos, las dimensiones del archivo y la presencia inicial de valores ausentes, sin realizar todavía ninguna transformación.

Esta exploración permitirá definir posteriormente una estrategia homogénea para la carga, limpieza y procesamiento del conjunto de archivos disponibles antes de su transformación a resolución diaria.

In [ ]:
# ==============================================================================
# 4.1 INSPECCIÓN INICIAL DE LOS DATOS DE CONTAMINACIÓN ACÚSTICA
# ==============================================================================

import pandas as pd
import os

# ------------------------------------------------------------------------------
# 1. Ruta al archivo seleccionado para la inspección inicial
# ------------------------------------------------------------------------------

ruta_acustica = (
    "/content/drive/MyDrive/TFM/07_Contaminacion_Acustica/"
    "BC 2024/2024_12Des_XarxaSoroll_EqMonitor_Dades_1Min.csv"
)

# ------------------------------------------------------------------------------
# 2. Comprobar que el archivo existe
# ------------------------------------------------------------------------------

if not os.path.exists(ruta_acustica):
    raise FileNotFoundError(
        f"No se ha encontrado el archivo:\n{ruta_acustica}"
    )

print("=" * 90)
print("4.1 INSPECCIÓN INICIAL DE CONTAMINACIÓN ACÚSTICA")
print("=" * 90)

print(f"\nArchivo: {os.path.basename(ruta_acustica)}")

# ------------------------------------------------------------------------------
# 3. Lectura del archivo
# ------------------------------------------------------------------------------

lecturas = [
    {"encoding": "utf-8", "sep": ","},
    {"encoding": "utf-8", "sep": ";"},
    {"encoding": "latin1", "sep": ","},
    {"encoding": "latin1", "sep": ";"}
]

df_acustica = None
configuracion_usada = None

for config in lecturas:
    try:
        df_prueba = pd.read_csv(
            ruta_acustica,
            encoding=config["encoding"],
            sep=config["sep"]
        )

        # La lectura se acepta únicamente si se detecta más de una columna
        if df_prueba.shape[1] > 1:
            df_acustica = df_prueba
            configuracion_usada = config
            break

    except Exception:
        pass

if df_acustica is None:
    raise ValueError(
        "No ha sido posible leer correctamente el archivo "
        "con las configuraciones probadas."
    )

# ------------------------------------------------------------------------------
# 4. Información básica
# ------------------------------------------------------------------------------

print(f"\nCodificación utilizada: {configuracion_usada['encoding']}")
print(f"Separador detectado: '{configuracion_usada['sep']}'")

print(f"\nDimensiones: {df_acustica.shape}")
print(f"Registros: {len(df_acustica):,}")
print(f"Variables: {df_acustica.shape[1]}")

# ------------------------------------------------------------------------------
# 5. Variables disponibles
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("VARIABLES DISPONIBLES")
print("-" * 90)

for i, columna in enumerate(df_acustica.columns, start=1):
    print(f"{i:02d}. {columna}")

# ------------------------------------------------------------------------------
# 6. Tipos de datos
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("TIPOS DE DATOS")
print("-" * 90)

print(df_acustica.dtypes)

# ------------------------------------------------------------------------------
# 7. Valores ausentes
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("VALORES AUSENTES")
print("-" * 90)

nulos = df_acustica.isna().sum()

df_nulos = pd.DataFrame({
    "Nulos": nulos,
    "Porcentaje": (nulos / len(df_acustica) * 100).round(2)
})

display(df_nulos)

# ------------------------------------------------------------------------------
# 8. Primeras observaciones
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("PRIMERAS OBSERVACIONES")
print("-" * 90)

display(df_acustica.head(10))

4.1 INSPECCIÓN INICIAL DE CONTAMINACIÓN ACÚSTICA

Archivo: 2024_12Des_XarxaSoroll_EqMonitor_Dades_1Min.csv

Codificación utilizada: utf-8
Separador detectado: ','

Dimensiones: (7163396, 4)
Registros: 7,163,396
Variables: 4

------------------------------------------------------------------------------------------
VARIABLES DISPONIBLES
------------------------------------------------------------------------------------------
01. Timestamp_UTC
02. Timestamp_local
03. Id_Instal
04. Nivell_LAeq_1min

------------------------------------------------------------------------------------------
TIPOS DE DATOS
------------------------------------------------------------------------------------------
Timestamp_UTC        object
Timestamp_local      object
Id_Instal             int64
Nivell_LAeq_1min    float64
dtype: object

------------------------------------------------------------------------------------------
VALORES AUSENTES
-----------------------------------------------------------------

,Nulos,Porcentaje
Timestamp_UTC,0,0.0
Timestamp_local,0,0.0
Id_Instal,0,0.0
Nivell_LAeq_1min,0,0.0



------------------------------------------------------------------------------------------
PRIMERAS OBSERVACIONES
------------------------------------------------------------------------------------------


,Timestamp_UTC,Timestamp_local,Id_Instal,Nivell_LAeq_1min
0,2024-11-30T23:00:52Z,2024-12-01T00:00:52+01:00,496,67.9
1,2024-11-30T23:01:52Z,2024-12-01T00:01:52+01:00,496,67.5
2,2024-11-30T23:02:52Z,2024-12-01T00:02:52+01:00,496,68.7
3,2024-11-30T23:03:52Z,2024-12-01T00:03:52+01:00,496,65.0
4,2024-11-30T23:04:52Z,2024-12-01T00:04:52+01:00,496,67.4
5,2024-11-30T23:05:52Z,2024-12-01T00:05:52+01:00,496,68.9
6,2024-11-30T23:06:52Z,2024-12-01T00:06:52+01:00,496,66.4
7,2024-11-30T23:07:52Z,2024-12-01T00:07:52+01:00,496,66.6
8,2024-11-30T23:08:52Z,2024-12-01T00:08:52+01:00,496,67.7
9,2024-11-30T23:09:52Z,2024-12-01T00:09:52+01:00,496,63.0


### Resultado de la inspección inicial

La inspección del archivo correspondiente a **diciembre de 2024** muestra una estructura formada por **7.163.396 registros y 4 variables**, lo que confirma el elevado volumen de información asociado a la resolución temporal de un minuto.

Las variables disponibles son:

- `Timestamp_UTC`: instante de la medición expresado en UTC;
- `Timestamp_local`: instante de la medición en hora local;
- `Id_Instal`: identificador de la instalación o punto de medida;
- `Nivell_LAeq_1min`: nivel sonoro continuo equivalente registrado durante un intervalo de un minuto, expresado en dB(A).

El archivo se encuentra correctamente estructurado utilizando codificación `UTF-8` y separador por comas. Las variables temporales se encuentran inicialmente almacenadas como texto, `Id_Instal` como variable entera y `Nivell_LAeq_1min` como variable numérica de tipo `float64`.

No se detectan valores ausentes en ninguna de las cuatro variables del archivo analizado.

La presencia simultánea de `Timestamp_UTC` y `Timestamp_local` permitirá conservar la referencia temporal original y utilizar posteriormente la **hora local de Barcelona** para la construcción de los indicadores diarios, evitando errores de asignación de fecha derivados del huso horario y de los cambios entre horario de invierno y verano.

La estructura observada resulta adecuada para continuar con la auditoría temporal, el análisis de las instalaciones de medida y el control de los valores acústicos antes de realizar cualquier agregación temporal.

## 4.2 Inventario y análisis de la estructura de los archivos históricos

Tras la inspección inicial de un archivo correspondiente a 2024, se analiza la estructura del conjunto de archivos disponibles para el periodo 2018–2024.

Dado que la organización y las variables de los archivos pueden haber cambiado a lo largo del periodo de estudio, antes de proceder a su unificación se realiza un inventario de las fuentes disponibles y se comparan sus esquemas de datos.

Para cada archivo se identifican el año de referencia, número de registros, número y nombre de las variables y tipos de datos. Este análisis permitirá determinar si existen diferentes estructuras históricas que requieran procedimientos específicos de tratamiento antes de construir una serie temporal homogénea.

En esta etapa no se realiza todavía ninguna agregación temporal ni transformación de las mediciones acústicas.

In [ ]:
# ==============================================================================
# 4.2 INVENTARIO Y ANÁLISIS DE LA ESTRUCTURA DE LOS ARCHIVOS HISTÓRICOS
# ==============================================================================

import pandas as pd
import glob
import os
import re

# ------------------------------------------------------------------------------
# 1. Ruta base
# ------------------------------------------------------------------------------

ruta_base_ruido = (
    "/content/drive/MyDrive/TFM/07_Contaminacion_Acustica/"
)

# ------------------------------------------------------------------------------
# 2. Localizar todos los CSV dentro de las carpetas BC 2018 - BC 2024
# ------------------------------------------------------------------------------

archivos_ruido = []

for anio in range(2018, 2025):

    patron = os.path.join(
        ruta_base_ruido,
        f"BC {anio}*",
        "*.csv"
    )

    archivos_anio = glob.glob(patron)

    for archivo in archivos_anio:
        archivos_ruido.append((anio, archivo))

archivos_ruido = sorted(
    archivos_ruido,
    key=lambda x: (x[0], x[1])
)

print("=" * 90)
print("4.2 INVENTARIO DE ARCHIVOS DE CONTAMINACIÓN ACÚSTICA")
print("=" * 90)

print(f"\nNúmero total de archivos encontrados: {len(archivos_ruido):,}")

for anio in range(2018, 2025):

    archivos_anio = [
        archivo
        for a, archivo in archivos_ruido
        if a == anio
    ]

    print(
        f"{anio}: "
        f"{len(archivos_anio):,} archivos"
    )

# ------------------------------------------------------------------------------
# 3. Inspeccionar la estructura de cada archivo
# ------------------------------------------------------------------------------

resumen_archivos = []

for anio, archivo in archivos_ruido:

    try:

        # Solo se leen unas pocas filas para estudiar la estructura.
        # Evitamos cargar innecesariamente todos los archivos en memoria.
        df_muestra = pd.read_csv(
            archivo,
            nrows=5,
            low_memory=False
        )

        columnas = tuple(df_muestra.columns.tolist())

        tipos = tuple(
            f"{col}: {dtype}"
            for col, dtype in df_muestra.dtypes.items()
        )

        resumen_archivos.append({
            "Año": anio,
            "Archivo": os.path.basename(archivo),
            "N_columnas": len(columnas),
            "Columnas": columnas,
            "Tipos_muestra": tipos
        })

    except Exception as e:

        resumen_archivos.append({
            "Año": anio,
            "Archivo": os.path.basename(archivo),
            "N_columnas": None,
            "Columnas": None,
            "Tipos_muestra": f"ERROR: {e}"
        })

df_inventario_ruido = pd.DataFrame(resumen_archivos)

# ------------------------------------------------------------------------------
# 4. Mostrar resumen de archivos
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("RESUMEN DE ARCHIVOS")
print("-" * 90)

display(
    df_inventario_ruido[
        [
            "Año",
            "Archivo",
            "N_columnas",
            "Columnas"
        ]
    ]
)

# ------------------------------------------------------------------------------
# 5. Identificar estructuras diferentes
# ------------------------------------------------------------------------------

estructuras = (
    df_inventario_ruido
    .dropna(subset=["Columnas"])
    .groupby("Columnas", dropna=False)
    .agg(
        N_archivos=("Archivo", "count"),
        Año_min=("Año", "min"),
        Año_max=("Año", "max")
    )
    .reset_index()
)

print("\n" + "-" * 90)
print("ESTRUCTURAS DIFERENTES DETECTADAS")
print("-" * 90)

print(
    f"Número de estructuras diferentes: "
    f"{len(estructuras):,}"
)

display(estructuras)

# ------------------------------------------------------------------------------
# 6. Relación año - estructura
# ------------------------------------------------------------------------------

# Las columnas están almacenadas como tuplas.
# En lugar de utilizar .map() directamente sobre ellas,
# asignamos el identificador de estructura mediante comparación explícita.

estructuras_unicas = (
    df_inventario_ruido["Columnas"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

def identificar_estructura(columnas):
    for i, estructura in enumerate(estructuras_unicas, start=1):
        if columnas == estructura:
            return i
    return pd.NA

df_inventario_ruido["Estructura"] = (
    df_inventario_ruido["Columnas"]
    .apply(identificar_estructura)
)

print("\n" + "-" * 90)
print("ESTRUCTURA UTILIZADA POR AÑO")
print("-" * 90)

resumen_estructura_anual = (
    df_inventario_ruido
    .groupby(["Año", "Estructura"], dropna=False)
    .agg(
        N_archivos=("Archivo", "count")
    )
    .reset_index()
)

display(resumen_estructura_anual)

# ------------------------------------------------------------------------------
# 7. Descripción de las estructuras detectadas
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("DESCRIPCIÓN DE LAS ESTRUCTURAS")
print("-" * 90)

for i, estructura in enumerate(estructuras_unicas, start=1):

    print(f"\nEstructura {i}:")

    for columna in estructura:
        print(f"  - {columna}")

4.2 INVENTARIO DE ARCHIVOS DE CONTAMINACIÓN ACÚSTICA

Número total de archivos encontrados: 24
2018: 2 archivos
2019: 2 archivos
2020: 2 archivos
2021: 2 archivos
2022: 2 archivos
2023: 2 archivos
2024: 12 archivos

------------------------------------------------------------------------------------------
RESUMEN DE ARCHIVOS
------------------------------------------------------------------------------------------


,Año,Archivo,N_columnas,Columnas
0,2018,2018_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
1,2018,2018_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
2,2019,2019_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
3,2019,2019_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
4,2020,2020_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
5,2020,2020_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
6,2021,2021_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
7,2021,2021_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
8,2022,2022_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"
9,2022,2022_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,6,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)"



------------------------------------------------------------------------------------------
ESTRUCTURAS DIFERENTES DETECTADAS
------------------------------------------------------------------------------------------
Número de estructuras diferentes: 2


,Columnas,N_archivos,Año_min,Año_max
0,"(Any, Mes, Dia, Hora, Id_Instal, Nivell_LAeq_1h)",12,2018,2023
1,"(Timestamp_UTC, Timestamp_local, Id_Instal, Ni...",12,2024,2024



------------------------------------------------------------------------------------------
ESTRUCTURA UTILIZADA POR AÑO
------------------------------------------------------------------------------------------


,Año,Estructura,N_archivos
0,2018,1,2
1,2019,1,2
2,2020,1,2
3,2021,1,2
4,2022,1,2
5,2023,1,2
6,2024,2,12



------------------------------------------------------------------------------------------
DESCRIPCIÓN DE LAS ESTRUCTURAS
------------------------------------------------------------------------------------------

Estructura 1:
  - Any
  - Mes
  - Dia
  - Hora
  - Id_Instal
  - Nivell_LAeq_1h

Estructura 2:
  - Timestamp_UTC
  - Timestamp_local
  - Id_Instal
  - Nivell_LAeq_1min


### Resultado del inventario y análisis estructural

El inventario identifica un total de **24 archivos de contaminación acústica** correspondientes al periodo 2018–2024. La organización de la fuente presenta dos configuraciones claramente diferenciadas a lo largo del periodo de estudio.

Para los años **2018–2023** se dispone de dos archivos semestrales por año, con una estructura común formada por seis variables:

- `Any`: año de la observación;
- `Mes`: mes;
- `Dia`: día;
- `Hora`: hora de la medición;
- `Id_Instal`: identificador de la instalación de medida;
- `Nivell_LAeq_1h`: nivel sonoro continuo equivalente correspondiente a un intervalo de una hora.

En **2024** la fuente modifica tanto su organización como su resolución temporal. Se dispone de doce archivos mensuales con cuatro variables:

- `Timestamp_UTC`: instante de medida en UTC;
- `Timestamp_local`: instante de medida en hora local;
- `Id_Instal`: identificador de la instalación;
- `Nivell_LAeq_1min`: nivel sonoro continuo equivalente correspondiente a un intervalo de un minuto.

Por tanto, se identifican **dos estructuras de datos diferentes**: una serie con resolución horaria para 2018–2023 y una serie con resolución de un minuto para 2024.

Esta heterogeneidad temporal deberá considerarse explícitamente durante el procesamiento. Los dos bloques serán tratados inicialmente de forma independiente y posteriormente transformados a una **resolución diaria común**, utilizando un procedimiento de agregación acústica adecuado y controles de cobertura temporal antes de su unificación.

La existencia de diferentes resoluciones en la fuente original no impide su integración posterior, dado que la escala temporal objetivo del estudio es diaria.

## 4.3 Carga y auditoría del bloque histórico 2018–2023

Una vez identificada la existencia de dos estructuras temporales diferentes, se procesa de forma independiente el bloque correspondiente al periodo **2018–2023**, caracterizado por disponer de mediciones de `LAeq` con resolución horaria.

Los doce archivos semestrales correspondientes a este periodo presentan una estructura homogénea, formada por las variables de año, mes, día, hora, identificador de instalación y nivel sonoro equivalente horario (`Nivell_LAeq_1h`).

En esta etapa se realiza la carga conjunta de los archivos conservando inicialmente el formato original de las variables. En particular, la variable `Hora` se inspecciona antes de efectuar cualquier conversión, con el fin de preservar correctamente la referencia temporal de las observaciones.

La auditoría comprende:

- número de registros incorporados por archivo;
- cobertura temporal del periodo 2018–2023;
- número de instalaciones de medida;
- formato y contenido de la variable horaria;
- presencia de valores ausentes;
- identificación de posibles duplicados temporales;
- rango y distribución de `Nivell_LAeq_1h`;
- cobertura de observaciones por año.

En esta fase no se realiza todavía ninguna agregación diaria ni imputación de valores.

In [ ]:
# ==============================================================================
# 4.3 CARGA Y AUDITORÍA DEL BLOQUE HISTÓRICO 2018-2023
# ==============================================================================

import pandas as pd
import numpy as np
import glob
import os

# ------------------------------------------------------------------------------
# 1. Localizar los archivos correspondientes a 2018-2023
# ------------------------------------------------------------------------------

archivos_18_23 = []

for anio in range(2018, 2024):

    patron = os.path.join(
        ruta_base_ruido,
        f"BC {anio}*",
        "*.csv"
    )

    archivos_18_23.extend(glob.glob(patron))

archivos_18_23 = sorted(archivos_18_23)

print("=" * 90)
print("4.3 CARGA Y AUDITORÍA DEL BLOQUE 2018-2023")
print("=" * 90)

print(f"\nArchivos encontrados: {len(archivos_18_23)}")

# ------------------------------------------------------------------------------
# 2. Cargar y concatenar los archivos
# ------------------------------------------------------------------------------

lista_df = []
resumen_carga = []

for archivo in archivos_18_23:

    df_temp = pd.read_csv(
        archivo,
        low_memory=False
    )

    resumen_carga.append({
        "Archivo": os.path.basename(archivo),
        "Registros": len(df_temp)
    })

    lista_df.append(df_temp)

df_ruido_18_23 = pd.concat(
    lista_df,
    ignore_index=True
)

del lista_df

print("\nCarga completada.")

df_resumen_carga = pd.DataFrame(resumen_carga)

display(df_resumen_carga)

# ------------------------------------------------------------------------------
# 3. Comprobar la estructura obtenida
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("ESTRUCTURA DEL DATASET")
print("-" * 90)

print(f"Dimensiones iniciales: {df_ruido_18_23.shape}")

print("\nVariables:")
print(df_ruido_18_23.columns.tolist())

print("\nTipos de datos originales:")
print(df_ruido_18_23.dtypes)

# ------------------------------------------------------------------------------
# 4. Diagnóstico de la variable Hora ANTES de transformarla
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("DIAGNÓSTICO DE LA VARIABLE HORA")
print("-" * 90)

print(f"Tipo original: {df_ruido_18_23['Hora'].dtype}")

print("\nPrimeros 30 valores originales:")
print(df_ruido_18_23["Hora"].head(30).tolist())

print("\nPrimeros valores únicos:")
print(
    df_ruido_18_23["Hora"]
    .drop_duplicates()
    .head(30)
    .tolist()
)

print(
    f"\nNúmero de valores diferentes: "
    f"{df_ruido_18_23['Hora'].nunique(dropna=False):,}"
)

print(
    f"Nulos originales en Hora: "
    f"{df_ruido_18_23['Hora'].isna().sum():,}"
)

print("\nValores de Hora más frecuentes:")

display(
    df_ruido_18_23["Hora"]
    .astype(str)
    .value_counts(dropna=False)
    .head(30)
    .to_frame("Frecuencia")
)

# ------------------------------------------------------------------------------
# 5. Conversión segura de las variables numéricas
# ------------------------------------------------------------------------------

# No transformamos todavía Hora.
# Primero preservamos exactamente el contenido original.

for columna in ["Any", "Mes", "Dia", "Id_Instal"]:

    df_ruido_18_23[columna] = pd.to_numeric(
        df_ruido_18_23[columna],
        errors="coerce"
    )

df_ruido_18_23["Nivell_LAeq_1h"] = pd.to_numeric(
    df_ruido_18_23["Nivell_LAeq_1h"],
    errors="coerce"
)

# ------------------------------------------------------------------------------
# 6. Construcción de la fecha
# ------------------------------------------------------------------------------

df_ruido_18_23["Fecha"] = pd.to_datetime(
    dict(
        year=df_ruido_18_23["Any"],
        month=df_ruido_18_23["Mes"],
        day=df_ruido_18_23["Dia"]
    ),
    errors="coerce"
)

# ------------------------------------------------------------------------------
# 7. Dimensiones y cobertura temporal
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("DIMENSIONES Y COBERTURA TEMPORAL")
print("-" * 90)

print(f"Dimensiones: {df_ruido_18_23.shape}")

print(
    f"Periodo: "
    f"{df_ruido_18_23['Fecha'].min()} - "
    f"{df_ruido_18_23['Fecha'].max()}"
)

print(
    f"Instalaciones únicas: "
    f"{df_ruido_18_23['Id_Instal'].nunique():,}"
)

print(
    f"Fechas no válidas: "
    f"{df_ruido_18_23['Fecha'].isna().sum():,}"
)

# ------------------------------------------------------------------------------
# 8. Valores ausentes
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("VALORES AUSENTES")
print("-" * 90)

nulos = df_ruido_18_23.isna().sum()

df_nulos_18_23 = pd.DataFrame({
    "Nulos": nulos,
    "Porcentaje": (
        nulos / len(df_ruido_18_23) * 100
    ).round(2)
})

display(df_nulos_18_23)

# ------------------------------------------------------------------------------
# 9. Comprobación preliminar de duplicados
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("CONTROL PRELIMINAR DE DUPLICADOS")
print("-" * 90)

# Solo se considera válida esta comprobación si Hora contiene información.

if df_ruido_18_23["Hora"].notna().any():

    duplicados = df_ruido_18_23.duplicated(
        subset=[
            "Any",
            "Mes",
            "Dia",
            "Hora",
            "Id_Instal"
        ]
    ).sum()

    print(
        f"Duplicados Año-Mes-Día-Hora-Id_Instal: "
        f"{duplicados:,}"
    )

else:

    print(
        "No se calcula todavía el número de duplicados temporales "
        "porque la variable Hora no contiene valores válidos."
    )

# ------------------------------------------------------------------------------
# 10. Control de la variable acústica
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("CONTROL DE NIVELL_LAEQ_1H")
print("-" * 90)

n_validos = df_ruido_18_23["Nivell_LAeq_1h"].notna().sum()
n_nulos = df_ruido_18_23["Nivell_LAeq_1h"].isna().sum()

print(f"Valores disponibles: {n_validos:,}")
print(f"Valores ausentes: {n_nulos:,}")

print(
    f"Mínimo observado: "
    f"{df_ruido_18_23['Nivell_LAeq_1h'].min():.2f} dB(A)"
)

print(
    f"Máximo observado: "
    f"{df_ruido_18_23['Nivell_LAeq_1h'].max():.2f} dB(A)"
)

print("\nEstadísticos descriptivos:")

display(
    df_ruido_18_23["Nivell_LAeq_1h"]
    .describe()
    .to_frame("Nivell_LAeq_1h")
)

# ------------------------------------------------------------------------------
# 11. Exploración inicial de valores extremos
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("EXPLORACIÓN INICIAL DE VALORES EXTREMOS")
print("-" * 90)

print(
    f"Valores < 20 dB(A): "
    f"{(df_ruido_18_23['Nivell_LAeq_1h'] < 20).sum():,}"
)

print(
    f"Valores > 100 dB(A): "
    f"{(df_ruido_18_23['Nivell_LAeq_1h'] > 100).sum():,}"
)

print(
    f"Valores > 120 dB(A): "
    f"{(df_ruido_18_23['Nivell_LAeq_1h'] > 120).sum():,}"
)

# Estos registros NO se eliminan.
# Únicamente se cuantifican para su posterior evaluación.

# ------------------------------------------------------------------------------
# 12. Cobertura por año
# ------------------------------------------------------------------------------

resumen_anual = (
    df_ruido_18_23
    .groupby("Any")
    .agg(
        Registros=("Nivell_LAeq_1h", "size"),
        Valores_validos=("Nivell_LAeq_1h", "count"),
        Instalaciones=("Id_Instal", "nunique")
    )
    .reset_index()
)

resumen_anual["Cobertura_valida_%"] = (
    resumen_anual["Valores_validos"]
    / resumen_anual["Registros"]
    * 100
).round(2)

print("\n" + "-" * 90)
print("RESUMEN POR AÑO")
print("-" * 90)

display(resumen_anual)

# ------------------------------------------------------------------------------
# 13. Primeras observaciones
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("PRIMERAS OBSERVACIONES")
print("-" * 90)

display(df_ruido_18_23.head(10))

4.3 CARGA Y AUDITORÍA DEL BLOQUE 2018-2023

Archivos encontrados: 12

Carga completada.


,Archivo,Registros
0,2018_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,354272
1,2018_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,463528
2,2019_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,482844
3,2019_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,515600
4,2020_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,470654
5,2020_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,472345
6,2021_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,495518
7,2021_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,521276
8,2022_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,490802
9,2022_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,602657



------------------------------------------------------------------------------------------
ESTRUCTURA DEL DATASET
------------------------------------------------------------------------------------------
Dimensiones iniciales: (6156724, 6)

Variables:
['Any', 'Mes', 'Dia', 'Hora', 'Id_Instal', 'Nivell_LAeq_1h']

Tipos de datos originales:
Any                 int64
Mes                 int64
Dia                 int64
Hora               object
Id_Instal           int64
Nivell_LAeq_1h    float64
dtype: object

------------------------------------------------------------------------------------------
DIAGNÓSTICO DE LA VARIABLE HORA
------------------------------------------------------------------------------------------
Tipo original: object

Primeros 30 valores originales:
['0:00', '1:00', '2:00', '3:00', '4:00', '5:00', '6:00', '7:00', '8:00', '9:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00', '18:00', '19:00', '20:00', '21:00', '22:00', '23:00', '0:00', '1

,Frecuencia
Hora,
0:00,257318
23:00,256997
1:00,256989
9:00,256830
21:00,256813
3:00,256806
22:00,256704
19:00,256688
10:00,256676



------------------------------------------------------------------------------------------
DIMENSIONES Y COBERTURA TEMPORAL
------------------------------------------------------------------------------------------
Dimensiones: (6156724, 7)
Periodo: 2018-01-01 00:00:00 - 2023-12-31 00:00:00
Instalaciones únicas: 532
Fechas no válidas: 0

------------------------------------------------------------------------------------------
VALORES AUSENTES
------------------------------------------------------------------------------------------


...

,Nivell_LAeq_1h
count,6.156724e+06
mean,6.233048e+01
std,7.145948e+00
min,1.000000e-01
25%,5.840000e+01
50%,6.310000e+01
75%,6.720000e+01
max,1.361000e+02



------------------------------------------------------------------------------------------
EXPLORACIÓN INICIAL DE VALORES EXTREMOS
------------------------------------------------------------------------------------------
Valores < 20 dB(A): 892
Valores > 100 dB(A): 103
Valores > 120 dB(A): 1

------------------------------------------------------------------------------------------
RESUMEN POR AÑO
------------------------------------------------------------------------------------------


,Any,Registros,Valores_validos,Instalaciones,Cobertura_valida_%
0,2018,817800,817800,174,100.0
1,2019,998444,998444,162,100.0
2,2020,942999,942999,160,100.0
3,2021,1016794,1016794,186,100.0
4,2022,1093459,1093459,241,100.0
5,2023,1287228,1287228,229,100.0



------------------------------------------------------------------------------------------
PRIMERAS OBSERVACIONES
------------------------------------------------------------------------------------------


,Any,Mes,Dia,Hora,Id_Instal,Nivell_LAeq_1h,Fecha
0,2018,1,1,0:00,2347,72.2,2018-01-01
1,2018,1,1,1:00,2347,69.3,2018-01-01
2,2018,1,1,2:00,2347,65.9,2018-01-01
3,2018,1,1,3:00,2347,64.1,2018-01-01
4,2018,1,1,4:00,2347,63.3,2018-01-01
5,2018,1,1,5:00,2347,60.5,2018-01-01
6,2018,1,1,6:00,2347,56.4,2018-01-01
7,2018,1,1,7:00,2347,54.9,2018-01-01
8,2018,1,1,8:00,2347,52.7,2018-01-01
9,2018,1,1,9:00,2347,52.7,2018-01-01


### Resultado de la auditoría 2018–2023

La carga conjunta de los doce archivos semestrales genera un conjunto de **6.156.724 observaciones horarias**, correspondientes al periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2023**, con un total de **532 instalaciones de medida** identificadas a lo largo del periodo.

La variable `Hora` se encuentra almacenada originalmente como texto, siguiendo el formato `0:00`–`23:00`. La inspección confirma la existencia de las **24 horas del día**, sin valores ausentes, por lo que se conserva esta información para la posterior construcción de la dimensión temporal.

La auditoría de calidad muestra una elevada integridad del conjunto:

- **0 fechas no válidas**;
- **0 valores ausentes** en las variables originales;
- **0 duplicados** para la combinación año-mes-día-hora-instalación;
- **100 % de disponibilidad de `Nivell_LAeq_1h`** en todos los años analizados.

El nivel sonoro equivalente horario presenta una mediana de **63,1 dB(A)** y un rango intercuartílico aproximado de **58,4–67,2 dB(A)**. Se identifican algunos valores extremos —892 observaciones inferiores a 20 dB(A), 103 superiores a 100 dB(A) y una superior a 120 dB(A)—, cuya frecuencia resulta muy reducida respecto al volumen total de datos. Estos registros se conservan inicialmente para evitar introducir modificaciones no justificadas sobre las mediciones originales.

El número de instalaciones disponibles varía entre años, pasando de 174 instalaciones en 2018 a un máximo de 241 en 2022. Esta variación deberá tenerse en cuenta posteriormente al analizar la cobertura espacial y temporal de la red.

El bloque **2018–2023 queda validado para continuar con su procesamiento temporal**, manteniendo los datos horarios originales hasta la fase de construcción del indicador acústico diario.

## 4.4 Procesamiento eficiente y auditoría del bloque 2024

Los datos acústicos correspondientes a 2024 presentan una resolución temporal de un minuto y un volumen aproximado de **81 millones de observaciones**, considerablemente superior al del periodo 2018–2023.

Para evitar la carga simultánea del conjunto completo en memoria, los doce archivos mensuales se procesan de forma secuencial. Cada archivo se carga, valida y reduce inmediatamente a escala instalación-día antes de liberar la memoria correspondiente.

Para cada combinación `Fecha–Id_Instal` se conserva:

- el número total de observaciones registradas;
- el número de mediciones acústicas válidas;
- la fracción de cobertura respecto a los 1.440 minutos teóricos de un día;
- la suma de las intensidades acústicas equivalentes, necesaria para obtener posteriormente el `LAeq` diario mediante agregación energética.

Esta estrategia permite conservar la información necesaria para evaluar la calidad y cobertura de cada serie diaria sin mantener en memoria las aproximadamente 81 millones de observaciones minutales originales.

In [ ]:
# ==============================================================================
# 4.4 PROCESAMIENTO EFICIENTE Y AUDITORÍA DEL BLOQUE 2024
# ==============================================================================

import pandas as pd
import numpy as np
import glob
import os
import gc

print("=" * 90)
print("4.4 PROCESAMIENTO EFICIENTE Y AUDITORÍA DEL BLOQUE 2024")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Localizar archivos
# ------------------------------------------------------------------------------

archivos_2024 = sorted(
    glob.glob(
        os.path.join(
            ruta_base_ruido,
            "BC 2024*",
            "*.csv"
        )
    )
)

print(f"\nArchivos encontrados: {len(archivos_2024)}")

for archivo in archivos_2024:
    print(" -", os.path.basename(archivo))

# ------------------------------------------------------------------------------
# 2. Contenedores de resultados REDUCIDOS
# ------------------------------------------------------------------------------

resultados_diarios_2024 = []
auditoria_mensual_2024 = []

# ------------------------------------------------------------------------------
# 3. Procesamiento secuencial
# ------------------------------------------------------------------------------

for i, archivo in enumerate(archivos_2024, start=1):

    nombre = os.path.basename(archivo)

    print("\n" + "-" * 90)
    print(f"[{i:02d}/{len(archivos_2024):02d}] Procesando: {nombre}")
    print("-" * 90)

    # --------------------------------------------------------------------------
    # 3.1 Carga SOLO de las columnas necesarias
    # --------------------------------------------------------------------------

    df_mes = pd.read_csv(
        archivo,
        usecols=[
            "Timestamp_UTC",
            "Timestamp_local",
            "Id_Instal",
            "Nivell_LAeq_1min"
        ],
        low_memory=False
    )

    n_registros = len(df_mes)

    # --------------------------------------------------------------------------
    # 3.2 Conversión del nivel acústico
    # --------------------------------------------------------------------------

    df_mes["Nivell_LAeq_1min"] = pd.to_numeric(
        df_mes["Nivell_LAeq_1min"],
        errors="coerce"
    )

    n_nulos_laeq = df_mes["Nivell_LAeq_1min"].isna().sum()

    # --------------------------------------------------------------------------
    # 3.3 Fecha LOCAL
    #
    # Extraemos YYYY-MM-DD directamente del Timestamp_local.
    # Así evitamos problemas derivados del cambio CET/CEST.
    # --------------------------------------------------------------------------

    df_mes["Fecha"] = pd.to_datetime(
        df_mes["Timestamp_local"]
        .astype(str)
        .str.slice(0, 10),
        errors="coerce"
    )

    n_fechas_invalidas = df_mes["Fecha"].isna().sum()

    # --------------------------------------------------------------------------
    # 3.4 Duplicados instalación-instante
    # --------------------------------------------------------------------------

    n_duplicados = df_mes.duplicated(
        subset=["Timestamp_UTC", "Id_Instal"]
    ).sum()

    # --------------------------------------------------------------------------
    # 3.5 Controles descriptivos
    # --------------------------------------------------------------------------

    minimo = df_mes["Nivell_LAeq_1min"].min()
    maximo = df_mes["Nivell_LAeq_1min"].max()

    menores_20 = (
        df_mes["Nivell_LAeq_1min"] < 20
    ).sum()

    mayores_100 = (
        df_mes["Nivell_LAeq_1min"] > 100
    ).sum()

    mayores_120 = (
        df_mes["Nivell_LAeq_1min"] > 120
    ).sum()

    # --------------------------------------------------------------------------
    # 3.6 Transformación energética
    #
    # Para niveles acústicos:
    #
    #       I = 10^(L/10)
    #
    # NO hacemos una media aritmética directa de los dB.
    # --------------------------------------------------------------------------

    df_mes["Energia_acustica"] = np.where(
        df_mes["Nivell_LAeq_1min"].notna(),
        10.0 ** (df_mes["Nivell_LAeq_1min"] / 10.0),
        np.nan
    )

    # --------------------------------------------------------------------------
    # 3.7 Reducción inmediata a Fecha × instalación
    # --------------------------------------------------------------------------

    diario_mes = (
        df_mes
        .groupby(
            ["Fecha", "Id_Instal"],
            as_index=False
        )
        .agg(
            N_registros=("Nivell_LAeq_1min", "size"),
            N_validos=("Nivell_LAeq_1min", "count"),
            Suma_energia=("Energia_acustica", "sum")
        )
    )

    # Cobertura respecto a 1440 minutos/día
    diario_mes["Cobertura_%"] = (
        diario_mes["N_validos"] / 1440 * 100
    ).round(2)

    resultados_diarios_2024.append(diario_mes)

    # --------------------------------------------------------------------------
    # 3.8 Auditoría del mes
    # --------------------------------------------------------------------------

    auditoria_mensual_2024.append({
        "Archivo": nombre,
        "Registros": n_registros,
        "Fecha_min": df_mes["Fecha"].min(),
        "Fecha_max": df_mes["Fecha"].max(),
        "Instalaciones": df_mes["Id_Instal"].nunique(),
        "Nulos_LAeq": n_nulos_laeq,
        "Fechas_invalidas": n_fechas_invalidas,
        "Duplicados_instante_sensor": n_duplicados,
        "LAeq_min": minimo,
        "LAeq_max": maximo,
        "LAeq_<20": menores_20,
        "LAeq_>100": mayores_100,
        "LAeq_>120": mayores_120,
        "Sensor_dia": len(diario_mes)
    })

    print(f"Registros originales : {n_registros:,}")
    print(f"Instalaciones        : {df_mes['Id_Instal'].nunique():,}")
    print(f"Sensor-día generados : {len(diario_mes):,}")
    print(f"Nulos LAeq           : {n_nulos_laeq:,}")
    print(f"Fechas inválidas     : {n_fechas_invalidas:,}")
    print(f"Duplicados           : {n_duplicados:,}")

    # --------------------------------------------------------------------------
    # 3.9 LIBERAR EL MES DE RAM
    # --------------------------------------------------------------------------

    del df_mes
    del diario_mes

    gc.collect()

    print("✓ Mes procesado y memoria liberada.")

# ==============================================================================
# 4. UNIÓN DE LOS RESULTADOS REDUCIDOS
# ==============================================================================

df_ruido_2024_sensor_dia = pd.concat(
    resultados_diarios_2024,
    ignore_index=True
)

del resultados_diarios_2024
gc.collect()

df_auditoria_2024 = pd.DataFrame(
    auditoria_mensual_2024
)

# ==============================================================================
# 5. RESULTADOS
# ==============================================================================

print("\n" + "=" * 90)
print("AUDITORÍA MENSUAL 2024")
print("=" * 90)

display(df_auditoria_2024)

print("\n" + "=" * 90)
print("DATASET REDUCIDO SENSOR-DÍA")
print("=" * 90)

print(
    f"Dimensiones: "
    f"{df_ruido_2024_sensor_dia.shape}"
)

print(
    f"Periodo: "
    f"{df_ruido_2024_sensor_dia['Fecha'].min().date()} - "
    f"{df_ruido_2024_sensor_dia['Fecha'].max().date()}"
)

print(
    f"Instalaciones únicas: "
    f"{df_ruido_2024_sensor_dia['Id_Instal'].nunique():,}"
)

print(
    f"Duplicados Fecha-Instalación: "
    f"{df_ruido_2024_sensor_dia.duplicated(['Fecha', 'Id_Instal']).sum():,}"
)

print("\nCobertura sensor-día:")

display(
    df_ruido_2024_sensor_dia["Cobertura_%"]
    .describe()
    .to_frame("Cobertura_%")
)

print("\nPrimeras observaciones:")

display(
    df_ruido_2024_sensor_dia.head(10)
)

print("\n✓ 2024 procesado completamente sin mantener los ~81 millones")
print("  de registros simultáneamente en memoria.")

4.4 PROCESAMIENTO EFICIENTE Y AUDITORÍA DEL BLOQUE 2024

Archivos encontrados: 12
 - 2024_01Gen_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_02Feb_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_03Mar_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_04Abr_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_05Mai_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_06Jun_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_07Jul_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_08Ago_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_09Set_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_10Oct_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_11Nov_XarxaSoroll_EqMonitor_Dades_1Min.csv
 - 2024_12Des_XarxaSoroll_EqMonitor_Dades_1Min.csv

------------------------------------------------------------------------------------------
[01/12] Procesando: 2024_01Gen_XarxaSoroll_EqMonitor_Dades_1Min.csv
------------------------------------------------------------------------------------------
Registros originales : 6,674,746
Instalaciones        :

,Archivo,Registros,Fecha_min,Fecha_max,Instalaciones,Nulos_LAeq,Fechas_invalidas,Duplicados_instante_sensor,LAeq_min,LAeq_max,LAeq_<20,LAeq_>100,LAeq_>120,Sensor_dia
0,2024_01Gen_XarxaSoroll_EqMonitor_Dades_1Min.csv,6674746,2023-12-31,2024-01-31,161,0,0,44,9.5,110.2,1053,53,0,4674
1,2024_02Feb_XarxaSoroll_EqMonitor_Dades_1Min.csv,6359738,2024-01-31,2024-02-29,160,0,0,48,9.4,114.3,3113,184,0,4455
2,2024_03Mar_XarxaSoroll_EqMonitor_Dades_1Min.csv,6879698,2024-02-29,2024-03-31,166,0,0,36,9.6,109.9,1388,159,0,4846
3,2024_04Abr_XarxaSoroll_EqMonitor_Dades_1Min.csv,6457453,2024-03-31,2024-04-30,158,0,0,36,9.7,112.8,2235,243,0,4544
4,2024_05Mai_XarxaSoroll_EqMonitor_Dades_1Min.csv,6831044,2024-04-30,2024-05-31,164,0,0,46,11.0,108.1,770,80,0,4839
5,2024_06Jun_XarxaSoroll_EqMonitor_Dades_1Min.csv,6503364,2024-05-31,2024-06-30,159,0,0,22,18.0,105.1,276,40,0,4570
6,2024_07Jul_XarxaSoroll_EqMonitor_Dades_1Min.csv,6867986,2024-06-30,2024-07-31,164,0,0,67,17.6,102.6,899,13,0,4853
7,2024_08Ago_XarxaSoroll_EqMonitor_Dades_1Min.csv,7034526,2024-07-31,2024-08-31,166,0,0,27,30.9,103.7,0,60,0,5031
8,2024_09Set_XarxaSoroll_EqMonitor_Dades_1Min.csv,6573829,2024-08-31,2024-09-30,169,0,0,28,16.0,112.6,186,57,0,4743
9,2024_10Oct_XarxaSoroll_EqMonitor_Dades_1Min.csv,6794839,2024-09-30,2024-10-31,177,0,0,49,24.3,109.4,0,24,0,4852



DATASET REDUCIDO SENSOR-DÍA
Dimensiones: (57272, 6)
Periodo: 2023-12-31 - 2024-12-31
Instalaciones únicas: 244
Duplicados Fecha-Instalación: 112

Cobertura sensor-día:


,Cobertura_%
count,57272.000000
mean,98.207631
std,9.261769
min,0.070000
25%,100.000000
50%,100.000000
75%,100.000000
max,104.510000



Primeras observaciones:


,Fecha,Id_Instal,N_registros,N_validos,Suma_energia,Cobertura_%
0,2023-12-31,8428,60,60,1.427913e+08,4.17
1,2023-12-31,8506,60,60,1.451369e+08,4.17
2,2023-12-31,8627,60,60,1.166716e+09,4.17
3,2023-12-31,8646,60,60,1.011204e+09,4.17
4,2023-12-31,8746,60,60,5.440292e+09,4.17
5,2023-12-31,8766,60,60,2.027901e+08,4.17
6,2024-01-01,496,1440,1440,1.115318e+10,100.00
7,2024-01-01,497,1440,1440,1.074107e+10,100.00
8,2024-01-01,651,1440,1440,4.518868e+09,100.00
9,2024-01-01,659,1440,1440,8.262497e+08,100.00



✓ 2024 procesado completamente sin mantener los ~81 millones
  de registros simultáneamente en memoria.


### Resultados del procesamiento del bloque 2024

El procesamiento secuencial de los 12 archivos mensuales permitió analizar un volumen de aproximadamente **81 millones de registros acústicos a resolución de un minuto** sin mantener simultáneamente el conjunto completo en memoria. Cada archivo mensual fue reducido inmediatamente a escala `Fecha–Id_Instal`, obteniéndose finalmente **57.272 observaciones sensor-día** correspondientes a **244 instalaciones únicas**.

La auditoría de los archivos originales muestra una elevada integridad de los datos: no se detectaron valores ausentes en `Nivell_LAeq_1min` ni fechas inválidas en ninguno de los doce meses. Los niveles registrados presentan valores máximos mensuales comprendidos aproximadamente entre 101 y 114 dB, sin observaciones superiores a 120 dB.

La cobertura temporal del conjunto reducido es, en general, muy elevada. La cobertura media alcanza el **98,21 %**, mientras que los percentiles 25, 50 y 75 presentan una cobertura del **100 %**, indicando que la mayor parte de las combinaciones sensor-día dispone de las 1.440 observaciones minutales teóricas.

La auditoría también identifica algunas particularidades que requieren una comprobación específica antes de calcular el indicador acústico diario definitivo:

- se detectan **112 duplicados** en la combinación `Fecha–Id_Instal`;
- existen observaciones con cobertura superior al 100 %, alcanzándose un máximo del **104,51 %**;
- los archivos mensuales presentan pequeños solapamientos en sus fechas límite, incluyendo observaciones correspondientes al último día del mes anterior;
- aparecen días frontera con coberturas muy reducidas, como las observaciones del 31/12/2023 incluidas en el archivo de enero de 2024.

Por este motivo, antes de obtener el `LAeq` diario definitivo se realizará una auditoría específica de los solapamientos temporales y de las coberturas anómalas. No se eliminan ni corrigen automáticamente estas observaciones en esta fase, preservando la trazabilidad de los datos originales.

La reducción realizada conserva para cada combinación sensor-día el número de registros, el número de observaciones válidas, la cobertura temporal y la suma de energía acústica. Esta última permitirá calcular posteriormente el **LAeq diario mediante agregación energética**, evitando realizar una media aritmética directa de niveles expresados en decibelios.

## 4.5 Auditoría y resolución de solapamientos temporales en 2024

Tras la reducción de los datos minutales de 2024 a escala sensor-día se detectaron algunos registros duplicados en la clave `Fecha–Id_Instal` y coberturas puntualmente superiores al 100 %.

Estas anomalías pueden estar relacionadas con el solapamiento temporal existente entre archivos mensuales consecutivos, que incluyen parcialmente observaciones correspondientes al día anterior o posterior al mes nominal.

En esta etapa se:

1. restringen los registros al periodo estricto comprendido entre el 1 de enero y el 31 de diciembre de 2024;
2. identifican las combinaciones `Fecha–Id_Instal` repetidas;
3. consolidan los registros duplicados sumando únicamente sus componentes aditivos (`N_registros`, `N_validos` y `Suma_energia`);
4. recalcula la cobertura temporal tras la consolidación;
5. auditan específicamente las coberturas superiores al 100 %.

No se realiza todavía ninguna imputación ni se establece un umbral mínimo de cobertura. El objetivo es obtener una estructura temporal única y trazable antes de definir los criterios de validez del indicador acústico diario.

In [ ]:
# ==============================================================================
# 4.5 AUDITORÍA Y RESOLUCIÓN DE SOLAPAMIENTOS TEMPORALES - 2024
# ==============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("4.5 AUDITORÍA Y RESOLUCIÓN DE SOLAPAMIENTOS TEMPORALES - 2024")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Copia de trabajo
# ------------------------------------------------------------------------------

df_2024 = df_ruido_2024_sensor_dia.copy()

df_2024["Fecha"] = pd.to_datetime(
    df_2024["Fecha"],
    errors="coerce"
)

print(f"\nRegistros iniciales sensor-día: {len(df_2024):,}")

# ------------------------------------------------------------------------------
# 2. Restringir estrictamente al año 2024
# ------------------------------------------------------------------------------

fuera_2024 = ~df_2024["Fecha"].between(
    "2024-01-01",
    "2024-12-31"
)

print(
    f"Registros sensor-día fuera de 2024: "
    f"{fuera_2024.sum():,}"
)

if fuera_2024.sum() > 0:

    print("\nFechas externas detectadas:")

    display(
        df_2024.loc[
            fuera_2024,
            ["Fecha", "Id_Instal", "N_registros",
             "N_validos", "Cobertura_%"]
        ]
        .sort_values(["Fecha", "Id_Instal"])
        .head(20)
    )

df_2024 = df_2024.loc[~fuera_2024].copy()

print(
    f"\nRegistros tras restringir al año 2024: "
    f"{len(df_2024):,}"
)

# ------------------------------------------------------------------------------
# 3. Identificar duplicados Fecha × Id_Instal
# ------------------------------------------------------------------------------

mascara_duplicados = df_2024.duplicated(
    subset=["Fecha", "Id_Instal"],
    keep=False
)

n_filas_duplicadas = mascara_duplicados.sum()

n_claves_duplicadas = (
    df_2024.loc[
        mascara_duplicados,
        ["Fecha", "Id_Instal"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "-" * 90)
print("DUPLICADOS FECHA-INSTALACIÓN")
print("-" * 90)

print(
    f"Filas pertenecientes a claves repetidas: "
    f"{n_filas_duplicadas:,}"
)

print(
    f"Combinaciones Fecha-Id_Instal repetidas: "
    f"{n_claves_duplicadas:,}"
)

if n_filas_duplicadas > 0:

    print("\nMuestra de registros repetidos:")

    display(
        df_2024.loc[mascara_duplicados]
        .sort_values(["Fecha", "Id_Instal"])
        .head(20)
    )

# ------------------------------------------------------------------------------
# 4. Consolidar solapamientos
#
# IMPORTANTE:
# N_registros, N_validos y Suma_energia son magnitudes aditivas.
# Por ello pueden sumarse cuando una misma combinación Fecha-Sensor
# aparece fragmentada entre dos archivos mensuales.
# ------------------------------------------------------------------------------

df_2024_consolidado = (
    df_2024
    .groupby(
        ["Fecha", "Id_Instal"],
        as_index=False
    )
    .agg(
        N_registros=("N_registros", "sum"),
        N_validos=("N_validos", "sum"),
        Suma_energia=("Suma_energia", "sum")
    )
)

# Recalcular cobertura DESPUÉS de consolidar
df_2024_consolidado["Cobertura_%"] = (
    df_2024_consolidado["N_validos"] / 1440 * 100
).round(2)

# ------------------------------------------------------------------------------
# 5. Comprobar unicidad
# ------------------------------------------------------------------------------

duplicados_finales = (
    df_2024_consolidado
    .duplicated(["Fecha", "Id_Instal"])
    .sum()
)

print("\n" + "-" * 90)
print("RESULTADO DE LA CONSOLIDACIÓN")
print("-" * 90)

print(
    f"Registros antes de consolidar: "
    f"{len(df_2024):,}"
)

print(
    f"Registros después de consolidar: "
    f"{len(df_2024_consolidado):,}"
)

print(
    f"Duplicados Fecha-Id_Instal restantes: "
    f"{duplicados_finales:,}"
)

# ------------------------------------------------------------------------------
# 6. Auditoría de cobertura
# ------------------------------------------------------------------------------

print("\n" + "-" * 90)
print("COBERTURA TRAS CONSOLIDACIÓN")
print("-" * 90)

display(
    df_2024_consolidado["Cobertura_%"]
    .describe()
    .to_frame("Cobertura_%")
)

cobertura_superior_100 = (
    df_2024_consolidado["Cobertura_%"] > 100
)

print(
    f"\nSensor-día con cobertura >100 %: "
    f"{cobertura_superior_100.sum():,}"
)

if cobertura_superior_100.sum() > 0:

    print("\nDistribución de casos con cobertura >100 %:")

    display(
        df_2024_consolidado.loc[
            cobertura_superior_100,
            ["Fecha", "Id_Instal",
             "N_registros", "N_validos", "Cobertura_%"]
        ]
        .sort_values(
            "Cobertura_%",
            ascending=False
        )
        .head(30)
    )

# ------------------------------------------------------------------------------
# 7. Comprobación especial de los cambios de hora
#
# En Barcelona:
# 31/03/2024 -> cambio a horario de verano
# 27/10/2024 -> cambio a horario de invierno
# ------------------------------------------------------------------------------

fechas_cambio_hora = pd.to_datetime([
    "2024-03-31",
    "2024-10-27"
])

print("\n" + "-" * 90)
print("CONTROL DE CAMBIOS DE HORA")
print("-" * 90)

display(
    df_2024_consolidado[
        df_2024_consolidado["Fecha"].isin(
            fechas_cambio_hora
        )
    ][
        ["Fecha", "Id_Instal",
         "N_registros", "N_validos", "Cobertura_%"]
    ]
    .sort_values(["Fecha", "Id_Instal"])
    .head(30)
)

# ------------------------------------------------------------------------------
# 8. Resumen final
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN 4.5")
print("=" * 90)

print(
    f"Periodo final: "
    f"{df_2024_consolidado['Fecha'].min().date()} - "
    f"{df_2024_consolidado['Fecha'].max().date()}"
)

print(
    f"Registros sensor-día: "
    f"{len(df_2024_consolidado):,}"
)

print(
    f"Instalaciones únicas: "
    f"{df_2024_consolidado['Id_Instal'].nunique():,}"
)

print(
    f"Duplicados Fecha-Instalación: "
    f"{duplicados_finales:,}"
)

print(
    f"Coberturas >100 %: "
    f"{cobertura_superior_100.sum():,}"
)

print("\n✓ Solapamientos mensuales consolidados.")
print("✓ Todavía no se han imputado ni descartado días por cobertura.")

4.5 AUDITORÍA Y RESOLUCIÓN DE SOLAPAMIENTOS TEMPORALES - 2024

Registros iniciales sensor-día: 57,272
Registros sensor-día fuera de 2024: 6

Fechas externas detectadas:


,Fecha,Id_Instal,N_registros,N_validos,Cobertura_%
0,2023-12-31,8428,60,60,4.17
1,2023-12-31,8506,60,60,4.17
2,2023-12-31,8627,60,60,4.17
3,2023-12-31,8646,60,60,4.17
4,2023-12-31,8746,60,60,4.17
5,2023-12-31,8766,60,60,4.17



Registros tras restringir al año 2024: 57,266

------------------------------------------------------------------------------------------
DUPLICADOS FECHA-INSTALACIÓN
------------------------------------------------------------------------------------------
Filas pertenecientes a claves repetidas: 224
Combinaciones Fecha-Id_Instal repetidas: 112

Muestra de registros repetidos:


,Fecha,Id_Instal,N_registros,N_validos,Suma_energia,Cobertura_%
4647,2024-01-31,8428,1380,1380,2.753763e+09,95.83
4674,2024-01-31,8428,60,60,1.364683e+08,4.17
4656,2024-01-31,8627,1380,1380,5.925307e+09,95.83
4675,2024-01-31,8627,60,60,8.155381e+07,4.17
4660,2024-01-31,8746,1380,1380,2.052859e+09,95.83
4676,2024-01-31,8746,60,60,3.568979e+07,4.17
4662,2024-01-31,8766,1380,1380,3.943367e+09,95.83
4677,2024-01-31,8766,60,60,2.493380e+08,4.17
4664,2024-01-31,8866,1380,1380,4.237353e+09,95.83
4678,2024-01-31,8866,60,60,6.505506e+07,4.17



------------------------------------------------------------------------------------------
RESULTADO DE LA CONSOLIDACIÓN
------------------------------------------------------------------------------------------
Registros antes de consolidar: 57,266
Registros después de consolidar: 57,154
Duplicados Fecha-Id_Instal restantes: 0

------------------------------------------------------------------------------------------
COBERTURA TRAS CONSOLIDACIÓN
------------------------------------------------------------------------------------------


...


Sensor-día con cobertura >100 %: 234

Distribución de casos con cobertura >100 %:


,Fecha,Id_Instal,N_registros,N_validos,Cobertura_%
55018,2024-12-18,9668,1505,1505,104.51
46530,2024-10-27,2886,1500,1500,104.17
46536,2024-10-27,3466,1500,1500,104.17
46533,2024-10-27,3086,1500,1500,104.17
46534,2024-10-27,3087,1500,1500,104.17
46535,2024-10-27,3106,1500,1500,104.17
46538,2024-10-27,3806,1500,1500,104.17
46537,2024-10-27,3469,1500,1500,104.17
46539,2024-10-27,3807,1500,1500,104.17
46531,2024-10-27,3027,1500,1500,104.17



------------------------------------------------------------------------------------------
CONTROL DE CAMBIOS DE HORA
------------------------------------------------------------------------------------------


,Fecha,Id_Instal,N_registros,N_validos,Cobertura_%
13803,2024-03-31,496,1380,1380,95.83
13804,2024-03-31,497,1380,1380,95.83
13805,2024-03-31,651,1380,1380,95.83
13806,2024-03-31,659,1380,1380,95.83
13807,2024-03-31,666,1380,1380,95.83
13808,2024-03-31,726,1380,1380,95.83
13809,2024-03-31,728,1380,1380,95.83
13810,2024-03-31,847,1380,1380,95.83
13811,2024-03-31,1026,1380,1380,95.83
13812,2024-03-31,1174,1379,1379,95.76



RESUMEN 4.5
Periodo final: 2024-01-01 - 2024-12-31
Registros sensor-día: 57,154
Instalaciones únicas: 244
Duplicados Fecha-Instalación: 0
Coberturas >100 %: 234

✓ Solapamientos mensuales consolidados.
✓ Todavía no se han imputado ni descartado días por cobertura.


### Resultados de la auditoría de solapamientos temporales

La auditoría permitió identificar y resolver los solapamientos existentes entre los archivos mensuales de 2024. Inicialmente se detectaron **112 combinaciones `Fecha–Id_Instal` repetidas**, correspondientes a 224 registros sensor-día.

El análisis mostró que estos duplicados procedían fundamentalmente de la fragmentación de determinados días entre archivos mensuales consecutivos. Por ejemplo, algunas observaciones correspondientes al último día de mes aparecían divididas en bloques de 1.380 y 60 minutos. Dado que las variables `N_registros`, `N_validos` y `Suma_energia` son aditivas, estos fragmentos pudieron consolidarse sin pérdida de información.

Tras la consolidación, el dataset quedó constituido por **57.154 observaciones sensor-día**, correspondientes a **244 instalaciones**, sin duplicados en la clave `Fecha–Id_Instal` y restringido estrictamente al periodo comprendido entre el 1 de enero y el 31 de diciembre de 2024.

La cobertura temporal continúa siendo elevada, con una media del **98,41 %** y una mediana del **100 %**. Se identificaron **234 sensor-día con coberturas superiores al 100 %**.

Una parte importante de estas coberturas se explica por los cambios oficiales de horario. El 31 de marzo de 2024, correspondiente al cambio al horario de verano, la duración teórica del día es de 23 horas (1.380 minutos), mientras que el 27 de octubre, correspondiente al retorno al horario de invierno, es de 25 horas (1.500 minutos). Por tanto, estos casos no deben interpretarse automáticamente como errores de medida.

En consecuencia, no se recortan artificialmente las coberturas al 100 % ni se eliminan observaciones en esta fase. El siguiente paso consistirá en establecer la duración teórica de cada día considerando los cambios horarios y definir un criterio mínimo de cobertura para determinar qué observaciones sensor-día proporcionan información suficiente para el cálculo del indicador acústico diario.

### Resultados de la auditoría de solapamientos temporales

La auditoría permitió identificar y resolver los solapamientos existentes entre los archivos mensuales de 2024. Inicialmente se detectaron **112 combinaciones `Fecha–Id_Instal` repetidas**, correspondientes a 224 registros sensor-día.

El análisis mostró que estos duplicados procedían fundamentalmente de la fragmentación de determinados días entre archivos mensuales consecutivos. Por ejemplo, algunas observaciones correspondientes al último día de mes aparecían divididas en bloques de 1.380 y 60 minutos. Dado que las variables `N_registros`, `N_validos` y `Suma_energia` son aditivas, estos fragmentos pudieron consolidarse sin pérdida de información.

Tras la consolidación, el dataset quedó constituido por **57.154 observaciones sensor-día**, correspondientes a **244 instalaciones**, sin duplicados en la clave `Fecha–Id_Instal` y restringido estrictamente al periodo comprendido entre el 1 de enero y el 31 de diciembre de 2024.

La cobertura temporal continúa siendo elevada, con una media del **98,41 %** y una mediana del **100 %**. Se identificaron **234 sensor-día con coberturas superiores al 100 %**.

Una parte importante de estas coberturas se explica por los cambios oficiales de horario. El 31 de marzo de 2024, correspondiente al cambio al horario de verano, la duración teórica del día es de 23 horas (1.380 minutos), mientras que el 27 de octubre, correspondiente al retorno al horario de invierno, es de 25 horas (1.500 minutos). Por tanto, estos casos no deben interpretarse automáticamente como errores de medida.

En consecuencia, no se recortan artificialmente las coberturas al 100 % ni se eliminan observaciones en esta fase. El siguiente paso consistirá en establecer la duración teórica de cada día considerando los cambios horarios y definir un criterio mínimo de cobertura para determinar qué observaciones sensor-día proporcionan información suficiente para el cálculo del indicador acústico diario.

In [ ]:
# ==============================================================================
# 4.6 CORRECCIÓN DE COBERTURA TEMPORAL Y DEFINICIÓN DE DÍAS VÁLIDOS
# ==============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("4.6 CORRECCIÓN DE COBERTURA TEMPORAL Y DEFINICIÓN DE DÍAS VÁLIDOS")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Copia de trabajo
# ------------------------------------------------------------------------------

df_2024_cobertura = df_2024_consolidado.copy()

df_2024_cobertura["Fecha"] = pd.to_datetime(
    df_2024_cobertura["Fecha"],
    errors="coerce"
)

# ------------------------------------------------------------------------------
# 2. Duración teórica de cada día
# ------------------------------------------------------------------------------

# Día convencional = 24 h = 1440 minutos
df_2024_cobertura["Minutos_teoricos"] = 1440

# Cambio al horario de verano:
# 31/03/2024 = 23 horas = 1380 minutos
df_2024_cobertura.loc[
    df_2024_cobertura["Fecha"] == pd.Timestamp("2024-03-31"),
    "Minutos_teoricos"
] = 1380

# Cambio al horario de invierno:
# 27/10/2024 = 25 horas = 1500 minutos
df_2024_cobertura.loc[
    df_2024_cobertura["Fecha"] == pd.Timestamp("2024-10-27"),
    "Minutos_teoricos"
] = 1500

# ------------------------------------------------------------------------------
# 3. Recalcular cobertura temporal real
# ------------------------------------------------------------------------------

df_2024_cobertura["Cobertura_corregida_%"] = (
    df_2024_cobertura["N_validos"]
    / df_2024_cobertura["Minutos_teoricos"]
    * 100
).round(2)

print("\n--- COBERTURA CORREGIDA ---")

display(
    df_2024_cobertura["Cobertura_corregida_%"]
    .describe()
    .to_frame("Cobertura_corregida_%")
)

# ------------------------------------------------------------------------------
# 4. Comprobar específicamente los cambios horarios
# ------------------------------------------------------------------------------

print("\n--- CONTROL DE LOS CAMBIOS DE HORA ---")

control_cambio_hora = (
    df_2024_cobertura[
        df_2024_cobertura["Fecha"].isin([
            pd.Timestamp("2024-03-31"),
            pd.Timestamp("2024-10-27")
        ])
    ][
        [
            "Fecha",
            "Id_Instal",
            "N_validos",
            "Minutos_teoricos",
            "Cobertura_corregida_%"
        ]
    ]
    .sort_values(["Fecha", "Id_Instal"])
)

display(control_cambio_hora.head(30))

# Resumen de cobertura por fecha de cambio horario
print("\nResumen por día de cambio horario:")

display(
    control_cambio_hora
    .groupby("Fecha")["Cobertura_corregida_%"]
    .describe()
)

# ------------------------------------------------------------------------------
# 5. Identificar coberturas realmente superiores al 100 %
# ------------------------------------------------------------------------------

exceso_cobertura = (
    df_2024_cobertura["Cobertura_corregida_%"] > 100
)

print("\n--- COBERTURAS SUPERIORES AL 100 % TRAS CORRECCIÓN ---")

print(
    f"Sensor-día con cobertura >100 %: "
    f"{exceso_cobertura.sum():,}"
)

if exceso_cobertura.sum() > 0:

    display(
        df_2024_cobertura.loc[
            exceso_cobertura,
            [
                "Fecha",
                "Id_Instal",
                "N_registros",
                "N_validos",
                "Minutos_teoricos",
                "Cobertura_corregida_%"
            ]
        ]
        .sort_values(
            "Cobertura_corregida_%",
            ascending=False
        )
        .head(30)
    )

# ------------------------------------------------------------------------------
# 6. Distribución de cobertura por intervalos
# ------------------------------------------------------------------------------

condiciones = [
    df_2024_cobertura["Cobertura_corregida_%"] < 25,

    (df_2024_cobertura["Cobertura_corregida_%"] >= 25) &
    (df_2024_cobertura["Cobertura_corregida_%"] < 50),

    (df_2024_cobertura["Cobertura_corregida_%"] >= 50) &
    (df_2024_cobertura["Cobertura_corregida_%"] < 75),

    (df_2024_cobertura["Cobertura_corregida_%"] >= 75) &
    (df_2024_cobertura["Cobertura_corregida_%"] < 90),

    (df_2024_cobertura["Cobertura_corregida_%"] >= 90) &
    (df_2024_cobertura["Cobertura_corregida_%"] <= 100),

    df_2024_cobertura["Cobertura_corregida_%"] > 100
]

categorias = [
    "<25 %",
    "25-50 %",
    "50-75 %",
    "75-90 %",
    "90-100 %",
    ">100 %"
]

df_2024_cobertura["Categoria_cobertura"] = np.select(
    condiciones,
    categorias,
    default="Sin clasificar"
)

tabla_cobertura = (
    df_2024_cobertura["Categoria_cobertura"]
    .value_counts()
    .reindex(categorias, fill_value=0)
    .to_frame("Sensor_dia")
)

tabla_cobertura["Porcentaje"] = (
    tabla_cobertura["Sensor_dia"]
    / len(df_2024_cobertura)
    * 100
).round(2)

print("\n--- DISTRIBUCIÓN DE COBERTURA ---")
display(tabla_cobertura)

# ------------------------------------------------------------------------------
# 7. Criterio preliminar de validez: cobertura >= 75 %
# ------------------------------------------------------------------------------

UMBRAL_COBERTURA = 75

df_2024_cobertura["Dia_valido"] = (
    df_2024_cobertura["Cobertura_corregida_%"]
    >= UMBRAL_COBERTURA
)

n_validos = df_2024_cobertura["Dia_valido"].sum()
n_no_validos = (~df_2024_cobertura["Dia_valido"]).sum()

print("\n--- CRITERIO DE VALIDEZ DIARIA ---")

print(
    f"Umbral mínimo de cobertura: "
    f"{UMBRAL_COBERTURA} %"
)

print(
    f"Sensor-día válidos: "
    f"{n_validos:,} "
    f"({n_validos / len(df_2024_cobertura) * 100:.2f} %)"
)

print(
    f"Sensor-día con cobertura insuficiente: "
    f"{n_no_validos:,} "
    f"({n_no_validos / len(df_2024_cobertura) * 100:.2f} %)"
)

# ------------------------------------------------------------------------------
# 8. Sensibilidad frente a distintos umbrales
#
# Esto nos permite justificar posteriormente la elección del 75 %.
# ------------------------------------------------------------------------------

umbrales = [50, 60, 70, 75, 80, 90, 95]

sensibilidad = []

for umbral in umbrales:

    n = (
        df_2024_cobertura["Cobertura_corregida_%"]
        >= umbral
    ).sum()

    sensibilidad.append({
        "Umbral_%": umbral,
        "Sensor_dia_validos": n,
        "Porcentaje_conservado": round(
            n / len(df_2024_cobertura) * 100,
            2
        )
    })

df_sensibilidad_cobertura = pd.DataFrame(sensibilidad)

print("\n--- ANÁLISIS DE SENSIBILIDAD DEL UMBRAL ---")
display(df_sensibilidad_cobertura)

# ------------------------------------------------------------------------------
# 9. Comprobaciones finales
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN 4.6")
print("=" * 90)

print(
    f"Registros sensor-día analizados: "
    f"{len(df_2024_cobertura):,}"
)

print(
    f"Cobertura media corregida: "
    f"{df_2024_cobertura['Cobertura_corregida_%'].mean():.2f} %"
)

print(
    f"Cobertura mediana corregida: "
    f"{df_2024_cobertura['Cobertura_corregida_%'].median():.2f} %"
)

print(
    f"Días válidos con umbral >=75 %: "
    f"{n_validos:,}"
)

print(
    f"Días con cobertura insuficiente: "
    f"{n_no_validos:,}"
)

print(
    f"Coberturas >100 % tras corrección horaria: "
    f"{exceso_cobertura.sum():,}"
)

print("\n✓ Cambios horarios incorporados al cálculo de cobertura.")
print("✓ Se conserva la totalidad de los registros.")
print("✓ Dia_valido actúa únicamente como bandera de calidad.")
print("✓ No se ha realizado ninguna imputación.")

4.6 CORRECCIÓN DE COBERTURA TEMPORAL Y DEFINICIÓN DE DÍAS VÁLIDOS

--- COBERTURA CORREGIDA ---


,Cobertura_corregida_%
count,57154.000000
mean,98.409541
std,8.279096
min,0.070000
25%,100.000000
50%,100.000000
75%,100.000000
max,104.510000



--- CONTROL DE LOS CAMBIOS DE HORA ---


,Fecha,Id_Instal,N_validos,Minutos_teoricos,Cobertura_corregida_%
13803,2024-03-31,496,1380,1380,100.00
13804,2024-03-31,497,1380,1380,100.00
13805,2024-03-31,651,1380,1380,100.00
13806,2024-03-31,659,1380,1380,100.00
13807,2024-03-31,666,1380,1380,100.00
13808,2024-03-31,726,1380,1380,100.00
13809,2024-03-31,728,1380,1380,100.00
13810,2024-03-31,847,1380,1380,100.00
13811,2024-03-31,1026,1380,1380,100.00
13812,2024-03-31,1174,1379,1380,99.93



Resumen por día de cambio horario:


...


--- DISTRIBUCIÓN DE COBERTURA ---


,Sensor_dia,Porcentaje
Categoria_cobertura,,
<25 %,189,0.33
25-50 %,296,0.52
50-75 %,968,1.69
75-90 %,830,1.45
90-100 %,54778,95.84
>100 %,93,0.16



--- CRITERIO DE VALIDEZ DIARIA ---
Umbral mínimo de cobertura: 75 %
Sensor-día válidos: 55,701 (97.46 %)
Sensor-día con cobertura insuficiente: 1,453 (2.54 %)

--- ANÁLISIS DE SENSIBILIDAD DEL UMBRAL ---


,Umbral_%,Sensor_dia_validos,Porcentaje_conservado
0,50,56669,99.15
1,60,56352,98.60
2,70,56171,98.28
3,75,55701,97.46
4,80,55449,97.02
5,90,54871,96.01
6,95,54314,95.03



RESUMEN 4.6
Registros sensor-día analizados: 57,154
Cobertura media corregida: 98.41 %
Cobertura mediana corregida: 100.00 %
Días válidos con umbral >=75 %: 55,701
Días con cobertura insuficiente: 1,453
Coberturas >100 % tras corrección horaria: 93

✓ Cambios horarios incorporados al cálculo de cobertura.
✓ Se conserva la totalidad de los registros.
✓ Dia_valido actúa únicamente como bandera de calidad.
✓ No se ha realizado ninguna imputación.


### Resultados de la evaluación de cobertura temporal

Una vez incorporada la duración teórica de los días afectados por los cambios oficiales de horario, la cobertura temporal media del conjunto de datos de 2024 se sitúa en el **98,41 %**, con una mediana del **100 %**. Los percentiles 25, 50 y 75 alcanzan igualmente el 100 %, mostrando una elevada completitud temporal de las series acústicas.

La corrección permite representar adecuadamente los días de cambio horario. En particular, el 31 de marzo de 2024 se consideran 1.380 minutos teóricos y el 27 de octubre 1.500 minutos, evitando interpretar sus duraciones de 23 y 25 horas como pérdidas o excesos de información.

La distribución de cobertura confirma la elevada calidad del conjunto: el **95,84 %** de las observaciones sensor-día presenta una cobertura comprendida entre el 90 y el 100 %. Únicamente el 2,54 % presenta una cobertura inferior al 75 %.

El análisis de sensibilidad muestra que un umbral mínimo del **75 %** permite conservar **55.701 observaciones sensor-día**, equivalentes al **97,46 %** del conjunto original. Este valor se adopta como criterio de calidad, manteniendo simultáneamente la variable `Dia_valido` para preservar la trazabilidad de la decisión.

Tras corregir los cambios horarios permanecen **93 observaciones con cobertura superior al 100 %**, equivalentes aproximadamente al **0,16 %** del conjunto. Dado su carácter excepcional, estas observaciones se mantienen identificadas para su control posterior y no se realiza un truncamiento artificial de la cobertura.

En esta etapa no se realiza ninguna imputación de valores acústicos ni se eliminan físicamente las observaciones con cobertura insuficiente. El criterio de cobertura se conserva mediante una bandera de calidad que permitirá seleccionar únicamente los días suficientemente representativos en las etapas analíticas posteriores.

## 4.7 Cálculo del nivel acústico equivalente diario (LAeq,d)

Los niveles sonoros expresados en decibelios se encuentran en una escala logarítmica, por lo que su agregación temporal no debe realizarse mediante una media aritmética directa.

Durante el procesamiento de los datos minutales se conservó para cada combinación `Fecha–Id_Instal` la suma de la energía acústica asociada a las observaciones válidas:

\[
E = \sum_{i=1}^{n} 10^{L_i/10}
\]

A partir de esta magnitud se calcula el nivel acústico equivalente diario mediante:

\[
LAeq_d = 10 \log_{10}
\left(
\frac{1}{n}
\sum_{i=1}^{n}10^{L_i/10}
\right)
\]

donde \(L_i\) representa el nivel `LAeq` de cada intervalo original y \(n\) el número de observaciones acústicas válidas disponibles para el sensor-día.

El cálculo se realiza utilizando `N_validos` como denominador y no el número teórico de minutos del día. De este modo, el indicador representa el nivel equivalente correspondiente al periodo efectivamente observado, mientras que la representatividad temporal de dicho periodo se controla independientemente mediante la variable de cobertura.

Se conserva el indicador para todas las observaciones técnicamente calculables y se utiliza la variable `Dia_valido`, definida mediante el umbral de cobertura del 75 %, para distinguir los valores suficientemente representativos para los análisis posteriores.

No se realiza imputación de niveles acústicos.

In [ ]:
# ==============================================================================
# 4.7 CÁLCULO DEL NIVEL ACÚSTICO EQUIVALENTE DIARIO (LAeq,d)
# ==============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("4.7 CÁLCULO DEL NIVEL ACÚSTICO EQUIVALENTE DIARIO (LAeq,d)")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Copia de trabajo
# ------------------------------------------------------------------------------

df_ruido_2024_diario = df_2024_cobertura.copy()

# ------------------------------------------------------------------------------
# 2. Comprobaciones previas
# ------------------------------------------------------------------------------

print("\n--- COMPROBACIONES PREVIAS ---")

print(
    f"Registros sensor-día: "
    f"{len(df_ruido_2024_diario):,}"
)

print(
    f"N_validos <= 0: "
    f"{(df_ruido_2024_diario['N_validos'] <= 0).sum():,}"
)

print(
    f"Suma_energia <= 0: "
    f"{(df_ruido_2024_diario['Suma_energia'] <= 0).sum():,}"
)

print(
    f"Valores nulos en Suma_energia: "
    f"{df_ruido_2024_diario['Suma_energia'].isna().sum():,}"
)

# ------------------------------------------------------------------------------
# 3. Cálculo energético del LAeq diario
#
# LAeq_d = 10 * log10(Suma_energia / N_validos)
# ------------------------------------------------------------------------------

condicion_calculo = (
    (df_ruido_2024_diario["N_validos"] > 0) &
    (df_ruido_2024_diario["Suma_energia"] > 0) &
    (df_ruido_2024_diario["Suma_energia"].notna())
)

df_ruido_2024_diario["LAeq_dia"] = np.nan

df_ruido_2024_diario.loc[
    condicion_calculo,
    "LAeq_dia"
] = (
    10 * np.log10(
        df_ruido_2024_diario.loc[
            condicion_calculo,
            "Suma_energia"
        ]
        /
        df_ruido_2024_diario.loc[
            condicion_calculo,
            "N_validos"
        ]
    )
)

# ------------------------------------------------------------------------------
# 4. Redondeo únicamente para presentación/almacenamiento
# ------------------------------------------------------------------------------

df_ruido_2024_diario["LAeq_dia"] = (
    df_ruido_2024_diario["LAeq_dia"]
    .round(2)
)

# ------------------------------------------------------------------------------
# 5. Auditoría del indicador obtenido
# ------------------------------------------------------------------------------

print("\n--- ESTADÍSTICOS DEL LAeq DIARIO ---")

display(
    df_ruido_2024_diario["LAeq_dia"]
    .describe()
    .to_frame("LAeq_dia")
)

print(
    f"\nLAeq_dia ausentes: "
    f"{df_ruido_2024_diario['LAeq_dia'].isna().sum():,}"
)

print(
    f"LAeq_dia disponibles: "
    f"{df_ruido_2024_diario['LAeq_dia'].notna().sum():,}"
)

# ------------------------------------------------------------------------------
# 6. Separar disponibilidad matemática y validez temporal
# ------------------------------------------------------------------------------

df_ruido_2024_diario["LAeq_dia_valido"] = (
    df_ruido_2024_diario["LAeq_dia"].notna()
    &
    df_ruido_2024_diario["Dia_valido"]
)

n_laeq_validos = df_ruido_2024_diario["LAeq_dia_valido"].sum()

print("\n--- VALIDEZ PARA ANÁLISIS POSTERIORES ---")

print(
    f"LAeq diarios calculados y con cobertura >=75 %: "
    f"{n_laeq_validos:,}"
)

print(
    f"Porcentaje sobre el dataset sensor-día: "
    f"{n_laeq_validos / len(df_ruido_2024_diario) * 100:.2f} %"
)

# ------------------------------------------------------------------------------
# 7. Comparación válidos vs cobertura insuficiente
# ------------------------------------------------------------------------------

print("\n--- LAeq SEGÚN CRITERIO DE COBERTURA ---")

resumen_validez = (
    df_ruido_2024_diario
    .groupby("Dia_valido")["LAeq_dia"]
    .agg(
        N="count",
        Media="mean",
        Mediana="median",
        Minimo="min",
        Maximo="max"
    )
    .round(2)
)

display(resumen_validez)

# ------------------------------------------------------------------------------
# 8. Comprobación de valores extremos
# ------------------------------------------------------------------------------

print("\n--- VALORES DIARIOS MÁS ALTOS ---")

display(
    df_ruido_2024_diario
    .nlargest(10, "LAeq_dia")
    [
        [
            "Fecha",
            "Id_Instal",
            "LAeq_dia",
            "N_validos",
            "Cobertura_corregida_%",
            "Dia_valido"
        ]
    ]
)

print("\n--- VALORES DIARIOS MÁS BAJOS ---")

display(
    df_ruido_2024_diario
    .nsmallest(10, "LAeq_dia")
    [
        [
            "Fecha",
            "Id_Instal",
            "LAeq_dia",
            "N_validos",
            "Cobertura_corregida_%",
            "Dia_valido"
        ]
    ]
)

# ------------------------------------------------------------------------------
# 9. Dataset analítico de días válidos
#
# NO modificamos el maestro.
# Creamos una vista específica para análisis posteriores.
# ------------------------------------------------------------------------------

df_ruido_2024_valido = (
    df_ruido_2024_diario[
        df_ruido_2024_diario["LAeq_dia_valido"]
    ]
    .copy()
)

print("\n--- DATASET ANALÍTICO 2024 ---")

print(
    f"Registros válidos: "
    f"{len(df_ruido_2024_valido):,}"
)

print(
    f"Instalaciones representadas: "
    f"{df_ruido_2024_valido['Id_Instal'].nunique():,}"
)

print(
    f"Periodo: "
    f"{df_ruido_2024_valido['Fecha'].min().date()} - "
    f"{df_ruido_2024_valido['Fecha'].max().date()}"
)

# ------------------------------------------------------------------------------
# 10. Muestra del resultado
# ------------------------------------------------------------------------------

print("\n--- MUESTRA DEL RESULTADO ---")

display(
    df_ruido_2024_diario[
        [
            "Fecha",
            "Id_Instal",
            "N_validos",
            "Minutos_teoricos",
            "Cobertura_corregida_%",
            "Dia_valido",
            "LAeq_dia",
            "LAeq_dia_valido"
        ]
    ]
    .head(10)
)

# ------------------------------------------------------------------------------
# 11. Resumen final
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN 4.7")
print("=" * 90)

print(
    f"Sensor-día totales: "
    f"{len(df_ruido_2024_diario):,}"
)

print(
    f"LAeq diarios calculados: "
    f"{df_ruido_2024_diario['LAeq_dia'].notna().sum():,}"
)

print(
    f"LAeq diarios válidos (cobertura >=75 %): "
    f"{n_laeq_validos:,}"
)

print(
    f"Porcentaje válido: "
    f"{n_laeq_validos / len(df_ruido_2024_diario) * 100:.2f} %"
)

print("\n✓ LAeq diario calculado mediante agregación energética.")
print("✓ La cobertura se mantiene como criterio independiente de calidad.")
print("✓ No se han imputado niveles acústicos.")
print("✓ El dataset maestro conserva todas las observaciones sensor-día.")

4.7 CÁLCULO DEL NIVEL ACÚSTICO EQUIVALENTE DIARIO (LAeq,d)

--- COMPROBACIONES PREVIAS ---
Registros sensor-día: 57,154
N_validos <= 0: 0
Suma_energia <= 0: 0
Valores nulos en Suma_energia: 0

--- ESTADÍSTICOS DEL LAeq DIARIO ---


,LAeq_dia
count,57154.000000
mean,64.279635
std,4.333728
min,16.950000
25%,61.770000
50%,64.000000
75%,66.740000
max,94.780000



LAeq_dia ausentes: 0
LAeq_dia disponibles: 57,154

--- VALIDEZ PARA ANÁLISIS POSTERIORES ---
LAeq diarios calculados y con cobertura >=75 %: 55,701
Porcentaje sobre el dataset sensor-día: 97.46 %

--- LAeq SEGÚN CRITERIO DE COBERTURA ---


,N,Media,Mediana,Minimo,Maximo
Dia_valido,,,,,
False,1453,63.42,63.44,16.95,90.71
True,55701,64.30,64.02,35.08,94.78



--- VALORES DIARIOS MÁS ALTOS ---


...


--- VALORES DIARIOS MÁS BAJOS ---


,Fecha,Id_Instal,LAeq_dia,N_validos,Cobertura_corregida_%,Dia_valido
49060,2024-11-11,9827,16.95,60,4.17,False
31303,2024-07-22,9527,17.72,120,8.33,False
31305,2024-07-22,9529,18.38,120,8.33,False
23926,2024-06-04,9328,20.41,120,8.33,False
49056,2024-11-11,9772,31.30,1,0.07,False
49058,2024-11-11,9806,34.16,60,4.17,False
46954,2024-10-29,9086,34.63,143,9.93,False
13778,2024-03-30,8686,35.08,1440,100.00,True
13624,2024-03-29,8686,35.37,1440,100.00,True
10439,2024-03-09,6759,36.07,391,27.15,False



--- DATASET ANALÍTICO 2024 ---
Registros válidos: 55,701
Instalaciones representadas: 242
Periodo: 2024-01-01 - 2024-12-31

--- MUESTRA DEL RESULTADO ---


,Fecha,Id_Instal,N_validos,Minutos_teoricos,Cobertura_corregida_%,Dia_valido,LAeq_dia,LAeq_dia_valido
0,2024-01-01,496,1440,1440,100.0,True,68.89,True
1,2024-01-01,497,1440,1440,100.0,True,68.73,True
2,2024-01-01,651,1440,1440,100.0,True,64.97,True
3,2024-01-01,659,1440,1440,100.0,True,57.59,True
4,2024-01-01,666,1440,1440,100.0,True,61.58,True
5,2024-01-01,726,1440,1440,100.0,True,62.72,True
6,2024-01-01,728,1440,1440,100.0,True,68.67,True
7,2024-01-01,847,1440,1440,100.0,True,62.74,True
8,2024-01-01,1026,1440,1440,100.0,True,53.48,True
9,2024-01-01,1174,1440,1440,100.0,True,63.68,True



RESUMEN 4.7
Sensor-día totales: 57,154
LAeq diarios calculados: 57,154
LAeq diarios válidos (cobertura >=75 %): 55,701
Porcentaje válido: 97.46 %

✓ LAeq diario calculado mediante agregación energética.
✓ La cobertura se mantiene como criterio independiente de calidad.
✓ No se han imputado niveles acústicos.
✓ El dataset maestro conserva todas las observaciones sensor-día.


### Resultados del cálculo del LAeq diario

El nivel acústico equivalente diario (`LAeq_dia`) se calculó mediante agregación energética para las **57.154 observaciones sensor-día** disponibles en 2024. Todas las observaciones presentaban un número positivo de medidas válidas y una suma de energía acústica válida, por lo que fue posible obtener el indicador diario para la totalidad del conjunto.

El `LAeq_dia` presenta una media de **64,28 dB** y una mediana de **64,00 dB**, con un rango global comprendido entre **16,95 y 94,78 dB**.

La aplicación independiente del criterio de cobertura temporal permite distinguir entre disponibilidad matemática del indicador y representatividad diaria. Con el umbral mínimo establecido del **75 %**, se consideran válidas **55.701 observaciones sensor-día**, equivalentes al **97,46 %** del conjunto.

Los valores diarios asociados a cobertura insuficiente muestran una mayor presencia de valores extremos, especialmente en el límite inferior. El mínimo global de 16,95 dB corresponde a una observación con únicamente 60 minutos válidos (4,17 % de cobertura), mientras que entre las observaciones consideradas temporalmente válidas el mínimo asciende a **35,08 dB**. Este comportamiento respalda la utilización de la cobertura como criterio de calidad independiente del cálculo acústico.

Los días válidos presentan un `LAeq_dia` medio de **64,30 dB** y una mediana de **64,02 dB**. El máximo observado dentro de este conjunto es **94,78 dB**.

Como resultado se mantienen dos niveles de información:

- un dataset maestro que conserva las **57.154 observaciones sensor-día**, manteniendo la trazabilidad completa;
- un subconjunto analítico formado por **55.701 observaciones válidas**, correspondientes a **242 instalaciones**, destinado a los análisis posteriores.

No se han imputado niveles acústicos ni eliminado observaciones del dataset maestro. La variable `LAeq_dia_valido` permite identificar de forma explícita qué valores cumplen simultáneamente los requisitos de cálculo y cobertura temporal.

## 4.8 Procesamiento energético diario del periodo 2018–2023

Los datos acústicos correspondientes al periodo 2018–2023 presentan una resolución temporal horaria, a diferencia de los registros minutales disponibles para 2024.

Con objeto de obtener un indicador diario metodológicamente homogéneo entre ambos periodos, los niveles horarios no se agregan mediante una media aritmética de decibelios. Para cada combinación `Fecha–Id_Instal` se conserva la suma de la energía acústica asociada a las observaciones horarias:

\[
E = \sum_{i=1}^{n} 10^{L_i/10}
\]

y posteriormente se calcula:

\[
LAeq_d = 10 \log_{10}
\left(
\frac{1}{n}
\sum_{i=1}^{n}10^{L_i/10}
\right)
\]

donde \(L_i\) corresponde al `LAeq` horario y \(n\) al número de observaciones horarias válidas disponibles.

Los archivos se procesan individualmente para reducir el consumo de memoria. Para cada sensor-día se conserva asimismo el número de observaciones válidas y se calcula su cobertura temporal.

La duración teórica se establece en 24 observaciones para los días convencionales, 23 para el cambio al horario de verano y 25 para el cambio al horario de invierno. Las fechas de cambio horario se determinan automáticamente para cada año.

Al igual que en 2024, se establece una cobertura mínima del **75 %** para identificar los días suficientemente representativos mediante una bandera de calidad, sin realizar imputación de niveles acústicos.

Este procedimiento permite obtener para 2018–2023 un indicador diario conceptualmente equivalente al calculado a partir de las observaciones minutales de 2024.

In [ ]:
# ==============================================================================
# 4.8 PROCESAMIENTO ENERGÉTICO DIARIO DEL PERIODO 2018-2023
# ==============================================================================

import pandas as pd
import numpy as np
import glob
import os
import gc
import calendar
from datetime import date

print("=" * 90)
print("4.8 PROCESAMIENTO ENERGÉTICO DIARIO DEL PERIODO 2018-2023")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Localizar los archivos horarios 2018-2023
# ------------------------------------------------------------------------------

archivos_18_23 = []

for anio in range(2018, 2024):
    archivos_18_23.extend(
        glob.glob(
            os.path.join(
                ruta_base_ruido,
                f"BC {anio}*",
                "*.csv"
            )
        )
    )

archivos_18_23 = sorted(archivos_18_23)

print(f"\nArchivos encontrados: {len(archivos_18_23)}")

for archivo in archivos_18_23:
    print("-", os.path.basename(archivo))

# ------------------------------------------------------------------------------
# 2. Función para obtener el último domingo de un mes
#
# Los cambios horarios europeos se producen el último domingo
# de marzo y el último domingo de octubre.
# ------------------------------------------------------------------------------

def ultimo_domingo(anio, mes):

    ultimo_dia = calendar.monthrange(anio, mes)[1]
    fecha = date(anio, mes, ultimo_dia)

    desplazamiento = (
        fecha.weekday() - calendar.SUNDAY
    ) % 7

    return pd.Timestamp(
        fecha - pd.Timedelta(days=desplazamiento)
    )

# Fechas de cambio horario para 2018-2023

cambios_verano = {
    ultimo_domingo(anio, 3)
    for anio in range(2018, 2024)
}

cambios_invierno = {
    ultimo_domingo(anio, 10)
    for anio in range(2018, 2024)
}

print("\nCambios a horario de verano:")
print(sorted(cambios_verano))

print("\nCambios a horario de invierno:")
print(sorted(cambios_invierno))

# ------------------------------------------------------------------------------
# 3. Procesamiento archivo a archivo
# ------------------------------------------------------------------------------

resumenes_18_23 = []
auditoria_archivos = []

for i, archivo in enumerate(archivos_18_23, start=1):

    nombre = os.path.basename(archivo)

    print("\n" + "-" * 90)
    print(
        f"[{i:02d}/{len(archivos_18_23):02d}] "
        f"Procesando: {nombre}"
    )

    # Leer únicamente las columnas necesarias
    df = pd.read_csv(
        archivo,
        usecols=[
            "Any",
            "Mes",
            "Dia",
            "Id_Instal",
            "Nivell_LAeq_1h"
        ],
        low_memory=False
    )

    n_original = len(df)

    # --------------------------------------------------------------------------
    # Construcción de Fecha
    # --------------------------------------------------------------------------

    df["Fecha"] = pd.to_datetime(
        dict(
            year=df["Any"],
            month=df["Mes"],
            day=df["Dia"]
        ),
        errors="coerce"
    )

    n_fecha_invalida = df["Fecha"].isna().sum()

    # --------------------------------------------------------------------------
    # Conversión del nivel acústico a numérico
    # --------------------------------------------------------------------------

    df["Nivell_LAeq_1h"] = pd.to_numeric(
        df["Nivell_LAeq_1h"],
        errors="coerce"
    )

    n_nulos_laeq = df["Nivell_LAeq_1h"].isna().sum()

    # Solo pueden participar en el cálculo energético
    # las observaciones con fecha y LAeq válidos

    df_val = df.dropna(
        subset=[
            "Fecha",
            "Id_Instal",
            "Nivell_LAeq_1h"
        ]
    ).copy()

    # --------------------------------------------------------------------------
    # Transformación a energía
    # --------------------------------------------------------------------------

    df_val["Energia"] = np.power(
        10.0,
        df_val["Nivell_LAeq_1h"] / 10.0
    )

    # --------------------------------------------------------------------------
    # Agregación sensor-día
    # --------------------------------------------------------------------------

    diario = (
        df_val
        .groupby(
            ["Fecha", "Id_Instal"],
            as_index=False
        )
        .agg(
            N_validos=("Nivell_LAeq_1h", "count"),
            Suma_energia=("Energia", "sum")
        )
    )

    resumenes_18_23.append(diario)

    auditoria_archivos.append({
        "Archivo": nombre,
        "Registros_originales": n_original,
        "Fechas_invalidas": n_fecha_invalida,
        "Nulos_LAeq": n_nulos_laeq,
        "Sensor_dia_generados": len(diario),
        "Instalaciones": diario["Id_Instal"].nunique()
    })

    print(f"Registros originales : {n_original:,}")
    print(f"Fechas inválidas     : {n_fecha_invalida:,}")
    print(f"Nulos LAeq           : {n_nulos_laeq:,}")
    print(f"Sensor-día generados : {len(diario):,}")
    print(
        f"Instalaciones        : "
        f"{diario['Id_Instal'].nunique():,}"
    )

    # Liberar memoria
    del df
    del df_val
    del diario
    gc.collect()

    print("✓ Archivo procesado y memoria liberada.")

# ------------------------------------------------------------------------------
# 4. Auditoría de archivos
# ------------------------------------------------------------------------------

df_auditoria_18_23 = pd.DataFrame(
    auditoria_archivos
)

print("\n" + "=" * 90)
print("AUDITORÍA DE LOS ARCHIVOS 2018-2023")
print("=" * 90)

display(df_auditoria_18_23)

# ------------------------------------------------------------------------------
# 5. Unión de los resúmenes diarios
# ------------------------------------------------------------------------------

df_ruido_18_23_base = pd.concat(
    resumenes_18_23,
    ignore_index=True
)

del resumenes_18_23
gc.collect()

print(
    f"\nRegistros sensor-día antes de consolidar: "
    f"{len(df_ruido_18_23_base):,}"
)

# ------------------------------------------------------------------------------
# 6. Consolidar posibles solapamientos entre semestres
# ------------------------------------------------------------------------------

duplicados_pre = (
    df_ruido_18_23_base
    .duplicated(
        ["Fecha", "Id_Instal"],
        keep=False
    )
    .sum()
)

print(
    f"Filas pertenecientes a claves Fecha-Id_Instal repetidas: "
    f"{duplicados_pre:,}"
)

df_ruido_18_23_diario = (
    df_ruido_18_23_base
    .groupby(
        ["Fecha", "Id_Instal"],
        as_index=False
    )
    .agg(
        N_validos=("N_validos", "sum"),
        Suma_energia=("Suma_energia", "sum")
    )
)

del df_ruido_18_23_base
gc.collect()

# ------------------------------------------------------------------------------
# 7. Número teórico de observaciones horarias
# ------------------------------------------------------------------------------

df_ruido_18_23_diario["Horas_teoricas"] = 24

df_ruido_18_23_diario.loc[
    df_ruido_18_23_diario["Fecha"].isin(
        cambios_verano
    ),
    "Horas_teoricas"
] = 23

df_ruido_18_23_diario.loc[
    df_ruido_18_23_diario["Fecha"].isin(
        cambios_invierno
    ),
    "Horas_teoricas"
] = 25

# ------------------------------------------------------------------------------
# 8. Cobertura temporal corregida
# ------------------------------------------------------------------------------

df_ruido_18_23_diario[
    "Cobertura_corregida_%"
] = (
    df_ruido_18_23_diario["N_validos"]
    /
    df_ruido_18_23_diario["Horas_teoricas"]
    * 100
).round(2)

# ------------------------------------------------------------------------------
# 9. Calcular LAeq diario mediante agregación energética
# ------------------------------------------------------------------------------

condicion_calculo = (
    (df_ruido_18_23_diario["N_validos"] > 0)
    &
    (df_ruido_18_23_diario["Suma_energia"] > 0)
)

df_ruido_18_23_diario["LAeq_dia"] = np.nan

df_ruido_18_23_diario.loc[
    condicion_calculo,
    "LAeq_dia"
] = (
    10 * np.log10(
        df_ruido_18_23_diario.loc[
            condicion_calculo,
            "Suma_energia"
        ]
        /
        df_ruido_18_23_diario.loc[
            condicion_calculo,
            "N_validos"
        ]
    )
)

df_ruido_18_23_diario["LAeq_dia"] = (
    df_ruido_18_23_diario["LAeq_dia"]
    .round(2)
)

# ------------------------------------------------------------------------------
# 10. Criterio de calidad temporal
# ------------------------------------------------------------------------------

UMBRAL_COBERTURA = 75

df_ruido_18_23_diario["Dia_valido"] = (
    df_ruido_18_23_diario[
        "Cobertura_corregida_%"
    ] >= UMBRAL_COBERTURA
)

df_ruido_18_23_diario["LAeq_dia_valido"] = (
    df_ruido_18_23_diario["Dia_valido"]
    &
    df_ruido_18_23_diario["LAeq_dia"].notna()
)

# ------------------------------------------------------------------------------
# 11. Añadir año para auditoría
# ------------------------------------------------------------------------------

df_ruido_18_23_diario["Any"] = (
    df_ruido_18_23_diario["Fecha"].dt.year
)

# ------------------------------------------------------------------------------
# 12. Auditoría por año
# ------------------------------------------------------------------------------

resumen_anual_18_23 = (
    df_ruido_18_23_diario
    .groupby("Any")
    .agg(
        Sensor_dia=("LAeq_dia", "size"),
        LAeq_disponibles=("LAeq_dia", "count"),
        Dias_validos=("LAeq_dia_valido", "sum"),
        Instalaciones=("Id_Instal", "nunique"),
        Cobertura_media=(
            "Cobertura_corregida_%",
            "mean"
        ),
        LAeq_medio=("LAeq_dia", "mean"),
        LAeq_mediana=("LAeq_dia", "median")
    )
    .round(2)
)

resumen_anual_18_23["Porcentaje_valido"] = (
    resumen_anual_18_23["Dias_validos"]
    /
    resumen_anual_18_23["Sensor_dia"]
    * 100
).round(2)

print("\n" + "=" * 90)
print("RESUMEN POR AÑO")
print("=" * 90)

display(resumen_anual_18_23)

# ------------------------------------------------------------------------------
# 13. Comprobación de cambios horarios
# ------------------------------------------------------------------------------

print("\n--- CONTROL DE CAMBIOS HORARIOS ---")

control_cambios = (
    df_ruido_18_23_diario[
        df_ruido_18_23_diario["Fecha"].isin(
            cambios_verano | cambios_invierno
        )
    ][
        [
            "Fecha",
            "Id_Instal",
            "N_validos",
            "Horas_teoricas",
            "Cobertura_corregida_%",
            "LAeq_dia",
            "Dia_valido"
        ]
    ]
    .sort_values(
        ["Fecha", "Id_Instal"]
    )
)

display(control_cambios.head(30))

# ------------------------------------------------------------------------------
# 14. Controles finales
# ------------------------------------------------------------------------------

duplicados_finales = (
    df_ruido_18_23_diario
    .duplicated(["Fecha", "Id_Instal"])
    .sum()
)

n_validos = (
    df_ruido_18_23_diario[
        "LAeq_dia_valido"
    ].sum()
)

print("\n" + "=" * 90)
print("RESUMEN 4.8")
print("=" * 90)

print(
    f"Periodo: "
    f"{df_ruido_18_23_diario['Fecha'].min().date()} - "
    f"{df_ruido_18_23_diario['Fecha'].max().date()}"
)

print(
    f"Sensor-día totales: "
    f"{len(df_ruido_18_23_diario):,}"
)

print(
    f"Instalaciones únicas: "
    f"{df_ruido_18_23_diario['Id_Instal'].nunique():,}"
)

print(
    f"Duplicados Fecha-Id_Instal: "
    f"{duplicados_finales:,}"
)

print(
    f"LAeq diarios calculados: "
    f"{df_ruido_18_23_diario['LAeq_dia'].notna().sum():,}"
)

print(
    f"LAeq diarios válidos (cobertura >=75 %): "
    f"{n_validos:,}"
)

print(
    f"Porcentaje válido: "
    f"{n_validos / len(df_ruido_18_23_diario) * 100:.2f} %"
)

print(
    f"Coberturas >100 %: "
    f"{(df_ruido_18_23_diario['Cobertura_corregida_%'] > 100).sum():,}"
)

print("\n--- ESTADÍSTICOS LAeq 2018-2023 ---")

display(
    df_ruido_18_23_diario["LAeq_dia"]
    .describe()
    .to_frame("LAeq_dia")
)

print("\n✓ Datos horarios 2018-2023 agregados energéticamente.")
print("✓ Cambios horarios incorporados a la cobertura.")
print("✓ Umbral de cobertura >=75 % aplicado como bandera.")
print("✓ No se han imputado niveles acústicos.")
print("✓ Dataset preparado para homogeneización con 2024.")

4.8 PROCESAMIENTO ENERGÉTICO DIARIO DEL PERIODO 2018-2023

Archivos encontrados: 12
- 2018_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2018_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2019_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2019_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2020_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2020_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2021_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2021_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2022_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2022_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2023_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv
- 2023_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv

Cambios a horario de verano:
[Timestamp('2018-03-25 00:00:00'), Timestamp('2019-03-31 00:00:00'), Timestamp('2020-03-29 00:00:00'), Timestamp('2021-03-28 00:00:00'), Timestamp('2022-03-27 00:00:00'), Timestamp('2023-03-26 00:00:00')]

Cambios a horario de invierno:
[Timestamp('2018-10-28 00:00:00'), Timestamp('2019-10-27 00:00:00'), Time

,Archivo,Registros_originales,Fechas_invalidas,Nulos_LAeq,Sensor_dia_generados,Instalaciones
0,2018_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,354272,0,0,15020,120
1,2018_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,463528,0,0,19715,133
2,2019_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,482844,0,0,20763,144
3,2019_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,515600,0,0,21650,139
4,2020_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,470654,0,0,19924,130
5,2020_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,472345,0,0,19871,138
6,2021_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,495518,0,0,20835,152
7,2021_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,521276,0,0,21878,158
8,2022_1S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,490802,0,0,20845,183
9,2022_2S_XarxaSoroll_EqMonitor_Dades_1Hora.csv,602657,0,0,25309,192



Registros sensor-día antes de consolidar: 259,797
Filas pertenecientes a claves Fecha-Id_Instal repetidas: 0

RESUMEN POR AÑO


,Sensor_dia,LAeq_disponibles,Dias_validos,Instalaciones,Cobertura_media,LAeq_medio,LAeq_mediana,Porcentaje_valido
Any,,,,,,,,
2018,34735,34735,33604,174,98.10,65.73,65.78,96.74
2019,42413,42413,41405,162,98.09,65.24,65.53,97.62
2020,39795,39795,38926,160,98.74,63.15,63.24,97.82
2021,42713,42713,42325,186,99.19,64.00,63.96,99.09
2022,46154,46154,45425,241,98.71,64.54,64.28,98.42
2023,53987,53987,53574,229,99.35,64.65,64.41,99.24



--- CONTROL DE CAMBIOS HORARIOS ---


,Fecha,Id_Instal,N_validos,Horas_teoricas,Cobertura_corregida_%,LAeq_dia,Dia_valido
7037,2018-03-25,495,23,23,100.0,71.53,True
7038,2018-03-25,496,23,23,100.0,71.20,True
7039,2018-03-25,497,23,23,100.0,71.49,True
7040,2018-03-25,651,23,23,100.0,69.63,True
7041,2018-03-25,666,23,23,100.0,61.82,True
7042,2018-03-25,667,23,23,100.0,62.19,True
7043,2018-03-25,686,23,23,100.0,65.52,True
7044,2018-03-25,726,23,23,100.0,60.14,True
7045,2018-03-25,727,23,23,100.0,71.34,True
7046,2018-03-25,728,23,23,100.0,70.15,True



RESUMEN 4.8
Periodo: 2018-01-01 - 2023-12-31
Sensor-día totales: 259,797
Instalaciones únicas: 532
Duplicados Fecha-Id_Instal: 0
LAeq diarios calculados: 259,797
LAeq diarios válidos (cobertura >=75 %): 255,259
Porcentaje válido: 98.25 %
Coberturas >100 %: 41

--- ESTADÍSTICOS LAeq 2018-2023 ---


,LAeq_dia
count,259797.000000
mean,64.535627
std,4.856573
min,10.910000
25%,61.640000
50%,64.500000
75%,67.720000
max,124.060000



✓ Datos horarios 2018-2023 agregados energéticamente.
✓ Cambios horarios incorporados a la cobertura.
✓ Umbral de cobertura >=75 % aplicado como bandera.
✓ No se han imputado niveles acústicos.
✓ Dataset preparado para homogeneización con 2024.


### Resultados del procesamiento acústico 2018–2023

El procesamiento del periodo 2018–2023 comprendió los 12 archivos semestrales disponibles, correspondientes a seis años completos de registros acústicos horarios. No se detectaron fechas inválidas ni valores ausentes en la variable original de nivel acústico durante la lectura de los archivos.

Tras la agregación energética se obtuvieron **259.797 observaciones sensor-día**, correspondientes a **532 instalaciones únicas**, sin duplicados en la clave `Fecha–Id_Instal`.

La corrección de la cobertura temporal incorporó específicamente los días de cambio oficial de horario, considerando 23, 24 o 25 observaciones horarias teóricas según la duración real de cada día. Las comprobaciones realizadas sobre dichas fechas confirman el funcionamiento adecuado de esta corrección.

Aplicando el criterio de cobertura mínima del **75 %**, se consideran temporalmente válidas **255.259 observaciones sensor-día**, equivalentes al **98,25 %** del conjunto. Por tanto, la pérdida potencial de información asociada al criterio de calidad es reducida.

La disponibilidad presenta una elevada estabilidad anual. El porcentaje de observaciones válidas oscila entre el **96,74 % en 2018** y el **99,24 % en 2023**, manteniéndose por encima del 97 % en todos los años restantes.

El `LAeq_dia` presenta una media global de **64,54 dB** y una mediana de **64,50 dB**. Los valores centrales muestran una distribución estable, con un primer cuartil de 61,64 dB y un tercer cuartil de 67,72 dB.

Se identifican valores extremos comprendidos entre **10,91 y 124,06 dB**, que se mantienen provisionalmente en el dataset para preservar la información original. Estos registros serán sometidos a una auditoría específica antes de la construcción del dataset acústico definitivo, evitando la eliminación automática de observaciones únicamente mediante criterios estadísticos.

Asimismo, permanecen 41 observaciones con coberturas superiores al 100 % tras la corrección horaria. Dada su escasa incidencia sobre el conjunto total, se conservan identificadas para su control posterior.

No se han imputado niveles acústicos. La variable de validez temporal permite mantener separadas la disponibilidad del indicador y su representatividad para los análisis posteriores.

## 4.9 Auditoría de valores acústicos extremos (2018–2023)

Antes de integrar los datos históricos con los registros de 2024 se realiza una auditoría específica de los valores extremos del `LAeq_dia`.

El objetivo de esta etapa no es eliminar automáticamente observaciones mediante criterios estadísticos, ya que los niveles acústicos elevados o reducidos pueden corresponder a situaciones reales. En su lugar, se analiza su relación con la cobertura temporal, la instalación de procedencia, la fecha y la persistencia temporal.

Se estudian especialmente:

- los valores diarios mínimos y máximos;
- la distribución de extremos entre días válidos y días con cobertura insuficiente;
- los registros con `LAeq_dia < 30 dB`;
- los registros con `LAeq_dia > 100 dB`;
- las instalaciones responsables de los valores extremos;
- la posible repetición de estos niveles en una misma estación.

Esta auditoría permite distinguir valores potencialmente anómalos de episodios acústicos reales antes de adoptar cualquier decisión de filtrado.

In [ ]:
# ==============================================================================
# 4.9 AUDITORÍA DE VALORES ACÚSTICOS EXTREMOS 2018-2023
# ==============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("4.9 AUDITORÍA DE VALORES ACÚSTICOS EXTREMOS 2018-2023")
print("=" * 90)

df_auditoria_extremos = df_ruido_18_23_diario.copy()

# ------------------------------------------------------------------------------
# 1. Estadísticos generales: total vs días válidos
# ------------------------------------------------------------------------------

print("\n--- DISTRIBUCIÓN GLOBAL ---")

display(
    df_auditoria_extremos["LAeq_dia"]
    .describe()
    .to_frame("Todos_los_dias")
)

print("\n--- DISTRIBUCIÓN SOLO EN DÍAS VÁLIDOS (COBERTURA >=75 %) ---")

display(
    df_auditoria_extremos.loc[
        df_auditoria_extremos["LAeq_dia_valido"],
        "LAeq_dia"
    ]
    .describe()
    .to_frame("Dias_validos")
)

# ------------------------------------------------------------------------------
# 2. Los 20 valores más bajos
# ------------------------------------------------------------------------------

print("\n--- 20 LAeq DIARIOS MÁS BAJOS ---")

columnas_control = [
    "Fecha",
    "Any",
    "Id_Instal",
    "LAeq_dia",
    "N_validos",
    "Horas_teoricas",
    "Cobertura_corregida_%",
    "Dia_valido"
]

display(
    df_auditoria_extremos
    .nsmallest(20, "LAeq_dia")[columnas_control]
)

# ------------------------------------------------------------------------------
# 3. Los 20 valores más altos
# ------------------------------------------------------------------------------

print("\n--- 20 LAeq DIARIOS MÁS ALTOS ---")

display(
    df_auditoria_extremos
    .nlargest(20, "LAeq_dia")[columnas_control]
)

# ------------------------------------------------------------------------------
# 4. Umbrales de auditoría
#
# IMPORTANTE:
# Estos límites NO constituyen todavía criterios de eliminación.
# Solo se utilizan para localizar observaciones que merecen revisión.
# ------------------------------------------------------------------------------

UMBRAL_BAJO = 30
UMBRAL_ALTO = 100

extremos_bajos = (
    df_auditoria_extremos["LAeq_dia"] < UMBRAL_BAJO
)

extremos_altos = (
    df_auditoria_extremos["LAeq_dia"] > UMBRAL_ALTO
)

print("\n--- FRECUENCIA DE EXTREMOS ---")

print(
    f"LAeq_dia < {UMBRAL_BAJO} dB: "
    f"{extremos_bajos.sum():,} "
    f"({extremos_bajos.mean() * 100:.4f} %)"
)

print(
    f"LAeq_dia > {UMBRAL_ALTO} dB: "
    f"{extremos_altos.sum():,} "
    f"({extremos_altos.mean() * 100:.4f} %)"
)

# ------------------------------------------------------------------------------
# 5. ¿Los extremos pasan el criterio de cobertura?
# ------------------------------------------------------------------------------

print("\n--- EXTREMOS SEGÚN VALIDEZ TEMPORAL ---")

tabla_extremos = pd.DataFrame({
    "Grupo": [
        f"< {UMBRAL_BAJO} dB",
        f"> {UMBRAL_ALTO} dB"
    ],
    "Total": [
        extremos_bajos.sum(),
        extremos_altos.sum()
    ],
    "Dias_validos": [
        (
            extremos_bajos
            & df_auditoria_extremos["Dia_valido"]
        ).sum(),
        (
            extremos_altos
            & df_auditoria_extremos["Dia_valido"]
        ).sum()
    ],
    "Dias_no_validos": [
        (
            extremos_bajos
            & ~df_auditoria_extremos["Dia_valido"]
        ).sum(),
        (
            extremos_altos
            & ~df_auditoria_extremos["Dia_valido"]
        ).sum()
    ]
})

tabla_extremos["%_validos"] = np.where(
    tabla_extremos["Total"] > 0,
    (
        tabla_extremos["Dias_validos"]
        / tabla_extremos["Total"]
        * 100
    ).round(2),
    np.nan
)

display(tabla_extremos)

# ------------------------------------------------------------------------------
# 6. Extremos bajos que además son días válidos
# ------------------------------------------------------------------------------

print(
    f"\n--- < {UMBRAL_BAJO} dB CON COBERTURA >=75 % ---"
)

bajos_validos = (
    df_auditoria_extremos[
        extremos_bajos
        & df_auditoria_extremos["Dia_valido"]
    ]
    .sort_values("LAeq_dia")
)

print(
    f"Número de observaciones: "
    f"{len(bajos_validos):,}"
)

display(
    bajos_validos[columnas_control].head(30)
)

# ------------------------------------------------------------------------------
# 7. Extremos altos que además son días válidos
# ------------------------------------------------------------------------------

print(
    f"\n--- > {UMBRAL_ALTO} dB CON COBERTURA >=75 % ---"
)

altos_validos = (
    df_auditoria_extremos[
        extremos_altos
        & df_auditoria_extremos["Dia_valido"]
    ]
    .sort_values(
        "LAeq_dia",
        ascending=False
    )
)

print(
    f"Número de observaciones: "
    f"{len(altos_validos):,}"
)

display(
    altos_validos[columnas_control].head(30)
)

# ------------------------------------------------------------------------------
# 8. Instalaciones responsables de los extremos bajos
# ------------------------------------------------------------------------------

print("\n--- INSTALACIONES CON MÁS VALORES <30 dB ---")

if extremos_bajos.sum() > 0:

    estaciones_bajas = (
        df_auditoria_extremos.loc[
            extremos_bajos
        ]
        .groupby("Id_Instal")
        .agg(
            N_extremos=("LAeq_dia", "size"),
            LAeq_min=("LAeq_dia", "min"),
            LAeq_mediana=("LAeq_dia", "median"),
            Cobertura_media=(
                "Cobertura_corregida_%",
                "mean"
            )
        )
        .sort_values(
            "N_extremos",
            ascending=False
        )
        .round(2)
    )

    display(estaciones_bajas.head(20))

# ------------------------------------------------------------------------------
# 9. Instalaciones responsables de los extremos altos
# ------------------------------------------------------------------------------

print("\n--- INSTALACIONES CON MÁS VALORES >100 dB ---")

if extremos_altos.sum() > 0:

    estaciones_altas = (
        df_auditoria_extremos.loc[
            extremos_altos
        ]
        .groupby("Id_Instal")
        .agg(
            N_extremos=("LAeq_dia", "size"),
            LAeq_max=("LAeq_dia", "max"),
            LAeq_mediana=("LAeq_dia", "median"),
            Cobertura_media=(
                "Cobertura_corregida_%",
                "mean"
            )
        )
        .sort_values(
            "N_extremos",
            ascending=False
        )
        .round(2)
    )

    display(estaciones_altas.head(20))

# ------------------------------------------------------------------------------
# 10. Comprobar persistencia de las estaciones extremas
#
# Para cada estación implicada calculamos sus estadísticas sobre TODA
# su serie, no únicamente sobre las observaciones extremas.
# ------------------------------------------------------------------------------

ids_extremos = df_auditoria_extremos.loc[
    extremos_bajos | extremos_altos,
    "Id_Instal"
].unique()

if len(ids_extremos) > 0:

    persistencia = (
        df_auditoria_extremos[
            df_auditoria_extremos["Id_Instal"].isin(ids_extremos)
        ]
        .groupby("Id_Instal")
        .agg(
            N_dias=("LAeq_dia", "size"),
            LAeq_min=("LAeq_dia", "min"),
            LAeq_media=("LAeq_dia", "mean"),
            LAeq_mediana=("LAeq_dia", "median"),
            LAeq_max=("LAeq_dia", "max"),
            Cobertura_media=(
                "Cobertura_corregida_%",
                "mean"
            )
        )
        .round(2)
    )

    print("\n--- COMPORTAMIENTO GLOBAL DE LAS ESTACIONES EXTREMAS ---")
    display(
        persistencia
        .sort_values(
            "LAeq_max",
            ascending=False
        )
    )

# ------------------------------------------------------------------------------
# 11. Extremos por año
# ------------------------------------------------------------------------------

print("\n--- EXTREMOS POR AÑO ---")

extremos_por_anio = (
    df_auditoria_extremos
    .assign(
        Extremo_bajo=extremos_bajos.astype(int),
        Extremo_alto=extremos_altos.astype(int)
    )
    .groupby("Any")
    .agg(
        Sensor_dia=("LAeq_dia", "size"),
        Extremos_bajos=("Extremo_bajo", "sum"),
        Extremos_altos=("Extremo_alto", "sum"),
        LAeq_min=("LAeq_dia", "min"),
        LAeq_max=("LAeq_dia", "max")
    )
)

display(extremos_por_anio)

# ------------------------------------------------------------------------------
# 12. Percentiles extremos
# ------------------------------------------------------------------------------

print("\n--- PERCENTILES DE LA DISTRIBUCIÓN ---")

percentiles = (
    df_auditoria_extremos["LAeq_dia"]
    .quantile([
        0.001,
        0.005,
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
        0.995,
        0.999
    ])
    .to_frame("LAeq_dia")
)

display(percentiles)

# ------------------------------------------------------------------------------
# 13. Resumen
# ------------------------------------------------------------------------------

print("\n" + "=" * 90)
print("RESUMEN 4.9")
print("=" * 90)

print(
    f"Sensor-día auditados: "
    f"{len(df_auditoria_extremos):,}"
)

print(
    f"LAeq <30 dB: "
    f"{extremos_bajos.sum():,}"
)

print(
    f"LAeq <30 dB y día válido: "
    f"{len(bajos_validos):,}"
)

print(
    f"LAeq >100 dB: "
    f"{extremos_altos.sum():,}"
)

print(
    f"LAeq >100 dB y día válido: "
    f"{len(altos_validos):,}"
)

print(
    f"Mínimo en días válidos: "
    f"{df_auditoria_extremos.loc[df_auditoria_extremos['Dia_valido'], 'LAeq_dia'].min():.2f} dB"
)

print(
    f"Máximo en días válidos: "
    f"{df_auditoria_extremos.loc[df_auditoria_extremos['Dia_valido'], 'LAeq_dia'].max():.2f} dB"
)

print("\n✓ Auditoría completada.")
print("✓ No se ha eliminado ni modificado ninguna observación.")
print("✓ Los extremos quedan identificados para decidir su tratamiento.")

4.9 AUDITORÍA DE VALORES ACÚSTICOS EXTREMOS 2018-2023

--- DISTRIBUCIÓN GLOBAL ---


,Todos_los_dias
count,259797.000000
mean,64.535627
std,4.856573
min,10.910000
25%,61.640000
50%,64.500000
75%,67.720000
max,124.060000



--- DISTRIBUCIÓN SOLO EN DÍAS VÁLIDOS (COBERTURA >=75 %) ---


,Dias_validos
count,255259.000000
mean,64.540482
std,4.826733
min,10.910000
25%,61.650000
50%,64.510000
75%,67.710000
max,107.180000



--- 20 LAeq DIARIOS MÁS BAJOS ---


...


--- EXTREMOS POR AÑO ---


,Sensor_dia,Extremos_bajos,Extremos_altos,LAeq_min,LAeq_max
Any,,,,,
2018,34735,2,0,27.67,93.98
2019,42413,2,2,28.06,107.18
2020,39795,12,0,12.62,90.49
2021,42713,5,0,10.91,96.76
2022,46154,2,0,28.32,94.93
2023,53987,2,1,27.90,124.06



--- PERCENTILES DE LA DISTRIBUCIÓN ---


,LAeq_dia
0.001,44.00000
0.005,48.58000
0.010,51.67000
0.050,56.70000
0.500,64.50000
0.950,72.02000
0.990,75.03000
0.995,77.43000
0.999,84.69224



RESUMEN 4.9
Sensor-día auditados: 259,797
LAeq <30 dB: 25
LAeq <30 dB y día válido: 20
LAeq >100 dB: 3
LAeq >100 dB y día válido: 2
Mínimo en días válidos: 10.91 dB
Máximo en días válidos: 107.18 dB

✓ Auditoría completada.
✓ No se ha eliminado ni modificado ninguna observación.
✓ Los extremos quedan identificados para decidir su tratamiento.


### Resultados de la auditoría de valores extremos

La auditoría comprendió **259.797 observaciones sensor-día** del periodo 2018–2023.

Se identificaron únicamente **25 observaciones con `LAeq_dia < 30 dB` (0,0096 %)** y **3 observaciones con `LAeq_dia > 100 dB` (0,0012 %)**, lo que confirma el carácter excepcional de estos registros dentro de la distribución global.

La comprobación conjunta con la cobertura temporal mostró que **20 de los 25 valores inferiores a 30 dB** y **2 de los 3 valores superiores a 100 dB** corresponden a días que cumplen el criterio mínimo de cobertura del 75 %. Por tanto, la presencia de valores extremos no puede atribuirse exclusivamente a una disponibilidad temporal insuficiente.

Asimismo, algunos niveles bajos presentan persistencia temporal en determinadas instalaciones, mientras que los dos valores superiores a 100 dB considerados temporalmente válidos se registraron el mismo día en dos instalaciones diferentes.

El máximo absoluto del conjunto, **124,06 dB**, presenta una cobertura del **66,67 %** y queda clasificado como día no válido mediante el criterio de calidad temporal establecido.

A partir de estos resultados se decide **no aplicar eliminación automática, winsorización ni sustitución de valores extremos en función exclusiva de su magnitud**. Los registros se conservan en el dataset original y la selección del conjunto analítico se realizará mediante el indicador `Dia_valido`, basado en la cobertura temporal.

Esta estrategia evita introducir modificaciones artificiales en la distribución acústica y mantiene la trazabilidad entre los datos originales, los controles de calidad y el dataset utilizado posteriormente en los análisis.

## 4.10 Homogeneización y unión del dataset acústico 2018–2024

Una vez procesados de forma independiente los registros horarios de 2018–2023 y los registros minutales de 2024, se procede a su homogeneización en una única estructura diaria.

Aunque la resolución temporal original difiere entre ambos periodos, el indicador diario ha sido calculado en ambos casos mediante agregación energética de los niveles acústicos. De esta forma, `LAeq_dia` mantiene una interpretación homogénea a lo largo de toda la serie temporal.

Para cada combinación `Fecha–Id_Instal` se conservan:

- la fecha de observación;
- el identificador de la instalación acústica;
- el año;
- el nivel equivalente diario `LAeq_dia`;
- el número de observaciones válidas utilizadas;
- el número teórico de observaciones esperado;
- la cobertura temporal corregida;
- la bandera `Dia_valido`, definida mediante una cobertura mínima del 75 %;
- la resolución temporal de origen, con objeto de mantener la trazabilidad metodológica.

La unión se realiza sin imputar niveles acústicos y sin eliminar observaciones por su magnitud. Los registros con cobertura insuficiente se mantienen en el dataset maestro, identificados mediante la variable `Dia_valido`, permitiendo diferenciar entre disponibilidad de información y validez para los análisis posteriores.

Finalmente, se comprueba la ausencia de duplicados `Fecha–Id_Instal`, la continuidad temporal de la serie y la coherencia de las variables entre ambos periodos.

In [ ]:
# ==============================================================================
# 4.10 HOMOGENEIZACIÓN Y UNIÓN DEL DATASET ACÚSTICO 2018-2024
# ==============================================================================

import pandas as pd
import numpy as np
import gc

print("=" * 90)
print("4.10 HOMOGENEIZACIÓN Y UNIÓN DEL DATASET ACÚSTICO 2018-2024")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Copias de trabajo
# ------------------------------------------------------------------------------

ruido_18_23 = df_ruido_18_23_diario.copy()
ruido_2024 = df_ruido_2024_diario.copy()

print("\nDimensiones iniciales:")
print(f"2018-2023: {ruido_18_23.shape}")
print(f"2024     : {ruido_2024.shape}")

print("\nColumnas 2018-2023:")
print(ruido_18_23.columns.tolist())

print("\nColumnas 2024:")
print(ruido_2024.columns.tolist())

# ------------------------------------------------------------------------------
# 2. Normalizar Fecha e Id_Instal
# ------------------------------------------------------------------------------

for df in [ruido_18_23, ruido_2024]:

    df["Fecha"] = pd.to_datetime(
        df["Fecha"],
        errors="coerce"
    )

    df["Id_Instal"] = pd.to_numeric(
        df["Id_Instal"],
        errors="coerce"
    )

# ------------------------------------------------------------------------------
# 3. Homogeneizar nombres de columnas de 2024
#
# Se contemplan posibles nombres utilizados durante el procesamiento anterior.
# ------------------------------------------------------------------------------

renombres_2024 = {}

# LAeq diario
if "Nivell_LAeq" in ruido_2024.columns and "LAeq_dia" not in ruido_2024.columns:
    renombres_2024["Nivell_LAeq"] = "LAeq_dia"

if "LAeq_diario" in ruido_2024.columns and "LAeq_dia" not in ruido_2024.columns:
    renombres_2024["LAeq_diario"] = "LAeq_dia"

# Número de observaciones válidas
if "N_min_validos" in ruido_2024.columns and "N_validos" not in ruido_2024.columns:
    renombres_2024["N_min_validos"] = "N_validos"

if "Minutos_validos" in ruido_2024.columns and "N_validos" not in ruido_2024.columns:
    renombres_2024["Minutos_validos"] = "N_validos"

# Número teórico
if "Minutos_teoricos" in ruido_2024.columns and "Observaciones_teoricas" not in ruido_2024.columns:
    renombres_2024["Minutos_teoricos"] = "Observaciones_teoricas"

# Cobertura
if "Cobertura_%" in ruido_2024.columns and "Cobertura_corregida_%" not in ruido_2024.columns:
    renombres_2024["Cobertura_%"] = "Cobertura_corregida_%"

ruido_2024 = ruido_2024.rename(
    columns=renombres_2024
)

print("\nRenombres aplicados a 2024:")
print(renombres_2024 if renombres_2024 else "No fueron necesarios.")

# ------------------------------------------------------------------------------
# 4. Homogeneizar número teórico de observaciones
#
# 2018-2023 -> horas
# 2024      -> minutos
#
# No deben compararse directamente como cantidades absolutas.
# La variable comparable entre periodos es la COBERTURA.
# ------------------------------------------------------------------------------

if "Horas_teoricas" in ruido_18_23.columns:

    ruido_18_23["Observaciones_teoricas"] = (
        ruido_18_23["Horas_teoricas"]
    )

elif "Observaciones_teoricas" not in ruido_18_23.columns:

    ruido_18_23["Observaciones_teoricas"] = np.nan


# En 2024 buscamos el nombre utilizado previamente

if "Observaciones_teoricas" not in ruido_2024.columns:

    candidatos_teoricos = [
        "N_teoricos",
        "N_min_teoricos",
        "Minutos_dia",
        "Minutos_esperados"
    ]

    encontrado = False

    for col in candidatos_teoricos:

        if col in ruido_2024.columns:

            ruido_2024["Observaciones_teoricas"] = (
                ruido_2024[col]
            )

            encontrado = True
            break

    if not encontrado:
        ruido_2024["Observaciones_teoricas"] = np.nan

# ------------------------------------------------------------------------------
# 5. Asegurar existencia de las variables esenciales
# ------------------------------------------------------------------------------

variables_esenciales = [
    "Fecha",
    "Id_Instal",
    "LAeq_dia",
    "N_validos",
    "Cobertura_corregida_%",
    "Dia_valido"
]

print("\n--- CONTROL DE VARIABLES ESENCIALES ---")

for nombre, df in [
    ("2018-2023", ruido_18_23),
    ("2024", ruido_2024)
]:

    faltantes = [
        col for col in variables_esenciales
        if col not in df.columns
    ]

    if len(faltantes) == 0:

        print(
            f"✓ {nombre}: todas las variables "
            f"esenciales disponibles."
        )

    else:

        print(
            f"⚠ {nombre}: faltan columnas: "
            f"{faltantes}"
        )

# Si falta una variable esencial detenemos aquí para no construir
# silenciosamente un dataset incorrecto.

faltantes_18 = [
    c for c in variables_esenciales
    if c not in ruido_18_23.columns
]

faltantes_24 = [
    c for c in variables_esenciales
    if c not in ruido_2024.columns
]

if faltantes_18 or faltantes_24:

    raise ValueError(
        "No se puede realizar todavía la unión. "
        f"Faltantes 2018-2023: {faltantes_18}; "
        f"faltantes 2024: {faltantes_24}"
    )

# ------------------------------------------------------------------------------
# 6. Añadir variables de trazabilidad
# ------------------------------------------------------------------------------

ruido_18_23["Resolucion_origen"] = "1 hora"
ruido_2024["Resolucion_origen"] = "1 minuto"

ruido_18_23["Periodo_fuente"] = "2018-2023"
ruido_2024["Periodo_fuente"] = "2024"

# Año calculado directamente desde Fecha para evitar incoherencias

ruido_18_23["Any"] = (
    ruido_18_23["Fecha"].dt.year
)

ruido_2024["Any"] = (
    ruido_2024["Fecha"].dt.year
)

# ------------------------------------------------------------------------------
# 7. Seleccionar estructura común
# ------------------------------------------------------------------------------

columnas_finales = [
    "Fecha",
    "Any",
    "Id_Instal",
    "LAeq_dia",
    "N_validos",
    "Observaciones_teoricas",
    "Cobertura_corregida_%",
    "Dia_valido",
    "Resolucion_origen",
    "Periodo_fuente"
]

ruido_18_23 = ruido_18_23[
    columnas_finales
].copy()

ruido_2024 = ruido_2024[
    columnas_finales
].copy()

print("\n--- ESTRUCTURA HOMOGENEIZADA ---")

print("\n2018-2023:")
print(ruido_18_23.dtypes)

print("\n2024:")
print(ruido_2024.dtypes)

# ------------------------------------------------------------------------------
# 8. Unión 2018-2024
# ------------------------------------------------------------------------------

df_ruido_diario_2018_2024 = pd.concat(
    [
        ruido_18_23,
        ruido_2024
    ],
    ignore_index=True
)

df_ruido_diario_2018_2024 = (
    df_ruido_diario_2018_2024
    .sort_values(
        ["Fecha", "Id_Instal"]
    )
    .reset_index(drop=True)
)

print("\n✓ Unión realizada.")

print(
    f"Dimensiones: "
    f"{df_ruido_diario_2018_2024.shape}"
)

# ------------------------------------------------------------------------------
# 9. Control temporal
# ------------------------------------------------------------------------------

print("\n--- CONTROL TEMPORAL ---")

print(
    "Periodo:",
    df_ruido_diario_2018_2024["Fecha"]
    .min()
    .date(),
    "-",
    df_ruido_diario_2018_2024["Fecha"]
    .max()
    .date()
)

print(
    "Años presentes:",
    sorted(
        df_ruido_diario_2018_2024[
            "Any"
        ].dropna().unique()
    )
)

# ------------------------------------------------------------------------------
# 10. Control de duplicados
# ------------------------------------------------------------------------------

duplicados = (
    df_ruido_diario_2018_2024
    .duplicated(
        ["Fecha", "Id_Instal"]
    )
    .sum()
)

print("\n--- DUPLICADOS ---")

print(
    f"Duplicados Fecha-Id_Instal: "
    f"{duplicados:,}"
)

# Si aparecieran duplicados, NO los eliminamos automáticamente.

if duplicados > 0:

    print(
        "⚠ Se han detectado duplicados. "
        "Deben auditarse antes de continuar."
    )

else:

    print(
        "✓ No existen duplicados "
        "Fecha-Id_Instal."
    )

# ------------------------------------------------------------------------------
# 11. Nulos
# ------------------------------------------------------------------------------

print("\n--- VALORES AUSENTES ---")

nulos_union = pd.DataFrame({
    "Nulos":
        df_ruido_diario_2018_2024
        .isna()
        .sum(),

    "Porcentaje":
        (
            df_ruido_diario_2018_2024
            .isna()
            .mean()
            * 100
        ).round(3)
})

display(nulos_union)

# ------------------------------------------------------------------------------
# 12. Resumen por año
# ------------------------------------------------------------------------------

print("\n--- RESUMEN POR AÑO ---")

resumen_anual_union = (
    df_ruido_diario_2018_2024
    .groupby("Any")
    .agg(
        Sensor_dia=(
            "LAeq_dia",
            "size"
        ),
        LAeq_disponibles=(
            "LAeq_dia",
            "count"
        ),
        Dias_validos=(
            "Dia_valido",
            "sum"
        ),
        Instalaciones=(
            "Id_Instal",
            "nunique"
        ),
        Cobertura_media=(
            "Cobertura_corregida_%",
            "mean"
        ),
        LAeq_medio=(
            "LAeq_dia",
            "mean"
        ),
        LAeq_mediana=(
            "LAeq_dia",
            "median"
        )
    )
    .round(2)
)

resumen_anual_union[
    "Porcentaje_valido"
] = (
    resumen_anual_union["Dias_validos"]
    /
    resumen_anual_union["Sensor_dia"]
    * 100
).round(2)

display(resumen_anual_union)

# ------------------------------------------------------------------------------
# 13. Control específico del salto 2023 -> 2024
# ------------------------------------------------------------------------------

print("\n--- CONTROL DE LA TRANSICIÓN 2023 → 2024 ---")

control_transicion = (
    df_ruido_diario_2018_2024[
        df_ruido_diario_2018_2024["Fecha"]
        .between(
            "2023-12-25",
            "2024-01-07"
        )
    ]
    .groupby("Fecha")
    .agg(
        Instalaciones=(
            "Id_Instal",
            "nunique"
        ),
        LAeq_medio=(
            "LAeq_dia",
            "mean"
        ),
        Cobertura_media=(
            "Cobertura_corregida_%",
            "mean"
        ),
        Dias_validos=(
            "Dia_valido",
            "sum"
        )
    )
    .round(2)
)

display(control_transicion)

# ------------------------------------------------------------------------------
# 14. Comprobar coherencia de la resolución de origen
# ------------------------------------------------------------------------------

print("\n--- RESOLUCIÓN TEMPORAL DE ORIGEN ---")

display(
    pd.crosstab(
        df_ruido_diario_2018_2024["Any"],
        df_ruido_diario_2018_2024[
            "Resolucion_origen"
        ]
    )
)

# ------------------------------------------------------------------------------
# 15. Estadísticos LAeq por periodo
# ------------------------------------------------------------------------------

print("\n--- LAeq POR PERIODO DE ORIGEN ---")

display(
    df_ruido_diario_2018_2024
    .groupby("Periodo_fuente")[
        "LAeq_dia"
    ]
    .describe()
    .round(2)
)

# ------------------------------------------------------------------------------
# 16. Resumen final
# ------------------------------------------------------------------------------

total = len(
    df_ruido_diario_2018_2024
)

validos = (
    df_ruido_diario_2018_2024[
        "Dia_valido"
    ].sum()
)

print("\n" + "=" * 90)
print("RESUMEN 4.10")
print("=" * 90)

print(
    f"Registros sensor-día: "
    f"{total:,}"
)

print(
    f"Instalaciones únicas: "
    f"{df_ruido_diario_2018_2024['Id_Instal'].nunique():,}"
)

print(
    f"Días válidos: "
    f"{validos:,}"
)

print(
    f"Porcentaje válido: "
    f"{validos / total * 100:.2f} %"
)

print(
    f"Duplicados Fecha-Id_Instal: "
    f"{duplicados:,}"
)

print(
    f"LAeq mínimo: "
    f"{df_ruido_diario_2018_2024['LAeq_dia'].min():.2f} dB"
)

print(
    f"LAeq máximo: "
    f"{df_ruido_diario_2018_2024['LAeq_dia'].max():.2f} dB"
)

print("\n✓ Periodos 2018-2023 y 2024 homogeneizados.")
print("✓ Resolución temporal original conservada como trazabilidad.")
print("✓ No se han imputado niveles acústicos.")
print("✓ No se han eliminado extremos por magnitud.")
print("✓ Dataset preparado para auditoría final.")

4.10 HOMOGENEIZACIÓN Y UNIÓN DEL DATASET ACÚSTICO 2018-2024

Dimensiones iniciales:
2018-2023: (259797, 10)
2024     : (57154, 12)

Columnas 2018-2023:
['Fecha', 'Id_Instal', 'N_validos', 'Suma_energia', 'Horas_teoricas', 'Cobertura_corregida_%', 'LAeq_dia', 'Dia_valido', 'LAeq_dia_valido', 'Any']

Columnas 2024:
['Fecha', 'Id_Instal', 'N_registros', 'N_validos', 'Suma_energia', 'Cobertura_%', 'Minutos_teoricos', 'Cobertura_corregida_%', 'Categoria_cobertura', 'Dia_valido', 'LAeq_dia', 'LAeq_dia_valido']

Renombres aplicados a 2024:
{'Minutos_teoricos': 'Observaciones_teoricas'}

--- CONTROL DE VARIABLES ESENCIALES ---
✓ 2018-2023: todas las variables esenciales disponibles.
✓ 2024: todas las variables esenciales disponibles.

--- ESTRUCTURA HOMOGENEIZADA ---

2018-2023:
Fecha                     datetime64[ns]
Any                                int32
Id_Instal                          int64
LAeq_dia                         float64
N_validos                          int64
Observaciones

,Nulos,Porcentaje
Fecha,0,0.0
Any,0,0.0
Id_Instal,0,0.0
LAeq_dia,0,0.0
N_validos,0,0.0
Observaciones_teoricas,0,0.0
Cobertura_corregida_%,0,0.0
Dia_valido,0,0.0
Resolucion_origen,0,0.0
Periodo_fuente,0,0.0



--- RESUMEN POR AÑO ---


,Sensor_dia,LAeq_disponibles,Dias_validos,Instalaciones,Cobertura_media,LAeq_medio,LAeq_mediana,Porcentaje_valido
Any,,,,,,,,
2018,34735,34735,33604,174,98.10,65.73,65.78,96.74
2019,42413,42413,41405,162,98.09,65.24,65.53,97.62
2020,39795,39795,38926,160,98.74,63.15,63.24,97.82
2021,42713,42713,42325,186,99.19,64.00,63.96,99.09
2022,46154,46154,45425,241,98.71,64.54,64.28,98.42
2023,53987,53987,53574,229,99.35,64.65,64.41,99.24
2024,57154,57154,55701,244,98.41,64.28,64.00,97.46



--- CONTROL DE LA TRANSICIÓN 2023 → 2024 ---


...


--- RESOLUCIÓN TEMPORAL DE ORIGEN ---


Resolucion_origen,1 hora,1 minuto
Any,,
2018,34735,0
2019,42413,0
2020,39795,0
2021,42713,0
2022,46154,0
2023,53987,0
2024,0,57154



--- LAeq POR PERIODO DE ORIGEN ---


,count,mean,std,min,25%,50%,75%,max
Periodo_fuente,,,,,,,,
2018-2023,259797.0,64.54,4.86,10.91,61.64,64.5,67.72,124.06
2024,57154.0,64.28,4.33,16.95,61.77,64.0,66.74,94.78



RESUMEN 4.10
Registros sensor-día: 316,951
Instalaciones únicas: 628
Días válidos: 310,960
Porcentaje válido: 98.11 %
Duplicados Fecha-Id_Instal: 0
LAeq mínimo: 10.91 dB
LAeq máximo: 124.06 dB

✓ Periodos 2018-2023 y 2024 homogeneizados.
✓ Resolución temporal original conservada como trazabilidad.
✓ No se han imputado niveles acústicos.
✓ No se han eliminado extremos por magnitud.
✓ Dataset preparado para auditoría final.


### Resultados de la homogeneización 2018–2024

La integración de los dos periodos temporales generó un dataset acústico unificado compuesto por **316.951 observaciones sensor-día**, correspondientes a **628 instalaciones únicas**, con cobertura temporal comprendida entre el 1 de enero de 2018 y el 31 de diciembre de 2024.

La estructura final contiene diez variables comunes y no presenta valores ausentes en ninguna de ellas. Asimismo, no se detectaron duplicados para la clave `Fecha–Id_Instal`.

La trazabilidad de la resolución temporal original se conserva mediante la variable `Resolucion_origen`: los registros de 2018–2023 proceden de mediciones horarias, mientras que los correspondientes a 2024 proceden de mediciones minutales. En ambos casos, el indicador `LAeq_dia` fue obtenido mediante agregación energética.

El criterio mínimo de cobertura temporal del 75 % clasifica como válidas **310.960 observaciones sensor-día**, equivalentes al **98,11 %** del conjunto unificado.

La disponibilidad anual permanece elevada durante todo el periodo, con porcentajes de observaciones válidas comprendidos entre el **96,74 % y el 99,24 %**. La cobertura media anual se mantiene asimismo próxima al 98–99 %.

La inspección específica de la transición entre diciembre de 2023 y enero de 2024 no muestra discontinuidades evidentes en los niveles acústicos medios coincidentes con el cambio de resolución de la fuente. Los valores diarios medios permanecen dentro de rangos comparables y la cobertura temporal continúa próxima al 100 %.

El `LAeq_dia` medio del periodo 2018–2023 es de **64,54 dB**, frente a **64,28 dB en 2024**, mientras que las medianas son respectivamente 64,50 y 64,00 dB.

Se mantienen en el dataset maestro los valores extremos previamente auditados, así como las observaciones con cobertura insuficiente, preservando la trazabilidad completa. No se han imputado niveles acústicos ni eliminado observaciones en función exclusiva de su magnitud.

El dataset resultante queda preparado para la auditoría final de calidad previa a su almacenamiento definitivo y posterior integración con el resto de bloques del TFM.

## 4.11 Auditoría final de calidad del dataset acústico 2018–2024

Antes del almacenamiento definitivo se realiza una auditoría global del dataset acústico unificado con objeto de verificar su integridad estructural, temporal y metodológica.

La validación comprende:

- dimensiones y periodo temporal;
- unicidad de la clave `Fecha–Id_Instal`;
- presencia de valores ausentes;
- coherencia entre fecha y año;
- integridad de la variable `Dia_valido`;
- distribución de la cobertura temporal;
- control de coberturas superiores al 100 %;
- distribución y rango del `LAeq_dia`;
- número de instalaciones y observaciones por año;
- trazabilidad de la resolución temporal original;
- comportamiento de las observaciones consideradas válidas y no válidas.

La auditoría no modifica, imputa ni elimina observaciones. Su finalidad es certificar la calidad del conjunto resultante y documentar las características del dataset antes de su almacenamiento definitivo y posterior integración con los restantes bloques de información.

In [ ]:
# ==============================================================================
# 4.11 AUDITORÍA FINAL DEL DATASET ACÚSTICO 2018-2024
# ==============================================================================

import pandas as pd
import numpy as np

print("=" * 90)
print("4.11 AUDITORÍA FINAL DEL DATASET ACÚSTICO 2018-2024")
print("=" * 90)

df_final = df_ruido_diario_2018_2024

# ------------------------------------------------------------------------------
# 1. DIMENSIONES Y PERIODO
# ------------------------------------------------------------------------------

print("\n--- 1. ESTRUCTURA GENERAL ---")

print(f"Dimensiones: {df_final.shape}")
print(f"Registros sensor-día: {len(df_final):,}")
print(f"Variables: {df_final.shape[1]}")
print(f"Instalaciones únicas: {df_final['Id_Instal'].nunique():,}")

print(
    f"Periodo: "
    f"{df_final['Fecha'].min().date()} - "
    f"{df_final['Fecha'].max().date()}"
)

# ------------------------------------------------------------------------------
# 2. DUPLICADOS
# ------------------------------------------------------------------------------

print("\n--- 2. UNICIDAD DE LA CLAVE FECHA–INSTALACIÓN ---")

duplicados = (
    df_final
    .duplicated(["Fecha", "Id_Instal"])
    .sum()
)

print(f"Duplicados Fecha-Id_Instal: {duplicados:,}")

# ------------------------------------------------------------------------------
# 3. VALORES AUSENTES
# ------------------------------------------------------------------------------

print("\n--- 3. VALORES AUSENTES ---")

tabla_nulos = pd.DataFrame({
    "Nulos": df_final.isna().sum(),
    "Porcentaje": (
        df_final.isna().mean() * 100
    ).round(4)
})

display(tabla_nulos)

# ------------------------------------------------------------------------------
# 4. COHERENCIA ENTRE FECHA Y AÑO
# ------------------------------------------------------------------------------

print("\n--- 4. COHERENCIA TEMPORAL ---")

incoherencia_anio = (
    df_final["Any"]
    != df_final["Fecha"].dt.year
).sum()

print(
    f"Incoherencias Fecha-Año: "
    f"{incoherencia_anio:,}"
)

print(
    "Años presentes:",
    sorted(df_final["Any"].unique())
)

# ------------------------------------------------------------------------------
# 5. CONTROL DE LA BANDERA Dia_valido
#
# Debe equivaler exactamente a cobertura >=75 %
# ------------------------------------------------------------------------------

print("\n--- 5. COHERENCIA DE LA BANDERA DE CALIDAD ---")

dia_valido_teorico = (
    df_final["Cobertura_corregida_%"] >= 75
)

incoherencias_bandera = (
    df_final["Dia_valido"]
    != dia_valido_teorico
).sum()

print(
    f"Incoherencias Dia_valido vs cobertura >=75 %: "
    f"{incoherencias_bandera:,}"
)

# ------------------------------------------------------------------------------
# 6. COBERTURA TEMPORAL
# ------------------------------------------------------------------------------

print("\n--- 6. DISTRIBUCIÓN DE COBERTURA ---")

display(
    df_final["Cobertura_corregida_%"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .to_frame("Cobertura_%")
)

# Clasificación descriptiva de cobertura

condiciones = [
    df_final["Cobertura_corregida_%"] < 50,

    (
        (df_final["Cobertura_corregida_%"] >= 50)
        &
        (df_final["Cobertura_corregida_%"] < 75)
    ),

    (
        (df_final["Cobertura_corregida_%"] >= 75)
        &
        (df_final["Cobertura_corregida_%"] < 90)
    ),

    (
        (df_final["Cobertura_corregida_%"] >= 90)
        &
        (df_final["Cobertura_corregida_%"] <= 100)
    ),

    df_final["Cobertura_corregida_%"] > 100
]

categorias = [
    "<50 %",
    "50-<75 %",
    "75-<90 %",
    "90-100 %",
    ">100 %"
]

df_control_cobertura = df_final.copy()

df_control_cobertura["Categoria_cobertura"] = np.select(
    condiciones,
    categorias,
    default="Sin clasificar"
)

tabla_cobertura = (
    df_control_cobertura[
        "Categoria_cobertura"
    ]
    .value_counts()
    .reindex(categorias, fill_value=0)
    .to_frame("N")
)

tabla_cobertura["Porcentaje"] = (
    tabla_cobertura["N"]
    / len(df_control_cobertura)
    * 100
).round(3)

display(tabla_cobertura)

# ------------------------------------------------------------------------------
# 7. COBERTURAS >100 %
# ------------------------------------------------------------------------------

print("\n--- 7. COBERTURAS SUPERIORES AL 100 % ---")

cobertura_superior_100 = df_final[
    df_final["Cobertura_corregida_%"] > 100
].copy()

print(
    f"Observaciones con cobertura >100 %: "
    f"{len(cobertura_superior_100):,}"
)

print(
    f"Porcentaje: "
    f"{len(cobertura_superior_100) / len(df_final) * 100:.4f} %"
)

if len(cobertura_superior_100) > 0:

    display(
        cobertura_superior_100[
            [
                "Fecha",
                "Any",
                "Id_Instal",
                "N_validos",
                "Observaciones_teoricas",
                "Cobertura_corregida_%",
                "Resolucion_origen"
            ]
        ]
        .sort_values(
            "Cobertura_corregida_%",
            ascending=False
        )
        .head(20)
    )

# ------------------------------------------------------------------------------
# 8. DISTRIBUCIÓN FINAL DE LAeq
# ------------------------------------------------------------------------------

print("\n--- 8. DISTRIBUCIÓN GLOBAL DEL LAeq DIARIO ---")

display(
    df_final["LAeq_dia"]
    .describe(
        percentiles=[
            0.001,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.999
        ]
    )
    .to_frame("LAeq_dia")
)

# ------------------------------------------------------------------------------
# 9. LAeq EN DÍAS VÁLIDOS
# ------------------------------------------------------------------------------

print("\n--- 9. DISTRIBUCIÓN DEL LAeq EN DÍAS VÁLIDOS ---")

df_validos = df_final[
    df_final["Dia_valido"]
]

display(
    df_validos["LAeq_dia"]
    .describe(
        percentiles=[
            0.001,
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
            0.999
        ]
    )
    .to_frame("LAeq_dia_valido")
)

# ------------------------------------------------------------------------------
# 10. CONTROL DE EXTREMOS EN EL DATASET ANALÍTICO
# ------------------------------------------------------------------------------

print("\n--- 10. EXTREMOS EN DÍAS VÁLIDOS ---")

bajos_validos = (
    df_validos["LAeq_dia"] < 30
).sum()

altos_validos = (
    df_validos["LAeq_dia"] > 100
).sum()

print(
    f"LAeq <30 dB en días válidos: "
    f"{bajos_validos:,}"
)

print(
    f"LAeq >100 dB en días válidos: "
    f"{altos_validos:,}"
)

print(
    f"Mínimo válido: "
    f"{df_validos['LAeq_dia'].min():.2f} dB"
)

print(
    f"Máximo válido: "
    f"{df_validos['LAeq_dia'].max():.2f} dB"
)

# ------------------------------------------------------------------------------
# 11. RESUMEN POR AÑO
# ------------------------------------------------------------------------------

print("\n--- 11. RESUMEN FINAL POR AÑO ---")

resumen_final_anual = (
    df_final
    .groupby("Any")
    .agg(
        Sensor_dia=("LAeq_dia", "size"),
        Instalaciones=("Id_Instal", "nunique"),
        Dias_validos=("Dia_valido", "sum"),
        Cobertura_media=(
            "Cobertura_corregida_%",
            "mean"
        ),
        LAeq_medio=("LAeq_dia", "mean"),
        LAeq_mediana=("LAeq_dia", "median"),
        LAeq_min=("LAeq_dia", "min"),
        LAeq_max=("LAeq_dia", "max")
    )
    .round(2)
)

resumen_final_anual["%_validos"] = (
    resumen_final_anual["Dias_validos"]
    /
    resumen_final_anual["Sensor_dia"]
    * 100
).round(2)

display(resumen_final_anual)

# ------------------------------------------------------------------------------
# 12. TRAZABILIDAD DE LA RESOLUCIÓN
# ------------------------------------------------------------------------------

print("\n--- 12. TRAZABILIDAD DE LA RESOLUCIÓN ORIGINAL ---")

tabla_resolucion = pd.crosstab(
    df_final["Any"],
    df_final["Resolucion_origen"]
)

display(tabla_resolucion)

# Comprobar que no se mezclen resoluciones dentro del mismo periodo

errores_resolucion = (
    (
        (df_final["Any"] <= 2023)
        &
        (df_final["Resolucion_origen"] != "1 hora")
    )
    |
    (
        (df_final["Any"] == 2024)
        &
        (df_final["Resolucion_origen"] != "1 minuto")
    )
).sum()

print(
    f"Incoherencias en Resolucion_origen: "
    f"{errores_resolucion:,}"
)

# ------------------------------------------------------------------------------
# 13. CONTROL DE TIPOS
# ------------------------------------------------------------------------------

print("\n--- 13. TIPOS DE DATOS FINALES ---")

print(df_final.dtypes)

# ------------------------------------------------------------------------------
# 14. MUESTRA DEL DATASET
# ------------------------------------------------------------------------------

print("\n--- 14. PRIMERAS OBSERVACIONES ---")

display(df_final.head(10))

print("\n--- 15. ÚLTIMAS OBSERVACIONES ---")

display(df_final.tail(10))

# ------------------------------------------------------------------------------
# 16. RESUMEN FINAL DE CALIDAD
# ------------------------------------------------------------------------------

n_total = len(df_final)
n_validos = df_final["Dia_valido"].sum()
n_no_validos = n_total - n_validos

print("\n" + "=" * 90)
print("RESUMEN FINAL 4.11")
print("=" * 90)

print(f"Registros totales: {n_total:,}")
print(f"Instalaciones únicas: {df_final['Id_Instal'].nunique():,}")

print(
    f"Periodo: "
    f"{df_final['Fecha'].min().date()} - "
    f"{df_final['Fecha'].max().date()}"
)

print(f"Duplicados Fecha-Id_Instal: {duplicados:,}")

print(
    f"Nulos totales: "
    f"{df_final.isna().sum().sum():,}"
)

print(
    f"Días válidos (cobertura >=75 %): "
    f"{n_validos:,}"
)

print(
    f"Días no válidos conservados: "
    f"{n_no_validos:,}"
)

print(
    f"Porcentaje válido: "
    f"{n_validos / n_total * 100:.2f} %"
)

print(
    f"Incoherencias Fecha-Año: "
    f"{incoherencia_anio:,}"
)

print(
    f"Incoherencias de bandera de calidad: "
    f"{incoherencias_bandera:,}"
)

print(
    f"Incoherencias de resolución: "
    f"{errores_resolucion:,}"
)

print(
    f"Coberturas >100 %: "
    f"{len(cobertura_superior_100):,}"
)

print(
    f"LAeq <30 dB entre días válidos: "
    f"{bajos_validos:,}"
)

print(
    f"LAeq >100 dB entre días válidos: "
    f"{altos_validos:,}"
)

print("\n✓ Auditoría estructural completada.")
print("✓ Auditoría temporal completada.")
print("✓ Trazabilidad metodológica comprobada.")
print("✓ No se han imputado ni eliminado observaciones.")
print("✓ Dataset preparado para almacenamiento definitivo.")

4.11 AUDITORÍA FINAL DEL DATASET ACÚSTICO 2018-2024

--- 1. ESTRUCTURA GENERAL ---
Dimensiones: (316951, 10)
Registros sensor-día: 316,951
Variables: 10
Instalaciones únicas: 628
Periodo: 2018-01-01 - 2024-12-31

--- 2. UNICIDAD DE LA CLAVE FECHA–INSTALACIÓN ---
Duplicados Fecha-Id_Instal: 0

--- 3. VALORES AUSENTES ---


,Nulos,Porcentaje
Fecha,0,0.0
Any,0,0.0
Id_Instal,0,0.0
LAeq_dia,0,0.0
N_validos,0,0.0
Observaciones_teoricas,0,0.0
Cobertura_corregida_%,0,0.0
Dia_valido,0,0.0
Resolucion_origen,0,0.0
Periodo_fuente,0,0.0



--- 4. COHERENCIA TEMPORAL ---
Incoherencias Fecha-Año: 0
Años presentes: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]

--- 5. COHERENCIA DE LA BANDERA DE CALIDAD ---
Incoherencias Dia_valido vs cobertura >=75 %: 0

--- 6. DISTRIBUCIÓN DE COBERTURA ---


,Cobertura_%
count,316951.000000
mean,98.682372
std,8.023455
min,0.070000
1%,52.290000
5%,97.080000
25%,100.000000
50%,100.000000
75%,100.000000
95%,100.000000


,N,Porcentaje
Categoria_cobertura,,
<50 %,2802,0.884
50-<75 %,3189,1.006
75-<90 %,3011,0.950
90-100 %,307815,97.118
>100 %,134,0.042


...

Incoherencias en Resolucion_origen: 0

--- 13. TIPOS DE DATOS FINALES ---
Fecha                     datetime64[ns]
Any                                int32
Id_Instal                          int64
LAeq_dia                         float64
N_validos                          int64
Observaciones_teoricas             int64
Cobertura_corregida_%            float64
Dia_valido                          bool
Resolucion_origen                 object
Periodo_fuente                    object
dtype: object

--- 14. PRIMERAS OBSERVACIONES ---


,Fecha,Any,Id_Instal,LAeq_dia,N_validos,Observaciones_teoricas,Cobertura_corregida_%,Dia_valido,Resolucion_origen,Periodo_fuente
0,2018-01-01,2018,495,72.05,24,24,100.0,True,1 hora,2018-2023
1,2018-01-01,2018,496,71.03,24,24,100.0,True,1 hora,2018-2023
2,2018-01-01,2018,497,71.32,24,24,100.0,True,1 hora,2018-2023
3,2018-01-01,2018,651,67.81,24,24,100.0,True,1 hora,2018-2023
4,2018-01-01,2018,653,67.43,24,24,100.0,True,1 hora,2018-2023
5,2018-01-01,2018,659,61.83,24,24,100.0,True,1 hora,2018-2023
6,2018-01-01,2018,666,61.75,24,24,100.0,True,1 hora,2018-2023
7,2018-01-01,2018,667,61.69,24,24,100.0,True,1 hora,2018-2023
8,2018-01-01,2018,686,75.53,24,24,100.0,True,1 hora,2018-2023
9,2018-01-01,2018,726,62.81,24,24,100.0,True,1 hora,2018-2023



--- 15. ÚLTIMAS OBSERVACIONES ---


,Fecha,Any,Id_Instal,LAeq_dia,N_validos,Observaciones_teoricas,Cobertura_corregida_%,Dia_valido,Resolucion_origen,Periodo_fuente
316941,2024-12-31,2024,9866,61.89,1440,1440,100.00,True,1 minuto,2024
316942,2024-12-31,2024,9886,64.75,1440,1440,100.00,True,1 minuto,2024
316943,2024-12-31,2024,9887,64.94,1439,1440,99.93,True,1 minuto,2024
316944,2024-12-31,2024,9888,65.80,1440,1440,100.00,True,1 minuto,2024
316945,2024-12-31,2024,9889,65.41,1440,1440,100.00,True,1 minuto,2024
316946,2024-12-31,2024,9906,60.93,1440,1440,100.00,True,1 minuto,2024
316947,2024-12-31,2024,9907,62.05,1440,1440,100.00,True,1 minuto,2024
316948,2024-12-31,2024,9908,59.15,1440,1440,100.00,True,1 minuto,2024
316949,2024-12-31,2024,9909,58.37,1440,1440,100.00,True,1 minuto,2024
316950,2024-12-31,2024,20041,55.93,1440,1440,100.00,True,1 minuto,2024



RESUMEN FINAL 4.11
Registros totales: 316,951
Instalaciones únicas: 628
Periodo: 2018-01-01 - 2024-12-31
Duplicados Fecha-Id_Instal: 0
Nulos totales: 0
Días válidos (cobertura >=75 %): 310,960
Días no válidos conservados: 5,991
Porcentaje válido: 98.11 %
Incoherencias Fecha-Año: 0
Incoherencias de bandera de calidad: 0
Incoherencias de resolución: 0
Coberturas >100 %: 134
LAeq <30 dB entre días válidos: 20
LAeq >100 dB entre días válidos: 2

✓ Auditoría estructural completada.
✓ Auditoría temporal completada.
✓ Trazabilidad metodológica comprobada.
✓ No se han imputado ni eliminado observaciones.
✓ Dataset preparado para almacenamiento definitivo.


### Resultados de la auditoría final

La auditoría final del bloque acústico confirma la integridad del dataset unificado para el periodo **2018–2024**.

El conjunto está formado por **316.951 observaciones sensor-día**, correspondientes a **628 instalaciones acústicas**, y comprende de forma continua el periodo entre el 1 de enero de 2018 y el 31 de diciembre de 2024. No se detectaron duplicados en la clave `Fecha–Id_Instal` ni valores ausentes en ninguna de las diez variables finales.

El control temporal no identificó incoherencias entre la fecha y el año de observación. Asimismo, la variable `Dia_valido` coincide íntegramente con el criterio establecido de cobertura temporal mínima del 75 %, sin detectarse inconsistencias en su asignación.

Del total de observaciones, **310.960 sensor-día (98,11 %)** cumplen el criterio de calidad temporal, mientras que **5.991 observaciones** presentan cobertura insuficiente. Estas últimas se conservan en el dataset maestro para mantener la trazabilidad, aunque podrán excluirse posteriormente de los análisis que requieran observaciones temporalmente representativas.

La cobertura temporal presenta una elevada calidad global: el **97,12 % de las observaciones** se sitúa entre el 90 y el 100 % de cobertura. Se identifican únicamente **134 registros (0,0423 %)** con cobertura superior al 100 %, asociados fundamentalmente a particularidades del registro temporal y los cambios oficiales de horario. Dada su incidencia marginal, se mantienen sin modificación.

El `LAeq_dia` presenta una media global de **64,49 dB** y una mediana de **64,40 dB**. Al restringir el análisis a los días temporalmente válidos, estos valores permanecen prácticamente inalterados, con una media de **64,50 dB** y una mediana de **64,41 dB**, lo que indica que la exclusión analítica de los días de baja cobertura no altera sustancialmente la distribución global.

Entre las observaciones temporalmente válidas permanecen únicamente **20 registros inferiores a 30 dB** y **2 superiores a 100 dB**. De acuerdo con la auditoría específica realizada previamente, estos valores se conservan y no se aplica eliminación automática, winsorización ni imputación basada exclusivamente en su magnitud.

La trazabilidad de la resolución temporal original también queda verificada: los registros correspondientes a **2018–2023 proceden de mediciones horarias**, mientras que los de **2024 proceden de mediciones minutales**, sin incoherencias en la asignación de la variable `Resolucion_origen`.

En consecuencia, el dataset supera los controles estructurales, temporales y metodológicos establecidos y queda preparado para su almacenamiento definitivo y posterior integración con los restantes bloques de información.

## 4.12 Guardado y verificación del dataset acústico definitivo

Una vez superados los controles estructurales, temporales y metodológicos, se procede al almacenamiento del dataset acústico diario definitivo.

El archivo conserva tanto las observaciones que cumplen el criterio mínimo de cobertura temporal como aquellas consideradas no válidas, identificadas mediante la variable `Dia_valido`. De esta forma se mantiene la trazabilidad completa de los datos originales y se evita eliminar información durante la fase de limpieza.

No se han imputado niveles acústicos ni eliminado valores extremos por su magnitud.

Tras el almacenamiento, el archivo se recarga desde disco y se compara con el dataset existente en memoria para verificar que el proceso de escritura no haya alterado el número de registros, las variables, la unicidad de la clave `Fecha–Id_Instal` ni los principales indicadores de calidad.

El archivo resultante constituye el dataset maestro de contaminación acústica que se utilizará en las posteriores etapas de preintegración e integración con los restantes bloques de información.

In [ ]:
# ==============================================================================
# 4.12 GUARDADO Y VERIFICACIÓN DEL DATASET ACÚSTICO DEFINITIVO
# ==============================================================================

import pandas as pd
import numpy as np
import os

print("=" * 90)
print("4.12 GUARDADO Y VERIFICACIÓN DEL DATASET ACÚSTICO DEFINITIVO")
print("=" * 90)

# ------------------------------------------------------------------------------
# 1. Definir ruta de salida
#
# El archivo se guarda dentro de la carpeta "DATOS LIMPIOS".
# Si ya existe un archivo con el mismo nombre, se sobrescribe automáticamente.
# ------------------------------------------------------------------------------

ruta_csv_limpio = os.path.join(
    ruta_base_ruido,
    "DATOS LIMPIOS",
    "df_ruido_diario_limpio.csv"
)

# Comprobar que existe la carpeta de destino
os.makedirs(
    os.path.dirname(ruta_csv_limpio),
    exist_ok=True
)

print("\nArchivo de salida:")
print(ruta_csv_limpio)

if os.path.exists(ruta_csv_limpio):
    print("✓ El archivo ya existe y será sobrescrito con la versión definitiva.")
else:
    print("✓ Se creará el archivo definitivo.")

# ------------------------------------------------------------------------------
# 2. Dataset que vamos a guardar
#
# Se guarda exactamente el dataset maestro auditado en 4.11.
# No se eliminan los Dia_valido=False.
# No se imputan niveles acústicos.
# ------------------------------------------------------------------------------

df_guardar = df_ruido_diario_2018_2024.copy()

# Orden final por fecha e instalación
df_guardar = (
    df_guardar
    .sort_values(["Fecha", "Id_Instal"])
    .reset_index(drop=True)
)

print("\n--- DATASET PREVIO AL GUARDADO ---")

print(f"Registros: {len(df_guardar):,}")
print(f"Variables: {df_guardar.shape[1]}")
print(f"Instalaciones: {df_guardar['Id_Instal'].nunique():,}")

print(
    f"Periodo: "
    f"{df_guardar['Fecha'].min().date()} - "
    f"{df_guardar['Fecha'].max().date()}"
)

print(
    f"Días válidos: "
    f"{df_guardar['Dia_valido'].sum():,}"
)

print(
    f"Días no válidos conservados: "
    f"{(~df_guardar['Dia_valido']).sum():,}"
)

# ------------------------------------------------------------------------------
# 3. Guardar CSV definitivo
#
# to_csv sobrescribe automáticamente el archivo existente.
# ------------------------------------------------------------------------------

df_guardar.to_csv(
    ruta_csv_limpio,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Archivo definitivo guardado/sobrescrito correctamente.")

# ------------------------------------------------------------------------------
# 4. Comprobar existencia y tamaño
# ------------------------------------------------------------------------------

if not os.path.exists(ruta_csv_limpio):
    raise FileNotFoundError(
        "El archivo no se ha encontrado después del guardado."
    )

tamano_mb = os.path.getsize(ruta_csv_limpio) / (1024 ** 2)

print(f"Tamaño del archivo: {tamano_mb:.2f} MB")

# ------------------------------------------------------------------------------
# 5. Recargar el archivo realmente escrito en Drive
#
# Se especifica dtype para Periodo_fuente para evitar el DtypeWarning
# observado en la comprobación anterior.
# ------------------------------------------------------------------------------

print("\n--- RECARGANDO ARCHIVO GUARDADO ---")

df_verificacion = pd.read_csv(
    ruta_csv_limpio,
    encoding="utf-8-sig",
    low_memory=False,
    dtype={
        "Periodo_fuente": "string"
    }
)

df_verificacion["Fecha"] = pd.to_datetime(
    df_verificacion["Fecha"],
    errors="coerce"
)

print("✓ Archivo recargado correctamente.")

# ------------------------------------------------------------------------------
# 6. Comparación de dimensiones
# ------------------------------------------------------------------------------

print("\n--- COMPROBACIÓN DE DIMENSIONES ---")

print(f"Dataset en memoria : {df_guardar.shape}")
print(f"Dataset recargado  : {df_verificacion.shape}")

mismas_dimensiones = (
    df_guardar.shape == df_verificacion.shape
)

print(
    f"Dimensiones coincidentes: "
    f"{mismas_dimensiones}"
)

# ------------------------------------------------------------------------------
# 7. Comparación de columnas
# ------------------------------------------------------------------------------

print("\n--- COMPROBACIÓN DE VARIABLES ---")

mismas_columnas = (
    df_guardar.columns.tolist()
    == df_verificacion.columns.tolist()
)

print(
    f"Columnas coincidentes: "
    f"{mismas_columnas}"
)

print(df_verificacion.columns.tolist())

# ------------------------------------------------------------------------------
# 8. Comprobación temporal
# ------------------------------------------------------------------------------

print("\n--- COMPROBACIÓN TEMPORAL ---")

print(
    f"Fecha mínima: "
    f"{df_verificacion['Fecha'].min().date()}"
)

print(
    f"Fecha máxima: "
    f"{df_verificacion['Fecha'].max().date()}"
)

print(
    f"Fechas inválidas: "
    f"{df_verificacion['Fecha'].isna().sum():,}"
)

# ------------------------------------------------------------------------------
# 9. Duplicados Fecha-Id_Instal
# ------------------------------------------------------------------------------

duplicados_verificacion = (
    df_verificacion
    .duplicated(["Fecha", "Id_Instal"])
    .sum()
)

print("\n--- COMPROBACIÓN DE DUPLICADOS ---")

print(
    f"Duplicados Fecha-Id_Instal: "
    f"{duplicados_verificacion:,}"
)

# ------------------------------------------------------------------------------
# 10. Valores ausentes
# ------------------------------------------------------------------------------

nulos_verificacion = (
    df_verificacion
    .isna()
    .sum()
    .sum()
)

print("\n--- COMPROBACIÓN DE NULOS ---")

print(
    f"Nulos totales: "
    f"{nulos_verificacion:,}"
)

# ------------------------------------------------------------------------------
# 11. Recuperar correctamente Dia_valido
# ------------------------------------------------------------------------------

if df_verificacion["Dia_valido"].dtype != bool:

    df_verificacion["Dia_valido"] = (
        df_verificacion["Dia_valido"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False
        })
    )

# ------------------------------------------------------------------------------
# 12. Indicadores esenciales del archivo guardado
# ------------------------------------------------------------------------------

print("\n--- INDICADORES DEL ARCHIVO RECARGADO ---")

n_total = len(df_verificacion)

n_validos = (
    df_verificacion["Dia_valido"]
    .sum()
)

n_no_validos = (
    n_total - n_validos
)

print(
    f"Registros totales: "
    f"{n_total:,}"
)

print(
    f"Instalaciones únicas: "
    f"{df_verificacion['Id_Instal'].nunique():,}"
)

print(
    f"Días válidos: "
    f"{n_validos:,}"
)

print(
    f"Días no válidos: "
    f"{n_no_validos:,}"
)

print(
    f"Porcentaje válido: "
    f"{n_validos / n_total * 100:.2f} %"
)

print(
    f"LAeq mínimo: "
    f"{df_verificacion['LAeq_dia'].min():.2f} dB"
)

print(
    f"LAeq máximo: "
    f"{df_verificacion['LAeq_dia'].max():.2f} dB"
)

# ------------------------------------------------------------------------------
# 13. Comparación entre dataframe en memoria y archivo de Drive
# ------------------------------------------------------------------------------

print("\n--- COMPARACIÓN MEMORIA VS ARCHIVO ---")

controles = {

    "Mismo número de filas":
        len(df_guardar)
        == len(df_verificacion),

    "Mismo número de columnas":
        df_guardar.shape[1]
        == df_verificacion.shape[1],

    "Mismas columnas":
        df_guardar.columns.tolist()
        == df_verificacion.columns.tolist(),

    "Mismo número de instalaciones":
        df_guardar["Id_Instal"].nunique()
        == df_verificacion["Id_Instal"].nunique(),

    "Mismos días válidos":
        df_guardar["Dia_valido"].sum()
        == df_verificacion["Dia_valido"].sum(),

    "Mismo LAeq mínimo":
        np.isclose(
            df_guardar["LAeq_dia"].min(),
            df_verificacion["LAeq_dia"].min()
        ),

    "Mismo LAeq máximo":
        np.isclose(
            df_guardar["LAeq_dia"].max(),
            df_verificacion["LAeq_dia"].max()
        ),

    "Sin duplicados":
        duplicados_verificacion == 0,

    "Sin nulos":
        nulos_verificacion == 0
}

tabla_controles = pd.DataFrame(
    controles.items(),
    columns=[
        "Control",
        "Resultado"
    ]
)

display(tabla_controles)

# ------------------------------------------------------------------------------
# 14. Validación automática final
# ------------------------------------------------------------------------------

todo_correcto = all(
    controles.values()
)

print("\n" + "=" * 90)
print("RESULTADO FINAL")
print("=" * 90)

if todo_correcto:

    print("✓ TODOS LOS CONTROLES SUPERADOS.")
    print("✓ El archivo existente en DATOS LIMPIOS ha sido actualizado.")
    print("✓ El archivo guardado coincide con el dataset auditado.")
    print("✓ No existen duplicados Fecha-Id_Instal.")
    print("✓ No existen valores ausentes.")
    print("✓ Se conserva la trazabilidad de la calidad temporal.")
    print("✓ Dataset acústico 2018-2024 FINALIZADO.")

else:

    print("⚠ ALGÚN CONTROL NO HA SIDO SUPERADO.")
    print("Revisar la tabla anterior antes de continuar.")

print("\nArchivo definitivo:")
print(ruta_csv_limpio)

4.12 GUARDADO Y VERIFICACIÓN DEL DATASET ACÚSTICO DEFINITIVO

Archivo de salida:
/content/drive/MyDrive/TFM/07_Contaminacion_Acustica/DATOS LIMPIOS/df_ruido_diario_limpio.csv
✓ El archivo ya existe y será sobrescrito con la versión definitiva.

--- DATASET PREVIO AL GUARDADO ---
Registros: 316,951
Variables: 10
Instalaciones: 628
Periodo: 2018-01-01 - 2024-12-31
Días válidos: 310,960
Días no válidos conservados: 5,991

✓ Archivo definitivo guardado/sobrescrito correctamente.
Tamaño del archivo: 18.44 MB

--- RECARGANDO ARCHIVO GUARDADO ---
✓ Archivo recargado correctamente.

--- COMPROBACIÓN DE DIMENSIONES ---
Dataset en memoria : (316951, 10)
Dataset recargado  : (316951, 10)
Dimensiones coincidentes: True

--- COMPROBACIÓN DE VARIABLES ---
Columnas coincidentes: True
['Fecha', 'Any', 'Id_Instal', 'LAeq_dia', 'N_validos', 'Observaciones_teoricas', 'Cobertura_corregida_%', 'Dia_valido', 'Resolucion_origen', 'Periodo_fuente']

--- COMPROBACIÓN TEMPORAL ---
Fecha mínima: 2018-01-01
Fecha

,Control,Resultado
0,Mismo número de filas,True
1,Mismo número de columnas,True
2,Mismas columnas,True
3,Mismo número de instalaciones,True
4,Mismos días válidos,True
5,Mismo LAeq mínimo,True
6,Mismo LAeq máximo,True
7,Sin duplicados,True
8,Sin nulos,True



RESULTADO FINAL
✓ TODOS LOS CONTROLES SUPERADOS.
✓ El archivo existente en DATOS LIMPIOS ha sido actualizado.
✓ El archivo guardado coincide con el dataset auditado.
✓ No existen duplicados Fecha-Id_Instal.
✓ No existen valores ausentes.
✓ Se conserva la trazabilidad de la calidad temporal.
✓ Dataset acústico 2018-2024 FINALIZADO.

Archivo definitivo:
/content/drive/MyDrive/TFM/07_Contaminacion_Acustica/DATOS LIMPIOS/df_ruido_diario_limpio.csv


### Resultado del almacenamiento definitivo

El dataset acústico diario fue almacenado correctamente como `df_ruido_diario_limpio.csv` y posteriormente recargado desde disco para comprobar la integridad del archivo generado.

El fichero definitivo contiene **316.951 observaciones sensor-día y 10 variables**, correspondientes a **628 instalaciones** y al periodo comprendido entre el **1 de enero de 2018 y el 31 de diciembre de 2024**.

La comparación entre el dataset existente en memoria y el archivo recargado confirmó:

- coincidencia exacta del número de filas y columnas;
- conservación de todas las variables;
- mantenimiento del número de instalaciones;
- conservación de las **310.960 observaciones temporalmente válidas**;
- ausencia de fechas inválidas;
- ausencia de valores nulos;
- ausencia de duplicados en la clave `Fecha–Id_Instal`;
- conservación de los valores mínimo y máximo de `LAeq_dia`.

El **98,11 %** de las observaciones cumple el criterio mínimo de cobertura temporal establecido. Las **5.991 observaciones con cobertura insuficiente** se mantienen en el dataset maestro mediante su identificación con `Dia_valido = False`, preservando así la trazabilidad de la información sin utilizarlas necesariamente en los posteriores análisis.

No se han imputado niveles acústicos ni eliminado observaciones en función de su magnitud.

Todos los controles de integridad posteriores al almacenamiento fueron superados, por lo que el archivo generado constituye el **dataset acústico diario definitivo 2018–2024**, preparado para la posterior fase de preintegración.

## 4.13 Conclusiones del procesamiento de contaminación acústica

El procesamiento de los datos de la red de monitorización acústica de Barcelona permitió construir un **dataset diario homogéneo para el periodo 2018–2024**, partiendo de fuentes con diferente resolución temporal.

Los registros correspondientes a **2018–2023**, originalmente horarios, y los registros de **2024**, originalmente minutales, fueron tratados inicialmente de forma independiente. En ambos casos, el indicador acústico diario (`LAeq_dia`) se calculó mediante **agregación energética**, garantizando la coherencia física del tratamiento de una variable expresada en decibelios.

Posteriormente, ambos periodos fueron homogeneizados y unidos en una estructura común, conservando mediante `Resolucion_origen` la trazabilidad del cambio de resolución existente en los datos fuente.

## Resultado final

El dataset acústico definitivo contiene:

- **316.951 observaciones sensor-día**;
- **628 instalaciones acústicas diferentes**;
- **10 variables finales**;
- periodo completo entre **01/01/2018 y 31/12/2024**;
- **0 valores ausentes**;
- **0 duplicados `Fecha–Id_Instal`**.

El control de cobertura temporal clasifica como válidas **310.960 observaciones**, equivalentes al **98,11 % del dataset**. Las restantes **5.991 observaciones** presentan una cobertura inferior al 75 % y se mantienen identificadas mediante `Dia_valido = False`, sin ser eliminadas del dataset maestro.

La elevada disponibilidad observada confirma una buena cobertura temporal del conjunto: aproximadamente el **97,1 % de todas las observaciones presenta coberturas comprendidas entre el 90 y el 100 %**.

## Control de valores extremos

La distribución global del indicador presenta una media próxima a **64,5 dB** y una mediana de aproximadamente **64,4 dB**.

Se realizó una auditoría específica de los valores acústicos extremos. Entre las observaciones que cumplen el criterio de cobertura temporal se identificaron únicamente:

- **20 registros con `LAeq_dia < 30 dB`**;
- **2 registros con `LAeq_dia > 100 dB`**.

Su incidencia es residual respecto al tamaño total del conjunto y algunos de estos registros presentan persistencia temporal o coincidencia entre instalaciones. Por este motivo, se decidió **no aplicar eliminación automática de outliers, winsorización ni sustitución basada exclusivamente en la magnitud del LAeq**.

Esta estrategia evita modificar artificialmente la distribución acústica y permite conservar posibles episodios reales de niveles excepcionalmente altos o bajos.

## Homogeneización temporal y trazabilidad

El cambio de resolución de la fuente se conserva explícitamente:

- 2018–2023 → resolución original de **1 hora**;
- 2024 → resolución original de **1 minuto**.

La auditoría final no detectó incoherencias en esta asignación ni discontinuidades estructurales asociadas a la unión de ambos periodos.

Asimismo, las medias globales obtenidas para 2018–2023 y 2024 presentan magnitudes comparables, lo que no evidencia un desplazamiento abrupto del indicador diario coincidente con el cambio de resolución de los datos de origen.

## Dataset resultante

El archivo maestro se almacena como:

`df_ruido_diario_limpio.csv`

y conserva las siguientes variables:

- `Fecha`
- `Any`
- `Id_Instal`
- `LAeq_dia`
- `N_validos`
- `Observaciones_teoricas`
- `Cobertura_corregida_%`
- `Dia_valido`
- `Resolucion_origen`
- `Periodo_fuente`

No se han imputado niveles acústicos y las observaciones con cobertura insuficiente permanecen disponibles para garantizar la trazabilidad completa del procesamiento.

La recarga y verificación posterior del archivo confirmó la coincidencia del número de registros, variables, instalaciones, observaciones válidas y rangos acústicos con el dataset previamente auditado en memoria.

Por tanto, el bloque de **contaminación acústica 2018–2024 queda validado y preparado para la fase de preintegración**, en la que se realizará su coordinación temporal y espacial con los restantes bloques de información del proyecto.

### 5 · TRÁFICO AÉREO: TRATAMIENTO Y CONSTRUCCIÓN DE INDICADORES TEMPORALES


### Fuente y objetivo del tratamiento

Los datos de tráfico aéreo utilizados en el estudio proceden del portal **Open Data BCN del Ajuntament de Barcelona**, a partir del conjunto de datos de tráfico aéreo basado en información de **FlightRadar** correspondiente al aeropuerto de Barcelona.

**Fuente:**  
[Open Data BCN — Trànsit aeri FlightRadar](https://opendata-ajuntament.barcelona.cat/data/es/dataset/transitaeri_flightradar_ppal_aeroport)

Los archivos disponibles contienen información diaria desagregada por **compañía aérea y zona geográfica**, incluyendo el número de vuelos registrado para cada combinación. La estructura original responde, por tanto, a la granularidad:

**día × compañía aérea × zona geográfica**

Para este trabajo se dispone de archivos correspondientes al periodo **2019–2024**. La fuente no proporciona información para 2018 y, además, la serie disponible de 2019 comienza el **1 de julio**. Estas limitaciones temporales se conservarán explícitamente durante todo el tratamiento, evitando cualquier reconstrucción o imputación de periodos no observados.

El objetivo de este bloque es transformar los archivos originales en una serie temporal diaria homogénea que pueda utilizarse como indicador de **movilidad y actividad humana** y posteriormente integrarse con las variables de contaminación atmosférica, meteorología, tráfico viario, ruido y restantes fuentes del estudio.

El tratamiento se diseña además con dos finalidades complementarias:

1. **Integración y modelado:** construcción de una tabla con una observación por día y variables representativas de la intensidad y composición del tráfico aéreo.
2. **Visualización interactiva:** conservación de una estructura desagregada por zona geográfica que permita explorar la evolución temporal del tráfico aéreo en el dashboard.

### Relevancia para el análisis COVID/no-COVID

La fuerte alteración de la movilidad aérea durante la pandemia convierte esta fuente en un indicador especialmente relevante para caracterizar los cambios de actividad humana producidos durante el periodo estudiado.

Se distinguirán tres etapas:

- **Pre-COVID:** julio–diciembre de 2019.
- **COVID:** 2020–2021.
- **Post-COVID:** 2022–2024.

Dado que 2019 presenta únicamente información correspondiente al segundo semestre, las comparaciones temporales no se basarán exclusivamente en totales anuales. Se utilizarán también métricas normalizadas y ventanas temporales homogéneas —especialmente julio–diciembre— para reducir el posible efecto derivado de las diferencias de cobertura y de la estacionalidad del tráfico aéreo.

El procesamiento seguirá una secuencia reproducible de **inspección, consolidación multianual, auditoría, agregación diaria, validación, comparación temporal y exportación**, manteniendo en todo momento los archivos originales sin modificaciones.

### 5.1. Inspección de un archivo anual de referencia

Como paso previo a la consolidación multianual del tráfico aéreo, se analiza el archivo correspondiente a **2024** como muestra representativa de la estructura original de la fuente.

El objetivo de esta inspección no es limitar el análisis a dicho año, sino caracterizar previamente la organización de los datos antes de automatizar la carga del periodo completo **2019–2024**.

Se examinan:

- dimensiones del archivo;
- variables disponibles y tipos de datos;
- presencia de valores nulos;
- cardinalidad de las variables categóricas;
- estructura de las primeras observaciones;
- granularidad temporal y categórica de los registros.

Esta revisión permitirá comprobar qué dimensiones deben conservarse durante el tratamiento posterior. En particular, además de preparar variables para su integración en el dataset maestro y el modelado predictivo, se preservará la información necesaria para alimentar las visualizaciones del **dashboard**, especialmente la evolución temporal del tráfico aéreo y su posible desagregación por zona geográfica.

> **Criterio metodológico:** en esta primera etapa no se realiza ninguna agregación, imputación ni eliminación de registros. El archivo de 2024 se utiliza exclusivamente para comprender la estructura de la fuente y definir posteriormente un procedimiento homogéneo para los seis años disponibles.

In [ ]:
# ============================================================
# 5.1. INSPECCIÓN DE UN ARCHIVO ANUAL DE REFERENCIA — 2024
# ============================================================

import pandas as pd
from pathlib import Path
from google.colab import drive

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

drive.mount('/content/drive', force_remount=False)

# ------------------------------------------------------------
# 2. Definir ruta del archivo anual de referencia
# ------------------------------------------------------------

RUTA_VUELOS = Path(
    "/content/drive/MyDrive/TFM/05_Transporte_Aereo"
)

archivo_referencia = (
    RUTA_VUELOS /
    "2024_TransitAeri_FlightRadar_Ppal_Comp_Zona.csv"
)

if not archivo_referencia.exists():
    raise FileNotFoundError(
        f"No se encuentra el archivo:\n{archivo_referencia}"
    )

print("Archivo anual de referencia:")
print(archivo_referencia)

# ------------------------------------------------------------
# 3. Cargar archivo
# ------------------------------------------------------------

try:
    df_vuelos_2024 = pd.read_csv(archivo_referencia)

    # Control por si el separador no fuese coma
    if df_vuelos_2024.shape[1] <= 1:
        df_vuelos_2024 = pd.read_csv(
            archivo_referencia,
            sep=";",
            encoding="latin1"
        )

except Exception:
    df_vuelos_2024 = pd.read_csv(
        archivo_referencia,
        sep=";",
        encoding="latin1"
    )

# ------------------------------------------------------------
# 4. Resumen estructural
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DIMENSIONES")
print("=" * 70)

print(f"Filas:    {df_vuelos_2024.shape[0]:,}")
print(f"Columnas: {df_vuelos_2024.shape[1]}")

print("\n" + "=" * 70)
print("COLUMNAS DISPONIBLES")
print("=" * 70)

print(df_vuelos_2024.columns.tolist())

# ------------------------------------------------------------
# 5. Tipos, nulos y cardinalidad
# ------------------------------------------------------------

resumen_estructura = pd.DataFrame({
    "dtype": df_vuelos_2024.dtypes.astype(str),
    "n_no_nulos": df_vuelos_2024.notna().sum(),
    "n_nulos": df_vuelos_2024.isna().sum(),
    "%_nulos": (
        df_vuelos_2024.isna().mean() * 100
    ).round(2),
    "n_unicos": df_vuelos_2024.nunique(dropna=True)
})

print("\n" + "=" * 70)
print("ESTRUCTURA DE VARIABLES")
print("=" * 70)

display(resumen_estructura)

# ------------------------------------------------------------
# 6. Primeras observaciones
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRIMERAS OBSERVACIONES")
print("=" * 70)

display(df_vuelos_2024.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo anual de referencia:
/content/drive/MyDrive/TFM/05_Transporte_Aereo/2024_TransitAeri_FlightRadar_Ppal_Comp_Zona.csv

DIMENSIONES
Filas:    13,125
Columnas: 6

COLUMNAS DISPONIBLES
['Data_Referencia', 'Codi_Companyia', 'Nom_Companyia', 'Codi_Zona', 'Nom_Zona', 'Nombre_Vols']

ESTRUCTURA DE VARIABLES


,dtype,n_no_nulos,n_nulos,%_nulos,n_unicos
Data_Referencia,object,13125,0,0.0,366
Codi_Companyia,int64,13125,0,0.0,25
Nom_Companyia,object,13125,0,0.0,25
Codi_Zona,int64,13125,0,0.0,5
Nom_Zona,object,13125,0,0.0,5
Nombre_Vols,int64,13125,0,0.0,194



PRIMERAS OBSERVACIONES


,Data_Referencia,Codi_Companyia,Nom_Companyia,Codi_Zona,Nom_Zona,Nombre_Vols
0,2024-01-01,1,Vueling Airlines,1,Espanya,160
1,2024-01-01,1,Vueling Airlines,2,Europa,182
2,2024-01-01,1,Vueling Airlines,5,Àfrica,11
3,2024-01-01,2,Ryanair,1,Espanya,32
4,2024-01-01,2,Ryanair,2,Europa,74


#### Resultados de la inspección del archivo anual de referencia

El archivo correspondiente a 2024 contiene **13.125 registros y 6 variables**, sin valores nulos en ninguna de ellas.

La estructura de la fuente está formada por:

- `Data_Referencia`: fecha asociada al registro;
- `Codi_Companyia` y `Nom_Companyia`: código y nombre de la compañía aérea;
- `Codi_Zona` y `Nom_Zona`: código y denominación de la zona geográfica;
- `Nombre_Vols`: número de vuelos registrado para cada combinación.

La variable temporal presenta **366 fechas únicas**, valor coherente con la duración completa del año 2024, que fue bisiesto. Asimismo, se identifican **25 compañías aéreas** y **5 zonas geográficas** distintas.

A partir de las primeras observaciones se confirma que la granularidad de la fuente es:

**día × compañía aérea × zona geográfica**

Por ejemplo, para una misma fecha aparecen diferentes registros correspondientes a distintas compañías y zonas, con `Nombre_Vols` como variable cuantitativa agregada.

La estructura observada resulta adecuada para el objetivo del estudio, ya que permitirá construir posteriormente:

- indicadores diarios de actividad aérea total;
- desagregaciones por zona geográfica;
- métricas de diversidad o número de compañías activas;
- series temporales adecuadas para el análisis pre-COVID, COVID y post-COVID;
- estructuras específicas para el dashboard sin necesidad de volver a procesar la fuente original.

> **Conclusión:** el archivo de referencia presenta una estructura limpia, completa y claramente interpretable. No se requieren transformaciones previas antes de pasar a la consolidación multianual del periodo 2019–2024.

### 5.2. Carga y consolidación multianual del tráfico aéreo (2019–2024)

Una vez caracterizada la estructura de la fuente mediante el archivo anual de referencia, se procede a la carga conjunta de todos los archivos disponibles de tráfico aéreo para el periodo **2019–2024**.

La fuente utilizada **no dispone de datos correspondientes a 2018**, por lo que la serie de tráfico aéreo comienza en 2019, a diferencia de otras variables del estudio cuya cobertura temporal puede ser más amplia. Esta diferencia se conservará explícitamente y se tendrá en cuenta durante la integración y los análisis longitudinales, **sin realizar imputaciones retrospectivas ni estimaciones artificiales para 2018**.

La consolidación multianual tiene como objetivo construir una única tabla longitudinal manteniendo inicialmente la granularidad original:

**día × compañía aérea × zona geográfica**

Para garantizar la trazabilidad del proceso, se incorpora una variable auxiliar con el **año del archivo de procedencia** de cada registro. Antes de concatenar los datos se comprueba además que los seis archivos presentan una estructura de columnas compatible.

La disponibilidad de información entre 2019 y 2024 permitirá estudiar la evolución de la actividad aérea durante tres etapas especialmente relevantes:

- **2019:** referencia pre-COVID;
- **2020–2021:** periodo afectado por la pandemia y las restricciones de movilidad;
- **2022–2024:** periodo post-COVID y recuperación de la actividad.

Esta dimensión temporal será conservada tanto para el análisis estadístico como para las visualizaciones longitudinales del dashboard.

> **Limitación de la fuente:** no se dispone de registros de tráfico aéreo para 2018. Por tanto, cualquier análisis que incorpore esta variable se restringirá al periodo con información observada disponible, evitando completar 2018 mediante valores supuestos.

> **Criterio metodológico:** los archivos se concatenan sin realizar todavía agregaciones, imputaciones ni generación de variables derivadas. La cobertura temporal y la calidad de los registros se auditarán sobre el conjunto multianual antes de construir los indicadores diarios.

In [ ]:
# ============================================================
# 5.2. CARGA Y CONSOLIDACIÓN MULTIANUAL — 2019-2024
# ============================================================

ANIOS_VUELOS = list(range(2019, 2025))

lista_vuelos = []
resumen_archivos = []

# Estructura esperada a partir del archivo de referencia 2024
columnas_referencia = set(df_vuelos_2024.columns)

print("=" * 70)
print("CARGA DE ARCHIVOS ANUALES DE TRÁFICO AÉREO")
print("=" * 70)

for anio in ANIOS_VUELOS:

    nombre_archivo = (
        f"{anio}_TransitAeri_"
        f"FlightRadar_Ppal_Comp_Zona.csv"
    )

    ruta_archivo = RUTA_VUELOS / nombre_archivo

    # --------------------------------------------------------
    # 1. Comprobar existencia del archivo
    # --------------------------------------------------------

    if not ruta_archivo.exists():

        print(f"❌ {anio}: archivo no encontrado")

        resumen_archivos.append({
            "Anio": anio,
            "Filas": None,
            "Columnas": None,
            "Estructura_compatible": False,
            "Cargado": False
        })

        continue

    # --------------------------------------------------------
    # 2. Lectura robusta
    # --------------------------------------------------------

    try:
        df_anio = pd.read_csv(ruta_archivo)

        # Control por si el separador no fuese coma
        if df_anio.shape[1] <= 1:
            df_anio = pd.read_csv(
                ruta_archivo,
                sep=";",
                encoding="latin1"
            )

    except Exception:

        df_anio = pd.read_csv(
            ruta_archivo,
            sep=";",
            encoding="latin1"
        )

    # --------------------------------------------------------
    # 3. Comprobar compatibilidad estructural
    # --------------------------------------------------------

    estructura_compatible = (
        set(df_anio.columns) == columnas_referencia
    )

    print(
        f"✓ {anio}: "
        f"{df_anio.shape[0]:,} filas × "
        f"{df_anio.shape[1]} columnas | "
        f"Estructura compatible: {estructura_compatible}"
    )

    resumen_archivos.append({
        "Anio": anio,
        "Filas": df_anio.shape[0],
        "Columnas": df_anio.shape[1],
        "Estructura_compatible": estructura_compatible,
        "Cargado": True
    })

    # --------------------------------------------------------
    # 4. Añadir variable de trazabilidad
    # --------------------------------------------------------

    df_anio = df_anio.copy()
    df_anio["anio_archivo"] = anio

    lista_vuelos.append(df_anio)


# ------------------------------------------------------------
# 5. Resumen de archivos encontrados
# ------------------------------------------------------------

df_resumen_archivos = pd.DataFrame(resumen_archivos)

print("\n" + "=" * 70)
print("RESUMEN DE ARCHIVOS")
print("=" * 70)

display(df_resumen_archivos)


# ------------------------------------------------------------
# 6. Verificar que estén los seis años disponibles
# ------------------------------------------------------------

if not df_resumen_archivos["Cargado"].all():

    faltantes = df_resumen_archivos.loc[
        ~df_resumen_archivos["Cargado"],
        "Anio"
    ].tolist()

    raise FileNotFoundError(
        "Faltan archivos de tráfico aéreo para los años: "
        f"{faltantes}"
    )


# ------------------------------------------------------------
# 7. Verificar compatibilidad de columnas
# ------------------------------------------------------------

if not df_resumen_archivos["Estructura_compatible"].all():

    incompatibles = df_resumen_archivos.loc[
        ~df_resumen_archivos["Estructura_compatible"],
        "Anio"
    ].tolist()

    raise ValueError(
        "Se han detectado estructuras de columnas diferentes "
        f"en los años: {incompatibles}"
    )


# ------------------------------------------------------------
# 8. Concatenación multianual
# ------------------------------------------------------------

df_vuelos = pd.concat(
    lista_vuelos,
    ignore_index=True
)

print("\n" + "=" * 70)
print("DATASET MULTIANUAL CONSOLIDADO")
print("=" * 70)

print(f"Filas totales: {df_vuelos.shape[0]:,}")
print(f"Columnas:      {df_vuelos.shape[1]}")

print(
    "Años incluidos:",
    sorted(df_vuelos["anio_archivo"].unique())
)

print(
    "\nNota: la fuente de tráfico aéreo no dispone "
    "de datos para 2018."
)


# ------------------------------------------------------------
# 9. Registros por año
# ------------------------------------------------------------

registros_por_anio = (
    df_vuelos["anio_archivo"]
    .value_counts()
    .sort_index()
    .rename("Registros")
    .to_frame()
)

print("\nRegistros por año:")

display(registros_por_anio)


# ------------------------------------------------------------
# 10. Comprobación final de columnas
# ------------------------------------------------------------

print("\nColumnas del dataset consolidado:")

print(df_vuelos.columns.tolist())


# ------------------------------------------------------------
# 11. Primeras observaciones
# ------------------------------------------------------------

print("\nPrimeras observaciones:")

display(df_vuelos.head())

CARGA DE ARCHIVOS ANUALES DE TRÁFICO AÉREO
✓ 2019: 5,046 filas × 6 columnas | Estructura compatible: True
✓ 2020: 6,468 filas × 6 columnas | Estructura compatible: True
✓ 2021: 8,305 filas × 6 columnas | Estructura compatible: True
✓ 2022: 11,083 filas × 6 columnas | Estructura compatible: True
✓ 2023: 12,269 filas × 6 columnas | Estructura compatible: True
✓ 2024: 13,125 filas × 6 columnas | Estructura compatible: True

RESUMEN DE ARCHIVOS


,Anio,Filas,Columnas,Estructura_compatible,Cargado
0,2019,5046,6,True,True
1,2020,6468,6,True,True
2,2021,8305,6,True,True
3,2022,11083,6,True,True
4,2023,12269,6,True,True
5,2024,13125,6,True,True



DATASET MULTIANUAL CONSOLIDADO
Filas totales: 56,296
Columnas:      7
Años incluidos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Nota: la fuente de tráfico aéreo no dispone de datos para 2018.

Registros por año:


,Registros
anio_archivo,
2019,5046
2020,6468
2021,8305
2022,11083
2023,12269
2024,13125



Columnas del dataset consolidado:
['Data_Referencia', 'Codi_Companyia', 'Nom_Companyia', 'Codi_Zona', 'Nom_Zona', 'Nombre_Vols', 'anio_archivo']

Primeras observaciones:


,Data_Referencia,Codi_Companyia,Nom_Companyia,Codi_Zona,Nom_Zona,Nombre_Vols,anio_archivo
0,2019-07-01,1,Vueling Airlines,1,Espanya,172,2019
1,2019-07-01,1,Vueling Airlines,2,Europa,211,2019
2,2019-07-01,1,Vueling Airlines,4,Àsia,4,2019
3,2019-07-01,1,Vueling Airlines,5,Àfrica,9,2019
4,2019-07-01,2,Ryanair,1,Espanya,24,2019


#### Resultados de la consolidación multianual

Se localizaron y cargaron correctamente los **seis archivos anuales disponibles de tráfico aéreo**, correspondientes al periodo **2019–2024**. Todos ellos presentan una estructura homogénea de seis variables originales, lo que permite su concatenación directa sin necesidad de armonizar previamente los esquemas de columnas.

El número de registros aumenta progresivamente a lo largo del periodo:

- 2019: 5.046 registros;
- 2020: 6.468 registros;
- 2021: 8.305 registros;
- 2022: 11.083 registros;
- 2023: 12.269 registros;
- 2024: 13.125 registros.

Tras la concatenación se obtiene un conjunto multianual formado por **56.296 registros y 7 variables**, incluyendo la variable auxiliar `anio_archivo`, incorporada para mantener la trazabilidad de la procedencia de cada observación.

La inspección de los primeros registros muestra que la información disponible para **2019 comienza el 1 de julio**, por lo que este año presenta una cobertura parcial. Esta circunstancia resulta especialmente relevante al utilizar 2019 como referencia pre-COVID y será considerada explícitamente en las comparaciones temporales posteriores.

Asimismo, la fuente de tráfico aéreo **no dispone de información para 2018**. En consecuencia, los análisis que incorporen esta variable comenzarán en julio de 2019 y no se realizarán imputaciones ni reconstrucciones retrospectivas para completar el periodo no observado.

La estructura original **día × compañía aérea × zona geográfica** se mantiene intacta en esta fase, permitiendo realizar a continuación una auditoría multianual de cobertura, duplicados, valores ausentes y consistencia categórica.

> **Conclusión:** los seis archivos disponibles son estructuralmente compatibles y permiten construir una serie longitudinal homogénea para 2019–2024. La principal limitación identificada hasta este punto es la ausencia de datos para 2018 y la cobertura parcial de 2019, cuyo alcance exacto se verificará en la siguiente etapa.

### 5.3. Auditoría multianual de cobertura temporal y calidad

Tras consolidar los seis archivos disponibles, se realiza una auditoría conjunta del periodo **2019–2024** antes de efectuar cualquier agregación diaria.

El objetivo es comprobar que la homogeneidad estructural observada entre los archivos se corresponde también con una adecuada consistencia temporal y categórica. En particular, se evalúan:

- conversión y validez de la variable temporal;
- coherencia entre el año indicado por el archivo y el año de la fecha registrada;
- fecha inicial y final disponible para cada año;
- número de días observados y posibles fechas ausentes dentro de cada periodo disponible;
- presencia de valores nulos o negativos en `Nombre_Vols`;
- existencia de duplicados para la clave natural **fecha × compañía × zona**;
- consistencia de las correspondencias código–nombre de compañías y zonas;
- evolución del número de compañías y zonas presentes en cada año.

La auditoría presta especial atención a **2019**, cuya primera observación se ha identificado el 1 de julio. Por tanto, se distinguirá entre una posible ausencia de fechas dentro del periodo efectivamente cubierto y la inexistencia de información anterior al comienzo de la serie.

La fuente tampoco dispone de registros para **2018**. Esta ausencia se considera una limitación de cobertura de la fuente y no será corregida mediante imputación o reconstrucción artificial.

> **Criterio metodológico:** una fecha no observada antes del inicio de la cobertura disponible no se interpreta como un valor perdido. Los posibles huecos se evaluarán únicamente dentro del intervalo comprendido entre la primera y la última fecha observada de cada archivo anual.

In [ ]:
# ============================================================
# 5.3. AUDITORÍA MULTIANUAL DE COBERTURA Y CALIDAD
# ============================================================

df_vuelos_audit = df_vuelos.copy()

# ------------------------------------------------------------
# 1. Conversión y normalización de variables fundamentales
# ------------------------------------------------------------

df_vuelos_audit["Fecha"] = pd.to_datetime(
    df_vuelos_audit["Data_Referencia"],
    errors="coerce"
).dt.normalize()

df_vuelos_audit["Nombre_Vols"] = pd.to_numeric(
    df_vuelos_audit["Nombre_Vols"],
    errors="coerce"
)

df_vuelos_audit["anio_fecha"] = (
    df_vuelos_audit["Fecha"].dt.year
)


# ------------------------------------------------------------
# 2. Calidad básica de las variables
# ------------------------------------------------------------

print("=" * 70)
print("CALIDAD BÁSICA")
print("=" * 70)

print(
    f"Fechas no interpretables: "
    f"{df_vuelos_audit['Fecha'].isna().sum():,}"
)

print(
    f"Valores nulos/no numéricos en Nombre_Vols: "
    f"{df_vuelos_audit['Nombre_Vols'].isna().sum():,}"
)

print(
    f"Valores negativos en Nombre_Vols: "
    f"{(df_vuelos_audit['Nombre_Vols'] < 0).sum():,}"
)


# ------------------------------------------------------------
# 3. Coherencia año del archivo ↔ año de la fecha
# ------------------------------------------------------------

incoherencia_anio = (
    df_vuelos_audit["anio_archivo"]
    != df_vuelos_audit["anio_fecha"]
)

print("\n" + "=" * 70)
print("COHERENCIA TEMPORAL")
print("=" * 70)

print(
    "Registros con año de archivo distinto "
    "del año de la fecha:",
    incoherencia_anio.sum()
)

if incoherencia_anio.any():
    display(
        df_vuelos_audit.loc[
            incoherencia_anio,
            [
                "Data_Referencia",
                "Fecha",
                "anio_archivo",
                "anio_fecha"
            ]
        ].head(20)
    )


# ------------------------------------------------------------
# 4. Cobertura temporal por año
# ------------------------------------------------------------

cobertura_anual = (
    df_vuelos_audit
    .groupby("anio_archivo")
    .agg(
        fecha_min=("Fecha", "min"),
        fecha_max=("Fecha", "max"),
        dias_observados=("Fecha", "nunique"),
        registros=("Fecha", "size"),
        companias=("Codi_Companyia", "nunique"),
        zonas=("Codi_Zona", "nunique")
    )
    .reset_index()
)

# Número de días naturales comprendidos entre
# la primera y la última fecha realmente disponibles
cobertura_anual["dias_intervalo_disponible"] = (
    cobertura_anual.apply(
        lambda fila: len(
            pd.date_range(
                fila["fecha_min"],
                fila["fecha_max"],
                freq="D"
            )
        ),
        axis=1
    )
)

cobertura_anual["dias_ausentes_intervalo"] = (
    cobertura_anual["dias_intervalo_disponible"]
    - cobertura_anual["dias_observados"]
)

cobertura_anual["cobertura_pct"] = (
    100
    * cobertura_anual["dias_observados"]
    / cobertura_anual["dias_intervalo_disponible"]
).round(2)

print("\n" + "=" * 70)
print("COBERTURA TEMPORAL POR AÑO")
print("=" * 70)

display(cobertura_anual)


# ------------------------------------------------------------
# 5. Identificación explícita de fechas ausentes
#    dentro del periodo disponible de cada año
# ------------------------------------------------------------

resumen_fechas_faltantes = []
detalle_fechas_faltantes = {}

for anio in ANIOS_VUELOS:

    sub = df_vuelos_audit.loc[
        df_vuelos_audit["anio_archivo"] == anio
    ].copy()

    fecha_min = sub["Fecha"].min()
    fecha_max = sub["Fecha"].max()

    calendario = pd.date_range(
        start=fecha_min,
        end=fecha_max,
        freq="D"
    )

    fechas_observadas = pd.DatetimeIndex(
        sub["Fecha"]
        .dropna()
        .unique()
    )

    fechas_faltantes = calendario.difference(
        fechas_observadas
    )

    detalle_fechas_faltantes[anio] = fechas_faltantes

    resumen_fechas_faltantes.append({
        "Anio": anio,
        "Fecha_inicio": fecha_min,
        "Fecha_fin": fecha_max,
        "Dias_esperados_intervalo": len(calendario),
        "Dias_observados": len(fechas_observadas),
        "Dias_ausentes": len(fechas_faltantes)
    })

df_fechas_faltantes = pd.DataFrame(
    resumen_fechas_faltantes
)

print("\n" + "=" * 70)
print("FECHAS AUSENTES DENTRO DEL PERIODO DISPONIBLE")
print("=" * 70)

display(df_fechas_faltantes)

# Mostrar únicamente años que tengan huecos
for anio, fechas in detalle_fechas_faltantes.items():

    if len(fechas) > 0:

        print(
            f"\n{anio}: {len(fechas)} fechas ausentes"
        )

        print(
            list(fechas[:20])
        )

        if len(fechas) > 20:
            print("...")


# ------------------------------------------------------------
# 6. Duplicados según la clave natural
# ------------------------------------------------------------

clave_natural = [
    "Fecha",
    "Codi_Companyia",
    "Codi_Zona"
]

duplicados = df_vuelos_audit.duplicated(
    subset=clave_natural,
    keep=False
)

print("\n" + "=" * 70)
print("DUPLICADOS")
print("=" * 70)

print(
    "Registros duplicados para "
    "Fecha × Compañía × Zona:",
    duplicados.sum()
)

if duplicados.any():

    display(
        df_vuelos_audit.loc[duplicados]
        .sort_values(clave_natural)
        .head(20)
    )


# ------------------------------------------------------------
# 7. Consistencia código ↔ nombre de compañía
# ------------------------------------------------------------

control_companias = (
    df_vuelos_audit
    .groupby("Codi_Companyia")
    ["Nom_Companyia"]
    .nunique()
)

companias_inconsistentes = (
    control_companias[
        control_companias > 1
    ]
)

print("\n" + "=" * 70)
print("CONSISTENCIA DE COMPAÑÍAS")
print("=" * 70)

print(
    "Códigos de compañía asociados a >1 nombre:",
    len(companias_inconsistentes)
)

if len(companias_inconsistentes) > 0:
    display(companias_inconsistentes)


# ------------------------------------------------------------
# 8. Consistencia código ↔ nombre de zona
# ------------------------------------------------------------

control_zonas = (
    df_vuelos_audit
    .groupby("Codi_Zona")
    ["Nom_Zona"]
    .nunique()
)

zonas_inconsistentes = (
    control_zonas[
        control_zonas > 1
    ]
)

print("\n" + "=" * 70)
print("CONSISTENCIA DE ZONAS")
print("=" * 70)

print(
    "Códigos de zona asociados a >1 nombre:",
    len(zonas_inconsistentes)
)

if len(zonas_inconsistentes) > 0:
    display(zonas_inconsistentes)


# ------------------------------------------------------------
# 9. Catálogo de zonas observado en todo el periodo
# ------------------------------------------------------------

catalogo_zonas = (
    df_vuelos_audit[
        ["Codi_Zona", "Nom_Zona"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Codi_Zona", "Nom_Zona"]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("ZONAS OBSERVADAS — 2019-2024")
print("=" * 70)

display(catalogo_zonas)


# ------------------------------------------------------------
# 10. Compañías y zonas disponibles por año
# ------------------------------------------------------------

diversidad_anual = (
    df_vuelos_audit
    .groupby("anio_archivo")
    .agg(
        companias_activas=(
            "Codi_Companyia",
            "nunique"
        ),
        zonas_activas=(
            "Codi_Zona",
            "nunique"
        )
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("COMPAÑÍAS Y ZONAS POR AÑO")
print("=" * 70)

display(diversidad_anual)

CALIDAD BÁSICA
Fechas no interpretables: 0
Valores nulos/no numéricos en Nombre_Vols: 0
Valores negativos en Nombre_Vols: 0

COHERENCIA TEMPORAL
Registros con año de archivo distinto del año de la fecha: 0

COBERTURA TEMPORAL POR AÑO


,anio_archivo,fecha_min,fecha_max,dias_observados,registros,companias,zonas,dias_intervalo_disponible,dias_ausentes_intervalo,cobertura_pct
0,2019,2019-07-01,2019-12-31,184,5046,23,5,184,0,100.0
1,2020,2020-01-01,2020-12-31,366,6468,23,5,366,0,100.0
2,2021,2021-01-01,2021-12-31,365,8305,24,5,365,0,100.0
3,2022,2022-01-01,2022-12-31,365,11083,25,5,365,0,100.0
4,2023,2023-01-01,2023-12-31,365,12269,25,5,365,0,100.0
5,2024,2024-01-01,2024-12-31,366,13125,25,5,366,0,100.0



FECHAS AUSENTES DENTRO DEL PERIODO DISPONIBLE


,Anio,Fecha_inicio,Fecha_fin,Dias_esperados_intervalo,Dias_observados,Dias_ausentes
0,2019,2019-07-01,2019-12-31,184,184,0
1,2020,2020-01-01,2020-12-31,366,366,0
2,2021,2021-01-01,2021-12-31,365,365,0
3,2022,2022-01-01,2022-12-31,365,365,0
4,2023,2023-01-01,2023-12-31,365,365,0
5,2024,2024-01-01,2024-12-31,366,366,0



DUPLICADOS
Registros duplicados para Fecha × Compañía × Zona: 0

CONSISTENCIA DE COMPAÑÍAS
Códigos de compañía asociados a >1 nombre: 0

CONSISTENCIA DE ZONAS
Códigos de zona asociados a >1 nombre: 0

ZONAS OBSERVADAS — 2019-2024


,Codi_Zona,Nom_Zona
0,1,Espanya
1,2,Europa
2,3,Amèrica
3,4,Àsia
4,5,Àfrica



COMPAÑÍAS Y ZONAS POR AÑO


,anio_archivo,companias_activas,zonas_activas
0,2019,23,5
1,2020,23,5
2,2021,24,5
3,2022,25,5
4,2023,25,5
5,2024,25,5


#### Resultados de la auditoría multianual

La auditoría confirma una **elevada calidad y consistencia del conjunto de datos de tráfico aéreo**. No se detectan fechas no interpretables, valores nulos o no numéricos en `Nombre_Vols`, ni registros con valores negativos. Asimismo, el año asociado a cada fecha coincide en todos los casos con el año del archivo de procedencia.

La cobertura temporal disponible es la siguiente:

| Año | Periodo disponible | Días observados | Cobertura |
|---|---|---:|---:|
| 2019 | 01/07/2019 – 31/12/2019 | 184 | 100 % |
| 2020 | 01/01/2020 – 31/12/2020 | 366 | 100 % |
| 2021 | 01/01/2021 – 31/12/2021 | 365 | 100 % |
| 2022 | 01/01/2022 – 31/12/2022 | 365 | 100 % |
| 2023 | 01/01/2023 – 31/12/2023 | 365 | 100 % |
| 2024 | 01/01/2024 – 31/12/2024 | 366 | 100 % |

No se detectan fechas ausentes dentro de ninguno de los intervalos disponibles. Por tanto, **2019 constituye una serie completa desde el 1 de julio hasta el 31 de diciembre**, mientras que los años 2020–2024 presentan cobertura anual completa.

La cobertura parcial de 2019 y la inexistencia de información para 2018 constituyen limitaciones propias de esta fuente y no serán corregidas mediante imputación o reconstrucción artificial.

Esta circunstancia se tendrá especialmente en cuenta en el análisis **COVID/no-COVID**. El año 2019 proporciona una referencia observada del periodo pre-COVID, pero únicamente durante el segundo semestre. Por ello, no se compararán directamente sus totales absolutos con los correspondientes a años completos. Las comparaciones interanuales utilizarán métricas normalizadas —como el número medio diario de vuelos— y, cuando sea necesario, ventanas temporales equivalentes entre años.

La estructura de los registros también presenta una elevada consistencia. No se identifican duplicados para la clave natural:

**fecha × compañía aérea × zona geográfica**

ni inconsistencias en las correspondencias entre códigos y nombres de compañías o zonas.

Las cinco zonas geográficas —**España, Europa, América, Asia y África**— están presentes durante todo el periodo analizado, permitiendo estudiar no solo la evolución del volumen global de actividad aérea, sino también posibles diferencias en su composición geográfica durante las etapas pre-COVID, COVID y post-COVID.

El número de compañías presentes en la fuente muestra una ligera evolución temporal: **23 compañías en 2019 y 2020, 24 en 2021 y 25 entre 2022 y 2024**. Esta variable se conservará como indicador complementario de la diversidad de operadores activos.

A partir de estos resultados se establecen tres periodos analíticos:

- **Pre-COVID:** julio–diciembre de 2019.
- **COVID:** 2020–2021.
- **Post-COVID:** 2022–2024.

> **Conclusión:** la fuente presenta continuidad temporal completa dentro de su periodo disponible, ausencia de problemas relevantes de calidad y una estructura homogénea entre años. Estas características permiten construir indicadores diarios robustos y estudiar posteriormente el impacto de la pandemia y la recuperación de la actividad aérea, tanto en términos globales como por zona geográfica.

### 5.4. Construcción de indicadores diarios de actividad aérea

Una vez verificada la calidad y continuidad temporal de la fuente, los registros originales se transforman a una resolución **diaria**, compatible con la escala temporal utilizada para la integración de las diferentes fuentes del estudio.

La granularidad original:

**día × compañía aérea × zona geográfica**

se agrega para obtener una única observación por fecha, conservando tanto el volumen global de actividad como su composición geográfica.

Se generan los siguientes indicadores:

- **vuelos totales diarios**;
- vuelos diarios asociados a **España**;
- vuelos diarios asociados a **Europa**;
- vuelos diarios asociados a **América**;
- vuelos diarios asociados a **Asia**;
- vuelos diarios asociados a **África**;
- **vuelos internacionales**, calculados como la diferencia entre el total y los vuelos asociados a España;
- **número de compañías activas** cada día;
- **número de zonas con actividad** cada día.

Además, se incorporan variables temporales auxiliares —año, mes y periodo de análisis— destinadas a facilitar las comparaciones longitudinales y la explotación posterior de los datos en el dashboard.

#### Periodización COVID/no-COVID

La disponibilidad temporal de la fuente permite distinguir tres etapas:

- **Pre-COVID:** julio–diciembre de 2019;
- **COVID:** 2020–2021;
- **Post-COVID:** 2022–2024.

La fuente no dispone de información para 2018 y el año 2019 comienza el **1 de julio**. Por este motivo, los totales anuales de 2019 no se compararán directamente con años completos. Las comparaciones entre periodos se realizarán mediante indicadores normalizados —como vuelos medios diarios— o utilizando ventanas temporales equivalentes cuando sea necesario.

Esta diferenciación permitirá estudiar tanto la reducción de la actividad aérea durante la pandemia como su posterior recuperación, además de comprobar si estos cambios presentan comportamientos diferentes según la zona geográfica.

#### Preparación para el dashboard

La transformación conservará también una estructura en formato largo **fecha × zona × vuelos**, adecuada para la visualización interactiva. Esto permitirá representar y filtrar la evolución de la actividad aérea según:

- año;
- mes;
- periodo pre-COVID, COVID o post-COVID;
- zona geográfica.

De este modo, el mismo proceso de tratamiento genera una salida optimizada para la integración y el modelado y otra destinada a las visualizaciones del dashboard, evitando reprocesar posteriormente los archivos originales.

> **Criterio metodológico:** en esta etapa únicamente se generan agregaciones e indicadores directamente derivados de los datos observados. No se incorporan todavía retardos temporales, medias móviles, transformaciones logarítmicas ni otras variables predictivas, que se reservarán para la fase específica de *feature engineering* y modelado.

In [ ]:
# ============================================================
# 5.4. CONSTRUCCIÓN DE INDICADORES DIARIOS DE ACTIVIDAD AÉREA
# ============================================================

df_vuelos_limpio = df_vuelos_audit.copy()

# ------------------------------------------------------------
# 1. Selección de registros válidos
# ------------------------------------------------------------

# La auditoría anterior ha confirmado que no existen fechas
# inválidas, valores nulos ni valores negativos. Se mantiene
# este filtro como control de seguridad y reproducibilidad.

df_vuelos_limpio = (
    df_vuelos_limpio
    .dropna(subset=["Fecha", "Nombre_Vols"])
    .loc[lambda x: x["Nombre_Vols"] >= 0]
    .copy()
)


# ------------------------------------------------------------
# 2. Normalización de las zonas geográficas
# ------------------------------------------------------------

mapa_zonas = {
    "Espanya": "Espanya",
    "Europa": "Europa",
    "Amèrica": "America",
    "Àsia": "Asia",
    "Àfrica": "Africa"
}

df_vuelos_limpio["Zona"] = (
    df_vuelos_limpio["Nom_Zona"]
    .map(mapa_zonas)
)

zonas_sin_mapear = (
    df_vuelos_limpio.loc[
        df_vuelos_limpio["Zona"].isna(),
        "Nom_Zona"
    ]
    .dropna()
    .unique()
)

if len(zonas_sin_mapear) > 0:
    raise ValueError(
        f"Se han detectado zonas sin mapear: "
        f"{zonas_sin_mapear.tolist()}"
    )


# ------------------------------------------------------------
# 3. Indicadores globales diarios
# ------------------------------------------------------------

vuelos_totales_dia = (
    df_vuelos_limpio
    .groupby("Fecha", as_index=False)
    .agg(
        Vuelos_total=("Nombre_Vols", "sum"),
        Companias_activas=("Codi_Companyia", "nunique"),
        Zonas_activas=("Codi_Zona", "nunique")
    )
)


# ------------------------------------------------------------
# 4. Actividad diaria por zona geográfica
# ------------------------------------------------------------

vuelos_zona_dia = (
    df_vuelos_limpio
    .groupby(
        ["Fecha", "Zona"],
        as_index=False
    )
    .agg(
        Vuelos=("Nombre_Vols", "sum")
    )
)


# ------------------------------------------------------------
# 5. Transformación a formato ancho
#    para integración y modelado
# ------------------------------------------------------------

vuelos_zona_wide = (
    vuelos_zona_dia
    .pivot(
        index="Fecha",
        columns="Zona",
        values="Vuelos"
    )
    .fillna(0)
    .add_prefix("Vuelos_")
    .reset_index()
)

vuelos_zona_wide.columns.name = None


# ------------------------------------------------------------
# 6. Integración de totales y zonas
# ------------------------------------------------------------

df_vuelos_diario = (
    vuelos_totales_dia
    .merge(
        vuelos_zona_wide,
        on="Fecha",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Fecha")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Garantizar las cinco zonas esperadas
# ------------------------------------------------------------

columnas_zona = [
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa"
]

for columna in columnas_zona:
    if columna not in df_vuelos_diario.columns:
        df_vuelos_diario[columna] = 0


# ------------------------------------------------------------
# 8. Actividad aérea internacional
# ------------------------------------------------------------

df_vuelos_diario["Vuelos_internacionales"] = (
    df_vuelos_diario["Vuelos_total"]
    - df_vuelos_diario["Vuelos_Espanya"]
)


# ------------------------------------------------------------
# 9. Variables temporales para análisis y dashboard
# ------------------------------------------------------------

df_vuelos_diario["Anio"] = (
    df_vuelos_diario["Fecha"].dt.year
)

df_vuelos_diario["Mes"] = (
    df_vuelos_diario["Fecha"].dt.month
)

df_vuelos_diario["Trimestre"] = (
    df_vuelos_diario["Fecha"].dt.quarter
)

df_vuelos_diario["Dia_semana"] = (
    df_vuelos_diario["Fecha"].dt.dayofweek
)

df_vuelos_diario["Fin_semana"] = (
    df_vuelos_diario["Dia_semana"]
    .isin([5, 6])
    .astype(int)
)


# ------------------------------------------------------------
# 10. Clasificación temporal COVID / no-COVID
# ------------------------------------------------------------

def clasificar_periodo_covid(fecha):

    anio = fecha.year

    if anio == 2019:
        return "Pre-COVID"

    elif anio in [2020, 2021]:
        return "COVID"

    elif anio in [2022, 2023, 2024]:
        return "Post-COVID"

    return "Fuera_periodo"


df_vuelos_diario["Periodo_COVID"] = (
    df_vuelos_diario["Fecha"]
    .apply(clasificar_periodo_covid)
)


# Variable binaria auxiliar para comparaciones
df_vuelos_diario["Es_COVID"] = (
    df_vuelos_diario["Periodo_COVID"]
    .eq("COVID")
    .astype(int)
)


# ------------------------------------------------------------
# 11. Identificación de ventana comparable julio-diciembre
# ------------------------------------------------------------

# 2019 solo dispone de información desde julio.
# Esta variable permitirá realizar comparaciones interanuales
# utilizando exactamente los mismos meses en todos los años.

df_vuelos_diario["Ventana_Jul_Dic"] = (
    df_vuelos_diario["Mes"]
    .between(7, 12)
    .astype(int)
)


# ------------------------------------------------------------
# 12. Orden definitivo de columnas
# ------------------------------------------------------------

columnas_finales = [
    "Fecha",
    "Anio",
    "Mes",
    "Trimestre",
    "Dia_semana",
    "Fin_semana",
    "Periodo_COVID",
    "Es_COVID",
    "Ventana_Jul_Dic",

    "Vuelos_total",
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa",
    "Vuelos_internacionales",

    "Companias_activas",
    "Zonas_activas"
]

df_vuelos_diario = (
    df_vuelos_diario[columnas_finales]
    .sort_values("Fecha")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 13. Dataset largo para dashboard
# ------------------------------------------------------------

df_vuelos_dashboard = (
    vuelos_zona_dia
    .copy()
)

df_vuelos_dashboard["Anio"] = (
    df_vuelos_dashboard["Fecha"].dt.year
)

df_vuelos_dashboard["Mes"] = (
    df_vuelos_dashboard["Fecha"].dt.month
)

df_vuelos_dashboard["Periodo_COVID"] = (
    df_vuelos_dashboard["Fecha"]
    .apply(clasificar_periodo_covid)
)

df_vuelos_dashboard["Ventana_Jul_Dic"] = (
    df_vuelos_dashboard["Mes"]
    .between(7, 12)
    .astype(int)
)

df_vuelos_dashboard = (
    df_vuelos_dashboard[
        [
            "Fecha",
            "Anio",
            "Mes",
            "Periodo_COVID",
            "Ventana_Jul_Dic",
            "Zona",
            "Vuelos"
        ]
    ]
    .sort_values(["Fecha", "Zona"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 14. Control de coherencia: total = suma de zonas
# ------------------------------------------------------------

suma_zonas = (
    df_vuelos_diario[columnas_zona]
    .sum(axis=1)
)

diferencia = (
    df_vuelos_diario["Vuelos_total"]
    - suma_zonas
)

n_discrepancias = (diferencia != 0).sum()


# ------------------------------------------------------------
# 15. Resumen de la transformación
# ------------------------------------------------------------

print("=" * 70)
print("DATASET DIARIO DE ACTIVIDAD AÉREA")
print("=" * 70)

print(
    f"Periodo disponible: "
    f"{df_vuelos_diario['Fecha'].min().date()} "
    f"→ {df_vuelos_diario['Fecha'].max().date()}"
)

print(
    f"Días disponibles: "
    f"{len(df_vuelos_diario):,}"
)

print(
    f"Fechas duplicadas: "
    f"{df_vuelos_diario['Fecha'].duplicated().sum()}"
)

print(
    f"Discrepancias total-suma de zonas: "
    f"{n_discrepancias}"
)

print("\nDistribución por periodo:")

display(
    df_vuelos_diario
    .groupby("Periodo_COVID")
    .agg(
        Fecha_inicio=("Fecha", "min"),
        Fecha_fin=("Fecha", "max"),
        Dias=("Fecha", "nunique"),
        Vuelos_total=("Vuelos_total", "sum"),
        Vuelos_media_diaria=("Vuelos_total", "mean")
    )
    .round(2)
)

print("\nPrimeras observaciones del dataset diario:")

display(df_vuelos_diario.head())


print("\n" + "=" * 70)
print("DATASET EN FORMATO LARGO PARA DASHBOARD")
print("=" * 70)

print(
    f"Filas: {len(df_vuelos_dashboard):,}"
)

print(
    f"Fechas: {df_vuelos_dashboard['Fecha'].nunique():,}"
)

print(
    f"Zonas: {df_vuelos_dashboard['Zona'].nunique()}"
)

display(df_vuelos_dashboard.head(10))

DATASET DIARIO DE ACTIVIDAD AÉREA
Periodo disponible: 2019-07-01 → 2024-12-31
Días disponibles: 2,011
Fechas duplicadas: 0
Discrepancias total-suma de zonas: 0

Distribución por periodo:


,Fecha_inicio,Fecha_fin,Dias,Vuelos_total,Vuelos_media_diaria
Periodo_COVID,,,,,
COVID,2020-01-01,2021-12-31,731,219575,300.38
Post-COVID,2022-01-01,2024-12-31,1096,766126,699.02
Pre-COVID,2019-07-01,2019-12-31,184,123358,670.42



Primeras observaciones del dataset diario:


,Fecha,Anio,Mes,Trimestre,Dia_semana,Fin_semana,Periodo_COVID,Es_COVID,Ventana_Jul_Dic,Vuelos_total,Vuelos_Espanya,Vuelos_Europa,Vuelos_America,Vuelos_Asia,Vuelos_Africa,Vuelos_internacionales,Companias_activas,Zonas_activas
0,2019-07-01,2019,7,3,0,0,Pre-COVID,0,1,742,244.0,458.0,10.0,19.0,11.0,498.0,20,5
1,2019-07-02,2019,7,3,1,0,Pre-COVID,0,1,715,246.0,435.0,10.0,17.0,7.0,469.0,20,5
2,2019-07-03,2019,7,3,2,0,Pre-COVID,0,1,757,260.0,458.0,10.0,16.0,13.0,497.0,20,5
3,2019-07-04,2019,7,3,3,0,Pre-COVID,0,1,728,253.0,428.0,10.0,16.0,21.0,475.0,19,5
4,2019-07-05,2019,7,3,4,0,Pre-COVID,0,1,737,250.0,454.0,10.0,17.0,6.0,487.0,19,5



DATASET EN FORMATO LARGO PARA DASHBOARD
Filas: 9,378
Fechas: 2,011
Zonas: 5


,Fecha,Anio,Mes,Periodo_COVID,Ventana_Jul_Dic,Zona,Vuelos
0,2019-07-01,2019,7,Pre-COVID,1,Africa,11
1,2019-07-01,2019,7,Pre-COVID,1,America,10
2,2019-07-01,2019,7,Pre-COVID,1,Asia,19
3,2019-07-01,2019,7,Pre-COVID,1,Espanya,244
4,2019-07-01,2019,7,Pre-COVID,1,Europa,458
5,2019-07-02,2019,7,Pre-COVID,1,Africa,7
6,2019-07-02,2019,7,Pre-COVID,1,America,10
7,2019-07-02,2019,7,Pre-COVID,1,Asia,17
8,2019-07-02,2019,7,Pre-COVID,1,Espanya,246
9,2019-07-02,2019,7,Pre-COVID,1,Europa,435


#### Resultados de la construcción de indicadores diarios

La transformación de los registros originales genera un conjunto temporal compuesto por **2.011 observaciones diarias**, comprendidas entre el **1 de julio de 2019 y el 31 de diciembre de 2024**.

Cada fecha constituye una observación única y no se detectan duplicados temporales. Asimismo, el control interno entre el número total de vuelos y la suma de las cinco zonas geográficas produce **0 discrepancias**, confirmando que la transformación conserva íntegramente la actividad registrada en la fuente.

El dataset diario resultante incorpora información sobre:

- volumen total de vuelos;
- actividad correspondiente a España;
- actividad correspondiente a Europa;
- actividad correspondiente a América;
- actividad correspondiente a Asia;
- actividad correspondiente a África;
- actividad internacional;
- número de compañías activas;
- número de zonas activas;
- variables temporales necesarias para el análisis longitudinal;
- clasificación pre-COVID, COVID y post-COVID.

#### Primera evidencia del impacto de la pandemia

La agregación diaria permite observar una diferencia muy marcada entre los tres periodos definidos.

| Periodo | Días disponibles | Vuelos registrados | Media diaria |
|---|---:|---:|---:|
| Pre-COVID | 184 | 123.358 | 670,42 |
| COVID | 731 | 219.575 | 300,38 |
| Post-COVID | 1.096 | 766.126 | 699,02 |

La actividad aérea media durante el periodo COVID se reduce aproximadamente un **55 % respecto a la referencia pre-COVID**, pasando de 670,42 a 300,38 vuelos diarios.

Durante el periodo post-COVID la media asciende hasta **699,02 vuelos diarios**, superando ligeramente el nivel medio observado durante el segundo semestre de 2019. Estos resultados constituyen una primera evidencia descriptiva de la fuerte alteración de la movilidad aérea durante la pandemia y de su posterior recuperación.

No obstante, esta comparación debe interpretarse con cautela. El periodo pre-COVID disponible únicamente comprende **julio–diciembre de 2019**, mientras que los periodos COVID y post-COVID contienen años completos. La estacionalidad propia del tráfico aéreo podría afectar, por tanto, a la comparación directa de las medias.

Por este motivo, el análisis específico del efecto COVID incorporará también una **ventana temporal homogénea julio–diciembre para todos los años**, permitiendo comparar periodos equivalentes y separar, en la medida de lo posible, el efecto de la pandemia de las variaciones estacionales.

#### Preparación para el análisis por zona

La desagregación diaria conserva las cinco zonas geográficas de la fuente. Esto permitirá analizar si la caída y posterior recuperación de la actividad aérea presentaron comportamientos diferentes entre tráfico nacional, europeo e intercontinental.

Esta dimensión resulta especialmente relevante para evaluar no solo la intensidad total de la movilidad aérea, sino también los posibles cambios en su composición durante las distintas fases del periodo estudiado.

#### Preparación para el dashboard

Paralelamente se genera una estructura en formato largo destinada a la visualización interactiva. Esta tabla conserva las dimensiones:

**fecha × periodo COVID × zona geográfica × número de vuelos**

y permite alimentar directamente filtros y gráficos temporales del dashboard.

De este modo será posible visualizar dinámicamente la evolución de la actividad aérea según año, periodo epidemiológico y zona geográfica, así como comparar los patrones pre-COVID, COVID y post-COVID sin reprocesar los archivos originales.

> **Conclusión:** la agregación produce una serie diaria consistente y preparada tanto para su integración con las variables ambientales como para el análisis temporal y el dashboard. Los resultados muestran ya una señal descriptiva muy marcada del impacto de la pandemia sobre la actividad aérea, cuya magnitud y evolución se estudiarán posteriormente mediante comparaciones temporales homogéneas y análisis estadístico.

### 5.5. Validación final y análisis comparativo de la actividad aérea

Una vez construidos los indicadores diarios, se realiza una última validación antes de incorporar la información de tráfico aéreo al dataset integrado.

Esta etapa tiene tres objetivos principales:

1. **Verificar la consistencia de las tablas generadas**, comprobando la continuidad temporal, la correspondencia entre vuelos totales y vuelos por zona y la existencia de posibles combinaciones fecha–zona ausentes.

2. **Preparar una estructura completa para el dashboard**, garantizando que cada fecha disponga de las cinco zonas geográficas. Cuando una combinación fecha–zona no figure en la fuente original, se comprobará si su ausencia representa falta de actividad y, en ese caso, se codificará explícitamente como cero para evitar discontinuidades artificiales en las visualizaciones.

3. **Realizar una primera comparación descriptiva COVID/no-COVID**, utilizando tanto toda la información disponible como una ventana temporal homogénea julio–diciembre. Esta segunda comparación resulta necesaria debido a que la serie pre-COVID disponible comienza el 1 de julio de 2019.

Se analizarán las diferencias entre:

- **Pre-COVID:** julio–diciembre de 2019;
- **COVID:** 2020–2021;
- **Post-COVID:** 2022–2024.

Para reducir el posible efecto de la estacionalidad derivado de la cobertura parcial de 2019, se calcularán adicionalmente indicadores utilizando exclusivamente los meses de **julio a diciembre de todos los años**.

El análisis se realizará tanto para el volumen total de vuelos como para las cinco zonas geográficas, permitiendo estudiar si la reducción y recuperación de la movilidad aérea presentan patrones diferenciados según el ámbito territorial.

> **Criterio metodológico:** esta etapa tiene carácter descriptivo y de control de calidad. Las diferencias observadas entre periodos no se interpretarán todavía como relaciones causales con la calidad del aire. La asociación entre tráfico aéreo, contaminación y restantes variables explicativas se evaluará posteriormente sobre el dataset integrado.

In [ ]:
# ============================================================
# 5.5. VALIDACIÓN FINAL Y COMPARACIÓN COVID / NO-COVID
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. CONTROL FINAL DEL DATASET DIARIO
# ------------------------------------------------------------

print("=" * 70)
print("CONTROL FINAL — DATASET DIARIO")
print("=" * 70)

print(f"Filas: {len(df_vuelos_diario):,}")
print(
    f"Periodo: {df_vuelos_diario['Fecha'].min().date()} "
    f"→ {df_vuelos_diario['Fecha'].max().date()}"
)
print(
    "Fechas duplicadas:",
    df_vuelos_diario["Fecha"].duplicated().sum()
)

print(
    "Valores nulos totales:",
    df_vuelos_diario.isna().sum().sum()
)


# ------------------------------------------------------------
# 2. CONTROL DE COHERENCIA ENTRE TOTAL Y ZONAS
# ------------------------------------------------------------

columnas_zona = [
    "Vuelos_Espanya",
    "Vuelos_Europa",
    "Vuelos_America",
    "Vuelos_Asia",
    "Vuelos_Africa"
]

suma_zonas = df_vuelos_diario[columnas_zona].sum(axis=1)

diferencias = (
    df_vuelos_diario["Vuelos_total"] - suma_zonas
)

print(
    "Discrepancias total vs suma de zonas:",
    (diferencias != 0).sum()
)


# ------------------------------------------------------------
# 3. COMPROBAR MALLA FECHA × ZONA DEL DASHBOARD
# ------------------------------------------------------------

zonas_esperadas = [
    "Espanya",
    "Europa",
    "America",
    "Asia",
    "Africa"
]

fechas_esperadas = pd.DataFrame({
    "Fecha": df_vuelos_diario["Fecha"].unique()
})

malla_dashboard = (
    fechas_esperadas
    .assign(_key=1)
    .merge(
        pd.DataFrame({
            "Zona": zonas_esperadas,
            "_key": 1
        }),
        on="_key"
    )
    .drop(columns="_key")
)

print("\n" + "=" * 70)
print("CONTROL FECHA × ZONA")
print("=" * 70)

print(
    "Combinaciones esperadas:",
    len(malla_dashboard)
)

print(
    "Combinaciones observadas:",
    len(df_vuelos_dashboard)
)

combinaciones_ausentes = (
    malla_dashboard
    .merge(
        df_vuelos_dashboard[["Fecha", "Zona"]],
        on=["Fecha", "Zona"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    [["Fecha", "Zona"]]
)

print(
    "Combinaciones fecha-zona ausentes:",
    len(combinaciones_ausentes)
)

if len(combinaciones_ausentes) > 0:

    print("\nDistribución de combinaciones ausentes por zona:")

    display(
        combinaciones_ausentes["Zona"]
        .value_counts()
        .rename("Dias_ausentes")
        .to_frame()
    )


# ------------------------------------------------------------
# 4. COMPLETAR DATASET DEL DASHBOARD
# ------------------------------------------------------------

# La tabla diaria ancha ya ha demostrado que la suma de las
# cinco zonas reproduce exactamente Vuelos_total.
#
# Por tanto, las combinaciones fecha-zona no presentes en la
# tabla larga representan ausencia de vuelos registrados para
# esa zona y se completan explícitamente con 0.

df_vuelos_dashboard_completo = (
    malla_dashboard
    .merge(
        df_vuelos_dashboard[
            ["Fecha", "Zona", "Vuelos"]
        ],
        on=["Fecha", "Zona"],
        how="left"
    )
    .sort_values(["Fecha", "Zona"])
    .reset_index(drop=True)
)

df_vuelos_dashboard_completo["Vuelos"] = (
    df_vuelos_dashboard_completo["Vuelos"]
    .fillna(0)
)


# ------------------------------------------------------------
# 5. RECUPERAR VARIABLES TEMPORALES
# ------------------------------------------------------------

variables_temporales = (
    df_vuelos_diario[
        [
            "Fecha",
            "Anio",
            "Mes",
            "Periodo_COVID",
            "Ventana_Jul_Dic"
        ]
    ]
    .drop_duplicates("Fecha")
)

df_vuelos_dashboard_completo = (
    df_vuelos_dashboard_completo
    .merge(
        variables_temporales,
        on="Fecha",
        how="left",
        validate="many_to_one"
    )
)

df_vuelos_dashboard_completo = (
    df_vuelos_dashboard_completo[
        [
            "Fecha",
            "Anio",
            "Mes",
            "Periodo_COVID",
            "Ventana_Jul_Dic",
            "Zona",
            "Vuelos"
        ]
    ]
)


# ------------------------------------------------------------
# 6. VALIDACIÓN DEL DASHBOARD COMPLETO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET COMPLETO PARA DASHBOARD")
print("=" * 70)

print(
    "Filas:",
    f"{len(df_vuelos_dashboard_completo):,}"
)

print(
    "Fechas:",
    f"{df_vuelos_dashboard_completo['Fecha'].nunique():,}"
)

print(
    "Zonas:",
    df_vuelos_dashboard_completo["Zona"].nunique()
)

print(
    "Nulos en Vuelos:",
    df_vuelos_dashboard_completo["Vuelos"].isna().sum()
)


# ------------------------------------------------------------
# 7. COMPARACIÓN GENERAL ENTRE PERIODOS
# ------------------------------------------------------------

comparacion_periodos = (
    df_vuelos_diario
    .groupby("Periodo_COVID")
    .agg(
        Dias=("Fecha", "nunique"),
        Vuelos_totales=("Vuelos_total", "sum"),
        Media_diaria=("Vuelos_total", "mean"),
        Mediana_diaria=("Vuelos_total", "median"),
        Desv_estandar=("Vuelos_total", "std"),
        Companias_media=("Companias_activas", "mean")
    )
    .round(2)
)

print("\n" + "=" * 70)
print("COMPARACIÓN GENERAL PRE / COVID / POST")
print("=" * 70)

display(comparacion_periodos)


# ------------------------------------------------------------
# 8. CAMBIO RELATIVO RESPECTO AL PRE-COVID
# ------------------------------------------------------------

media_pre = comparacion_periodos.loc[
    "Pre-COVID",
    "Media_diaria"
]

comparacion_periodos["Cambio_vs_Pre_pct"] = (
    (
        comparacion_periodos["Media_diaria"]
        / media_pre
        - 1
    ) * 100
).round(2)

print("\nCambio relativo respecto al periodo pre-COVID:")

display(
    comparacion_periodos[
        [
            "Media_diaria",
            "Cambio_vs_Pre_pct"
        ]
    ]
)


# ------------------------------------------------------------
# 9. COMPARACIÓN HOMOGÉNEA JULIO-DICIEMBRE
# ------------------------------------------------------------

df_jul_dic = (
    df_vuelos_diario.loc[
        df_vuelos_diario["Ventana_Jul_Dic"] == 1
    ]
    .copy()
)

comparacion_jul_dic = (
    df_jul_dic
    .groupby("Anio")
    .agg(
        Dias=("Fecha", "nunique"),
        Vuelos_totales=("Vuelos_total", "sum"),
        Media_diaria=("Vuelos_total", "mean"),
        Mediana_diaria=("Vuelos_total", "median"),
        Companias_media=("Companias_activas", "mean")
    )
    .round(2)
)

media_2019 = comparacion_jul_dic.loc[
    2019,
    "Media_diaria"
]

comparacion_jul_dic["Cambio_vs_2019_pct"] = (
    (
        comparacion_jul_dic["Media_diaria"]
        / media_2019
        - 1
    ) * 100
).round(2)

print("\n" + "=" * 70)
print("COMPARACIÓN HOMOGÉNEA — JULIO A DICIEMBRE")
print("=" * 70)

display(comparacion_jul_dic)


# ------------------------------------------------------------
# 10. COMPARACIÓN POR ZONA Y PERIODO
# ------------------------------------------------------------

comparacion_zonas = (
    df_vuelos_dashboard_completo
    .groupby(
        ["Periodo_COVID", "Zona"]
    )
    .agg(
        Media_diaria=("Vuelos", "mean"),
        Mediana_diaria=("Vuelos", "median"),
        Vuelos_totales=("Vuelos", "sum")
    )
    .reset_index()
)

media_zona_pre = (
    comparacion_zonas.loc[
        comparacion_zonas["Periodo_COVID"]
        == "Pre-COVID",
        ["Zona", "Media_diaria"]
    ]
    .rename(
        columns={
            "Media_diaria": "Media_Pre"
        }
    )
)

comparacion_zonas = (
    comparacion_zonas
    .merge(
        media_zona_pre,
        on="Zona",
        how="left"
    )
)

comparacion_zonas["Cambio_vs_Pre_pct"] = (
    (
        comparacion_zonas["Media_diaria"]
        / comparacion_zonas["Media_Pre"]
        - 1
    ) * 100
).round(2)

print("\n" + "=" * 70)
print("COMPARACIÓN POR ZONA GEOGRÁFICA")
print("=" * 70)

display(
    comparacion_zonas.round(2)
)


# ------------------------------------------------------------
# 11. CONTROL FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDACIÓN FINAL")
print("=" * 70)

print(
    "Dataset diario preparado para integración:",
    df_vuelos_diario.shape
)

print(
    "Dataset largo preparado para dashboard:",
    df_vuelos_dashboard_completo.shape
)

print(
    "Duplicados Fecha-Zona dashboard:",
    df_vuelos_dashboard_completo
    .duplicated(["Fecha", "Zona"])
    .sum()
)

print(
    "Nulos totales dataset diario:",
    df_vuelos_diario.isna().sum().sum()
)

print(
    "Nulos totales dataset dashboard:",
    df_vuelos_dashboard_completo.isna().sum().sum()
)

CONTROL FINAL — DATASET DIARIO
Filas: 2,011
Periodo: 2019-07-01 → 2024-12-31
Fechas duplicadas: 0
Valores nulos totales: 0
Discrepancias total vs suma de zonas: 0

CONTROL FECHA × ZONA
Combinaciones esperadas: 10055
Combinaciones observadas: 9378
Combinaciones fecha-zona ausentes: 677

Distribución de combinaciones ausentes por zona:


,Dias_ausentes
Zona,
Africa,409
America,192
Asia,76



DATASET COMPLETO PARA DASHBOARD
Filas: 10,055
Fechas: 2,011
Zonas: 5
Nulos en Vuelos: 0

COMPARACIÓN GENERAL PRE / COVID / POST


,Dias,Vuelos_totales,Media_diaria,Mediana_diaria,Desv_estandar,Companias_media
Periodo_COVID,,,,,,
COVID,731,219575,300.38,274.0,208.48,15.92
Post-COVID,1096,766126,699.02,721.0,115.59,23.47
Pre-COVID,184,123358,670.42,693.5,85.99,19.84



Cambio relativo respecto al periodo pre-COVID:


...


COMPARACIÓN HOMOGÉNEA — JULIO A DICIEMBRE


,Dias,Vuelos_totales,Media_diaria,Mediana_diaria,Companias_media,Cambio_vs_2019_pct
Anio,,,,,,
2019,184,123358,670.42,693.5,19.84,0.00
2020,184,43780,237.93,233.5,15.61,-64.51
2021,184,95710,520.16,527.0,20.91,-22.41
2022,184,125063,679.69,704.0,22.60,1.38
2023,184,136608,742.43,758.5,24.29,10.74
2024,184,145244,789.37,803.0,24.54,17.74



COMPARACIÓN POR ZONA GEOGRÁFICA


,Periodo_COVID,Zona,Media_diaria,Mediana_diaria,Vuelos_totales,Media_Pre,Cambio_vs_Pre_pct
0,COVID,Africa,3.64,0.0,2659.0,12.15,-70.05
1,COVID,America,3.04,2.0,2225.0,10.27,-70.37
2,COVID,Asia,5.42,5.0,3959.0,15.30,-64.61
3,COVID,Espanya,133.74,138.0,97765.0,224.20,-40.35
4,COVID,Europa,154.54,124.0,112967.0,408.51,-62.17
5,Post-COVID,Africa,13.11,13.0,14366.0,12.15,7.91
6,Post-COVID,America,14.68,15.0,16086.0,10.27,42.89
7,Post-COVID,Asia,12.98,13.0,14231.0,15.30,-15.16
8,Post-COVID,Espanya,228.22,232.0,250129.0,224.20,1.79
9,Post-COVID,Europa,430.03,443.0,471314.0,408.51,5.27



VALIDACIÓN FINAL
Dataset diario preparado para integración: (2011, 18)
Dataset largo preparado para dashboard: (10055, 7)
Duplicados Fecha-Zona dashboard: 0
Nulos totales dataset diario: 0
Nulos totales dataset dashboard: 0


#### Resultados de la validación final y comparación COVID/no-COVID

La validación final confirma la consistencia del dataset diario de tráfico aéreo. La serie contiene **2.011 observaciones diarias**, comprendidas entre el 1 de julio de 2019 y el 31 de diciembre de 2024, sin fechas duplicadas, valores nulos ni discrepancias entre el volumen total de vuelos y la suma de las cinco zonas geográficas.

##### Validación de la estructura para el dashboard

La comprobación de la estructura `fecha × zona` identifica inicialmente **677 combinaciones no presentes explícitamente en los registros originales**. Estas ausencias se concentran exclusivamente en las zonas de menor frecuencia de actividad: África, América y Asia.

Dado que la tabla diaria agregada reproduce exactamente el número total de vuelos mediante la suma de las cinco zonas, estas combinaciones corresponden a días sin actividad registrada para dichas zonas y se representan explícitamente mediante valores cero.

Tras esta operación, la estructura destinada al dashboard queda formada por **10.055 registros**, correspondientes exactamente a:

**2.011 fechas × 5 zonas geográficas**

sin valores nulos ni duplicados fecha-zona. Esta estructura garantiza la continuidad de las series y evita discontinuidades artificiales durante la representación interactiva.

##### Comparación general entre periodos

La comparación descriptiva inicial muestra una fuerte alteración de la actividad aérea durante el periodo COVID.

La media diaria pasa de **670,42 vuelos/día en el periodo pre-COVID** a **300,38 vuelos/día durante 2020–2021**, lo que representa una reducción del **55,20 %**.

Durante 2022–2024 la actividad alcanza una media de **699,02 vuelos/día**, un **4,27 % superior** a la referencia pre-COVID disponible.

Sin embargo, debido a que el periodo pre-COVID únicamente dispone de información correspondiente a julio–diciembre de 2019, esta comparación general puede estar parcialmente condicionada por la estacionalidad del tráfico aéreo.

##### Comparación temporal homogénea: julio–diciembre

Para controlar esta diferencia de cobertura se realiza una segunda comparación utilizando exactamente la misma ventana temporal —**1 de julio a 31 de diciembre**— para cada uno de los seis años disponibles.

Los resultados muestran una secuencia temporal muy definida:

| Año | Media diaria de vuelos | Cambio respecto a 2019 |
|---|---:|---:|
| 2019 | 670,42 | referencia |
| 2020 | 237,93 | −64,51 % |
| 2021 | 520,16 | −22,41 % |
| 2022 | 679,69 | +1,38 % |
| 2023 | 742,43 | +10,74 % |
| 2024 | 789,37 | +17,74 % |

La utilización de una ventana temporal homogénea refuerza la señal observada inicialmente. Durante el segundo semestre de 2020 la actividad aérea se sitúa aproximadamente un **65 % por debajo del nivel de 2019**. En 2021 se observa una recuperación importante, aunque la actividad permanece todavía alrededor de un **22 % por debajo de la referencia pre-COVID**.

En 2022 se alcanza prácticamente el nivel previo a la pandemia, mientras que 2023 y 2024 muestran una actividad progresivamente superior a la registrada en 2019.

El patrón observado permite distinguir claramente una secuencia de **impacto, recuperación y expansión post-COVID**, que podrá contrastarse posteriormente con la evolución de las concentraciones de contaminantes atmosféricos.

##### Diferencias según zona geográfica

El impacto de la pandemia no presenta la misma intensidad en todas las zonas.

Durante el periodo COVID, respecto a la referencia pre-COVID, las reducciones medias son aproximadamente:

- **África:** −70,05 %;
- **América:** −70,37 %;
- **Asia:** −64,61 %;
- **Europa:** −62,17 %;
- **España:** −40,35 %.

Los resultados muestran, por tanto, una reducción considerablemente mayor en los vuelos internacionales e intercontinentales que en los asociados al territorio nacional.

Durante el periodo post-COVID aparecen además patrones de recuperación diferenciados. España y Europa recuperan y superan ligeramente sus niveles previos, mientras que América presenta un incremento considerable respecto a la referencia disponible. Asia, por el contrario, mantiene una media inferior a la observada durante el segundo semestre de 2019.

Estas diferencias justifican conservar la desagregación geográfica del tráfico aéreo en el análisis posterior, en lugar de utilizar exclusivamente el número total de vuelos.

##### Implicaciones para el análisis integrado

Los resultados permiten caracterizar el tráfico aéreo como una variable temporal especialmente útil para el estudio del periodo COVID. La fuerte caída observada en 2020, la recuperación parcial de 2021 y el retorno a niveles pre-COVID durante 2022 proporcionan una señal temporal claramente diferenciada.

No obstante, estas variaciones se interpretan en esta etapa exclusivamente como **patrones descriptivos de movilidad**. La posible asociación entre actividad aérea y concentraciones de contaminantes se evaluará posteriormente junto con meteorología, tráfico viario y restantes variables explicativas.

> **Conclusión:** el bloque de tráfico aéreo queda validado tanto para su integración en el dataset maestro como para su utilización en el dashboard. La comparación mediante ventanas temporales homogéneas permite identificar de forma robusta el impacto de la pandemia y la posterior recuperación de la actividad aérea, conservando además diferencias relevantes según la zona geográfica.

### 5.6. Exportación de los datos procesados

Finalizado el proceso de limpieza, agregación y validación, se exportan las tablas definitivas de tráfico aéreo en la carpeta `DATOS LIMPIOS` de la fuente original.

Se generan dos archivos diferenciados:

- `df_vuelos_diario_limpio.csv`: dataset diario en formato ancho, destinado a la posterior integración con el resto de variables temporales del estudio.
- `df_vuelos_dashboard_limpio.csv`: dataset en formato largo `fecha × zona`, preparado específicamente para las visualizaciones interactivas del dashboard.

Esta separación permite mantener una tabla compacta para el modelado y la integración, conservando simultáneamente el nivel de detalle geográfico necesario para el análisis visual.

Ambos archivos mantienen las variables de periodización necesarias para distinguir las etapas **pre-COVID, COVID y post-COVID**, así como la ventana temporal homogénea julio–diciembre utilizada para las comparaciones interanuales.

> Los archivos originales permanecen inalterados y las tablas procesadas se almacenan de forma independiente, garantizando la trazabilidad y reproducibilidad del tratamiento.

In [ ]:
# ============================================================
# 5.6. EXPORTACIÓN DE LOS DATASETS LIMPIOS DE VUELOS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Carpeta de salida
# ------------------------------------------------------------

RUTA_SALIDA_VUELOS = (
    Path("/content/drive/MyDrive/TFM/05_Transporte_Aereo")
    / "DATOS LIMPIOS"
)

RUTA_SALIDA_VUELOS.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Rutas de los archivos definitivos
# ------------------------------------------------------------

ruta_vuelos_diario = (
    RUTA_SALIDA_VUELOS
    / "df_vuelos_diario_limpio.csv"
)

ruta_vuelos_dashboard = (
    RUTA_SALIDA_VUELOS
    / "df_vuelos_dashboard_limpio.csv"
)


# ------------------------------------------------------------
# 3. Exportación
# ------------------------------------------------------------

df_vuelos_diario.to_csv(
    ruta_vuelos_diario,
    index=False,
    encoding="utf-8-sig"
)

df_vuelos_dashboard_completo.to_csv(
    ruta_vuelos_dashboard,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. Verificación
# ------------------------------------------------------------

print("=" * 70)
print("EXPORTACIÓN FINAL — TRÁFICO AÉREO")
print("=" * 70)

print("\n✓ Dataset diario para integración:")
print(ruta_vuelos_diario)
print(f"  Dimensiones: {df_vuelos_diario.shape}")

print("\n✓ Dataset específico para dashboard:")
print(ruta_vuelos_dashboard)
print(f"  Dimensiones: {df_vuelos_dashboard_completo.shape}")

print("\nArchivos generados correctamente.")

EXPORTACIÓN FINAL — TRÁFICO AÉREO

✓ Dataset diario para integración:
/content/drive/MyDrive/TFM/05_Transporte_Aereo/DATOS LIMPIOS/df_vuelos_diario_limpio.csv
  Dimensiones: (2011, 18)

✓ Dataset específico para dashboard:
/content/drive/MyDrive/TFM/05_Transporte_Aereo/DATOS LIMPIOS/df_vuelos_dashboard_limpio.csv
  Dimensiones: (10055, 7)

Archivos generados correctamente.


#### Resultados de la exportación

El procesamiento del tráfico aéreo finaliza con la generación de **dos datasets limpios y validados**, almacenados en la carpeta `DATOS LIMPIOS` de la fuente original.

Se exportan las siguientes tablas:

- **`df_vuelos_diario_limpio.csv`**: dataset principal para la integración posterior con el resto de fuentes del estudio. Contiene **2.011 observaciones diarias y 18 variables**, correspondientes al periodo comprendido entre el 1 de julio de 2019 y el 31 de diciembre de 2024.

- **`df_vuelos_dashboard_limpio.csv`**: dataset auxiliar específicamente preparado para la visualización interactiva. Contiene **10.055 registros y 7 variables**, correspondientes a la estructura completa de **2.011 fechas × 5 zonas geográficas**.

El primer archivo conserva los indicadores diarios necesarios para el análisis integrado —volumen total de vuelos, distribución geográfica, actividad internacional, compañías activas y variables temporales—, mientras que el segundo mantiene una estructura larga `fecha × zona` optimizada para filtros, comparaciones y series temporales en el dashboard.

Ambas tablas incorporan la clasificación **pre-COVID, COVID y post-COVID**, así como la identificación de la ventana homogénea julio–diciembre utilizada para realizar comparaciones interanuales compatibles con la cobertura parcial de 2019.

Los controles realizados previamente garantizan que las tablas exportadas no presentan valores nulos ni duplicados en sus respectivas claves temporales y que la suma de la actividad por zonas reproduce correctamente el volumen total diario de vuelos.

> **Resultado final:** el bloque de tráfico aéreo queda completamente procesado, validado y preparado para su incorporación al dataset maestro y para su explotación independiente en el dashboard, manteniendo los archivos originales sin modificaciones y garantizando la trazabilidad del proceso.

### Cierre del bloque de tráfico aéreo

El tratamiento de los datos de tráfico aéreo permite transformar los archivos originales desagregados por **fecha, compañía y zona geográfica** en dos estructuras finales consistentes y directamente utilizables en las siguientes fases del proyecto.

La auditoría realizada confirma una elevada calidad de la fuente dentro de su periodo disponible: no se detectan fechas inválidas, valores negativos, inconsistencias entre códigos y denominaciones ni duplicados para la clave natural `fecha × compañía × zona`. Asimismo, existe continuidad diaria completa desde el comienzo de la serie disponible.

La principal limitación temporal corresponde a la **ausencia de información para 2018** y al inicio de los registros disponibles el **1 de julio de 2019**. Esta circunstancia se mantiene explícitamente y no se han generado valores artificiales para completar los periodos no observados.

La agregación permite obtener una serie de **2.011 días**, comprendida entre el 1 de julio de 2019 y el 31 de diciembre de 2024, incorporando indicadores de actividad aérea total, distribución por zona geográfica, vuelos internacionales y número de compañías activas.

El análisis descriptivo muestra además una señal temporal claramente asociada al periodo de pandemia. Para evitar comparaciones condicionadas por la cobertura parcial de 2019, se utiliza una ventana homogénea julio–diciembre, en la que se observa una reducción de la actividad aérea del **64,51 % en 2020 respecto a 2019**, seguida de una recuperación progresiva: **−22,41 % en 2021, +1,38 % en 2022, +10,74 % en 2023 y +17,74 % en 2024**.

La desagregación geográfica muestra asimismo que el impacto no fue homogéneo entre zonas, justificando la conservación de esta dimensión para los análisis exploratorios posteriores.

Como resultado del procesamiento se generan dos archivos definitivos:

- **`df_vuelos_diario_limpio.csv`** — 2.011 observaciones × 18 variables. Constituye la tabla destinada a la integración con el dataset maestro.
- **`df_vuelos_dashboard_limpio.csv`** — 10.055 observaciones × 7 variables. Mantiene una malla completa `fecha × zona` destinada a la visualización interactiva.

De este modo, el bloque de tráfico aéreo queda **limpio, validado, documentado y preparado para su integración**, conservando simultáneamente una salida específica para el dashboard.

> **Resultado del bloque:** se obtiene un indicador diario robusto de actividad aérea que permitirá estudiar conjuntamente movilidad, periodo COVID/no-COVID y calidad del aire, manteniendo separadas las etapas de preparación de datos y posterior análisis de asociaciones.

### 6 · TRÁFICO MARÍTIMO: TRATAMIENTO Y CONSTRUCCIÓN DE INDICADORES TEMPORALES

## 6.1. Fuente de datos y planteamiento del tratamiento

Los datos de tráfico marítimo utilizados en el estudio proceden del portal de **Datos Abiertos del Port de Barcelona**, desde el que se descargaron los registros correspondientes a las escalas de buques.

**Fuente oficial:**  
https://opendata.portdebarcelona.cat/ca/dataset

Para cada año se dispone de dos archivos diferenciados:

- **arribades**, correspondientes a las llegadas de buques;
- **sortides**, correspondientes a las salidas de buques.

La información disponible permite reconstruir la actividad portuaria a partir de las fechas y horas estimadas de llegada (`ETA`) y salida (`ETD`) de cada escala, incorporando además características de las embarcaciones como el tipo de buque, la eslora, el calado, la manga y diferentes identificadores de la escala y del buque.

A diferencia de otras fuentes del proyecto, los datos marítimos presentan cobertura completa para el intervalo **2018–2024**, por lo que permiten analizar la evolución de la actividad portuaria durante todo el horizonte temporal considerado.

El tratamiento se plantea con tres objetivos complementarios:

1. **Integración y modelado:** obtener una serie diaria compatible con la resolución temporal utilizada para contaminación atmosférica y el resto de variables explicativas.
2. **Análisis temporal:** conservar la información necesaria para comparar los periodos **pre-COVID (2018–2019), COVID (2020–2021) y post-COVID (2022–2024)**.
3. **Dashboard:** mantener un segundo dataset con mayor granularidad que permita explorar de forma interactiva las llegadas, salidas, categorías y características de los buques.

El proceso ETL comprende la consolidación de los archivos anuales, normalización de fechas y variables físicas, identificación y eliminación de duplicados, clasificación funcional de los tipos de buque, reconstrucción independiente de llegadas y salidas, agregación diaria y controles finales de calidad.

La generación de variables específicamente destinadas al aprendizaje automático, como retardos o medias móviles, se reserva para una etapa posterior sobre el dataset integrado, permitiendo controlar adecuadamente su construcción y evitar posibles problemas de fuga de información (*data leakage*).

> **Objetivo del bloque:** transformar los registros originales de escalas del Port de Barcelona en una representación temporal diaria, consistente y reproducible de la actividad marítima, conservando simultáneamente el nivel de detalle necesario para el análisis exploratorio y el dashboard.

### 6.1. Inspección de los archivos anuales de referencia

Como paso previo al tratamiento multianual del tráfico marítimo, se inspeccionan los archivos de **arribadas y salidas correspondientes a 2024**.

A diferencia de otras fuentes del estudio, la información portuaria se distribuye en dos conjuntos de datos que describen eventos asociados a las escalas de los buques. Por este motivo, antes de concatenar los años disponibles resulta necesario conocer su estructura, variables comunes y posible solapamiento.

La inspección inicial tiene como objetivos:

- identificar las variables disponibles;
- comprobar dimensiones, tipos y valores ausentes;
- caracterizar las variables temporales `ETA` y `ETD`;
- identificar los campos disponibles para reconocer de forma única una escala;
- examinar las variables físicas del buque, como eslora, manga y calado;
- identificar la información disponible sobre tipo de buque, terminal, muelle, origen y destino.

Esta revisión permitirá posteriormente construir un procedimiento de deduplicación que evite contabilizar dos veces una misma escala cuando aparezca tanto en el archivo de arribadas como en el de salidas.

> **Criterio metodológico:** en esta primera etapa no se modifican los datos. Los archivos de 2024 se utilizan únicamente para caracterizar la fuente antes de desarrollar el tratamiento homogéneo del periodo completo 2018–2024.

In [ ]:
# ============================================================
# 6.1. INSPECCIÓN DE ARCHIVOS DE REFERENCIA — 2024
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

BASE_TFM = Path("/content/drive/MyDrive/TFM")
RUTA_PUERTO = BASE_TFM / "03_Transporte_Maritimo"

archivos_2024 = {
    "arribadas": RUTA_PUERTO / "2024" / "escales_arribades_2024.csv",
    "salidas": RUTA_PUERTO / "2024" / "escales_sortides_2024.csv"
}

def leer_csv_robusto(ruta):
    configuraciones = [
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "utf-8-sig"},
        {"sep": ";", "encoding": "utf-8-sig"},
        {"sep": ",", "encoding": "latin-1"},
        {"sep": ";", "encoding": "latin-1"},
    ]

    ultimo_error = None

    for config in configuraciones:
        try:
            df = pd.read_csv(ruta, low_memory=False, **config)
            if df.shape[1] > 1:
                return df
        except Exception as error:
            ultimo_error = error

    raise RuntimeError(
        f"No se pudo leer {ruta}\nÚltimo error: {ultimo_error}"
    )

dfs_referencia = {}

for tipo, ruta in archivos_2024.items():

    if not ruta.exists():
        raise FileNotFoundError(ruta)

    df_ref = leer_csv_robusto(ruta)
    dfs_referencia[tipo] = df_ref

    print("\n" + "=" * 75)
    print(f"{tipo.upper()} — 2024")
    print("=" * 75)

    print(f"Filas: {df_ref.shape[0]:,}")
    print(f"Columnas: {df_ref.shape[1]}")

    resumen = pd.DataFrame({
        "dtype": df_ref.dtypes.astype(str),
        "n_nulos": df_ref.isna().sum(),
        "%_nulos": (df_ref.isna().mean() * 100).round(2),
        "n_unicos": df_ref.nunique(dropna=True)
    })

    display(resumen)
    display(df_ref.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

ARRIBADAS — 2024
Filas: 10,313
Columnas: 32


,dtype,n_nulos,%_nulos,n_unicos
ANYESCALA,int64,0,0.00,2
TERMINALCODI,object,1490,14.45,36
VAIXELLBANDERACODI,object,0,0.00,60
ETD,object,0,0.00,10067
VAIXELLNOM,object,0,0.00,1748
CALAT_METRES,object,0,0.00,515
MANEGA_METRES,object,6634,64.33,217
MESINFO,object,14,0.14,1727
TERMINALNOM,object,1490,14.45,36
ETAHORA,object,0,0.00,1430


,ANYESCALA,TERMINALCODI,VAIXELLBANDERACODI,ETD,VAIXELLNOM,CALAT_METRES,MANEGA_METRES,MESINFO,TERMINALNOM,ETAHORA,...,PORTORIGENNOM,PORTDESTINOM,ESTOPERATIUID,MOLLMODULS,ESCALANUM,ETDDIA,ETA,VAIXELLBANDERANOM,CONSIGNATARI,CALLSIGN
0,2023,0010,PT,2024-01-12 06:40:00,CB PACIFIC,"10,5","31,99",https://www.marinetraffic.com/en/ais/details/s...,"EXOLUM CORPORATION, SA",15:12,...,Aliaga,Aliaga,F,2-5,51375-1,2024-01-12,2024-01-10 15:12:00,Portugal,BERGE MARITIMA SL,CQAS7
1,2023,0014,AG,2024-01-05 16:01:00,CATANIA,"4,1","11,41",https://www.marinetraffic.com/en/ais/details/s...,COMA Y RIBAS SL,11:27,...,Szczecin,Cartagena,F,3-4,51001-1,2024-01-05,2024-01-04 11:27:00,Antigua i Barbuda,COMA Y RIBAS SL,V2GH3
2,2023,0014,LV,2024-01-05 14:44:00,MOONRAY,"3,7","11,4",https://www.marinetraffic.com/en/ais/details/s...,COMA Y RIBAS SL,19:45,...,Villefranche-sur-Saone,Lyon,F,7-8,51294-1,2024-01-05,2024-01-04 19:45:00,Letònia,TRANSITAINER SA,YLOU



SALIDAS — 2024
Filas: 10,330
Columnas: 32


,dtype,n_nulos,%_nulos,n_unicos
ANYESCALA,int64,0,0.00,2
TERMINALCODI,object,1502,14.54,36
VAIXELLBANDERACODI,object,0,0.00,60
ETD,object,0,0.00,10083
VAIXELLNOM,object,0,0.00,1745
CALAT_METRES,object,0,0.00,514
MANEGA_METRES,object,6642,64.30,217
MESINFO,object,14,0.14,1724
TERMINALNOM,object,1502,14.54,36
ETAHORA,object,0,0.00,1430


,ANYESCALA,TERMINALCODI,VAIXELLBANDERACODI,ETD,VAIXELLNOM,CALAT_METRES,MANEGA_METRES,MESINFO,TERMINALNOM,ETAHORA,...,PORTORIGENNOM,PORTDESTINOM,ESTOPERATIUID,MOLLMODULS,ESCALANUM,ETDDIA,ETA,VAIXELLBANDERANOM,CONSIGNATARI,CALLSIGN
0,2023,0010,PT,2024-01-12 06:40:00,CB PACIFIC,"10,5","31,99",https://www.marinetraffic.com/en/ais/details/s...,"EXOLUM CORPORATION, SA",15:12,...,Aliaga,Aliaga,F,2-5,51375-1,2024-01-12,2024-01-10 15:12:00,Portugal,BERGE MARITIMA SL,CQAS7
1,2023,0014,AG,2024-01-05 16:01:00,CATANIA,"4,1","11,41",https://www.marinetraffic.com/en/ais/details/s...,COMA Y RIBAS SL,11:27,...,Szczecin,Cartagena,F,3-4,51001-1,2024-01-05,2024-01-04 11:27:00,Antigua i Barbuda,COMA Y RIBAS SL,V2GH3
2,2023,0014,LV,2024-01-05 14:44:00,MOONRAY,"3,7","11,4",https://www.marinetraffic.com/en/ais/details/s...,COMA Y RIBAS SL,19:45,...,Villefranche-sur-Saone,Lyon,F,7-8,51294-1,2024-01-05,2024-01-04 19:45:00,Letònia,TRANSITAINER SA,YLOU


#### Resultados de la inspección de los archivos de referencia

La inspección de los archivos de **arribadas y salidas correspondientes a 2024** confirma que ambas fuentes presentan una estructura muy similar y contienen un elevado nivel de detalle sobre cada escala portuaria.

El archivo de arribadas contiene **10.313 registros y 32 variables**, mientras que el archivo de salidas contiene **10.330 registros y 32 variables**. Ambas tablas comparten las mismas variables principales y permiten caracterizar cada escala mediante información temporal, identificadores del buque, dimensiones físicas, tipología, terminal, muelle y puertos de origen y destino.

Entre las variables más relevantes para el estudio se encuentran:

- `ETA` y `ETD`, correspondientes a las fechas y horas estimadas de llegada y salida;
- `ESCALANUM`, identificador de la escala;
- `IMO` y `MMSI`, identificadores del buque;
- `VAIXELLNOM`, nombre del buque;
- `VAIXELLTIPUS`, tipo de embarcación;
- `ESLORA_METRES`, `CALAT_METRES` y `MANEGA_METRES`, correspondientes a dimensiones físicas;
- `TERMINALCODI` y `TERMINALNOM`, vinculadas a la terminal;
- `MOLLCODI`, correspondiente al muelle;
- `PORTORIGENNOM` y `PORTDESTINOM`, correspondientes a los puertos de origen y destino.

La mayor parte de las variables esenciales presenta una elevada completitud. En particular, `ETA`, `ETD`, `VAIXELLTIPUS`, `VAIXELLNOM`, `IMO`, `ESLORA_METRES` y `CALAT_METRES` no presentan valores ausentes en los archivos inspeccionados.

Por el contrario, algunas variables auxiliares presentan una proporción relevante de valores nulos. La variable `MANEGA_METRES` alcanza aproximadamente un **64 % de valores ausentes**, mientras que la información relativa a terminal y determinados campos asociados al muelle presentan en torno a un **14 % de ausencia**. Estas diferencias se tendrán en cuenta durante la selección posterior de variables, evitando utilizar como indicadores principales aquellos campos cuya cobertura resulte insuficiente.

El campo `VAIXELLTIPUS` presenta **9 categorías diferentes en 2024**, lo que sugiere una cardinalidad suficientemente reducida para construir posteriormente grupos analíticos interpretables y variables específicas por tipología de buque.

La inspección también revela una característica importante de la dimensión temporal. Aunque los archivos corresponden nominalmente a 2024, la variable `ANYESCALA` contiene más de un año y algunas escalas registradas en enero de 2024 aparecen asociadas administrativamente a 2023. Por tanto, el año indicado por el archivo o por `ANYESCALA` no debe utilizarse directamente como referencia temporal del análisis.

En consecuencia, la dimensión temporal del estudio se definirá a partir de las fechas reales de **ETA y ETD**, utilizando el año del archivo y `ANYESCALA` únicamente como variables de trazabilidad y control.

Asimismo, las primeras observaciones muestran que una misma escala puede aparecer tanto en el archivo de arribadas como en el de salidas con idénticos identificadores y fechas. Esto confirma la existencia de un solapamiento potencial entre ambas fuentes y la necesidad de realizar una **deduplicación de escalas antes de construir los indicadores diarios**.

> **Conclusión:** los archivos marítimos contienen información rica y estructuralmente adecuada para construir indicadores diarios de actividad portuaria. No obstante, la posible duplicación de escalas entre arribadas y salidas y la diferencia entre el año administrativo y las fechas reales de operación hacen necesario aplicar una fase específica de normalización temporal y deduplicación antes de cualquier agregación.

### 6.2. Consolidación multianual de los registros marítimos (2018–2024)

Tras caracterizar la estructura de los archivos de referencia, se procede a la carga conjunta de los datos de tráfico marítimo correspondientes al periodo **2018–2024**.

Para cada año se dispone de dos archivos independientes:

- `escales_arribades_AAAA.csv`
- `escales_sortides_AAAA.csv`

Por tanto, el proceso incorpora un total esperado de **14 archivos**, correspondientes a siete años de observación y dos tipos de registro por año.

Antes de realizar la concatenación se verifica la existencia de todos los archivos y se registra su número de observaciones y variables. Posteriormente se incorporan campos auxiliares de trazabilidad que permiten identificar el año nominal, el tipo de archivo y el fichero original de procedencia de cada registro.

En esta etapa los registros de arribadas y salidas se conservan íntegramente y **no se realiza todavía ninguna deduplicación**. Esta decisión es especialmente importante porque la inspección inicial ha mostrado que una misma escala puede encontrarse representada en ambos archivos.

Asimismo, el año nominal del archivo no se utilizará como referencia temporal definitiva. La asignación temporal de las operaciones se realizará posteriormente a partir de las fechas efectivas `ETA` y `ETD`.

A diferencia del tráfico aéreo, la fuente marítima dispone de información desde **2018**, por lo que se conservará inicialmente la serie completa 2018–2024. Esto permitirá disponer de dos años completos de referencia pre-COVID y realizar posteriormente comparaciones entre:

- **Pre-COVID:** 2018–2019.
- **COVID:** 2020–2021.
- **Post-COVID:** 2022–2024.

El eventual recorte al periodo temporal común con otras fuentes se realizará únicamente durante la construcción del dataset integrado, evitando perder información útil durante el procesamiento independiente del tráfico marítimo.

> **Objetivo de la etapa:** obtener una tabla multianual trazable que reúna todos los registros originales de arribadas y salidas y sirva como punto de partida para la auditoría temporal, reconstrucción de escalas únicas y posterior generación de indicadores diarios.

In [ ]:
# ============================================================
# 6.2. CONSOLIDACIÓN MULTIANUAL — TRÁFICO MARÍTIMO 2018-2024
# ============================================================

ANIOS_PUERTO = list(range(2018, 2025))

lista_puerto = []
control_archivos = []

print("=" * 75)
print("CARGA DE ARCHIVOS MARÍTIMOS — 2018-2024")
print("=" * 75)


# ------------------------------------------------------------
# 1. Localización y lectura de los 14 archivos
# ------------------------------------------------------------

for anio in ANIOS_PUERTO:

    carpeta_anio = RUTA_PUERTO / str(anio)

    archivos_anio = {
        "arribadas": (
            carpeta_anio /
            f"escales_arribades_{anio}.csv"
        ),
        "salidas": (
            carpeta_anio /
            f"escales_sortides_{anio}.csv"
        )
    }

    for tipo_archivo, ruta in archivos_anio.items():

        # ----------------------------------------------------
        # Comprobar existencia
        # ----------------------------------------------------

        if not ruta.exists():

            control_archivos.append({
                "Anio": anio,
                "Tipo": tipo_archivo,
                "Existe": False,
                "Filas": np.nan,
                "Columnas": np.nan,
                "Archivo": ruta.name
            })

            continue


        # ----------------------------------------------------
        # Lectura robusta
        # ----------------------------------------------------

        df_anio = leer_csv_robusto(ruta)


        # ----------------------------------------------------
        # Variables de trazabilidad
        # ----------------------------------------------------

        df_anio["anio_archivo"] = anio
        df_anio["tipo_archivo"] = tipo_archivo
        df_anio["archivo_origen"] = ruta.name


        # ----------------------------------------------------
        # Control del archivo
        # ----------------------------------------------------

        control_archivos.append({
            "Anio": anio,
            "Tipo": tipo_archivo,
            "Existe": True,
            "Filas": len(df_anio),
            "Columnas": df_anio.shape[1] - 3,
            "Archivo": ruta.name
        })


        # ----------------------------------------------------
        # Incorporar a la lista multianual
        # ----------------------------------------------------

        lista_puerto.append(df_anio)


# ------------------------------------------------------------
# 2. Tabla de control de archivos
# ------------------------------------------------------------

df_control_archivos = pd.DataFrame(control_archivos)

print("\nCONTROL DE ARCHIVOS:")

display(df_control_archivos)


# ------------------------------------------------------------
# 3. Verificar que están los 14 archivos esperados
# ------------------------------------------------------------

n_esperados = len(ANIOS_PUERTO) * 2
n_encontrados = int(df_control_archivos["Existe"].sum())

print(
    f"\nArchivos esperados:   {n_esperados}"
)

print(
    f"Archivos encontrados: {n_encontrados}"
)

if n_encontrados != n_esperados:

    faltantes = df_control_archivos.loc[
        ~df_control_archivos["Existe"]
    ]

    display(faltantes)

    raise FileNotFoundError(
        "Falta al menos un archivo marítimo. "
        "Se detiene la consolidación."
    )


# ------------------------------------------------------------
# 4. Consolidación de los 14 archivos
# ------------------------------------------------------------

df_puerto_raw = pd.concat(
    lista_puerto,
    ignore_index=True,
    sort=False
)


# ------------------------------------------------------------
# 5. Resumen del dataset consolidado
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATASET MARÍTIMO CONSOLIDADO")
print("=" * 75)

print(
    f"Filas totales:    {len(df_puerto_raw):,}"
)

print(
    f"Columnas totales: {df_puerto_raw.shape[1]}"
)

print(
    f"Años nominales:   "
    f"{sorted(df_puerto_raw['anio_archivo'].unique())}"
)

print("\nRegistros por año y archivo:")

resumen_anual = (
    df_puerto_raw
    .groupby(
        ["anio_archivo", "tipo_archivo"]
    )
    .size()
    .unstack(fill_value=0)
)

resumen_anual["Total"] = (
    resumen_anual.sum(axis=1)
)

display(resumen_anual)


# ------------------------------------------------------------
# 6. Comprobar homogeneidad de las columnas originales
# ------------------------------------------------------------

columnas_por_archivo = (
    df_control_archivos
    .loc[df_control_archivos["Existe"]]
    [["Anio", "Tipo", "Columnas"]]
)

print("\nNúmero de columnas originales por archivo:")

display(columnas_por_archivo)


# ------------------------------------------------------------
# 7. Control de trazabilidad
# ------------------------------------------------------------

print("\nCONTROL DE TRAZABILIDAD:")

print(
    "Nulos en anio_archivo:",
    df_puerto_raw["anio_archivo"].isna().sum()
)

print(
    "Nulos en tipo_archivo:",
    df_puerto_raw["tipo_archivo"].isna().sum()
)

print(
    "Nulos en archivo_origen:",
    df_puerto_raw["archivo_origen"].isna().sum()
)


# ------------------------------------------------------------
# 8. Resultado
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CONSOLIDACIÓN FINALIZADA")
print("=" * 75)

print(
    f"✓ {n_encontrados} archivos cargados."
)

print(
    f"✓ {len(df_puerto_raw):,} registros "
    "consolidados."
)

print(
    "✓ Serie nominal disponible: 2018-2024."
)

print(
    "✓ Procedencia de cada registro conservada."
)

print(
    "\nLa deduplicación NO se ha realizado todavía."
)

CARGA DE ARCHIVOS MARÍTIMOS — 2018-2024

CONTROL DE ARCHIVOS:


,Anio,Tipo,Existe,Filas,Columnas,Archivo
0,2018,arribadas,True,10626,32,escales_arribades_2018.csv
1,2018,salidas,True,10626,32,escales_sortides_2018.csv
2,2019,arribadas,True,10749,32,escales_arribades_2019.csv
3,2019,salidas,True,10740,32,escales_sortides_2019.csv
4,2020,arribadas,True,8418,32,escales_arribades_2020.csv
5,2020,salidas,True,8422,32,escales_sortides_2020.csv
6,2021,arribadas,True,9454,32,escales_arribades_2021.csv
7,2021,salidas,True,9451,32,escales_sortides_2021.csv
8,2022,arribadas,True,10731,32,escales_arribades_2022.csv
9,2022,salidas,True,10738,32,escales_sortides_2022.csv



Archivos esperados:   14
Archivos encontrados: 14

DATASET MARÍTIMO CONSOLIDADO
Filas totales:    141,560
Columnas totales: 35
Años nominales:   [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Registros por año y archivo:


tipo_archivo,arribadas,salidas,Total
anio_archivo,,,
2018,10626,10626,21252
2019,10749,10740,21489
2020,8418,8422,16840
2021,9454,9451,18905
2022,10731,10738,21469
2023,10476,10486,20962
2024,10313,10330,20643



Número de columnas originales por archivo:


,Anio,Tipo,Columnas
0,2018,arribadas,32
1,2018,salidas,32
2,2019,arribadas,32
3,2019,salidas,32
4,2020,arribadas,32
5,2020,salidas,32
6,2021,arribadas,32
7,2021,salidas,32
8,2022,arribadas,32
9,2022,salidas,32



CONTROL DE TRAZABILIDAD:
Nulos en anio_archivo: 0
Nulos en tipo_archivo: 0
Nulos en archivo_origen: 0

CONSOLIDACIÓN FINALIZADA
✓ 14 archivos cargados.
✓ 141,560 registros consolidados.
✓ Serie nominal disponible: 2018-2024.
✓ Procedencia de cada registro conservada.

La deduplicación NO se ha realizado todavía.


#### Resultados de la consolidación multianual

La carga multianual confirma la disponibilidad completa de los **14 archivos esperados** para el periodo 2018–2024, correspondientes a un archivo anual de arribadas y otro de salidas.

Todos los archivos presentan una estructura homogénea de **32 variables originales**, lo que permite realizar su concatenación directa sin necesidad de armonizar previamente diferencias de esquema entre años.

Tras la consolidación se obtiene una tabla inicial de **141.560 registros y 35 variables**, incluyendo las tres variables adicionales incorporadas para garantizar la trazabilidad (`anio_archivo`, `tipo_archivo` y `archivo_origen`).

La distribución de registros presenta diferencias temporales relevantes. Los años 2018 y 2019 contienen aproximadamente 21.000 registros anuales, mientras que en 2020 se observa una reducción hasta **16.840 registros**, seguida de una recuperación progresiva durante 2021 y 2022. Esta evolución preliminar resulta compatible con una alteración de la actividad marítima durante el periodo COVID, aunque todavía no debe interpretarse como una variación directa del número de escalas, dado que los registros de arribadas y salidas no han sido deduplicados.

La trazabilidad del proceso queda completamente preservada, sin valores ausentes en el año nominal, tipo de archivo ni archivo de procedencia.

Asimismo, las pequeñas diferencias observadas entre el número de registros de arribadas y salidas de determinados años refuerzan la necesidad de reconstruir las escalas a partir de sus identificadores y fechas reales, en lugar de asumir una correspondencia directa entre ambos archivos.

> **Resultado:** se obtiene una tabla multianual estructuralmente homogénea y trazable para 2018–2024. En esta etapa no se ha eliminado ningún posible solapamiento entre arribadas y salidas; la identificación de escalas únicas se realizará en la siguiente fase mediante la normalización de `ETA` y `ETD` y el análisis de los identificadores disponibles.

### 6.3. Normalización temporal y reconstrucción de escalas marítimas únicas

Los archivos originales diferencian entre **arribadas y salidas**, pero ambas tablas pueden contener información relativa a una misma escala portuaria. Por este motivo, la concatenación realizada en la etapa anterior no puede interpretarse directamente como un conjunto de movimientos independientes.

El objetivo de esta etapa es reconstruir una tabla de **escalas marítimas únicas**, que constituirá la unidad básica a partir de la cual se generarán posteriormente los eventos diarios de llegada y salida.

En primer lugar, se normalizan las variables temporales:

- `ETA`: fecha y hora estimada de llegada.
- `ETD`: fecha y hora estimada de salida.

A partir de ellas se obtienen `Fecha_llegada` y `Fecha_salida`. Estas fechas, y no el año nominal del archivo ni `ANYESCALA`, constituirán la referencia temporal del análisis.

Posteriormente se analiza el grado de solapamiento entre los archivos de arribadas y salidas utilizando `ESCALANUM` como identificador principal de la escala y contrastándolo con `IMO`, `ETA` y `ETD`.

La auditoría incluye:

- conversión y validez de `ETA` y `ETD`;
- cobertura temporal real de los registros;
- coherencia entre las fechas y el periodo de estudio 2018–2024;
- presencia de escalas en ambos tipos de archivo;
- consistencia de `ESCALANUM` entre años;
- identificación de registros exactamente coincidentes;
- detección de escalas con `ETD < ETA`;
- cálculo preliminar de la duración de estancia.

La deduplicación se realiza de forma conservadora. Se consideran equivalentes los registros que representan la misma escala mediante la combinación de identificador y fechas, evitando eliminar observaciones únicamente porque compartan un identificador administrativo.

Esta reconstrucción es especialmente relevante para las fases posteriores del estudio. Una vez obtenida una escala única será posible derivar de forma independiente:

- **llegadas**, utilizando la fecha `ETA`;
- **salidas**, utilizando la fecha `ETD`;
- **movimientos diarios**, como combinación de ambos eventos.

De este modo se evita el doble conteo y se conserva simultáneamente la información necesaria para el análisis **pre-COVID / COVID / post-COVID**, la construcción de variables para los modelos predictivos y la visualización diferenciada de llegadas y salidas en el dashboard.

> **Objetivo de la etapa:** transformar los registros consolidados de arribadas y salidas en una estructura fiable de escalas portuarias únicas, preservando la información temporal necesaria para reconstruir posteriormente la actividad marítima diaria.

In [ ]:
# ============================================================
# 6.3. NORMALIZACIÓN TEMPORAL Y ESCALAS MARÍTIMAS ÚNICAS
# ============================================================

print("=" * 80)
print("AUDITORÍA Y RECONSTRUCCIÓN DE ESCALAS ÚNICAS")
print("=" * 80)

df_puerto = df_puerto_raw.copy()


# ------------------------------------------------------------
# 1. Conversión robusta de ETA y ETD
# ------------------------------------------------------------

df_puerto["ETA_datetime"] = pd.to_datetime(
    df_puerto["ETA"],
    errors="coerce"
)

df_puerto["ETD_datetime"] = pd.to_datetime(
    df_puerto["ETD"],
    errors="coerce"
)

df_puerto["Fecha_llegada"] = (
    df_puerto["ETA_datetime"].dt.normalize()
)

df_puerto["Fecha_salida"] = (
    df_puerto["ETD_datetime"].dt.normalize()
)


print("\n1. CONTROL DE FECHAS")
print("-" * 50)

print(
    "ETA no interpretables:",
    df_puerto["ETA_datetime"].isna().sum()
)

print(
    "ETD no interpretables:",
    df_puerto["ETD_datetime"].isna().sum()
)

print(
    "\nETA mínima:",
    df_puerto["ETA_datetime"].min()
)

print(
    "ETA máxima:",
    df_puerto["ETA_datetime"].max()
)

print(
    "ETD mínima:",
    df_puerto["ETD_datetime"].min()
)

print(
    "ETD máxima:",
    df_puerto["ETD_datetime"].max()
)


# ------------------------------------------------------------
# 2. Cobertura real respecto al periodo de estudio
# ------------------------------------------------------------

FECHA_INICIO_PUERTO = pd.Timestamp("2018-01-01")
FECHA_FIN_PUERTO = pd.Timestamp("2024-12-31")

eta_en_periodo = df_puerto["Fecha_llegada"].between(
    FECHA_INICIO_PUERTO,
    FECHA_FIN_PUERTO
)

etd_en_periodo = df_puerto["Fecha_salida"].between(
    FECHA_INICIO_PUERTO,
    FECHA_FIN_PUERTO
)

alguna_fecha_en_periodo = (
    eta_en_periodo | etd_en_periodo
)

print("\n2. COBERTURA TEMPORAL")
print("-" * 50)

print(
    "Registros con ETA dentro de 2018-2024:",
    eta_en_periodo.sum()
)

print(
    "Registros con ETD dentro de 2018-2024:",
    etd_en_periodo.sum()
)

print(
    "Registros con ETA o ETD dentro del periodo:",
    alguna_fecha_en_periodo.sum()
)

print(
    "Registros completamente fuera del periodo:",
    (~alguna_fecha_en_periodo).sum()
)


# Mantener registros relacionados con el periodo analítico
df_puerto = df_puerto.loc[
    alguna_fecha_en_periodo
].copy()


# ------------------------------------------------------------
# 3. Auditoría de ESCALANUM
# ------------------------------------------------------------

print("\n3. AUDITORÍA DEL IDENTIFICADOR ESCALANUM")
print("-" * 50)

print(
    "ESCALANUM nulos:",
    df_puerto["ESCALANUM"].isna().sum()
)

print(
    "ESCALANUM únicos:",
    df_puerto["ESCALANUM"].nunique(dropna=True)
)


# ¿Un ESCALANUM aparece asociado a más de un año nominal?
escala_anios = (
    df_puerto
    .dropna(subset=["ESCALANUM"])
    .groupby("ESCALANUM")["anio_archivo"]
    .nunique()
)

escalas_varios_anios = (
    escala_anios > 1
).sum()

print(
    "ESCALANUM asociados a varios años de archivo:",
    escalas_varios_anios
)


# ------------------------------------------------------------
# 4. Solapamiento arribadas / salidas
# ------------------------------------------------------------

print("\n4. SOLAPAMIENTO ENTRE FUENTES")
print("-" * 50)

solapamiento = (
    df_puerto
    .dropna(subset=["ESCALANUM"])
    .groupby("ESCALANUM")
    .agg(
        n_registros=("ESCALANUM", "size"),
        n_fuentes=("tipo_archivo", "nunique")
    )
)

n_escalas_ambas_fuentes = (
    solapamiento["n_fuentes"] == 2
).sum()

n_escalas_una_fuente = (
    solapamiento["n_fuentes"] == 1
).sum()

print(
    "Escalas presentes en arribadas Y salidas:",
    n_escalas_ambas_fuentes
)

print(
    "Escalas presentes en un solo tipo de archivo:",
    n_escalas_una_fuente
)


# ------------------------------------------------------------
# 5. Coincidencia exacta de escala + ETA + ETD
# ------------------------------------------------------------

clave_escala = [
    "ESCALANUM",
    "ETA_datetime",
    "ETD_datetime"
]

duplicados_clave = df_puerto.duplicated(
    subset=clave_escala,
    keep=False
)

print("\n5. DUPLICACIÓN DE LA CLAVE DE ESCALA")
print("-" * 50)

print(
    "Filas pertenecientes a claves repetidas:",
    duplicados_clave.sum()
)

print(
    "Claves escala+ETA+ETD únicas:",
    df_puerto[clave_escala]
    .drop_duplicates()
    .shape[0]
)


# ------------------------------------------------------------
# 6. Comprobar discrepancias entre copias de una misma escala
# ------------------------------------------------------------

columnas_comparacion = [
    c for c in [
        "IMO",
        "VAIXELLNOM",
        "VAIXELLTIPUS",
        "TERMINALCODI",
        "TERMINALNOM",
        "PORTORIGENNOM",
        "PORTDESTINOM"
    ]
    if c in df_puerto.columns
]

control_discrepancias = []

for columna in columnas_comparacion:

    n_distintos = (
        df_puerto
        .groupby(clave_escala, dropna=False)[columna]
        .nunique(dropna=True)
    )

    control_discrepancias.append({
        "Variable": columna,
        "Escalas_con_valores_distintos":
            int((n_distintos > 1).sum())
    })

df_discrepancias = pd.DataFrame(
    control_discrepancias
)

print("\n6. CONSISTENCIA ENTRE COPIAS DE UNA MISMA ESCALA")
print("-" * 50)

display(df_discrepancias)


# ------------------------------------------------------------
# 7. Reconstrucción de escalas únicas
# ------------------------------------------------------------

filas_antes = len(df_puerto)

df_escalas = (
    df_puerto
    .sort_values(
        [
            "ETA_datetime",
            "ETD_datetime",
            "tipo_archivo"
        ]
    )
    .drop_duplicates(
        subset=clave_escala,
        keep="first"
    )
    .reset_index(drop=True)
)

print("\n7. RECONSTRUCCIÓN DE ESCALAS")
print("-" * 50)

print(
    f"Registros antes de deduplicar: {filas_antes:,}"
)

print(
    f"Escalas únicas resultantes:    {len(df_escalas):,}"
)

print(
    f"Registros redundantes eliminados: "
    f"{filas_antes - len(df_escalas):,}"
)


# ------------------------------------------------------------
# 8. Duración de estancia
# ------------------------------------------------------------

df_escalas["Estancia_horas"] = (
    (
        df_escalas["ETD_datetime"]
        - df_escalas["ETA_datetime"]
    )
    .dt.total_seconds()
    / 3600
)

print("\n8. COHERENCIA ETA / ETD")
print("-" * 50)

print(
    "ETD anterior a ETA:",
    (df_escalas["Estancia_horas"] < 0).sum()
)

print(
    "Estancias no calculables:",
    df_escalas["Estancia_horas"].isna().sum()
)

print(
    "Estancias iguales a 0 h:",
    (df_escalas["Estancia_horas"] == 0).sum()
)

print("\nDistribución de estancia (horas):")

display(
    df_escalas["Estancia_horas"]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .to_frame("Estancia_horas")
)


# ------------------------------------------------------------
# 9. Cobertura de escalas únicas por año real de llegada
# ------------------------------------------------------------

df_escalas["Anio_llegada"] = (
    df_escalas["Fecha_llegada"].dt.year
)

resumen_escalas_anual = (
    df_escalas
    .loc[
        df_escalas["Anio_llegada"]
        .between(2018, 2024)
    ]
    .groupby("Anio_llegada")
    .agg(
        Escalas=("ESCALANUM", "size"),
        Dias_con_llegadas=("Fecha_llegada", "nunique"),
        Fecha_min=("Fecha_llegada", "min"),
        Fecha_max=("Fecha_llegada", "max")
    )
)

print("\n9. COBERTURA ANUAL — ESCALAS ÚNICAS")
print("-" * 50)

display(resumen_escalas_anual)


# ------------------------------------------------------------
# 10. Resultado
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AUDITORÍA 6.3 FINALIZADA")
print("=" * 80)

print(
    f"✓ Escalas únicas provisionales: {len(df_escalas):,}"
)

print(
    "✓ ETA y ETD conservadas para reconstruir "
    "posteriormente llegadas y salidas."
)

print(
    "✓ No se ha realizado todavía ninguna agregación diaria."
)

AUDITORÍA Y RECONSTRUCCIÓN DE ESCALAS ÚNICAS

1. CONTROL DE FECHAS
--------------------------------------------------
ETA no interpretables: 0
ETD no interpretables: 0

ETA mínima: 2017-09-20 08:47:00
ETA máxima: 2024-12-31 21:55:00
ETD mínima: 2018-01-01 08:24:00
ETD máxima: 2025-01-12 12:00:00

2. COBERTURA TEMPORAL
--------------------------------------------------
Registros con ETA dentro de 2018-2024: 141537
Registros con ETD dentro de 2018-2024: 141544
Registros con ETA o ETD dentro del periodo: 141560
Registros completamente fuera del periodo: 0

3. AUDITORÍA DEL IDENTIFICADOR ESCALANUM
--------------------------------------------------
ESCALANUM nulos: 0
ESCALANUM únicos: 69952
ESCALANUM asociados a varios años de archivo: 203

4. SOLAPAMIENTO ENTRE FUENTES
--------------------------------------------------
Escalas presentes en arribadas Y salidas: 69913
Escalas presentes en un solo tipo de archivo: 39

5. DUPLICACIÓN DE LA CLAVE DE ESCALA
--------------------------------------

,Variable,Escalas_con_valores_distintos
0,IMO,0
1,VAIXELLNOM,0
2,VAIXELLTIPUS,0
3,TERMINALCODI,690
4,TERMINALNOM,691
5,PORTORIGENNOM,0
6,PORTDESTINOM,0



7. RECONSTRUCCIÓN DE ESCALAS
--------------------------------------------------
Registros antes de deduplicar: 141,560
Escalas únicas resultantes:    70,044
Registros redundantes eliminados: 71,516

8. COHERENCIA ETA / ETD
--------------------------------------------------
ETD anterior a ETA: 0
Estancias no calculables: 0
Estancias iguales a 0 h: 1

Distribución de estancia (horas):


,Estancia_horas
count,70044.000000
mean,21.596093
std,39.058665
min,0.000000
1%,1.633333
25%,4.683333
50%,12.450000
75%,24.566667
95%,68.833333
99%,155.516333



9. COBERTURA ANUAL — ESCALAS ÚNICAS
--------------------------------------------------


,Escalas,Dias_con_llegadas,Fecha_min,Fecha_max
Anio_llegada,,,,
2018,10539,365,2018-01-01,2018-12-31
2019,10647,365,2019-01-01,2019-12-31
2020,8292,365,2020-01-01,2020-12-31
2021,9319,365,2021-01-01,2021-12-31
2022,10624,365,2022-01-01,2022-12-31
2023,10404,365,2023-01-01,2023-12-31
2024,10196,366,2024-01-01,2024-12-31



AUDITORÍA 6.3 FINALIZADA
✓ Escalas únicas provisionales: 70,044
✓ ETA y ETD conservadas para reconstruir posteriormente llegadas y salidas.
✓ No se ha realizado todavía ninguna agregación diaria.


#### Resultados de la normalización temporal y reconstrucción de escalas

La auditoría temporal muestra una elevada calidad de los registros marítimos. La totalidad de las variables `ETA` y `ETD` pudo convertirse correctamente a formato temporal, sin detectarse fechas no interpretables.

Los límites temporales observados exceden ligeramente el intervalo nominal 2018–2024, debido a escalas cuya llegada o salida se produce en fechas próximas a los límites anuales. No obstante, los **141.560 registros** presentan al menos uno de los dos eventos (`ETA` o `ETD`) dentro del periodo de estudio, por lo que se conservan inicialmente para evitar la pérdida de operaciones que atraviesan dichos límites.

El análisis de `ESCALANUM` confirma que este identificador no debe utilizarse aisladamente como clave global de deduplicación. Aunque se identifican 69.952 valores distintos, 203 aparecen asociados a más de un año nominal. Por este motivo se emplea la combinación `ESCALANUM + ETA + ETD` como clave operativa para la reconstrucción de las escalas.

El solapamiento entre los archivos de arribadas y salidas resulta muy elevado: 69.913 identificadores aparecen en ambas fuentes y únicamente 39 se encuentran en una sola. Esto confirma que la concatenación directa de ambos conjuntos produciría un doble conteo sistemático de la actividad portuaria.

Tras aplicar la clave compuesta se obtienen **70.044 escalas únicas provisionales**, eliminándose 71.516 registros redundantes.

La comparación de los registros duplicados muestra una elevada consistencia en las características principales de cada escala. No se detectan discrepancias en el IMO, nombre o tipo de buque, ni en los puertos de origen y destino. Sí aparecen diferencias en aproximadamente 690 escalas para la terminal registrada, circunstancia que se conservará y analizará separadamente para evitar la pérdida de información potencialmente asociada a las operaciones de llegada y salida.

La coherencia cronológica es igualmente elevada: no existen escalas con `ETD` anterior a `ETA` ni estancias no calculables. La duración mediana de estancia es aproximadamente **12,45 horas**, aunque se observa una distribución asimétrica con algunas estancias excepcionalmente prolongadas.

Finalmente, la cobertura anual basada en la fecha real de llegada es completa para todos los años del periodo. Se observan 10.539 escalas en 2018 y 10.647 en 2019, frente a una reducción hasta 8.292 en 2020 y 9.319 en 2021. A partir de 2022 la actividad recupera valores próximos a los niveles pre-COVID.

Este comportamiento proporciona una primera evidencia descriptiva de la alteración de la actividad marítima durante la pandemia, que será posteriormente contrastada conjuntamente con las restantes variables de movilidad y calidad del aire.

> **Resultado:** se obtiene una estructura consistente de aproximadamente 70.000 escalas portuarias únicas, con información completa de llegada y salida y cobertura temporal continua entre 2018 y 2024. Esta estructura permitirá generar posteriormente indicadores diarios diferenciados de llegadas, salidas, tipología de buque e intensidad de actividad marítima para el modelo predictivo, el análisis COVID y el dashboard.

### 6.4. Caracterización de las escalas e ingeniería de variables marítimas

Una vez reconstruidas las escalas portuarias únicas, se procede a caracterizar cada operación mediante las propiedades físicas del buque y su tipología.

Las variables originales contienen información sobre las dimensiones de los buques, su categoría administrativa, terminal de operación y características de la escala. Estas variables presentan un elevado interés potencial para explicar la actividad portuaria con mayor precisión que el simple número diario de embarcaciones.

En esta etapa se realizan las siguientes operaciones:

- conversión a formato numérico de la **eslora, manga y calado**;
- control de valores físicamente anómalos;
- análisis de valores ausentes;
- normalización de las denominaciones de los tipos de buque;
- agrupación de las categorías originales en grupos operativos interpretables;
- conservación de la información temporal de llegada y salida;
- incorporación de la duración de estancia como característica de cada escala.

La tipología de buque se agrupa en categorías de mayor utilidad analítica:

- cruceros;
- ferris y buques de pasajeros;
- portacontenedores;
- buques tanque;
- Ro-Ro;
- carga;
- servicios portuarios;
- otros.

Esta clasificación permitirá posteriormente generar indicadores diarios diferenciados por tipo de actividad. De este modo, el tráfico marítimo no quedará representado únicamente mediante el número de escalas, sino también mediante variables capaces de aproximar su **intensidad y composición**.

Para el modelo predictivo se conservarán principalmente variables numéricas agregables, mientras que la información categórica y de terminal tendrá especial utilidad para el análisis exploratorio y el dashboard.

Asimismo, se mantiene la separación entre `Fecha_llegada` y `Fecha_salida`, ya que ambas serán utilizadas en la siguiente etapa para reconstruir los eventos diarios de entrada y salida del puerto.

> **Objetivo de la etapa:** transformar la tabla de escalas únicas en una base analítica enriquecida, interpretable y preparada para generar indicadores diarios de actividad marítima destinados al modelado, al análisis pre-COVID/COVID/post-COVID y al dashboard.

In [ ]:
# ============================================================
# 6.4. CARACTERIZACIÓN E INGENIERÍA DE VARIABLES MARÍTIMAS
# ============================================================

import re
import unicodedata

print("=" * 80)
print("CARACTERIZACIÓN DE LAS ESCALAS MARÍTIMAS")
print("=" * 80)

df_escalas_enriquecido = df_escalas.copy()


# ------------------------------------------------------------
# 1. Función robusta para variables numéricas
# ------------------------------------------------------------

def convertir_numero_europeo(serie):

    texto = (
        serie.astype(str)
        .str.strip()
        .replace({
            "": np.nan,
            "nan": np.nan,
            "None": np.nan,
            "null": np.nan
        })
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.replace(
            r"[^0-9.\-]",
            "",
            regex=True
        )
    )

    return pd.to_numeric(
        texto,
        errors="coerce"
    )


# ------------------------------------------------------------
# 2. Conversión de dimensiones físicas
# ------------------------------------------------------------

variables_fisicas = {
    "ESLORA_METRES": "Eslora_m",
    "MANEGA_METRES": "Manga_m",
    "CALAT_METRES": "Calado_m"
}

for original, nueva in variables_fisicas.items():

    if original in df_escalas_enriquecido.columns:

        df_escalas_enriquecido[nueva] = (
            convertir_numero_europeo(
                df_escalas_enriquecido[original]
            )
        )


# ------------------------------------------------------------
# 3. Control inicial de dimensiones
# ------------------------------------------------------------

print("\n1. VARIABLES FÍSICAS — ANTES DEL CONTROL")
print("-" * 55)

cols_fisicas = [
    c for c in [
        "Eslora_m",
        "Manga_m",
        "Calado_m"
    ]
    if c in df_escalas_enriquecido.columns
]

display(
    df_escalas_enriquecido[
        cols_fisicas
    ].describe().T
)


# ------------------------------------------------------------
# 4. Control de valores físicamente anómalos
# ------------------------------------------------------------

limites_fisicos = {
    "Eslora_m": (5, 500),
    "Manga_m": (1, 100),
    "Calado_m": (0.5, 30)
}

control_anomalias = []

for variable, (lim_inf, lim_sup) in limites_fisicos.items():

    if variable not in df_escalas_enriquecido.columns:
        continue

    serie = df_escalas_enriquecido[variable]

    mascara_anomala = (
        serie.notna()
        &
        ~serie.between(lim_inf, lim_sup)
    )

    control_anomalias.append({
        "Variable": variable,
        "Valores_anomalos": int(
            mascara_anomala.sum()
        ),
        "Limite_inferior": lim_inf,
        "Limite_superior": lim_sup
    })

    # No imputamos: los valores imposibles pasan a NaN
    df_escalas_enriquecido.loc[
        mascara_anomala,
        variable
    ] = np.nan


df_control_anomalias = pd.DataFrame(
    control_anomalias
)

print("\n2. CONTROL DE VALORES FÍSICAMENTE ANÓMALOS")
print("-" * 55)

display(df_control_anomalias)


# ------------------------------------------------------------
# 5. Control de valores ausentes
# ------------------------------------------------------------

control_nulos_fisicos = pd.DataFrame({
    "Nulos":
        df_escalas_enriquecido[
            cols_fisicas
        ].isna().sum(),

    "%_nulos":
        (
            df_escalas_enriquecido[
                cols_fisicas
            ].isna().mean() * 100
        ).round(2)
})

print("\n3. VALORES AUSENTES — VARIABLES FÍSICAS")
print("-" * 55)

display(control_nulos_fisicos)


# ------------------------------------------------------------
# 6. Normalización de texto
# ------------------------------------------------------------

def normalizar_texto(valor):

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


# ------------------------------------------------------------
# 7. Clasificación funcional del tipo de buque
# ------------------------------------------------------------

def categorizar_buque(tipo_original):

    tipo = normalizar_texto(tipo_original)

    # Ferris / pasajeros
    if re.search(
        r"transbord|ferri|ferry|passatge|passatger|"
        r"pasajer|passenger|ro[\s\-]?pax|ropax",
        tipo
    ):
        return "ferry_pasajeros"

    # Portacontenedores
    if re.search(
        r"portaconten|contenidor|contenedor|container",
        tipo
    ):
        return "portacontenedores"

    # Tanques
    if re.search(
        r"tanc|tanque|tanker|petrol|petrolier|oil|"
        r"quimic|chemical|gaser|gas carrier|lng|lpg",
        tipo
    ):
        return "tanque"

    # Ro-Ro
    if re.search(
        r"ro[\s\-]?ro|roro|car[\s\-]?carrier|"
        r"vehicle|automobil",
        tipo
    ):
        return "ro_ro"

    # Carga / granel / frigorífico
    if re.search(
        r"carrega|carga|cargo|lo[\s\-]?lo|mercant|"
        r"freighter|granel|graneler|bulk|"
        r"general cargo|frigor",
        tipo
    ):
        return "carga"

    # Cruceros
    if re.search(
        r"creuer|crucer|cruise",
        tipo
    ):
        return "crucero"

    # Servicios portuarios
    if re.search(
        r"remolc|remolcador|tug|servei|servicio|service|"
        r"pilot|supply|drag|dredger|auxiliar",
        tipo
    ):
        return "servicio_portuario"

    return "otros"


df_escalas_enriquecido["Categoria_buque"] = (
    df_escalas_enriquecido[
        "VAIXELLTIPUS"
    ].apply(categorizar_buque)
)


# ------------------------------------------------------------
# 8. Catálogo tipo original → categoría analítica
# ------------------------------------------------------------

catalogo_tipos = (
    df_escalas_enriquecido
    .groupby(
        [
            "VAIXELLTIPUS",
            "Categoria_buque"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="N_escalas"
    )
    .sort_values(
        "N_escalas",
        ascending=False
    )
)

print("\n4. CATÁLOGO DE TIPOS DE BUQUE")
print("-" * 55)

display(catalogo_tipos)


# ------------------------------------------------------------
# 9. Distribución de las categorías analíticas
# ------------------------------------------------------------

resumen_categorias = (
    df_escalas_enriquecido
    .groupby("Categoria_buque")
    .agg(
        N_escalas=("Categoria_buque", "size"),
        Eslora_media=("Eslora_m", "mean"),
        Calado_medio=("Calado_m", "mean"),
        Estancia_media_h=("Estancia_horas", "mean")
    )
    .sort_values(
        "N_escalas",
        ascending=False
    )
)

resumen_categorias["Porcentaje"] = (
    resumen_categorias["N_escalas"]
    / len(df_escalas_enriquecido)
    * 100
)

print("\n5. DISTRIBUCIÓN POR CATEGORÍA")
print("-" * 55)

display(
    resumen_categorias.round(2)
)


# ------------------------------------------------------------
# 10. Revisar específicamente la categoría "otros"
# ------------------------------------------------------------

tipos_otros = (
    catalogo_tipos.loc[
        catalogo_tipos["Categoria_buque"]
        == "otros"
    ]
)

print("\n6. TIPOS CLASIFICADOS COMO 'OTROS'")
print("-" * 55)

if tipos_otros.empty:

    print(
        "✓ No existen tipos pendientes de clasificación."
    )

else:

    display(tipos_otros)


# ------------------------------------------------------------
# 11. Variables temporales útiles para análisis posterior
# ------------------------------------------------------------

df_escalas_enriquecido["Anio_llegada"] = (
    df_escalas_enriquecido[
        "Fecha_llegada"
    ].dt.year
)

df_escalas_enriquecido["Mes_llegada"] = (
    df_escalas_enriquecido[
        "Fecha_llegada"
    ].dt.month
)

df_escalas_enriquecido["Dia_semana_llegada"] = (
    df_escalas_enriquecido[
        "Fecha_llegada"
    ].dt.dayofweek
)


# ------------------------------------------------------------
# 12. Periodo COVID
# ------------------------------------------------------------

def asignar_periodo_covid(anio):

    if pd.isna(anio):
        return np.nan

    if anio <= 2019:
        return "pre_covid"

    elif anio <= 2021:
        return "covid"

    else:
        return "post_covid"


df_escalas_enriquecido["Periodo_COVID"] = (
    df_escalas_enriquecido[
        "Anio_llegada"
    ].apply(asignar_periodo_covid)
)


# ------------------------------------------------------------
# 13. Control temporal por periodo COVID
# ------------------------------------------------------------

resumen_covid = (
    df_escalas_enriquecido
    .loc[
        df_escalas_enriquecido[
            "Anio_llegada"
        ].between(2018, 2024)
    ]
    .groupby("Periodo_COVID")
    .agg(
        N_escalas=("ESCALANUM", "size"),
        Eslora_media=("Eslora_m", "mean"),
        Calado_medio=("Calado_m", "mean"),
        Estancia_media_h=("Estancia_horas", "mean")
    )
)

print("\n7. RESUMEN POR PERIODO COVID")
print("-" * 55)

display(
    resumen_covid.round(2)
)


# ------------------------------------------------------------
# 14. Resultado
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CARACTERIZACIÓN 6.4 FINALIZADA")
print("=" * 80)

print(
    f"✓ Escalas caracterizadas: "
    f"{len(df_escalas_enriquecido):,}"
)

print(
    f"✓ Categorías de buque: "
    f"{df_escalas_enriquecido['Categoria_buque'].nunique()}"
)

print(
    "✓ Fechas de llegada y salida conservadas."
)

print(
    "✓ Periodo COVID incorporado para análisis descriptivo."
)

print(
    "✓ Datos preparados para construir la actividad diaria."
)

CARACTERIZACIÓN DE LAS ESCALAS MARÍTIMAS

1. VARIABLES FÍSICAS — ANTES DEL CONTROL
-------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
Eslora_m,70044.0,187.503974,64.013185,40.90,142.10,183.00,204.0000,400.00
Manga_m,24499.0,27.698939,8.534928,9.46,22.60,26.00,32.2000,61.51
Calado_m,70024.0,8.440556,2.759273,0.00,6.55,7.41,9.6025,48.20



2. CONTROL DE VALORES FÍSICAMENTE ANÓMALOS
-------------------------------------------------------


,Variable,Valores_anomalos,Limite_inferior,Limite_superior
0,Eslora_m,0,5.0,500
1,Manga_m,0,1.0,100
2,Calado_m,14,0.5,30



3. VALORES AUSENTES — VARIABLES FÍSICAS
-------------------------------------------------------


...


5. DISTRIBUCIÓN POR CATEGORÍA
-------------------------------------------------------


,N_escalas,Eslora_media,Calado_medio,Estancia_media_h,Porcentaje
Categoria_buque,,,,,
ferry_pasajeros,25444,201.22,6.59,8.71,36.33
portacontenedores,19082,209.76,10.83,20.63,27.24
tanque,12685,148.18,9.16,40.65,18.11
ro_ro,7237,194.44,7.93,12.46,10.33
carga,5596,129.42,7.74,52.12,7.99



6. TIPOS CLASIFICADOS COMO 'OTROS'
-------------------------------------------------------
✓ No existen tipos pendientes de clasificación.

7. RESUMEN POR PERIODO COVID
-------------------------------------------------------


,N_escalas,Eslora_media,Calado_medio,Estancia_media_h
Periodo_COVID,,,,
covid,17611,180.80,8.46,22.18
post_covid,31224,191.64,8.46,22.36
pre_covid,21186,187.00,8.40,19.79



CARACTERIZACIÓN 6.4 FINALIZADA
✓ Escalas caracterizadas: 70,044
✓ Categorías de buque: 5
✓ Fechas de llegada y salida conservadas.
✓ Periodo COVID incorporado para análisis descriptivo.
✓ Datos preparados para construir la actividad diaria.


#### Resultados de la caracterización de las escalas marítimas

La caracterización de las 70.044 escalas únicas muestra una elevada calidad de las principales variables físicas asociadas a los buques.

La **eslora** presenta cobertura completa, con una media aproximada de 187,5 m y valores comprendidos entre 40,9 y 400 m. El **calado** también presenta una cobertura prácticamente completa. Se identificaron únicamente 14 valores situados fuera del intervalo físico establecido, que fueron considerados anómalos y transformados en valores ausentes. Tras este control, el porcentaje de ausencia del calado es de apenas un 0,05 %.

La **manga**, por el contrario, presenta aproximadamente un 65 % de valores ausentes. Debido a esta elevada incompletitud, se conservará como información auxiliar pero no se utilizará inicialmente como predictor principal en los modelos.

La clasificación funcional de los tipos de buque permite reducir las categorías administrativas originales a **cinco grupos analíticos**, sin quedar ningún tipo pendiente dentro de la categoría residual `otros`.

La actividad portuaria está dominada por los **ferris y buques de pasajeros**, que representan aproximadamente el 36,3 % de las escalas, seguidos por los **portacontenedores** (27,2 %), los **buques tanque** (18,1 %), los **Ro-Ro** (10,3 %) y los buques de **carga** (8,0 %).

Esta clasificación permite representar no solo el volumen total de actividad marítima, sino también su composición. Las diferencias observadas entre categorías en variables como eslora, calado y duración de estancia justifican la conservación de esta información para evaluar posteriormente posibles relaciones diferenciales con la calidad del aire.

La comparación preliminar por periodos identifica 21.186 escalas en el periodo pre-COVID, 17.611 durante 2020–2021 y 31.224 durante el periodo post-COVID. Además de la reducción del número de operaciones durante la pandemia, se observan ligeras variaciones en las características medias de los buques y en la duración de las estancias.

Estas diferencias serán analizadas posteriormente dentro del bloque específico de comparación temporal, evitando extraer conclusiones causales en esta fase de preparación de los datos.

> **Resultado:** se obtiene una tabla enriquecida de 70.044 escalas marítimas, con cinco categorías funcionales de buque, variables físicas depuradas y clasificación temporal pre-COVID/COVID/post-COVID. La información queda preparada para reconstruir en la siguiente etapa los eventos diarios de llegada y salida y generar los indicadores destinados al dataset maestro, al análisis temporal y al dashboard.

### 6.5. Reconstrucción de la actividad marítima diaria

Una vez depuradas y caracterizadas las escalas portuarias, se transforma la información a la resolución temporal diaria utilizada por el resto de fuentes del estudio.

Cada escala contiene dos eventos temporales diferenciados:

- **llegada**, determinada por `ETA`;
- **salida**, determinada por `ETD`.

Por tanto, una misma escala puede generar dos movimientos portuarios en fechas diferentes. Esta transformación no constituye una duplicación de registros, sino la reconstrucción explícita de los eventos de entrada y salida asociados a cada operación portuaria.

A partir de estos eventos se generan dos estructuras complementarias.

**1. Dataset diario para integración y modelado**

Se construye una tabla con una observación por fecha que resume:

- número de llegadas;
- número de salidas;
- movimientos marítimos totales;
- movimientos por categoría de buque;
- eslora total movilizada;
- eslora media;
- calado medio;
- número de buques identificados.

El número de movimientos representa la intensidad operativa básica, mientras que la eslora acumulada permite incorporar una aproximación sencilla al volumen físico de los buques movilizados. Estas variables se conservarán inicialmente como candidatas, dejando la selección definitiva de predictores para la fase de modelado.

**2. Dataset en formato largo para dashboard**

Paralelamente se conserva una tabla de eventos que mantiene la fecha, el tipo de movimiento y la categoría del buque. Esta estructura permitirá realizar filtros y agregaciones dinámicas sin reconstruir posteriormente los datos originales.

La serie diaria se completa mediante un calendario continuo entre 2018 y 2024. Los días sin movimientos se representan mediante valores cero en las variables de conteo, mientras que las variables medias se mantienen como ausentes cuando no existe ninguna observación sobre la que calcularlas.

Finalmente, se incorpora la clasificación temporal **pre-COVID (2018–2019), COVID (2020–2021) y post-COVID (2022–2024)** para facilitar el posterior análisis comparativo.

> **Objetivo de la etapa:** obtener una representación diaria, continua y libre de doble conteo de la actividad marítima, manteniendo simultáneamente una estructura compacta para integración/modelado y otra flexible para el dashboard.

In [ ]:
# ============================================================
# 6.5. CONSTRUCCIÓN DE LA ACTIVIDAD MARÍTIMA DIARIA
# ============================================================

print("=" * 80)
print("CONSTRUCCIÓN DE LA ACTIVIDAD MARÍTIMA DIARIA")
print("=" * 80)

df_base = df_escalas_enriquecido.copy()


# ------------------------------------------------------------
# 1. CREAR EVENTOS DE LLEGADA
# ------------------------------------------------------------

llegadas = pd.DataFrame({
    "fecha": df_base["Fecha_llegada"],
    "movimiento": "llegada",
    "ESCALANUM": df_base["ESCALANUM"],
    "IMO": df_base["IMO"],
    "VAIXELLNOM": df_base["VAIXELLNOM"],
    "Categoria_buque": df_base["Categoria_buque"],
    "Eslora_m": df_base["Eslora_m"],
    "Calado_m": df_base["Calado_m"],
    "Estancia_horas": df_base["Estancia_horas"]
})


# ------------------------------------------------------------
# 2. CREAR EVENTOS DE SALIDA
# ------------------------------------------------------------

salidas = pd.DataFrame({
    "fecha": df_base["Fecha_salida"],
    "movimiento": "salida",
    "ESCALANUM": df_base["ESCALANUM"],
    "IMO": df_base["IMO"],
    "VAIXELLNOM": df_base["VAIXELLNOM"],
    "Categoria_buque": df_base["Categoria_buque"],
    "Eslora_m": df_base["Eslora_m"],
    "Calado_m": df_base["Calado_m"],
    "Estancia_horas": df_base["Estancia_horas"]
})


# ------------------------------------------------------------
# 3. CONCATENAR EVENTOS
# ------------------------------------------------------------

df_eventos_puerto = pd.concat(
    [llegadas, salidas],
    ignore_index=True
)

df_eventos_puerto["fecha"] = pd.to_datetime(
    df_eventos_puerto["fecha"],
    errors="coerce"
).dt.normalize()


# ------------------------------------------------------------
# 4. LIMITAR AL PERIODO ANALÍTICO 2018-2024
# ------------------------------------------------------------

FECHA_INICIO = pd.Timestamp("2018-01-01")
FECHA_FIN = pd.Timestamp("2024-12-31")

n_eventos_antes = len(df_eventos_puerto)

df_eventos_puerto = (
    df_eventos_puerto
    .loc[
        df_eventos_puerto["fecha"].between(
            FECHA_INICIO,
            FECHA_FIN
        )
    ]
    .copy()
)

print("\n1. RECONSTRUCCIÓN DE EVENTOS")
print("-" * 55)

print(
    f"Escalas originales:        {len(df_base):,}"
)

print(
    f"Eventos teóricos (2/escala): "
    f"{len(df_base) * 2:,}"
)

print(
    f"Eventos antes del recorte: {n_eventos_antes:,}"
)

print(
    f"Eventos dentro 2018-2024:  "
    f"{len(df_eventos_puerto):,}"
)

print(
    f"Eventos fuera del periodo: "
    f"{n_eventos_antes - len(df_eventos_puerto):,}"
)


# ------------------------------------------------------------
# 5. CONTROL DE LLEGADAS Y SALIDAS
# ------------------------------------------------------------

resumen_movimientos = (
    df_eventos_puerto
    .groupby("movimiento")
    .size()
    .to_frame("N_eventos")
)

resumen_movimientos["Porcentaje"] = (
    resumen_movimientos["N_eventos"]
    / len(df_eventos_puerto)
    * 100
)

print("\n2. DISTRIBUCIÓN POR TIPO DE MOVIMIENTO")
print("-" * 55)

display(
    resumen_movimientos.round(2)
)


# ------------------------------------------------------------
# 6. VARIABLES TEMPORALES PARA DASHBOARD / COVID
# ------------------------------------------------------------

df_eventos_puerto["anio"] = (
    df_eventos_puerto["fecha"].dt.year
)

df_eventos_puerto["mes"] = (
    df_eventos_puerto["fecha"].dt.month
)

df_eventos_puerto["dia_semana"] = (
    df_eventos_puerto["fecha"].dt.dayofweek
)


def asignar_periodo_covid(anio):

    if anio <= 2019:
        return "pre_covid"

    elif anio <= 2021:
        return "covid"

    else:
        return "post_covid"


df_eventos_puerto["Periodo_COVID"] = (
    df_eventos_puerto["anio"]
    .apply(asignar_periodo_covid)
)


# ------------------------------------------------------------
# 7. AGREGACIÓN DIARIA BÁSICA
# ------------------------------------------------------------

puerto_diario = (
    df_eventos_puerto
    .groupby("fecha")
    .agg(
        Puerto_movimientos=(
            "ESCALANUM",
            "size"
        ),

        Puerto_n_buques=(
            "IMO",
            "nunique"
        ),

        Puerto_eslora_total=(
            "Eslora_m",
            "sum"
        ),

        Puerto_eslora_media=(
            "Eslora_m",
            "mean"
        ),

        Puerto_calado_medio=(
            "Calado_m",
            "mean"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 8. LLEGADAS Y SALIDAS POR DÍA
# ------------------------------------------------------------

movimientos_dia = (
    df_eventos_puerto
    .pivot_table(
        index="fecha",
        columns="movimiento",
        values="ESCALANUM",
        aggfunc="size",
        fill_value=0
    )
    .reset_index()
)


# Garantizar ambas columnas aunque algún caso extremo faltara
for columna in ["llegada", "salida"]:

    if columna not in movimientos_dia.columns:
        movimientos_dia[columna] = 0


movimientos_dia = movimientos_dia.rename(
    columns={
        "llegada": "Puerto_llegadas",
        "salida": "Puerto_salidas"
    }
)


puerto_diario = puerto_diario.merge(
    movimientos_dia[
        [
            "fecha",
            "Puerto_llegadas",
            "Puerto_salidas"
        ]
    ],
    on="fecha",
    how="left"
)


# ------------------------------------------------------------
# 9. MOVIMIENTOS POR CATEGORÍA DE BUQUE
# ------------------------------------------------------------

categorias_dia = (
    df_eventos_puerto
    .pivot_table(
        index="fecha",
        columns="Categoria_buque",
        values="ESCALANUM",
        aggfunc="size",
        fill_value=0
    )
)


categorias_dia.columns = [
    f"Puerto_{col}"
    for col in categorias_dia.columns
]

categorias_dia = (
    categorias_dia
    .reset_index()
)


puerto_diario = puerto_diario.merge(
    categorias_dia,
    on="fecha",
    how="left"
)


# ------------------------------------------------------------
# 10. COMPLETAR CALENDARIO DIARIO
# ------------------------------------------------------------

calendario = pd.DataFrame({
    "fecha": pd.date_range(
        FECHA_INICIO,
        FECHA_FIN,
        freq="D"
    )
})


puerto_diario = (
    calendario
    .merge(
        puerto_diario,
        on="fecha",
        how="left"
    )
    .sort_values("fecha")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. TRATAMIENTO DE DÍAS SIN MOVIMIENTOS
# ------------------------------------------------------------

columnas_conteo = [
    "Puerto_movimientos",
    "Puerto_n_buques",
    "Puerto_llegadas",
    "Puerto_salidas"
]

columnas_conteo += [
    c for c in puerto_diario.columns
    if c.startswith("Puerto_")
    and c not in columnas_conteo
    and c not in [
        "Puerto_eslora_total",
        "Puerto_eslora_media",
        "Puerto_calado_medio"
    ]
]


# Eliminar posibles duplicados de la lista
columnas_conteo = list(
    dict.fromkeys(columnas_conteo)
)


puerto_diario[columnas_conteo] = (
    puerto_diario[columnas_conteo]
    .fillna(0)
)


# Eslora total = 0 cuando no hubo movimientos
puerto_diario["Puerto_eslora_total"] = (
    puerto_diario["Puerto_eslora_total"]
    .fillna(0)
)


# Las medias NO se rellenan con cero:
# un día sin barcos no implica eslora/calado igual a cero.


# ------------------------------------------------------------
# 12. COMPROBACIÓN DE IDENTIDAD
# ------------------------------------------------------------

puerto_diario["Control_movimientos"] = (
    puerto_diario["Puerto_llegadas"]
    +
    puerto_diario["Puerto_salidas"]
)


diferencias = (
    puerto_diario["Puerto_movimientos"]
    != puerto_diario["Control_movimientos"]
).sum()

print("\n3. CONTROL DE CONSISTENCIA DIARIA")
print("-" * 55)

print(
    "Días donde movimientos != llegadas + salidas:",
    diferencias
)


# ------------------------------------------------------------
# 13. VARIABLES TEMPORALES
# ------------------------------------------------------------

puerto_diario["anio"] = (
    puerto_diario["fecha"].dt.year
)

puerto_diario["mes"] = (
    puerto_diario["fecha"].dt.month
)

puerto_diario["dia_semana"] = (
    puerto_diario["fecha"].dt.dayofweek
)

puerto_diario["Periodo_COVID"] = (
    puerto_diario["anio"]
    .apply(asignar_periodo_covid)
)


# ------------------------------------------------------------
# 14. RESUMEN ANUAL
# ------------------------------------------------------------

resumen_anual_puerto = (
    puerto_diario
    .groupby("anio")
    .agg(
        Dias=("fecha", "size"),
        Movimientos=(
            "Puerto_movimientos",
            "sum"
        ),
        Llegadas=(
            "Puerto_llegadas",
            "sum"
        ),
        Salidas=(
            "Puerto_salidas",
            "sum"
        ),
        Movimientos_medios_dia=(
            "Puerto_movimientos",
            "mean"
        ),
        Eslora_total=(
            "Puerto_eslora_total",
            "sum"
        )
    )
)

print("\n4. ACTIVIDAD MARÍTIMA POR AÑO")
print("-" * 55)

display(
    resumen_anual_puerto.round(2)
)


# ------------------------------------------------------------
# 15. RESUMEN POR PERIODO COVID
# ------------------------------------------------------------

resumen_covid_puerto = (
    puerto_diario
    .groupby("Periodo_COVID")
    .agg(
        Dias=("fecha", "size"),
        Movimientos=(
            "Puerto_movimientos",
            "sum"
        ),
        Movimientos_medios_dia=(
            "Puerto_movimientos",
            "mean"
        ),
        Llegadas_medias_dia=(
            "Puerto_llegadas",
            "mean"
        ),
        Salidas_medias_dia=(
            "Puerto_salidas",
            "mean"
        ),
        Eslora_media_movimiento=(
            "Puerto_eslora_media",
            "mean"
        )
    )
)

print("\n5. ACTIVIDAD POR PERIODO COVID")
print("-" * 55)

display(
    resumen_covid_puerto.round(2)
)


# ------------------------------------------------------------
# 16. CONTROL DE COBERTURA
# ------------------------------------------------------------

print("\n6. COBERTURA DEL DATASET DIARIO")
print("-" * 55)

print(
    "Fecha inicial:",
    puerto_diario["fecha"].min()
)

print(
    "Fecha final:",
    puerto_diario["fecha"].max()
)

print(
    "Número de días:",
    len(puerto_diario)
)

print(
    "Fechas duplicadas:",
    puerto_diario["fecha"]
    .duplicated()
    .sum()
)

print(
    "Días sin movimientos:",
    (
        puerto_diario["Puerto_movimientos"]
        == 0
    ).sum()
)


# ------------------------------------------------------------
# 17. DATASET ESPECÍFICO PARA DASHBOARD
# ------------------------------------------------------------

columnas_dashboard = [
    "fecha",
    "movimiento",
    "Categoria_buque",
    "Eslora_m",
    "Calado_m",
    "Estancia_horas",
    "Periodo_COVID"
]

df_puerto_dashboard = (
    df_eventos_puerto[
        columnas_dashboard
    ]
    .copy()
    .sort_values("fecha")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 18. ELIMINAR VARIABLE AUXILIAR DE CONTROL
# ------------------------------------------------------------

puerto_diario = puerto_diario.drop(
    columns="Control_movimientos"
)


# ------------------------------------------------------------
# 19. RESULTADO FINAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONSTRUCCIÓN DIARIA 6.5 FINALIZADA")
print("=" * 80)

print(
    f"✓ Dataset diario: "
    f"{puerto_diario.shape}"
)

print(
    f"✓ Dataset dashboard: "
    f"{df_puerto_dashboard.shape}"
)

print(
    "✓ Llegadas y salidas reconstruidas independientemente."
)

print(
    "✓ Tipologías de buque conservadas."
)

print(
    "✓ Serie diaria continua 2018-2024."
)

print(
    "✓ Periodos COVID incorporados."
)

CONSTRUCCIÓN DE LA ACTIVIDAD MARÍTIMA DIARIA

1. RECONSTRUCCIÓN DE EVENTOS
-------------------------------------------------------
Escalas originales:        70,044
Eventos teóricos (2/escala): 140,088
Eventos antes del recorte: 140,088
Eventos dentro 2018-2024:  140,049
Eventos fuera del periodo: 39

2. DISTRIBUCIÓN POR TIPO DE MOVIMIENTO
-------------------------------------------------------


,N_eventos,Porcentaje
movimiento,,
llegada,70021,50.0
salida,70028,50.0



3. CONTROL DE CONSISTENCIA DIARIA
-------------------------------------------------------
Días donde movimientos != llegadas + salidas: 0

4. ACTIVIDAD MARÍTIMA POR AÑO
-------------------------------------------------------


,Dias,Movimientos,Llegadas,Salidas,Movimientos_medios_dia,Eslora_total
anio,,,,,,
2018,365,21078.0,10539.0,10539.0,57.75,3948922.62
2019,365,21288.0,10647.0,10641.0,58.32,3973346.52
2020,366,16586.0,8292.0,8294.0,45.32,2989586.07
2021,365,18634.0,9319.0,9315.0,51.05,3378408.10
2022,365,21225.0,10624.0,10601.0,58.15,4015261.24
2023,365,20808.0,10404.0,10404.0,57.01,4005858.93
2024,366,20430.0,10196.0,10234.0,55.82,3948526.07



5. ACTIVIDAD POR PERIODO COVID
-------------------------------------------------------


,Dias,Movimientos,Movimientos_medios_dia,Llegadas_medias_dia,Salidas_medias_dia,Eslora_media_movimiento
Periodo_COVID,,,,,,
covid,731,35220.0,48.18,24.09,24.09,181.36
post_covid,1096,62463.0,56.99,28.49,28.50,191.99
pre_covid,730,42366.0,58.04,29.02,29.01,187.38



6. COBERTURA DEL DATASET DIARIO
-------------------------------------------------------
Fecha inicial: 2018-01-01 00:00:00
Fecha final: 2024-12-31 00:00:00
Número de días: 2557
Fechas duplicadas: 0
Días sin movimientos: 1

CONSTRUCCIÓN DIARIA 6.5 FINALIZADA
✓ Dataset diario: (2557, 17)
✓ Dataset dashboard: (140049, 7)
✓ Llegadas y salidas reconstruidas independientemente.
✓ Tipologías de buque conservadas.
✓ Serie diaria continua 2018-2024.
✓ Periodos COVID incorporados.


#### Resultados de la reconstrucción de la actividad marítima diaria

A partir de las 70.044 escalas portuarias previamente identificadas se reconstruyeron de forma independiente los eventos asociados a sus fechas de llegada (`ETA`) y salida (`ETD`).

La transformación genera inicialmente 140.088 eventos potenciales. Tras restringir las fechas al intervalo analítico 2018–2024 se conservan **140.049 movimientos portuarios**, quedando únicamente 39 eventos fuera de los límites temporales del estudio.

La distribución entre ambos tipos de movimiento resulta prácticamente simétrica, con **70.021 llegadas y 70.028 salidas**. Asimismo, la comprobación diaria confirma que el número total de movimientos coincide en todos los casos con la suma de llegadas y salidas, descartándose inconsistencias en la reconstrucción.

La serie diaria obtenida contiene **2.557 días consecutivos entre el 1 de enero de 2018 y el 31 de diciembre de 2024**, sin fechas duplicadas y con un único día sin actividad registrada.

La evolución temporal muestra una alteración clara de la actividad marítima durante el periodo COVID. La media diaria pasa de aproximadamente **58,04 movimientos/día en el periodo pre-COVID** a **48,18 durante 2020–2021**, recuperándose posteriormente hasta **56,99 movimientos/día en 2022–2024**.

El descenso resulta especialmente acusado en 2020, con aproximadamente 45,32 movimientos diarios frente a los 58,32 registrados en 2019. En 2022 la actividad vuelve a alcanzar aproximadamente 58,15 movimientos diarios, próxima a los niveles previos a la pandemia.

Además del número de operaciones, se conserva información sobre la composición de la actividad por tipología de buque y sobre las dimensiones físicas de las embarcaciones, permitiendo representar la intensidad marítima mediante indicadores complementarios al simple conteo de movimientos.

Como resultado se generan dos estructuras diferenciadas:

- un **dataset diario de 2.557 observaciones**, destinado a la integración con las restantes fuentes y al modelado;
- un **dataset de 140.049 eventos**, destinado al análisis detallado y a la construcción del dashboard.

Esta separación permite mantener un dataset maestro compacto para los modelos sin renunciar al nivel de detalle necesario para las visualizaciones interactivas.

> **Resultado:** la actividad marítima queda representada mediante una serie diaria continua, coherente y diferenciada por llegadas, salidas y tipología de buque, preparada para su integración con calidad del aire, meteorología, tráfico viario y tráfico aéreo.

### 6.6. Control de calidad final y exportación de los datos marítimos

Una vez reconstruida la actividad portuaria a escala diaria, se realiza un último control de calidad antes de considerar cerrado el bloque de tráfico marítimo.

Esta etapa tiene como objetivo verificar la consistencia de las estructuras generadas y almacenar los datasets definitivos que se utilizarán posteriormente en la integración, el modelado y el dashboard.

Se comprueban específicamente:

- la cobertura temporal completa del periodo **2018–2024**;
- la ausencia de fechas duplicadas en la serie diaria;
- la coherencia entre el número total de movimientos y la suma de llegadas y salidas;
- la ausencia de valores negativos en las variables de conteo;
- los valores ausentes presentes en las variables finales;
- la conservación de las categorías funcionales de buque;
- la correcta clasificación de los periodos **pre-COVID, COVID y post-COVID**;
- las dimensiones y estructura final de ambos datasets.

Se mantienen dos productos diferenciados:

1. **Dataset marítimo diario**, con una observación por fecha y las variables agregadas necesarias para su posterior integración con contaminación atmosférica, meteorología, tráfico viario y tráfico aéreo.

2. **Dataset marítimo para dashboard**, que conserva el detalle de los eventos de llegada y salida, la categoría del buque y sus principales características físicas, permitiendo realizar posteriormente visualizaciones y filtros interactivos.

No se incorporan todavía variables derivadas específicas para los modelos, como retardos temporales o medias móviles. Estas transformaciones se realizarán posteriormente sobre el dataset integrado, evitando generar variables redundantes y permitiendo controlar adecuadamente la información temporal utilizada durante el entrenamiento.

Los resultados definitivos se almacenan en la carpeta `DATOS LIMPIOS` del bloque de transporte marítimo, utilizando una nomenclatura homogénea con el resto de fuentes del proyecto y el sufijo `_limpio`.

> **Objetivo de la etapa:** cerrar el proceso ETL del tráfico marítimo obteniendo datasets validados, documentados y directamente reutilizables en las siguientes fases del estudio, sin necesidad de volver a procesar los archivos originales.

In [ ]:
# ============================================================
# 6.6. CONTROL DE CALIDAD FINAL Y EXPORTACIÓN
#      TRÁFICO MARÍTIMO
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 75)
print("CONTROL FINAL Y EXPORTACIÓN — TRÁFICO MARÍTIMO")
print("=" * 75)


# ------------------------------------------------------------
# 1. CREAR COPIAS FINALES
# ------------------------------------------------------------

df_puerto_diario_limpio = puerto_diario.copy()
df_puerto_dashboard_limpio = df_puerto_dashboard.copy()


# ------------------------------------------------------------
# 2. NORMALIZAR Y ORDENAR FECHAS
# ------------------------------------------------------------

df_puerto_diario_limpio["fecha"] = pd.to_datetime(
    df_puerto_diario_limpio["fecha"],
    errors="coerce"
).dt.normalize()

df_puerto_dashboard_limpio["fecha"] = pd.to_datetime(
    df_puerto_dashboard_limpio["fecha"],
    errors="coerce"
).dt.normalize()


df_puerto_diario_limpio = (
    df_puerto_diario_limpio
    .sort_values("fecha")
    .reset_index(drop=True)
)

df_puerto_dashboard_limpio = (
    df_puerto_dashboard_limpio
    .sort_values("fecha")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. CONTROL DE COBERTURA TEMPORAL
# ------------------------------------------------------------

FECHA_INICIO = pd.Timestamp("2018-01-01")
FECHA_FIN = pd.Timestamp("2024-12-31")

calendario_esperado = pd.date_range(
    FECHA_INICIO,
    FECHA_FIN,
    freq="D"
)

fechas_observadas = pd.DatetimeIndex(
    df_puerto_diario_limpio["fecha"].dropna()
)

fechas_faltantes = calendario_esperado.difference(
    fechas_observadas
)

fechas_fuera_periodo = df_puerto_diario_limpio.loc[
    ~df_puerto_diario_limpio["fecha"].between(
        FECHA_INICIO,
        FECHA_FIN
    ),
    "fecha"
]

print("\n1. COBERTURA TEMPORAL")
print("-" * 55)

print(
    "Fecha inicial:",
    df_puerto_diario_limpio["fecha"].min()
)

print(
    "Fecha final:",
    df_puerto_diario_limpio["fecha"].max()
)

print(
    "Días esperados:",
    len(calendario_esperado)
)

print(
    "Días observados:",
    df_puerto_diario_limpio["fecha"].nunique()
)

print(
    "Fechas ausentes:",
    len(fechas_faltantes)
)

print(
    "Fechas duplicadas:",
    df_puerto_diario_limpio["fecha"]
    .duplicated()
    .sum()
)

print(
    "Registros fuera de 2018-2024:",
    len(fechas_fuera_periodo)
)


# ------------------------------------------------------------
# 4. CONTROL DE CONSISTENCIA DE MOVIMIENTOS
# ------------------------------------------------------------

control_movimientos = (
    df_puerto_diario_limpio["Puerto_llegadas"]
    +
    df_puerto_diario_limpio["Puerto_salidas"]
)

n_inconsistencias = (
    control_movimientos
    != df_puerto_diario_limpio["Puerto_movimientos"]
).sum()

movimientos_totales = int(
    df_puerto_diario_limpio[
        "Puerto_movimientos"
    ].sum()
)

llegadas_totales = int(
    df_puerto_diario_limpio[
        "Puerto_llegadas"
    ].sum()
)

salidas_totales = int(
    df_puerto_diario_limpio[
        "Puerto_salidas"
    ].sum()
)


print("\n2. CONSISTENCIA DE MOVIMIENTOS")
print("-" * 55)

print(
    "Días con movimientos != llegadas + salidas:",
    n_inconsistencias
)

print(
    "Movimientos totales:",
    f"{movimientos_totales:,}"
)

print(
    "Llegadas totales:",
    f"{llegadas_totales:,}"
)

print(
    "Salidas totales:",
    f"{salidas_totales:,}"
)


# ------------------------------------------------------------
# 5. IDENTIFICAR VARIABLES DE CONTEO
# ------------------------------------------------------------

columnas_excluir_conteo = [
    "Puerto_eslora_total",
    "Puerto_eslora_media",
    "Puerto_calado_medio"
]

columnas_conteo = [
    c
    for c in df_puerto_diario_limpio.columns
    if c.startswith("Puerto_")
    and c not in columnas_excluir_conteo
]


# ------------------------------------------------------------
# 6. CONTROL DE VALORES NEGATIVOS
# ------------------------------------------------------------

control_negativos = []

for columna in columnas_conteo:

    control_negativos.append({
        "Variable": columna,
        "Valores_negativos": int(
            (
                df_puerto_diario_limpio[columna] < 0
            ).sum()
        )
    })

df_negativos = pd.DataFrame(
    control_negativos
)

print("\n3. CONTROL DE VALORES NEGATIVOS")
print("-" * 55)

display(df_negativos)


# ------------------------------------------------------------
# 7. CONTROL DE NULOS — DATASET DIARIO
# ------------------------------------------------------------

nulos_diario = pd.DataFrame({
    "Nulos":
        df_puerto_diario_limpio
        .isna()
        .sum()
})

nulos_diario["%_nulos"] = (
    nulos_diario["Nulos"]
    / len(df_puerto_diario_limpio)
    * 100
).round(3)


print("\n4. NULOS — DATASET DIARIO")
print("-" * 55)

nulos_diario_detectados = nulos_diario.loc[
    nulos_diario["Nulos"] > 0
]

if nulos_diario_detectados.empty:

    print(
        "✓ No se detectan valores nulos."
    )

else:

    display(
        nulos_diario_detectados
    )


# ------------------------------------------------------------
# 8. CONTROL ESPECÍFICO DE VARIABLES FÍSICAS
# ------------------------------------------------------------

print("\n5. VARIABLES FÍSICAS")
print("-" * 55)

columnas_fisicas = [
    "Puerto_eslora_total",
    "Puerto_eslora_media",
    "Puerto_calado_medio"
]

control_fisicas = pd.DataFrame({
    "Variable": columnas_fisicas,
    "Nulos": [
        df_puerto_diario_limpio[c]
        .isna()
        .sum()
        for c in columnas_fisicas
    ],
    "%_nulos": [
        round(
            df_puerto_diario_limpio[c]
            .isna()
            .mean() * 100,
            3
        )
        for c in columnas_fisicas
    ]
})

display(control_fisicas)


# ------------------------------------------------------------
# 9. CONTROL DE CATEGORÍAS DEL DASHBOARD
# ------------------------------------------------------------

categorias_dashboard = sorted(
    df_puerto_dashboard_limpio[
        "Categoria_buque"
    ]
    .dropna()
    .unique()
)

movimientos_dashboard = sorted(
    df_puerto_dashboard_limpio[
        "movimiento"
    ]
    .dropna()
    .unique()
)


print("\n6. ESTRUCTURA DEL DATASET PARA DASHBOARD")
print("-" * 55)

print(
    "Categorías de buque:"
)

for categoria in categorias_dashboard:
    print(
        f"  ✓ {categoria}"
    )

print(
    "\nTipos de movimiento:"
)

for movimiento in movimientos_dashboard:
    print(
        f"  ✓ {movimiento}"
    )

print(
    "\nNúmero total de eventos:",
    f"{len(df_puerto_dashboard_limpio):,}"
)


# ------------------------------------------------------------
# 10. CONTROL DE LOS PERIODOS COVID
# ------------------------------------------------------------

control_periodos = (
    df_puerto_diario_limpio
    .groupby(
        "Periodo_COVID",
        observed=True
    )
    .agg(
        Fecha_inicio=(
            "fecha",
            "min"
        ),

        Fecha_fin=(
            "fecha",
            "max"
        ),

        Dias=(
            "fecha",
            "size"
        ),

        Movimientos=(
            "Puerto_movimientos",
            "sum"
        ),

        Movimientos_medios_dia=(
            "Puerto_movimientos",
            "mean"
        )
    )
)

print("\n7. PERIODOS DE ESTUDIO")
print("-" * 55)

display(
    control_periodos.round(2)
)


# ------------------------------------------------------------
# 11. CONTROL ANUAL
# ------------------------------------------------------------

control_anual = (
    df_puerto_diario_limpio
    .groupby("anio")
    .agg(
        Dias=(
            "fecha",
            "size"
        ),

        Movimientos=(
            "Puerto_movimientos",
            "sum"
        ),

        Llegadas=(
            "Puerto_llegadas",
            "sum"
        ),

        Salidas=(
            "Puerto_salidas",
            "sum"
        ),

        Movimientos_medios_dia=(
            "Puerto_movimientos",
            "mean"
        )
    )
)

print("\n8. CONTROL ANUAL")
print("-" * 55)

display(
    control_anual.round(2)
)


# ------------------------------------------------------------
# 12. CONTROL DE DIMENSIONES FINALES
# ------------------------------------------------------------

print("\n9. DIMENSIONES FINALES")
print("-" * 55)

print(
    "Dataset diario:",
    df_puerto_diario_limpio.shape
)

print(
    "Dataset dashboard:",
    df_puerto_dashboard_limpio.shape
)


# ------------------------------------------------------------
# 13. MOSTRAR COLUMNAS FINALES
# ------------------------------------------------------------

print("\n10. VARIABLES DEL DATASET DIARIO")
print("-" * 55)

for columna in df_puerto_diario_limpio.columns:
    print(
        f"  • {columna}"
    )


print("\n11. VARIABLES DEL DATASET DASHBOARD")
print("-" * 55)

for columna in df_puerto_dashboard_limpio.columns:
    print(
        f"  • {columna}"
    )


# ------------------------------------------------------------
# 14. CARPETA FINAL — DATOS LIMPIOS
# ------------------------------------------------------------

RUTA_DATOS_LIMPIOS = (
    Path("/content/drive/MyDrive/TFM")
    / "03_Transporte_Maritimo"
    / "DATOS LIMPIOS"
)

RUTA_DATOS_LIMPIOS.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 15. NOMBRES DEFINITIVOS
# ------------------------------------------------------------

RUTA_PUERTO_DIARIO = (
    RUTA_DATOS_LIMPIOS
    / "df_puerto_diario_limpio.csv"
)

RUTA_PUERTO_DASHBOARD = (
    RUTA_DATOS_LIMPIOS
    / "df_puerto_dashboard_limpio.csv"
)


# ------------------------------------------------------------
# 16. EXPORTACIÓN
# ------------------------------------------------------------

df_puerto_diario_limpio.to_csv(
    RUTA_PUERTO_DIARIO,
    index=False
)

df_puerto_dashboard_limpio.to_csv(
    RUTA_PUERTO_DASHBOARD,
    index=False
)


# ------------------------------------------------------------
# 17. VERIFICAR EXPORTACIÓN
# ------------------------------------------------------------

if not RUTA_PUERTO_DIARIO.exists():

    raise FileNotFoundError(
        "No se ha generado correctamente "
        "df_puerto_diario_limpio.csv"
    )


if not RUTA_PUERTO_DASHBOARD.exists():

    raise FileNotFoundError(
        "No se ha generado correctamente "
        "df_puerto_dashboard_limpio.csv"
    )


# ------------------------------------------------------------
# 18. COMPROBACIÓN MEDIANTE RELECTURA
# ------------------------------------------------------------

df_control_diario = pd.read_csv(
    RUTA_PUERTO_DIARIO
)

df_control_dashboard = pd.read_csv(
    RUTA_PUERTO_DASHBOARD
)

if df_control_diario.shape != df_puerto_diario_limpio.shape:

    raise ValueError(
        "Las dimensiones del dataset diario "
        "han cambiado durante la exportación."
    )


if df_control_dashboard.shape != df_puerto_dashboard_limpio.shape:

    raise ValueError(
        "Las dimensiones del dataset dashboard "
        "han cambiado durante la exportación."
    )


# ------------------------------------------------------------
# 19. RESULTADO FINAL
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXPORTACIÓN FINAL — TRÁFICO MARÍTIMO")
print("=" * 75)

print(
    "\n✓ Dataset diario para integración:"
)

print(
    RUTA_PUERTO_DIARIO
)

print(
    "  Dimensiones:",
    df_puerto_diario_limpio.shape
)


print(
    "\n✓ Dataset específico para dashboard:"
)

print(
    RUTA_PUERTO_DASHBOARD
)

print(
    "  Dimensiones:",
    df_puerto_dashboard_limpio.shape
)


print("\n✓ Verificación mediante relectura superada.")

print(
    "✓ Ambos archivos almacenados "
    "en la carpeta DATOS LIMPIOS."
)

print(
    "✓ Bloque de tráfico marítimo "
    "preparado para integración."
)

print("\nArchivos generados correctamente.")

CONTROL FINAL Y EXPORTACIÓN — TRÁFICO MARÍTIMO

1. COBERTURA TEMPORAL
-------------------------------------------------------
Fecha inicial: 2018-01-01 00:00:00
Fecha final: 2024-12-31 00:00:00
Días esperados: 2557
Días observados: 2557
Fechas ausentes: 0
Fechas duplicadas: 0
Registros fuera de 2018-2024: 0

2. CONSISTENCIA DE MOVIMIENTOS
-------------------------------------------------------
Días con movimientos != llegadas + salidas: 0
Movimientos totales: 140,049
Llegadas totales: 70,021
Salidas totales: 70,028

3. CONTROL DE VALORES NEGATIVOS
-------------------------------------------------------


,Variable,Valores_negativos
0,Puerto_movimientos,0
1,Puerto_n_buques,0
2,Puerto_llegadas,0
3,Puerto_salidas,0
4,Puerto_carga,0
5,Puerto_ferry_pasajeros,0
6,Puerto_portacontenedores,0
7,Puerto_ro_ro,0
8,Puerto_tanque,0



4. NULOS — DATASET DIARIO
-------------------------------------------------------


,Nulos,%_nulos
Puerto_eslora_media,1,0.039
Puerto_calado_medio,1,0.039



5. VARIABLES FÍSICAS
-------------------------------------------------------


...


6. ESTRUCTURA DEL DATASET PARA DASHBOARD
-------------------------------------------------------
Categorías de buque:
  ✓ carga
  ✓ ferry_pasajeros
  ✓ portacontenedores
  ✓ ro_ro
  ✓ tanque

Tipos de movimiento:
  ✓ llegada
  ✓ salida

Número total de eventos: 140,049

7. PERIODOS DE ESTUDIO
-------------------------------------------------------


,Fecha_inicio,Fecha_fin,Dias,Movimientos,Movimientos_medios_dia
Periodo_COVID,,,,,
covid,2020-01-01,2021-12-31,731,35220.0,48.18
post_covid,2022-01-01,2024-12-31,1096,62463.0,56.99
pre_covid,2018-01-01,2019-12-31,730,42366.0,58.04



8. CONTROL ANUAL
-------------------------------------------------------


,Dias,Movimientos,Llegadas,Salidas,Movimientos_medios_dia
anio,,,,,
2018,365,21078.0,10539.0,10539.0,57.75
2019,365,21288.0,10647.0,10641.0,58.32
2020,366,16586.0,8292.0,8294.0,45.32
2021,365,18634.0,9319.0,9315.0,51.05
2022,365,21225.0,10624.0,10601.0,58.15
2023,365,20808.0,10404.0,10404.0,57.01
2024,366,20430.0,10196.0,10234.0,55.82



9. DIMENSIONES FINALES
-------------------------------------------------------
Dataset diario: (2557, 17)
Dataset dashboard: (140049, 7)

10. VARIABLES DEL DATASET DIARIO
-------------------------------------------------------
  • fecha
  • Puerto_movimientos
  • Puerto_n_buques
  • Puerto_eslora_total
  • Puerto_eslora_media
  • Puerto_calado_medio
  • Puerto_llegadas
  • Puerto_salidas
  • Puerto_carga
  • Puerto_ferry_pasajeros
  • Puerto_portacontenedores
  • Puerto_ro_ro
  • Puerto_tanque
  • anio
  • mes
  • dia_semana
  • Periodo_COVID

11. VARIABLES DEL DATASET DASHBOARD
-------------------------------------------------------
  • fecha
  • movimiento
  • Categoria_buque
  • Eslora_m
  • Calado_m
  • Estancia_horas
  • Periodo_COVID

EXPORTACIÓN FINAL — TRÁFICO MARÍTIMO

✓ Dataset diario para integración:
/content/drive/MyDrive/TFM/03_Transporte_Maritimo/DATOS LIMPIOS/df_puerto_diario_limpio.csv
  Dimensiones: (2557, 17)

✓ Dataset específico para dashboard:
/content/drive/MyDr

#### Resultados del control final y cierre del bloque marítimo

El control final confirma la consistencia y completitud de los datasets generados para representar la actividad marítima del Puerto de Barcelona.

La serie diaria presenta una **cobertura temporal completa entre el 1 de enero de 2018 y el 31 de diciembre de 2024**, con 2.557 observaciones consecutivas, sin fechas ausentes, duplicadas ni registros situados fuera del periodo de estudio.

Durante este intervalo se identifican **140.049 movimientos portuarios**, correspondientes a 70.021 llegadas y 70.028 salidas. La comprobación de consistencia confirma que, para todos los días de la serie, el número total de movimientos coincide exactamente con la suma de llegadas y salidas.

No se detectan valores negativos en ninguna de las variables de actividad. Las únicas ausencias del dataset diario corresponden a un valor de eslora media y otro de calado medio, equivalentes al 0,039 % de las observaciones. Estas ausencias se mantienen sin imputar, al corresponder a una jornada sin movimientos sobre la que no resulta conceptualmente adecuado calcular características físicas medias de los buques.

La estructura final conserva cinco categorías funcionales de embarcaciones: **carga, ferry/pasajeros, portacontenedores, Ro-Ro y tanque**, además de la diferenciación entre movimientos de llegada y salida.

La comparación preliminar entre periodos muestra una reducción clara de la actividad durante la pandemia. La intensidad media pasa de **58,04 movimientos diarios en 2018–2019** a **48,18 movimientos diarios durante 2020–2021**, recuperándose hasta **56,99 movimientos diarios en 2022–2024**. Este patrón justifica la conservación explícita de la variable temporal `Periodo_COVID` para los análisis comparativos posteriores.

Como resultado del proceso ETL se generan dos datasets complementarios:

- `df_puerto_diario_limpio.csv`: **2.557 observaciones y 17 variables**, destinado a la integración con el resto de fuentes y al posterior modelado.
- `df_puerto_dashboard_limpio.csv`: **140.049 eventos y 7 variables**, destinado a las visualizaciones interactivas y al análisis detallado de la actividad portuaria.

Ambos archivos han sido verificados mediante relectura después de su exportación y almacenados en la carpeta `DATOS LIMPIOS` correspondiente al bloque de transporte marítimo.

> **Conclusión del bloque:** el tráfico marítimo queda transformado desde los registros originales de escalas a una estructura temporal diaria continua, validada y compatible con el resto de fuentes del estudio. Paralelamente se conserva una estructura de eventos con mayor nivel de detalle para el dashboard. El bloque queda preparado para su incorporación al proceso de integración multifuente.

## Cierre del bloque — Tráfico marítimo

El tratamiento del tráfico marítimo permite transformar los archivos originales de llegadas y salidas del Port de Barcelona en dos estructuras analíticas complementarias y directamente reutilizables en las siguientes fases del proyecto.

Tras consolidar los archivos correspondientes al periodo **2018–2024**, se normalizaron las variables temporales y físicas, se identificaron las escalas únicas y se reconstruyeron independientemente los eventos asociados a sus fechas de llegada (`ETA`) y salida (`ETD`).

Los tipos originales de embarcaciones se agruparon en cinco categorías funcionales:

- carga;
- ferry/pasajeros;
- portacontenedores;
- Ro-Ro;
- tanque.

Esta clasificación permite representar no solo la intensidad global de la actividad portuaria, sino también su composición, conservando información potencialmente relevante para estudiar su relación con la calidad del aire.

La reconstrucción temporal genera **140.049 movimientos portuarios** dentro del periodo 2018–2024, correspondientes a **70.021 llegadas y 70.028 salidas**.

La posterior agregación genera una serie diaria continua de **2.557 días**, comprendida entre el 1 de enero de 2018 y el 31 de diciembre de 2024, sin fechas ausentes ni duplicadas. Los controles realizados confirman además que, para todas las fechas, el número total de movimientos coincide con la suma de llegadas y salidas.

La clasificación temporal permite diferenciar explícitamente los periodos **pre-COVID, COVID y post-COVID**. La exploración inicial muestra una reducción de la actividad portuaria durante 2020–2021 y una posterior recuperación, patrón que será analizado conjuntamente con la evolución de la contaminación atmosférica y las restantes variables de movilidad y actividad urbana.

Como resultado final se generan dos archivos:

- `df_puerto_diario_limpio.csv`: dataset diario destinado a la integración multifuente y al posterior modelado.
- `df_puerto_dashboard_limpio.csv`: dataset a nivel de evento destinado al análisis exploratorio y a las visualizaciones interactivas.

Ambos datasets se almacenan en la carpeta `DATOS LIMPIOS` del bloque de transporte marítimo y han superado los controles de cobertura temporal, duplicados, coherencia de movimientos, valores negativos y consistencia tras la exportación.

> **Resultado final:** el bloque de tráfico marítimo queda completamente procesado y validado, proporcionando una medida diaria de la intensidad y composición de la actividad portuaria compatible con el dataset maestro, junto con una estructura detallada específicamente preparada para el dashboard.